# 06 — Bulk Intelligence, Smart Assistant & Dashboard Integration

**Responsibility:** new-batch K-Means/DBSCAN, bulk CSV processing, three-model integration,
investigation recommendations, and application/dashboard development.





In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

assert (
    PROJECT_ROOT / "src"
).exists(), "Project source folder not found."

assert (
    PROJECT_ROOT / "models"
).exists(), "Project models folder not found."

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("SMART ASSISTANT DEVELOPMENT")
print("=" * 60)
print("Project root:", PROJECT_ROOT)
print("Notebook: 06_smart_assistant_bulk_intelligence")
print("Existing models: PRESERVED")
print("Status: READY")

SMART ASSISTANT DEVELOPMENT
Project root: D:\Urban-Infrastructure-Pulse-Intelligence
Notebook: 06_smart_assistant_bulk_intelligence
Existing models: PRESERVED
Status: READY


In [3]:

from pathlib import Path
from io import StringIO
import sys
import pandas as pd

print("OPTIONAL ZIP NORMALIZATION FIX")
print("=" * 70)

root = Path(PROJECT_ROOT).resolve()

module_path = (
    root / "src/integration/bulk_complaint_contract.py"
)

template_path = (
    root / "outputs/templates/new_complaints_template.csv"
)

assert "module_source" in globals()
assert "sample" in globals()
assert "categories" in globals()
assert "batch_500" in globals()

# --------------------------------------------------
# 1. Fix the reusable validator
# --------------------------------------------------

old_code = "    return pd.DataFrame.from_records(records)"

new_code = """    result = pd.DataFrame.from_records(records)

    # Pandas may convert optional missing ZIPs
    # into NaN. The backend expects Python None.
    result["incident_zip"] = (
        result["incident_zip"]
        .astype(object)
        .where(result["incident_zip"].notna(), None)
    )

    return result"""

assert module_source.count(old_code) == 1, (
    "Expected validator return statement not found."
)

corrected_source = module_source.replace(
    old_code, new_code
)

compile(
    corrected_source,
    str(module_path),
    "exec"
)

namespace = {}

exec(
    compile(
        corrected_source,
        str(module_path),
        "exec"
    ),
    namespace
)

validate_batch = namespace["validate_batch"]
read_batch_csv = namespace["read_batch_csv"]

# --------------------------------------------------
# 2. Verify original sample validation
# --------------------------------------------------

validated = validate_batch(sample, categories)

ready = validated.loc[
    validated["validation_status"]
    == "ready_for_geographic_validation"
]

rejected = validated.loc[
    validated["validation_status"]
    == "invalid_input"
]

assert len(validated) == 7
assert len(ready) == 3
assert len(rejected) == 4

first = ready.loc[
    ready["complaint_id"] == "NEW-001"
].iloc[0]

zip_value = first["incident_zip"]

print("\n1. OPTIONAL ZIP DIAGNOSTIC")
print("ZIP value:", repr(zip_value))
print("ZIP Python type:", type(zip_value).__name__)

assert zip_value is None, (
    "Missing ZIP was not normalized to None."
)

# --------------------------------------------------
# 3. Recheck the existing backend
# --------------------------------------------------

print("\n2. BACKEND COMPATIBILITY TEST")

from src.ui.service_factory import get_gui_adapter

adapter = get_gui_adapter()

backend_result = adapter.evaluate(
    float(first["latitude"]),
    float(first["longitude"]),
    str(first["complaint_category"]),
    zip_value
)

assert backend_result["geography"]["borough"] == (
    "MANHATTAN"
)

assert backend_result["dbscan"]["status"] == (
    "established_cluster"
)

assert backend_result["dbscan"]["cluster_id"] == 1421

assert backend_result["kmeans"]["display_allowed"] is False

print("Backend: CONNECTED")
print("Borough: MANHATTAN")
print("DBSCAN Cluster: 1421")
print("K-Means missing ZIP safeguard: PASSED")

# --------------------------------------------------
# 4. Verify all 500 incoming records
# --------------------------------------------------

print("\n3. SIMULATED 500-COMPLAINT VALIDATION")

result_500 = validate_batch(
    batch_500,
    categories
)

assert len(result_500) == 500

assert result_500["complaint_id"].is_unique

assert result_500["validation_status"].eq(
    "ready_for_geographic_validation"
).all()

assert result_500["incident_zip"].map(
    lambda value: value is None
).all()

print("Total complaints:", len(result_500))
print("Input-valid complaints: 500")
print("Missing ZIP values normalized: 500")

# --------------------------------------------------
# 5. Preserve ZIPs with leading zeros
# --------------------------------------------------

csv_example = StringIO(
    "complaint_id,created_date,latitude,"
    "longitude,complaint_category,incident_zip\n"
    "ZIP-001,2026-11-09,40.75,-73.99,"
    "STREET CONDITION,00501\n"
)

zip_test = validate_batch(
    read_batch_csv(csv_example),
    categories
)

assert zip_test.iloc[0]["incident_zip"] == "00501"

print("Leading-zero ZIP preservation: PASSED")

# --------------------------------------------------
# 6. Recheck malformed and duplicate records
# --------------------------------------------------

assert (
    "invalid_latitude"
    in validated.iloc[2]["validation_errors"]
)

assert (
    "invalid_incident_zip"
    in validated.iloc[3]["validation_errors"]
)

assert (
    "unsupported_complaint_category"
    in validated.iloc[5]["validation_errors"]
)

assert (
    "invalid_created_date"
    in validated.iloc[6]["validation_errors"]
)

duplicate_sample = sample.copy()

duplicate_sample.loc[
    6, "complaint_id"
] = "NEW-002"

duplicate_result = validate_batch(
    duplicate_sample,
    categories
)

duplicate_rows = duplicate_result.loc[
    duplicate_result["complaint_id"] == "NEW-002"
]

assert len(duplicate_rows) == 2

assert duplicate_rows["validation_errors"].str.contains(
    "duplicate_complaint_id"
).all()

print("Malformed and duplicate record handling: PASSED")

# --------------------------------------------------
# 7. Prepare reusable template
# --------------------------------------------------

template_columns = [
    "complaint_id",
    "created_date",
    "latitude",
    "longitude",
    "complaint_category",
    "incident_zip"
]

template = sample.iloc[:2][
    template_columns
].copy()

template_csv = template.to_csv(
    index=False,
    lineterminator="\n"
)

# --------------------------------------------------
# 8. Protect existing project files
# --------------------------------------------------

if module_path.exists():
    assert module_path.read_text(
        encoding="utf-8"
    ) == corrected_source, (
        "Existing module differs. No overwrite."
    )

if template_path.exists():
    assert template_path.read_text(
        encoding="utf-8"
    ) == template_csv, (
        "Existing template differs. No overwrite."
    )

# --------------------------------------------------
# 9. Save new module and CSV template
# --------------------------------------------------

module_path.parent.mkdir(
    parents=True,
    exist_ok=True
)

template_path.parent.mkdir(
    parents=True,
    exist_ok=True
)

if not module_path.exists():
    module_path.write_text(
        corrected_source,
        encoding="utf-8"
    )

if not template_path.exists():
    template_path.write_text(
        template_csv,
        encoding="utf-8"
    )

module_source = corrected_source

print("\n4. CREATED / VERIFIED FILES")
print("Validator:", module_path)
print("Template:", template_path)

print("\nSTATUS: PASSED")
print("INPUT CONTRACT VERIFIED")
print("OPTIONAL ZIP NORMALIZATION VERIFIED")
print("500-ROW INPUT VALIDATION VERIFIED")
print("EXISTING MODELS UNMODIFIED")


OPTIONAL ZIP NORMALIZATION FIX

1. OPTIONAL ZIP DIAGNOSTIC
ZIP value: None
ZIP Python type: NoneType

2. BACKEND COMPATIBILITY TEST
Backend: CONNECTED
Borough: MANHATTAN
DBSCAN Cluster: 1421
K-Means missing ZIP safeguard: PASSED

3. SIMULATED 500-COMPLAINT VALIDATION
Total complaints: 500
Input-valid complaints: 500
Missing ZIP values normalized: 500
Leading-zero ZIP preservation: PASSED
Malformed and duplicate record handling: PASSED

4. CREATED / VERIFIED FILES
Validator: D:\Urban-Infrastructure-Pulse-Intelligence\src\integration\bulk_complaint_contract.py
Template: D:\Urban-Infrastructure-Pulse-Intelligence\outputs\templates\new_complaints_template.csv

STATUS: PASSED
INPUT CONTRACT VERIFIED
OPTIONAL ZIP NORMALIZATION VERIFIED
500-ROW INPUT VALIDATION VERIFIED
EXISTING MODELS UNMODIFIED


In [4]:

from pathlib import Path
from textwrap import dedent
import sys
import json
import pandas as pd

print(" BULK COMPLAINT BACKEND PROCESSING")
print("=" * 75)

root = Path(PROJECT_ROOT).resolve()

if str(root) not in sys.path:
    sys.path.insert(0, str(root))

module_path = (
    root / "src/integration/bulk_complaint_processor.py"
)

taxonomy_path = (
    root / "outputs/tables/dbscan"
    / "dbscan_action_taxonomy_v1.csv"
)

from src.integration.bulk_complaint_contract import (
    validate_batch
)
from src.ui.service_factory import get_gui_adapter

taxonomy = pd.read_csv(taxonomy_path)

categories = set(
    taxonomy["complaint_category"]
    .dropna()
    .astype(str)
    .str.strip()
    .str.upper()
)

assert len(categories) == 15

# --------------------------------------------------
# 1. Define the reusable batch processor
# --------------------------------------------------

processor_source = dedent('''
"""Batch processing using saved historical services.

Each input record receives a traceable outcome.
Existing models are never retrained or updated.
"""

import pandas as pd

from src.integration.bulk_complaint_contract import (
    validate_batch
)


def process_complaint_batch(
    frame,
    supported_categories,
    adapter,
    use_cache=True
):
    validated = validate_batch(
        frame,
        supported_categories
    )

    results = []
    responses_by_source_row = {}
    cache = {}
    backend_evaluations = 0

    for record in validated.to_dict("records"):

        output = dict(record)

        output.update({
            "processing_status": None,
            "system_status": None,
            "borough": None,
            "geography_status": None,
            "zip_validation_status": None,
            "dbscan_status": None,
            "dbscan_cluster_id": None,
            "kmeans_status": None,
            "kmeans_area_id": None,
            "kmeans_cluster_id": None,
            "kmeans_reporting_profile": None,
            "kmeans_display_allowed": False,
            "featured_count": 0,
            "supporting_count": 0,
            "limited_evidence_count": 0,
            "warning_count": 0,
            "processing_error": None
        })

        if record["validation_status"] != (
            "ready_for_geographic_validation"
        ):
            output["processing_status"] = (
                "input_rejected"
            )

            results.append(output)
            continue

        zip_value = record["incident_zip"]

        if pd.isna(zip_value):
            zip_value = None

        # The historical backend is deterministic
        # for identical normalized input values.
        # Caching avoids repeating costly lookups.
        key = (
            float(record["latitude"]),
            float(record["longitude"]),
            str(record["complaint_category"]),
            zip_value
        )

        try:

            if use_cache and key in cache:
                response = cache[key]

            else:
                backend_evaluations += 1

                response = adapter.evaluate(
                    key[0],
                    key[1],
                    key[2],
                    key[3]
                )

                if use_cache:
                    cache[key] = response

            geography = response.get(
                "geography"
            ) or {}

            dbscan = response.get(
                "dbscan"
            ) or {}

            kmeans = response.get(
                "kmeans"
            ) or {}

            zip_info = response.get(
                "zip_validation"
            ) or {}

            recommendations = response.get(
                "recommendations"
            ) or {}

            warnings = response.get(
                "warnings"
            ) or []

            needs_review = (
                response.get("status")
                == "geographic_review_required"
            )

            output.update({
                "processing_status": (
                    "geographic_review_required"
                    if needs_review
                    else "processed"
                ),
                "system_status": response.get(
                    "status"
                ),
                "borough": geography.get(
                    "borough"
                ),
                "geography_status": geography.get(
                    "status"
                ),
                "zip_validation_status":
                    zip_info.get("status"),
                "dbscan_status": dbscan.get(
                    "status"
                ),
                "dbscan_cluster_id":
                    dbscan.get("cluster_id"),
                "kmeans_status": kmeans.get(
                    "status"
                ),
                "kmeans_area_id": kmeans.get(
                    "area_id"
                ),
                "kmeans_cluster_id":
                    kmeans.get("cluster_id"),
                "kmeans_reporting_profile":
                    kmeans.get("reporting_profile"),
                "kmeans_display_allowed": (
                    kmeans.get("display_allowed")
                    is True
                ),
                "featured_count": len(
                    recommendations.get(
                        "featured", []
                    )
                ),
                "supporting_count": len(
                    recommendations.get(
                        "supporting", []
                    )
                ),
                "limited_evidence_count": len(
                    recommendations.get(
                        "limited_evidence", []
                    )
                ),
                "warning_count": len(warnings)
            })

            # Preserve full historical evidence
            # separately, indexed by source row.
            responses_by_source_row[
                int(record["source_row"])
            ] = response

        except Exception as exc:

            output["processing_status"] = (
                "backend_error"
            )

            output["processing_error"] = (
                type(exc).__name__
                + ": "
                + str(exc)[:250]
            )

        results.append(output)

    table = pd.DataFrame.from_records(results)

    return {
        "records": table,
        "responses_by_source_row":
            responses_by_source_row,
        "stats": {
            "total": len(table),
            "processed": int(
                table["processing_status"]
                .eq("processed").sum()
            ),
            "geographic_review": int(
                table["processing_status"]
                .eq("geographic_review_required")
                .sum()
            ),
            "input_rejected": int(
                table["processing_status"]
                .eq("input_rejected").sum()
            ),
            "backend_errors": int(
                table["processing_status"]
                .eq("backend_error").sum()
            ),
            "backend_evaluations":
                backend_evaluations,
            "cache_entries": len(cache)
        }
    }
''').lstrip()

compile(
    processor_source,
    str(module_path),
    "exec"
)

namespace = {}

exec(
    compile(
        processor_source,
        str(module_path),
        "exec"
    ),
    namespace
)

process_complaint_batch = namespace[
    "process_complaint_batch"
]

# --------------------------------------------------
# 2. Construct 500 synthetic incoming complaints
# --------------------------------------------------

print("\n1. PREPARING MIXED 500-COMPLAINT BATCH")

new_batch = pd.DataFrame([
    {
        "complaint_id": f"NEW-{i:04d}",
        "created_date": "2026-11-09",
        "latitude": "40.702106342862",
        "longitude": "-74.013109867312",
        "complaint_category": "STREET CONDITION",
        "incident_zip": ""
    }
    for i in range(1, 501)
])

# 125 geographically consistent Manhattan records.
new_batch.loc[
    200:324, "latitude"
] = "40.75169000334706"

new_batch.loc[
    200:324, "longitude"
] = "-74.00466101019599"

new_batch.loc[
    200:324, "incident_zip"
] = "10001"

# 100 syntactically valid records outside NYC.
new_batch.loc[
    325:424, "latitude"
] = "0.0"

new_batch.loc[
    325:424, "longitude"
] = "0.0"

# 50 records with a historical ZIP whose
# ZCTA polygon is unavailable.
new_batch.loc[
    425:474, "incident_zip"
] = "10000"

# 20 records with a mismatched Manhattan ZIP.
new_batch.loc[
    475:494, "incident_zip"
] = "10002"

# Five malformed records.
new_batch.loc[495, "latitude"] = "invalid"

new_batch.loc[
    496, "complaint_category"
] = "UNKNOWN CATEGORY"

new_batch.loc[
    497, "incident_zip"
] = "83"

new_batch.loc[
    498, "created_date"
] = "2026-02-30"

new_batch.loc[
    499, "complaint_id"
] = ""

assert len(new_batch) == 500

# --------------------------------------------------
# 3. Process the full incoming batch
# --------------------------------------------------

print("\n2. PROCESSING COMPLAINTS")

adapter = get_gui_adapter()

batch_output = process_complaint_batch(
    new_batch,
    categories,
    adapter,
    use_cache=True
)

batch_results = batch_output["records"]
batch_stats = batch_output["stats"]
batch_responses = batch_output[
    "responses_by_source_row"
]

print("\nBATCH PROCESSING SUMMARY")

for name, value in batch_stats.items():
    print(f"{name}: {value}")

print(
    "\nNote: Identical complaint locations and "
    "categories reuse deterministic historical "
    "lookup results. Every complaint still "
    "receives its own output row."
)

# --------------------------------------------------
# 4. Verify all 500 records are accounted for
# --------------------------------------------------

checks = []

def check(name, condition):
    checks.append({
        "test": name,
        "passed": bool(condition)
    })

check(
    "All 500 complaints retained",
    len(batch_results) == 500
)

check(
    "All source rows remain unique",
    batch_results["source_row"].is_unique
)

check(
    "Five invalid complaints rejected",
    batch_stats["input_rejected"] == 5
)

check(
    "100 complaints require geographic review",
    batch_stats["geographic_review"] == 100
)

check(
    "395 complaints processed normally",
    batch_stats["processed"] == 395
)

check(
    "No backend processing errors",
    batch_stats["backend_errors"] == 0
)

check(
    "Every record has a processing outcome",
    batch_results["processing_status"].notna().all()
)

check(
    "All valid complaints have historical responses",
    len(batch_responses) == 495
)

# --------------------------------------------------
# 5. Verify known hotspot association
# --------------------------------------------------

print("\n3. HISTORICAL DBSCAN VALIDATION")

known_group = batch_results.iloc[:200]

check(
    "Known hotspot complaints associate with 1421",
    known_group["dbscan_cluster_id"].eq(
        1421
    ).all()
)

check(
    "Known hotspot complaints show three Featured actions",
    known_group["featured_count"].eq(
        3
    ).all()
)

check(
    "Missing ZIP never invents K-Means profile",
    known_group[
        "kmeans_display_allowed"
    ].eq(False).all()
)

# --------------------------------------------------
# 6. Verify consistent ZIP / historical K-Means
# --------------------------------------------------

print("\n4. HISTORICAL K-MEANS VALIDATION")

consistent_group = batch_results.iloc[
    200:325
]

check(
    "Consistent ZIP group identifies Manhattan",
    consistent_group["borough"].eq(
        "MANHATTAN"
    ).all()
)

check(
    "Consistent ZIP group passes ZCTA validation",
    consistent_group[
        "zip_validation_status"
    ].eq("zcta_consistent").all()
)

check(
    "Historical area MANHATTAN_10001 retrieved",
    consistent_group[
        "kmeans_area_id"
    ].eq("MANHATTAN_10001").all()
)

check(
    "Consistent ZIP permits historical area profile",
    consistent_group[
        "kmeans_display_allowed"
    ].eq(True).all()
)

# --------------------------------------------------
# 7. Verify geographic review safeguards
# --------------------------------------------------

print("\n5. GEOGRAPHIC REVIEW VALIDATION")

outside_group = batch_results.iloc[
    325:425
]

check(
    "Outside-NYC complaints require review",
    outside_group["processing_status"].eq(
        "geographic_review_required"
    ).all()
)

check(
    "Outside-NYC complaints have no DBSCAN cluster",
    outside_group[
        "dbscan_cluster_id"
    ].isna().all()
)

check(
    "Outside-NYC complaints have no K-Means display",
    outside_group[
        "kmeans_display_allowed"
    ].eq(False).all()
)

# --------------------------------------------------
# 8. Verify unavailable and mismatched ZCTAs
# --------------------------------------------------

print("\n6. ZIP REFERENCE SAFEGUARDS")

unavailable_group = batch_results.iloc[
    425:475
]

mismatch_group = batch_results.iloc[
    475:495
]

check(
    "Unavailable ZCTA explicitly identified",
    unavailable_group[
        "zip_validation_status"
    ].eq("zcta_unavailable").all()
)

check(
    "Unavailable ZCTA uses historical lookup",
    unavailable_group[
        "kmeans_display_allowed"
    ].eq(True).all()
)

check(
    "Mismatched ZIP explicitly identified",
    mismatch_group[
        "zip_validation_status"
    ].eq("zcta_mismatch").all()
)

check(
    "Mismatched ZIP K-Means profile withheld",
    mismatch_group[
        "kmeans_display_allowed"
    ].eq(False).all()
)

check(
    "ZIP mismatch does not change DBSCAN hotspot",
    mismatch_group[
        "dbscan_cluster_id"
    ].eq(1421).all()
)

# --------------------------------------------------
# 9. Verify invalid records did not reach backend
# --------------------------------------------------

print("\n7. INVALID RECORD ISOLATION")

invalid_group = batch_results.iloc[
    495:500
]

check(
    "Malformed records remain visible",
    invalid_group["processing_status"].eq(
        "input_rejected"
    ).all()
)

check(
    "Malformed records have no model assignments",
    invalid_group[
        "dbscan_cluster_id"
    ].isna().all()
    and invalid_group[
        "kmeans_cluster_id"
    ].isna().all()
)

# --------------------------------------------------
# 10. Validate response integrity
# --------------------------------------------------

check(
    "All returned backend responses are JSON compatible",
    all(
        isinstance(
            json.dumps(
                response,
                allow_nan=False
            ),
            str
        )
        for response in {
            id(response): response
            for response in batch_responses.values()
        }.values()
    )
)

check(
    "Original historical model period preserved",
    all(
        response["historical_period"]
        == "2023-2025"
        for response in batch_responses.values()
    )
)

# --------------------------------------------------
# 11. Display complaint outcome summaries
# --------------------------------------------------

print("\n8. COMPLAINT PROCESSING STATUS")

display(
    batch_results.groupby(
        "processing_status",
        dropna=False
    ).size().rename(
        "complaints"
    ).reset_index()
)

print("\n9. SAMPLE COMPLAINT OUTPUTS")

sample_rows = [
    0, 200, 325, 425, 475, 495
]

display(
    batch_results.iloc[sample_rows][[
        "complaint_id",
        "processing_status",
        "borough",
        "zip_validation_status",
        "dbscan_cluster_id",
        "kmeans_area_id",
        "kmeans_display_allowed",
        "featured_count",
        "validation_errors"
    ]]
)

# --------------------------------------------------
# 12. Verify test results before writing module
# --------------------------------------------------

results_17_2a = pd.DataFrame(checks)

print("\n10. VALIDATION CHECKLIST")

display(results_17_2a)

passed = int(results_17_2a["passed"].sum())
total = len(results_17_2a)

print(f"\nChecks passed: {passed}/{total}")

failures = results_17_2a.loc[
    ~results_17_2a["passed"],
    "test"
].tolist()

assert not failures, (
    "Phase 17.2A failures: "
    + "; ".join(failures)
)

# --------------------------------------------------
# 13. Save reusable processor safely
# --------------------------------------------------

if module_path.exists():

    assert module_path.read_text(
        encoding="utf-8"
    ) == processor_source, (
        "Existing bulk processor module differs. "
        "No overwrite performed."
    )

else:

    module_path.parent.mkdir(
        parents=True,
        exist_ok=True
    )

    module_path.write_text(
        processor_source,
        encoding="utf-8"
    )

print("\n11. MODULE CREATED / VERIFIED")

print("Processor:", module_path)

print("\nSTATUS: PASSED")
print("BULK PROCESSING VERIFIED")
print("500 COMPLAINT OUTCOMES GENERATED")
print("HISTORICAL DBSCAN AND K-MEANS SERVICES REUSED")
print("EXISTING MODEL ARTIFACTS UNMODIFIED")


 BULK COMPLAINT BACKEND PROCESSING

1. PREPARING MIXED 500-COMPLAINT BATCH

2. PROCESSING COMPLAINTS

BATCH PROCESSING SUMMARY
total: 500
processed: 395
geographic_review: 100
input_rejected: 5
backend_errors: 0
backend_evaluations: 5
cache_entries: 5

Note: Identical complaint locations and categories reuse deterministic historical lookup results. Every complaint still receives its own output row.

3. HISTORICAL DBSCAN VALIDATION

4. HISTORICAL K-MEANS VALIDATION

5. GEOGRAPHIC REVIEW VALIDATION

6. ZIP REFERENCE SAFEGUARDS

7. INVALID RECORD ISOLATION

8. COMPLAINT PROCESSING STATUS


,processing_status,complaints
0,geographic_review_required,100
1,input_rejected,5
2,processed,395



9. SAMPLE COMPLAINT OUTPUTS


,complaint_id,processing_status,borough,zip_validation_status,dbscan_cluster_id,kmeans_area_id,kmeans_display_allowed,featured_count,validation_errors
0,NEW-0001,processed,MANHATTAN,zip_not_provided,1421.0,NaN,False,3,
200,NEW-0201,processed,MANHATTAN,zcta_consistent,1964.0,MANHATTAN_10001,True,2,
325,NEW-0326,geographic_review_required,NaN,skipped_geographic_review,NaN,NaN,False,0,
425,NEW-0426,processed,MANHATTAN,zcta_unavailable,1421.0,MANHATTAN_10000,True,3,
475,NEW-0476,processed,MANHATTAN,zcta_mismatch,1421.0,NaN,False,3,
495,NEW-0496,input_rejected,NaN,NaN,NaN,NaN,False,0,invalid_latitude



10. VALIDATION CHECKLIST


,test,passed
0,All 500 complaints retained,True
1,All source rows remain unique,True
2,Five invalid complaints rejected,True
3,100 complaints require geographic review,True
4,395 complaints processed normally,True
5,No backend processing errors,True
6,Every record has a processing outcome,True
7,All valid complaints have historical responses,True
8,Known hotspot complaints associate with 1421,True
9,Known hotspot complaints show three Featured a...,True



Checks passed: 27/27

11. MODULE CREATED / VERIFIED
Processor: D:\Urban-Infrastructure-Pulse-Intelligence\src\integration\bulk_complaint_processor.py

STATUS: PASSED
BULK PROCESSING VERIFIED
500 COMPLAINT OUTCOMES GENERATED
HISTORICAL DBSCAN AND K-MEANS SERVICES REUSED
EXISTING MODEL ARTIFACTS UNMODIFIED


In [5]:

from pathlib import Path
from textwrap import dedent
from io import BytesIO
import json
import sys
import pandas as pd

print(" BULK EXPORTS AND REVIEW QUEUES")
print("=" * 75)

root = Path(PROJECT_ROOT).resolve()

if str(root) not in sys.path:
    sys.path.insert(0, str(root))

module_path = (
    root / "src/integration/bulk_complaint_export.py"
)

export_dir = (
    root / "outputs/tests"
    / "phase_17_2b_synthetic_exports"
)

assert "batch_output" in globals(), (
    "Run Phase 17.2A first."
)

# --------------------------------------------------
# 1. Create reusable export module
# --------------------------------------------------

module_source = dedent('''
"""Safe CSV exports for bulk complaint outcomes.

Exported review queues are not operational
risk rankings or repair assignments.
"""

from io import BytesIO
import json
import pandas as pd


REVIEW_ZIP_STATUSES = {
    "zcta_mismatch",
    "boundary_review_required"
}


def _safe_cell(value):
    if value is None:
        return ""

    try:
        if pd.isna(value):
            return ""
    except (TypeError, ValueError):
        pass

    text = str(value)

    # Prevent spreadsheet formula execution
    # when exported text is opened in Excel.
    if text.lstrip().startswith(
        ("=", "+", "-", "@")
    ) or text.startswith(("\\t", "\\r", "\\n")):
        return "'" + text

    return text


def csv_bytes(frame):
    work = frame.copy()

    for column in work.columns:

        if pd.api.types.is_numeric_dtype(
            work[column]
        ):
            continue

        work[column] = work[column].map(
            _safe_cell
        )

    return work.to_csv(
        index=False,
        na_rep="",
        lineterminator="\\n"
    ).encode("utf-8-sig")


def build_bulk_export_package(batch_output):

    records = batch_output["records"].copy()

    required = {
        "source_row",
        "complaint_id",
        "processing_status",
        "validation_errors",
        "zip_validation_status",
        "dbscan_cluster_id",
        "kmeans_display_allowed"
    }

    if not required.issubset(records.columns):
        raise ValueError(
            "Bulk processor output schema changed."
        )

    if records.empty:
        raise ValueError("No complaint outcomes.")

    if records["source_row"].duplicated().any():
        raise ValueError(
            "Duplicate source-row identifiers."
        )

    statuses = records["processing_status"]

    allowed = {
        "processed",
        "geographic_review_required",
        "input_rejected",
        "backend_error"
    }

    if (
        statuses.isna().any()
        or not set(statuses).issubset(allowed)
    ):
        raise ValueError(
            "Unexpected processing status."
        )

    geo_mask = statuses.eq(
        "geographic_review_required"
    )

    invalid_mask = statuses.eq(
        "input_rejected"
    )

    error_mask = statuses.eq(
        "backend_error"
    )

    processed_mask = statuses.eq(
        "processed"
    )

    zip_mask = records[
        "zip_validation_status"
    ].isin(REVIEW_ZIP_STATUSES)

    # Review queues are subsets. An otherwise
    # processed complaint may need ZIP review.
    review_mask = (
        geo_mask | zip_mask | error_mask
    )

    records["review_reason"] = ""

    records.loc[
        geo_mask, "review_reason"
    ] = "geographic_review"

    records.loc[
        zip_mask & ~geo_mask,
        "review_reason"
    ] = "zip_validation_review"

    records.loc[
        error_mask, "review_reason"
    ] = "backend_processing_error"

    review = records.loc[
        review_mask
    ].copy()

    rejected = records.loc[
        invalid_mask
    ].copy()

    processed = records.loc[
        processed_mask
    ].copy()

    zip_review = records.loc[
        zip_mask
    ].copy()

    errors = records.loc[
        error_mask
    ].copy()

    stats = batch_output.get("stats", {})

    if int(stats.get("total", -1)) != len(records):
        raise ValueError(
            "Batch input count mismatch."
        )

    status_counts = {
        name: int(statuses.eq(name).sum())
        for name in sorted(allowed)
    }

    for status, stat_name in [
        ("processed", "processed"),
        ("geographic_review_required",
         "geographic_review"),
        ("input_rejected", "input_rejected"),
        ("backend_error", "backend_errors")
    ]:
        if status_counts[status] != int(
            stats.get(stat_name, -1)
        ):
            raise ValueError(
                "Batch status totals do not agree."
            )

    summary = {
        "schema_version": "bulk_export_v1",
        "input_type": "synthetic_test_batch",
        "historical_reference_period": "2023-2025",
        "total_complaints": len(records),
        "status_counts": status_counts,
        "review_queue_rows": len(review),
        "zip_review_rows": len(zip_review),
        "rejected_input_rows": len(rejected),
        "backend_error_rows": len(errors),
        "historical_hotspot_associations": int(
            records["dbscan_cluster_id"]
            .notna().sum()
        ),
        "historical_kmeans_profiles_displayed": int(
            records["kmeans_display_allowed"]
            .eq(True).sum()
        ),
        "backend_evaluations": int(
            stats.get("backend_evaluations", 0)
        ),
        "notes": [
            "This batch contains synthetic test records.",
            "Review queues are not safety rankings.",
            "ZIP review can coexist with processed status.",
            "Historical clusters are not retrained.",
            "No new hotspots are discovered in this phase."
        ]
    }

    tables = {
        "all_complaint_outcomes.csv": records,
        "processed_complaints.csv": processed,
        "review_queue.csv": review,
        "zip_review.csv": zip_review,
        "rejected_inputs.csv": rejected,
        "backend_errors.csv": errors
    }

    files = {
        filename: csv_bytes(table)
        for filename, table in tables.items()
    }

    files["batch_summary.json"] = json.dumps(
        summary,
        indent=2,
        allow_nan=False
    ).encode("utf-8")

    return {
        "files": files,
        "tables": tables,
        "summary": summary
    }
''').lstrip()

compile(
    module_source,
    str(module_path),
    "exec"
)

namespace = {}

exec(
    compile(
        module_source,
        str(module_path),
        "exec"
    ),
    namespace
)

build_bulk_export_package = namespace[
    "build_bulk_export_package"
]

csv_bytes = namespace["csv_bytes"]

# --------------------------------------------------
# 2. Generate export package in memory
# --------------------------------------------------

print("\n1. BUILDING EXPORT PACKAGE")

package = build_bulk_export_package(
    batch_output
)

files = package["files"]
tables = package["tables"]
summary = package["summary"]

display(
    pd.DataFrame([
        {
            "file": filename,
            "rows": (
                len(tables[filename])
                if filename in tables
                else "-"
            ),
            "size_bytes": len(content)
        }
        for filename, content in files.items()
    ])
)

# --------------------------------------------------
# 3. Validate counts and review logic
# --------------------------------------------------

checks = []

def check(name, passed):
    checks.append({
        "test": name,
        "passed": bool(passed)
    })

check(
    "All 500 outcomes exported",
    len(tables["all_complaint_outcomes.csv"])
    == 500
)

check(
    "395 processed records preserved",
    len(tables["processed_complaints.csv"])
    == 395
)

check(
    "100 geographic review records preserved",
    tables["review_queue.csv"][
        "processing_status"
    ].eq(
        "geographic_review_required"
    ).sum() == 100
)

check(
    "20 ZIP mismatch records identified",
    len(tables["zip_review.csv"]) == 20
)

check(
    "Review queue includes 120 records",
    len(tables["review_queue.csv"]) == 120
)

check(
    "All 5 malformed records exported",
    len(tables["rejected_inputs.csv"]) == 5
)

check(
    "No backend errors recorded",
    len(tables["backend_errors.csv"]) == 0
)

check(
    "Original complaint records preserved",
    set(
        tables["all_complaint_outcomes.csv"][
            "source_row"
        ]
    ) == set(batch_results["source_row"])
)

check(
    "ZIP review does not hide DBSCAN results",
    tables["zip_review.csv"][
        "dbscan_cluster_id"
    ].notna().all()
)

check(
    "Review reasons are traceable",
    tables["review_queue.csv"][
        "review_reason"
    ].ne("").all()
)

# --------------------------------------------------
# 4. Verify CSV export round trips
# --------------------------------------------------

print("\n2. TESTING CSV READABILITY")

roundtrip = {}

for filename, content in files.items():

    if not filename.endswith(".csv"):
        continue

    roundtrip[filename] = pd.read_csv(
        BytesIO(content),
        dtype=str,
        keep_default_na=False,
        encoding="utf-8-sig"
    )

    check(
        f"CSV reopens with correct row count: {filename}",
        len(roundtrip[filename])
        == len(tables[filename])
    )

check(
    "Complaint IDs survive CSV export",
    roundtrip["all_complaint_outcomes.csv"][
        "complaint_id"
    ].iloc[0] == "NEW-0001"
)

check(
    "Historical ZIP 10001 preserved",
    roundtrip["all_complaint_outcomes.csv"][
        "incident_zip"
    ].iloc[200] == "10001"
)

check(
    "Missing ZIP remains blank",
    roundtrip["all_complaint_outcomes.csv"][
        "incident_zip"
    ].iloc[0] == ""
)

# --------------------------------------------------
# 5. Verify spreadsheet formula safety
# --------------------------------------------------

dangerous = pd.DataFrame([
    {
        "complaint_id": "=2+2",
        "created_date": "2026-11-09",
        "latitude": "=2+2",
        "longitude": -74.01
    }
])

dangerous_csv = csv_bytes(
    dangerous
).decode("utf-8-sig")

check(
    "Formula-like complaint IDs safely escaped",
    "'=2+2" in dangerous_csv
)

check(
    "Formula-like invalid coordinates safely escaped",
    dangerous_csv.count("'=2+2") == 2
)

# --------------------------------------------------
# 6. Verify summary JSON
# --------------------------------------------------

loaded_summary = json.loads(
    files["batch_summary.json"].decode("utf-8")
)

check(
    "Summary JSON total is 500",
    loaded_summary["total_complaints"] == 500
)

check(
    "Summary preserves original history period",
    loaded_summary[
        "historical_reference_period"
    ] == "2023-2025"
)

check(
    "Summary distinguishes ZIP review from rejection",
    loaded_summary["zip_review_rows"] == 20
    and loaded_summary["rejected_input_rows"] == 5
)

check(
    "Historical K-Means display count is 175",
    loaded_summary[
        "historical_kmeans_profiles_displayed"
    ] == 175
)

check(
    "Historical DBSCAN association count is 395",
    loaded_summary[
        "historical_hotspot_associations"
    ] == 395
)

# --------------------------------------------------
# 7. Verify all results before saving
# --------------------------------------------------

results_17_2b = pd.DataFrame(checks)

print("\n3. TEST CHECKLIST")

display(results_17_2b)

passed = int(results_17_2b["passed"].sum())
total = len(results_17_2b)

print(f"\nChecks passed: {passed}/{total}")

failures = results_17_2b.loc[
    ~results_17_2b["passed"],
    "test"
].tolist()

assert not failures, (
    "Phase 17.2B failed: "
    + "; ".join(failures)
)

# --------------------------------------------------
# 8. Preflight existing files to avoid overwrites
# --------------------------------------------------

if module_path.exists():
    assert module_path.read_text(
        encoding="utf-8"
    ) == module_source, (
        "Existing export module differs. "
        "No overwrite performed."
    )

for filename, content in files.items():

    target = export_dir / filename

    if target.exists():
        assert target.read_bytes() == content, (
            f"Existing export differs: {filename}. "
            "No overwrite performed."
        )

# --------------------------------------------------
# 9. Save reusable module and synthetic test exports
# --------------------------------------------------

module_path.parent.mkdir(
    parents=True,
    exist_ok=True
)

export_dir.mkdir(
    parents=True,
    exist_ok=True
)

if not module_path.exists():
    module_path.write_text(
        module_source,
        encoding="utf-8"
    )

for filename, content in files.items():

    target = export_dir / filename

    if not target.exists():
        target.write_bytes(content)

print("\n4. SAVED FILES")
print("Exporter module:", module_path)
print("Synthetic export folder:", export_dir)

for filename in files:
    print(" -", filename)

print("\n5. BATCH SUMMARY")
print(json.dumps(summary, indent=2))

print("\nSTATUS: PASSED")
print("CSV EXPORT AND REVIEW QUEUES VERIFIED")
print("ALL 500 SYNTHETIC COMPLAINTS ACCOUNTED FOR")
print("EXISTING MODEL ARTIFACTS UNMODIFIED")


 BULK EXPORTS AND REVIEW QUEUES

1. BUILDING EXPORT PACKAGE


,file,rows,size_bytes
0,all_complaint_outcomes.csv,500,138192
1,processed_complaints.csv,395,108452
2,review_queue.csv,120,34933
3,zip_review.csv,20,5933
4,rejected_inputs.csv,5,1173
5,backend_errors.csv,0,433
6,batch_summary.json,-,779



2. TESTING CSV READABILITY

3. TEST CHECKLIST


,test,passed
0,All 500 outcomes exported,True
1,395 processed records preserved,True
2,100 geographic review records preserved,True
3,20 ZIP mismatch records identified,True
4,Review queue includes 120 records,True
5,All 5 malformed records exported,True
6,No backend errors recorded,True
7,Original complaint records preserved,True
8,ZIP review does not hide DBSCAN results,True
9,Review reasons are traceable,True



Checks passed: 26/26

4. SAVED FILES
Exporter module: D:\Urban-Infrastructure-Pulse-Intelligence\src\integration\bulk_complaint_export.py
Synthetic export folder: D:\Urban-Infrastructure-Pulse-Intelligence\outputs\tests\phase_17_2b_synthetic_exports
 - all_complaint_outcomes.csv
 - processed_complaints.csv
 - review_queue.csv
 - zip_review.csv
 - rejected_inputs.csv
 - backend_errors.csv
 - batch_summary.json

5. BATCH SUMMARY
{
  "schema_version": "bulk_export_v1",
  "input_type": "synthetic_test_batch",
  "historical_reference_period": "2023-2025",
  "total_complaints": 500,
  "status_counts": {
    "backend_error": 0,
    "geographic_review_required": 100,
    "input_rejected": 5,
    "processed": 395
  },
  "review_queue_rows": 120,
  "zip_review_rows": 20,
  "rejected_input_rows": 5,
  "backend_error_rows": 0,
  "historical_hotspot_associations": 395,
  "historical_kmeans_profiles_displayed": 175,
  "backend_evaluations": 5,
  "notes": [
    "This batch contains synthetic test re

In [6]:

from pathlib import Path
from textwrap import dedent
import sys
import pandas as pd

print(" COMPLAINT GROUPING AND SUMMARIES")
print("=" * 75)

root = Path(PROJECT_ROOT).resolve()

if str(root) not in sys.path:
    sys.path.insert(0, str(root))

module_path = (
    root / "src/integration/bulk_complaint_grouping.py"
)

hotspot_path = (
    root / "outputs/tables/dbscan"
    / "dbscan_primary_f_gui_hotspots.csv"
)

assert "batch_output" in globals(), (
    "Phase 17.2A batch output is missing."
)

assert hotspot_path.is_file()

hotspot_reference = pd.read_csv(hotspot_path)

# --------------------------------------------------
# 1. Create reusable grouping module
# --------------------------------------------------

module_source = dedent('''
"""Summaries of incoming complaints and historical context.

These are descriptive reporting groups, not
operational safety priorities or new ML clusters.
"""

import pandas as pd


def summarize_bulk_complaints(
    batch_output,
    hotspot_reference
):

    records = batch_output["records"].copy()
    reference = hotspot_reference.copy()

    required = {
        "source_row",
        "complaint_id",
        "created_date",
        "processing_status",
        "borough",
        "complaint_category",
        "latitude",
        "longitude",
        "dbscan_status",
        "dbscan_cluster_id",
        "zip_validation_status",
        "kmeans_display_allowed",
        "kmeans_area_id",
        "kmeans_cluster_id",
        "kmeans_reporting_profile"
    }

    if not required.issubset(records.columns):
        raise ValueError(
            "Bulk processor output schema changed."
        )

    if records.empty:
        raise ValueError("Empty batch results.")

    if records["source_row"].duplicated().any():
        raise ValueError("Duplicate source rows.")

    if not {
        "dbscan_cluster_id",
        "total_complaints",
        "primary_borough",
        "dominant_category"
    }.issubset(reference.columns):
        raise ValueError(
            "Historical hotspot reference changed."
        )

    if reference[
        "dbscan_cluster_id"
    ].duplicated().any():
        raise ValueError(
            "Duplicate historical hotspot IDs."
        )

    records["dbscan_cluster_id"] = pd.to_numeric(
        records["dbscan_cluster_id"],
        errors="coerce"
    ).astype("Int64")

    records["kmeans_cluster_id"] = pd.to_numeric(
        records["kmeans_cluster_id"],
        errors="coerce"
    ).astype("Int64")

    reference["dbscan_cluster_id"] = pd.to_numeric(
        reference["dbscan_cluster_id"],
        errors="raise"
    ).astype("Int64")

    # ----------------------------------------------
    # 2. Separate input processing outcomes
    # ----------------------------------------------

    processed = records.loc[
        records["processing_status"].eq(
            "processed"
        )
    ].copy()

    geographic_review = records.loc[
        records["processing_status"].eq(
            "geographic_review_required"
        )
    ].copy()

    rejected = records.loc[
        records["processing_status"].eq(
            "input_rejected"
        )
    ].copy()

    backend_errors = records.loc[
        records["processing_status"].eq(
            "backend_error"
        )
    ].copy()

    if (
        len(processed)
        + len(geographic_review)
        + len(rejected)
        + len(backend_errors)
        != len(records)
    ):
        raise ValueError(
            "Batch processing totals do not reconcile."
        )

    # ----------------------------------------------
    # 3. Group processed complaints by borough
    # ----------------------------------------------

    by_borough = (
        processed.groupby("borough")
        .size()
        .reset_index(name="new_complaints")
        .sort_values(
            ["new_complaints", "borough"],
            ascending=[False, True]
        )
        .reset_index(drop=True)
    )

    # ----------------------------------------------
    # 4. Group processed complaints by category
    # ----------------------------------------------

    by_category = (
        processed.groupby("complaint_category")
        .size()
        .reset_index(name="new_complaints")
        .sort_values(
            ["new_complaints", "complaint_category"],
            ascending=[False, True]
        )
        .reset_index(drop=True)
    )

    if len(processed):
        by_category["share_of_processed_pct"] = (
            100
            * by_category["new_complaints"]
            / len(processed)
        ).round(2)
    else:
        by_category["share_of_processed_pct"] = (
            pd.Series(dtype=float)
        )

    # ----------------------------------------------
    # 5. Group new reports by historical DBSCAN ID
    # ----------------------------------------------

    associated = processed.loc[
        processed["dbscan_status"].eq(
            "established_cluster"
        )
    ].copy()

    if associated[
        "dbscan_cluster_id"
    ].isna().any():
        raise ValueError(
            "Established hotspot has no cluster ID."
        )

    # Exact coordinates are descriptive only.
    # Multiple reports at the same coordinate
    # do not prove multiple physical defects.
    associated["location_key"] = [
        repr(float(lat)) + "|" + repr(float(lon))
        for lat, lon in zip(
            associated["latitude"],
            associated["longitude"]
        )
    ]

    hotspot_groups = (
        associated.groupby("dbscan_cluster_id")
        .agg(
            incoming_complaints=(
                "source_row", "size"
            ),
            distinct_submitted_locations=(
                "location_key", "nunique"
            ),
            incoming_categories=(
                "complaint_category", "nunique"
            )
        )
        .reset_index()
    )

    historical_columns = reference[[
        "dbscan_cluster_id",
        "total_complaints",
        "primary_borough",
        "dominant_category"
    ]].rename(columns={
        "total_complaints":
            "historical_complaints_2023_2025",
        "primary_borough":
            "historical_primary_borough",
        "dominant_category":
            "historical_dominant_category"
    })

    hotspot_groups = hotspot_groups.merge(
        historical_columns,
        on="dbscan_cluster_id",
        how="left",
        validate="one_to_one"
    )

    if hotspot_groups[
        "historical_complaints_2023_2025"
    ].isna().any():
        raise ValueError(
            "Associated hotspot missing from "
            "historical reference."
        )

    hotspot_groups = hotspot_groups.sort_values(
        ["incoming_complaints", "dbscan_cluster_id"],
        ascending=[False, True]
    ).reset_index(drop=True)

    # ----------------------------------------------
    # 6. Category distribution within each hotspot
    # ----------------------------------------------

    by_hotspot_category = (
        associated.groupby([
            "dbscan_cluster_id",
            "complaint_category"
        ])
        .size()
        .reset_index(name="new_complaints")
    )

    if not by_hotspot_category.empty:

        totals = (
            hotspot_groups.set_index(
                "dbscan_cluster_id"
            )["incoming_complaints"]
        )

        by_hotspot_category[
            "share_within_incoming_hotspot_pct"
        ] = (
            100
            * by_hotspot_category["new_complaints"]
            / by_hotspot_category[
                "dbscan_cluster_id"
            ].map(totals)
        ).round(2)

    # ----------------------------------------------
    # 7. Group displayed historical K-Means profiles
    # ----------------------------------------------

    shown = processed.loc[
        processed[
            "kmeans_display_allowed"
        ].eq(True)
    ].copy()

    if shown[
        [
            "kmeans_area_id",
            "kmeans_cluster_id",
            "kmeans_reporting_profile"
        ]
    ].isna().any().any():
        raise ValueError(
            "Displayed area profile is incomplete."
        )

    area_groups = (
        shown.groupby([
            "kmeans_area_id",
            "kmeans_cluster_id",
            "kmeans_reporting_profile",
            "zip_validation_status"
        ], dropna=False)
        .size()
        .reset_index(name="new_complaints")
        .sort_values(
            "new_complaints",
            ascending=False
        )
        .reset_index(drop=True)
    )

    # ----------------------------------------------
    # 8. Review cases are not a safety ranking
    # ----------------------------------------------

    zip_review = records.loc[
        records["zip_validation_status"].isin([
            "zcta_mismatch",
            "boundary_review_required"
        ])
    ].copy()

    review_mask = (
        records["processing_status"].isin([
            "geographic_review_required",
            "backend_error"
        ])
        | records["zip_validation_status"].isin([
            "zcta_mismatch",
            "boundary_review_required"
        ])
    )

    review_queue = records.loc[
        review_mask
    ].copy()

    # ----------------------------------------------
    # 9. Daily incoming complaint counts
    # ----------------------------------------------

    by_date = (
        records.groupby([
            "created_date",
            "processing_status"
        ], dropna=False)
        .size()
        .reset_index(name="complaints")
    )

    # ----------------------------------------------
    # 10. Summary with explicit denominators
    # ----------------------------------------------

    summary = {
        "total_input": len(records),
        "processed": len(processed),
        "geographic_review": len(
            geographic_review
        ),
        "input_rejected": len(rejected),
        "backend_errors": len(backend_errors),
        "associated_with_historical_hotspots":
            len(associated),
        "distinct_associated_historical_hotspots":
            len(hotspot_groups),
        "historical_area_profiles_displayed":
            len(shown),
        "zip_review": len(zip_review),
        "review_queue_rows": len(review_queue)
    }

    return {
        "summary": summary,
        "by_borough": by_borough,
        "by_category": by_category,
        "historical_hotspots": hotspot_groups,
        "by_hotspot_category":
            by_hotspot_category,
        "historical_area_profiles":
            area_groups,
        "by_date": by_date,
        "review_queue": review_queue
    }
''').lstrip()

compile(
    module_source,
    str(module_path),
    "exec"
)

namespace = {}

exec(
    compile(
        module_source,
        str(module_path),
        "exec"
    ),
    namespace
)

summarize_bulk_complaints = namespace[
    "summarize_bulk_complaints"
]

# --------------------------------------------------
# 2. Generate grouped results
# --------------------------------------------------

print("\n1. GROUPING NEW COMPLAINTS")

grouped = summarize_bulk_complaints(
    batch_output,
    hotspot_reference
)

summary = grouped["summary"]

print("\nBATCH GROUPING SUMMARY")

for key, value in summary.items():
    print(f"{key}: {value}")

# --------------------------------------------------
# 3. Display borough and category summaries
# --------------------------------------------------

print("\n2. NEW COMPLAINTS BY BOROUGH")

display(grouped["by_borough"])

print("\n3. NEW COMPLAINTS BY CATEGORY")

display(grouped["by_category"])

# --------------------------------------------------
# 4. Display historical hotspot groups
# --------------------------------------------------

print("\n4. INCOMING REPORTS BY HISTORICAL HOTSPOT")

display(grouped["historical_hotspots"])

print("\n5. CATEGORY SHARES WITHIN HOTSPOT GROUPS")

display(grouped["by_hotspot_category"])

# --------------------------------------------------
# 5. Display historical K-Means area profiles
# --------------------------------------------------

print("\n6. HISTORICAL AREA PROFILES")

display(grouped["historical_area_profiles"])

# --------------------------------------------------
# 6. Validate all grouping outcomes
# --------------------------------------------------

checks = []

def check(name, condition):
    checks.append({
        "test": name,
        "passed": bool(condition)
    })

check(
    "All 500 complaints accounted for",
    summary["total_input"] == 500
)

check(
    "395 complaints processed",
    summary["processed"] == 395
)

check(
    "100 geographic review cases retained",
    summary["geographic_review"] == 100
)

check(
    "Five invalid records retained",
    summary["input_rejected"] == 5
)

check(
    "No backend processing errors",
    summary["backend_errors"] == 0
)

check(
    "395 new reports associated with historical hotspots",
    summary[
        "associated_with_historical_hotspots"
    ] == 395
)

check(
    "Two distinct historical hotspots represented",
    summary[
        "distinct_associated_historical_hotspots"
    ] == 2
)

check(
    "Borough grouping contains 395 complaints",
    int(
        grouped["by_borough"][
            "new_complaints"
        ].sum()
    ) == 395
)

check(
    "Manhattan has all 395 processed complaints",
    int(
        grouped["by_borough"].loc[
            grouped["by_borough"][
                "borough"
            ] == "MANHATTAN",
            "new_complaints"
        ].sum()
    ) == 395
)

check(
    "Category grouping contains 395 complaints",
    int(
        grouped["by_category"][
            "new_complaints"
        ].sum()
    ) == 395
)

check(
    "Hotspot grouping preserves 395 complaints",
    int(
        grouped["historical_hotspots"][
            "incoming_complaints"
        ].sum()
    ) == 395
)

# --------------------------------------------------
# 7. Independently verify exact hotspot grouping
# --------------------------------------------------

hotspot_counts = {
    int(row.dbscan_cluster_id):
        int(row.incoming_complaints)
    for row in grouped[
        "historical_hotspots"
    ].itertuples(index=False)
}

check(
    "Historical hotspot 1421 contains 270 incoming records",
    hotspot_counts.get(1421) == 270
)

check(
    "Historical hotspot 1964 contains 125 incoming records",
    hotspot_counts.get(1964) == 125
)

historical_1421 = hotspot_reference.loc[
    hotspot_reference[
        "dbscan_cluster_id"
    ] == 1421,
    "total_complaints"
].iloc[0]

grouped_1421 = grouped[
    "historical_hotspots"
].loc[
    grouped["historical_hotspots"][
        "dbscan_cluster_id"
    ] == 1421
].iloc[0]

check(
    "Historical 1421 count remains 12773",
    int(historical_1421) == 12773
    and int(
        grouped_1421[
            "historical_complaints_2023_2025"
        ]
    ) == 12773
)

check(
    "Historical and incoming totals remain separate",
    int(
        grouped_1421["incoming_complaints"]
    ) == 270
    and int(
        grouped_1421[
            "historical_complaints_2023_2025"
        ]
    ) == 12773
)

# --------------------------------------------------
# 8. Verify K-Means area grouping
# --------------------------------------------------

area_groups = grouped[
    "historical_area_profiles"
]

area_counts = {
    str(row.kmeans_area_id):
        int(row.new_complaints)
    for row in area_groups.itertuples(index=False)
}

check(
    "175 incoming records have displayed area profiles",
    summary[
        "historical_area_profiles_displayed"
    ] == 175
)

check(
    "Manhattan 10001 has 125 incoming records",
    area_counts.get("MANHATTAN_10001")
    == 125
)

check(
    "Manhattan 10000 has 50 incoming records",
    area_counts.get("MANHATTAN_10000")
    == 50
)

check(
    "Unavailable ZCTA remains distinguishable",
    area_groups.loc[
        area_groups[
            "kmeans_area_id"
        ] == "MANHATTAN_10000",
        "zip_validation_status"
    ].eq("zcta_unavailable").all()
)

# --------------------------------------------------
# 9. Verify review queue accounting
# --------------------------------------------------

check(
    "20 ZIP review records preserved",
    summary["zip_review"] == 20
)

check(
    "Review queue contains 120 records",
    summary["review_queue_rows"] == 120
)

check(
    "Review queue source rows remain unique",
    grouped["review_queue"][
        "source_row"
    ].is_unique
)

check(
    "Incoming category shares sum to 100%",
    abs(
        grouped["by_category"][
            "share_of_processed_pct"
        ].sum() - 100
    ) < 0.01
)

# --------------------------------------------------
# 10. Display verification results
# --------------------------------------------------

results_17_3a = pd.DataFrame(checks)

print("\n7. TEST CHECKLIST")

display(results_17_3a)

passed = int(results_17_3a["passed"].sum())
total = len(results_17_3a)

print(f"\nChecks passed: {passed}/{total}")

failures = results_17_3a.loc[
    ~results_17_3a["passed"],
    "test"
].tolist()

assert not failures, (
    "Phase 17.3A failures: "
    + "; ".join(failures)
)

# --------------------------------------------------
# 11. Safely save the reusable grouping module
# --------------------------------------------------

if module_path.exists():

    assert module_path.read_text(
        encoding="utf-8"
    ) == module_source, (
        "Existing grouping module differs. "
        "No overwrite performed."
    )

else:

    module_path.parent.mkdir(
        parents=True,
        exist_ok=True
    )

    module_path.write_text(
        module_source,
        encoding="utf-8"
    )

print("\n8. MODULE CREATED / VERIFIED")

print("Grouping module:", module_path)

print("\nSTATUS: PASSED")
print("COMPLAINT GROUPING VERIFIED")
print("INCOMING REPORTS LINKED TO HISTORICAL HOTSPOTS")
print("K-MEANS HISTORICAL AREA SUMMARIES VERIFIED")
print("NO NEW HOTSPOTS OR RISK SCORES CLAIMED")
print("EXISTING MODEL ARTIFACTS UNMODIFIED")


 COMPLAINT GROUPING AND SUMMARIES

1. GROUPING NEW COMPLAINTS

BATCH GROUPING SUMMARY
total_input: 500
processed: 395
geographic_review: 100
input_rejected: 5
backend_errors: 0
associated_with_historical_hotspots: 395
distinct_associated_historical_hotspots: 2
historical_area_profiles_displayed: 175
zip_review: 20
review_queue_rows: 120

2. NEW COMPLAINTS BY BOROUGH


,borough,new_complaints
0,MANHATTAN,395



3. NEW COMPLAINTS BY CATEGORY


,complaint_category,new_complaints,share_of_processed_pct
0,STREET CONDITION,395,100.0



4. INCOMING REPORTS BY HISTORICAL HOTSPOT


,dbscan_cluster_id,incoming_complaints,distinct_submitted_locations,incoming_categories,historical_complaints_2023_2025,historical_primary_borough,historical_dominant_category
0,1421,270,1,1,12773,MANHATTAN,STREET CONDITION
1,1964,125,1,1,43,MANHATTAN,STREET CONDITION


... (3 outputs left out)

,kmeans_area_id,kmeans_cluster_id,kmeans_reporting_profile,zip_validation_status,new_complaints
0,MANHATTAN_10001,0,Sustained High-Volume Reporting,zcta_consistent,125
1,MANHATTAN_10000,2,Lower-Volume Persistence-Oriented Reporting,zcta_unavailable,50



7. TEST CHECKLIST


,test,passed
0,All 500 complaints accounted for,True
1,395 complaints processed,True
2,100 geographic review cases retained,True
3,Five invalid records retained,True
4,No backend processing errors,True
5,395 new reports associated with historical hot...,True
6,Two distinct historical hotspots represented,True
7,Borough grouping contains 395 complaints,True
8,Manhattan has all 395 processed complaints,True
9,Category grouping contains 395 complaints,True



Checks passed: 23/23

8. MODULE CREATED / VERIFIED
Grouping module: D:\Urban-Infrastructure-Pulse-Intelligence\src\integration\bulk_complaint_grouping.py

STATUS: PASSED
COMPLAINT GROUPING VERIFIED
INCOMING REPORTS LINKED TO HISTORICAL HOTSPOTS
K-MEANS HISTORICAL AREA SUMMARIES VERIFIED
NO NEW HOTSPOTS OR RISK SCORES CLAIMED
EXISTING MODEL ARTIFACTS UNMODIFIED


In [7]:

from pathlib import Path
from textwrap import dedent
import sys
import pandas as pd

print(" SMART INVESTIGATION SUMMARIES")
print("=" * 75)

root = Path(PROJECT_ROOT).resolve()

if str(root) not in sys.path:
    sys.path.insert(0, str(root))

module_path = (
    root / "src/integration/bulk_investigation_summary.py"
)

assert "batch_output" in globals()
assert "hotspot_reference" in globals()

# --------------------------------------------------
# 1. Build reusable investigation summary module
# --------------------------------------------------

module_source = dedent('''
"""Evidence-based briefs for incoming complaint batches.

Historical hotspot association is not proof of
a defect, repair urgency, or safety risk.
"""

import pandas as pd


def build_investigation_briefs(
    batch_output,
    historical_hotspots
):
    records = batch_output["records"].copy()
    responses = batch_output[
        "responses_by_source_row"
    ]

    reference = historical_hotspots.copy()

    required = {
        "source_row",
        "processing_status",
        "borough",
        "complaint_category",
        "latitude",
        "longitude",
        "dbscan_status",
        "dbscan_cluster_id",
        "zip_validation_status",
        "kmeans_display_allowed",
        "featured_count",
        "supporting_count",
        "limited_evidence_count"
    }

    if not required.issubset(records.columns):
        raise ValueError(
            "Unexpected bulk processor schema."
        )

    if records["source_row"].duplicated().any():
        raise ValueError("Duplicate source rows.")

    reference = reference.set_index(
        "dbscan_cluster_id",
        verify_integrity=True
    )

    # Only established historical associations
    # become historical hotspot briefs.
    established = records.loc[
        records["processing_status"].eq(
            "processed"
        )
        & records["dbscan_status"].eq(
            "established_cluster"
        )
    ].copy()

    established["dbscan_cluster_id"] = (
        pd.to_numeric(
            established["dbscan_cluster_id"],
            errors="raise"
        ).astype(int)
    )

    briefs = []

    for (cluster_id, category), group in (
        established.groupby([
            "dbscan_cluster_id",
            "complaint_category"
        ])
    ):
        cluster_id = int(cluster_id)

        if cluster_id not in reference.index:
            raise ValueError(
                "Historical hotspot reference missing."
            )

        historical = reference.loc[cluster_id]

        incoming_count = len(group)

        location_count = len(set(
            zip(
                group["latitude"].astype(float),
                group["longitude"].astype(float)
            )
        ))

        historical_count = int(
            historical["total_complaints"]
        )

        representative_row = int(
            group["source_row"].iloc[0]
        )

        response = responses.get(
            representative_row
        )

        if response is None:
            raise ValueError(
                "Missing representative backend evidence."
            )

        if (
            response["dbscan"]["cluster_id"]
            != cluster_id
        ):
            raise ValueError(
                "Representative hotspot mismatch."
            )

        category_context = (
            response["dbscan"].get(
                "category_context"
            ) or {}
        )

        historical_category_count = (
            category_context.get(
                "category_complaints"
            )
        )

        if historical_category_count is not None:
            historical_category_count = int(
                historical_category_count
            )

        actions = response.get(
            "recommendations"
        ) or {}

        featured = len(
            actions.get("featured", [])
        )

        supporting = len(
            actions.get("supporting", [])
        )

        limited = len(
            actions.get("limited_evidence", [])
        )

        for column, expected in [
            ("featured_count", featured),
            ("supporting_count", supporting),
            ("limited_evidence_count", limited)
        ]:
            if not group[column].eq(expected).all():
                raise ValueError(
                    "Recommendation count inconsistency."
                )

        zip_review_count = int(
            group["zip_validation_status"].isin([
                "zcta_mismatch",
                "boundary_review_required"
            ]).sum()
        )

        unverified_zip_count = int(
            group["zip_validation_status"].eq(
                "zcta_unavailable"
            ).sum()
        )

        displayed_area_count = int(
            group["kmeans_display_allowed"]
            .eq(True).sum()
        )

        # Avoid asserting that repeated input
        # records represent distinct defects.
        repeated_location_review = (
            incoming_count > location_count
        )

        evidence = (
            f"{incoming_count} incoming complaint "
            f"records in category {category} were "
            f"associated with historical hotspot "
            f"{cluster_id}, across "
            f"{location_count} distinct submitted "
            f"coordinate(s). The hotspot contains "
            f"{historical_count} historical reports "
            f"from 2023-2025."
        )

        if historical_category_count is not None:
            evidence += (
                f" Of these historical reports, "
                f"{historical_category_count} were "
                f"in category {category}."
            )

        safeguards = []

        if repeated_location_review:
            safeguards.append(
                "Check whether repeated coordinates "
                "represent duplicate reports or "
                "different incidents."
            )

        if zip_review_count:
            safeguards.append(
                f"{zip_review_count} incoming records "
                f"require ZIP geographic review."
            )

        if unverified_zip_count:
            safeguards.append(
                f"{unverified_zip_count} incoming "
                f"records use an unavailable ZCTA "
                f"reference."
            )

        safeguards.append(
            "Historical hotspot association does "
            "not establish physical defect severity."
        )

        suggested_review = (
            "Review related complaint descriptions, "
            "maintenance history and potential "
            "duplicate submissions. Consider "
            "coordinated field investigation when "
            "supported by verified operational "
            "information."
        )

        briefs.append({
            "historical_hotspot_id": cluster_id,
            "complaint_category": category,
            "incoming_complaints": incoming_count,
            "distinct_submitted_locations":
                location_count,
            "historical_complaints_2023_2025":
                historical_count,
            "historical_category_complaints":
                historical_category_count,
            "historical_primary_borough":
                historical["primary_borough"],
            "historical_dominant_category":
                historical["dominant_category"],
            "zip_review_records":
                zip_review_count,
            "zcta_unavailable_records":
                unverified_zip_count,
            "historical_area_profiles_displayed":
                displayed_area_count,
            "featured_investigations":
                featured,
            "supporting_investigations":
                supporting,
            "limited_evidence_investigations":
                limited,
            "representative_source_row":
                representative_row,
            "repeat_location_review":
                repeated_location_review,
            "evidence_explanation": evidence,
            "review_safeguards": " ".join(
                safeguards
            ),
            "suggested_investigation":
                suggested_review,
            "operational_priority":
                "not_assessed"
        })

    brief_table = pd.DataFrame.from_records(
        briefs
    )

    if not brief_table.empty:
        brief_table = brief_table.sort_values(
            [
                "incoming_complaints",
                "historical_hotspot_id"
            ],
            ascending=[False, True]
        ).reset_index(drop=True)

    # Complaints outside established historical
    # clusters must not disappear from the queue.
    unmatched = records.loc[
        records["processing_status"].eq(
            "processed"
        )
        & ~records["dbscan_status"].eq(
            "established_cluster"
        )
    ].copy()

    unmatched_groups = (
        unmatched.groupby(
            [
                "borough",
                "complaint_category",
                "dbscan_status"
            ],
            dropna=False
        )
        .size()
        .reset_index(name="incoming_complaints")
    )

    review_counts = {
        "geographic_review": int(
            records["processing_status"].eq(
                "geographic_review_required"
            ).sum()
        ),
        "zip_review": int(
            records["zip_validation_status"]
            .isin([
                "zcta_mismatch",
                "boundary_review_required"
            ]).sum()
        ),
        "invalid_input": int(
            records["processing_status"].eq(
                "input_rejected"
            ).sum()
        ),
        "backend_error": int(
            records["processing_status"].eq(
                "backend_error"
            ).sum()
        )
    }

    summary = {
        "total_complaints": len(records),
        "established_hotspot_reports":
            len(established),
        "historical_hotspot_category_briefs":
            len(brief_table),
        "unmatched_processed_reports":
            len(unmatched),
        "review_counts": review_counts,
        "interpretation": (
            "Briefs are descriptive investigation "
            "aids, not safety rankings or confirmed "
            "repair instructions."
        )
    }

    return {
        "briefs": brief_table,
        "unmatched_groups": unmatched_groups,
        "summary": summary
    }
''').lstrip()

compile(
    module_source,
    str(module_path),
    "exec"
)

namespace = {}

exec(
    compile(
        module_source,
        str(module_path),
        "exec"
    ),
    namespace
)

build_investigation_briefs = namespace[
    "build_investigation_briefs"
]

# --------------------------------------------------
# 2. Generate investigation briefs
# --------------------------------------------------

print("\n1. BUILDING INVESTIGATION BRIEFS")

investigation_output = build_investigation_briefs(
    batch_output,
    hotspot_reference
)

briefs = investigation_output["briefs"]
brief_summary = investigation_output["summary"]

display(briefs[[
    "historical_hotspot_id",
    "complaint_category",
    "incoming_complaints",
    "distinct_submitted_locations",
    "historical_complaints_2023_2025",
    "historical_category_complaints",
    "featured_investigations",
    "zip_review_records",
    "operational_priority"
]])

# --------------------------------------------------
# 3. Display the natural-language briefs
# --------------------------------------------------

print("\n2. SMART ASSISTANT INVESTIGATION EXPLANATIONS")

for item in briefs.to_dict("records"):

    print("\n" + "-" * 70)
    print(
        "HISTORICAL HOTSPOT",
        item["historical_hotspot_id"]
    )
    print("Category:", item["complaint_category"])
    print("\nEVIDENCE:")
    print(item["evidence_explanation"])

    print("\nRECOMMENDATION AVAILABILITY:")
    print(
        "Featured:",
        item["featured_investigations"]
    )
    print(
        "Supporting:",
        item["supporting_investigations"]
    )
    print(
        "Limited Evidence:",
        item["limited_evidence_investigations"]
    )

    print("\nSAFEGUARDS:")
    print(item["review_safeguards"])

    print("\nSUGGESTED INVESTIGATION:")
    print(item["suggested_investigation"])

# --------------------------------------------------
# 4. Verify evidence and accounting
# --------------------------------------------------

checks = []

def check(name, passed):
    checks.append({
        "test": name,
        "passed": bool(passed)
    })

check(
    "Two historical hotspot investigation briefs",
    len(briefs) == 2
)

check(
    "Incoming hotspot complaints total 395",
    int(briefs["incoming_complaints"].sum())
    == 395
)

check(
    "No processed complaints lost from summaries",
    (
        brief_summary["established_hotspot_reports"]
        + brief_summary["unmatched_processed_reports"]
    ) == 395
)

by_id = briefs.set_index(
    "historical_hotspot_id"
)

check(
    "Hotspot 1421 contains 270 incoming reports",
    int(
        by_id.loc[1421, "incoming_complaints"]
    ) == 270
)

check(
    "Hotspot 1964 contains 125 incoming reports",
    int(
        by_id.loc[1964, "incoming_complaints"]
    ) == 125
)

check(
    "Hotspot 1421 historical total remains 12773",
    int(
        by_id.loc[
            1421,
            "historical_complaints_2023_2025"
        ]
    ) == 12773
)

check(
    "Hotspot 1964 historical total remains 43",
    int(
        by_id.loc[
            1964,
            "historical_complaints_2023_2025"
        ]
    ) == 43
)

check(
    "Street Condition historical evidence retained",
    int(
        by_id.loc[
            1421,
            "historical_category_complaints"
        ]
    ) == 3870
)

check(
    "Hotspot 1421 has three Featured investigations",
    int(
        by_id.loc[
            1421,
            "featured_investigations"
        ]
    ) == 3
)

check(
    "Both hotspot groups contain one submitted location",
    briefs[
        "distinct_submitted_locations"
    ].eq(1).all()
)

check(
    "Repeated coordinate review is highlighted",
    briefs[
        "repeat_location_review"
    ].eq(True).all()
)

check(
    "20 ZIP mismatch records flagged in hotspot 1421",
    int(
        by_id.loc[
            1421, "zip_review_records"
        ]
    ) == 20
)

check(
    "50 unavailable ZCTA records remain qualified",
    int(
        by_id.loc[
            1421,
            "zcta_unavailable_records"
        ]
    ) == 50
)

check(
    "No automatic repair priorities assigned",
    briefs["operational_priority"].eq(
        "not_assessed"
    ).all()
)

check(
    "Geographic review remains accounted for",
    brief_summary["review_counts"][
        "geographic_review"
    ] == 100
)

check(
    "Five invalid inputs remain accounted for",
    brief_summary["review_counts"][
        "invalid_input"
    ] == 5
)

check(
    "No backend errors remain",
    brief_summary["review_counts"][
        "backend_error"
    ] == 0
)

# --------------------------------------------------
# 5. Display results and save verified module
# --------------------------------------------------

results_17_3b = pd.DataFrame(checks)

print("\n3. TEST CHECKLIST")

display(results_17_3b)

passed = int(results_17_3b["passed"].sum())
total = len(results_17_3b)

print(f"\nChecks passed: {passed}/{total}")

failures = results_17_3b.loc[
    ~results_17_3b["passed"], "test"
].tolist()

assert not failures, (
    "Phase 17.3B failures: "
    + "; ".join(failures)
)

if module_path.exists():

    assert module_path.read_text(
        encoding="utf-8"
    ) == module_source, (
        "Existing investigation module differs. "
        "No overwrite performed."
    )

else:

    module_path.parent.mkdir(
        parents=True,
        exist_ok=True
    )

    module_path.write_text(
        module_source,
        encoding="utf-8"
    )

print("\n4. MODULE CREATED / VERIFIED")
print("Investigation summaries:", module_path)

print("\nSTATUS: PASSED")
print("INVESTIGATION SUMMARIES VERIFIED")
print("HISTORICAL EVIDENCE PRESERVED")
print("REPEATED LOCATIONS FLAGGED FOR REVIEW")
print("NO AUTOMATIC SAFETY PRIORITY ASSIGNED")
print("EXISTING MODELS UNMODIFIED")


 SMART INVESTIGATION SUMMARIES

1. BUILDING INVESTIGATION BRIEFS


,historical_hotspot_id,complaint_category,incoming_complaints,distinct_submitted_locations,historical_complaints_2023_2025,historical_category_complaints,featured_investigations,zip_review_records,operational_priority
0,1421,STREET CONDITION,270,1,12773,3870,3,20,not_assessed
1,1964,STREET CONDITION,125,1,43,16,2,0,not_assessed



2. SMART ASSISTANT INVESTIGATION EXPLANATIONS

----------------------------------------------------------------------
HISTORICAL HOTSPOT 1421
Category: STREET CONDITION

EVIDENCE:
270 incoming complaint records in category STREET CONDITION were associated with historical hotspot 1421, across 1 distinct submitted coordinate(s). The hotspot contains 12773 historical reports from 2023-2025. Of these historical reports, 3870 were in category STREET CONDITION.

RECOMMENDATION AVAILABILITY:
Featured: 3
Supporting: 3
Limited Evidence: 8

SAFEGUARDS:
Check whether repeated coordinates represent duplicate reports or different incidents. 20 incoming records require ZIP geographic review. 50 incoming records use an unavailable ZCTA reference. Historical hotspot association does not establish physical defect severity.

SUGGESTED INVESTIGATION:
Review related complaint descriptions, maintenance history and potential duplicate submissions. Consider coordinated field investigation when supported by 

,test,passed
0,Two historical hotspot investigation briefs,True
1,Incoming hotspot complaints total 395,True
2,No processed complaints lost from summaries,True
3,Hotspot 1421 contains 270 incoming reports,True
4,Hotspot 1964 contains 125 incoming reports,True
5,Hotspot 1421 historical total remains 12773,True
6,Hotspot 1964 historical total remains 43,True
7,Street Condition historical evidence retained,True
8,Hotspot 1421 has three Featured investigations,True
9,Both hotspot groups contain one submitted loca...,True



Checks passed: 17/17

4. MODULE CREATED / VERIFIED
Investigation summaries: D:\Urban-Infrastructure-Pulse-Intelligence\src\integration\bulk_investigation_summary.py

STATUS: PASSED
INVESTIGATION SUMMARIES VERIFIED
HISTORICAL EVIDENCE PRESERVED
REPEATED LOCATIONS FLAGGED FOR REVIEW
NO AUTOMATIC SAFETY PRIORITY ASSIGNED
EXISTING MODELS UNMODIFIED


In [8]:

from pathlib import Path
from shapely.geometry import shape
import json
import re
import sys
import pandas as pd

print(" DIVERSE NYC AREA REFERENCE AUDIT")
print("=" * 75)

root = Path(PROJECT_ROOT).resolve()

if str(root) not in sys.path:
    sys.path.insert(0, str(root))

zcta_path = (
    root / "models/geography"
    / "nyc_zcta_35j5-n34v_snapshot_2026_v1.geojson"
)

profile_path = (
    root / "outputs/tables/kmeans"
    / "kmeans_full_data_k3_descriptive_profiles.csv"
)

taxonomy_path = (
    root / "outputs/tables/dbscan"
    / "dbscan_action_taxonomy_v1.csv"
)

for path in [zcta_path, profile_path, taxonomy_path]:
    assert path.is_file(), f"Missing: {path}"

# --------------------------------------------------
# 1. Inspect the saved NYC reference data
# --------------------------------------------------

with open(zcta_path, encoding="utf-8") as file:
    zcta_geojson = json.load(file)

features = zcta_geojson["features"]

profiles = pd.read_csv(
    profile_path,
    dtype={"area_id": str}
)

taxonomy = pd.read_csv(taxonomy_path)

categories_17_4a = sorted({
    str(value).strip().upper()
    for value in taxonomy[
        "complaint_category"
    ].dropna()
})

print("\n1. REFERENCE INVENTORY")
print("NYC ZCTA polygons:", len(features))
print("Historical K-Means areas:", len(profiles))
print("Supported categories:", len(categories_17_4a))

print("\nZCTA property fields:")
print(list(features[0]["properties"].keys()))

print("\nSupported complaint categories:")
print(categories_17_4a)

# --------------------------------------------------
# 2. Detect the ZCTA identifier field
# --------------------------------------------------

def is_five_digit_zip(value):
    if value is None:
        return False

    return bool(
        re.fullmatch(
            r"\d{5}",
            str(value).strip()
        )
    )


property_names = list(
    features[0]["properties"].keys()
)

candidate_fields = [
    name
    for name in property_names
    if any(
        token in name.lower()
        for token in ("zcta", "zip", "postal")
    )
]

field_scores = {}

for field in candidate_fields:
    field_scores[field] = sum(
        is_five_digit_zip(
            item["properties"].get(field)
        )
        for item in features
    )

print("\n2. IDENTIFIER FIELD SCORES")
print(field_scores)

assert field_scores, (
    "No ZCTA identifier field found. "
    "Share the property field names."
)

zip_field = max(
    field_scores,
    key=field_scores.get
)

assert field_scores[zip_field] >= 150, (
    "Unable to reliably identify the ZCTA code. "
    "Share field scores before continuing."
)

print("Selected ZCTA field:", zip_field)

# --------------------------------------------------
# 3. Generate candidate points inside ZCTA areas
# --------------------------------------------------

zcta_points = {}

for item in features:

    zip_code = str(
        item["properties"].get(zip_field, "")
    ).strip()

    if not is_five_digit_zip(zip_code):
        continue

    if item.get("geometry") is None:
        continue

    polygon = shape(item["geometry"])

    if polygon.is_empty:
        continue

    # A representative point lies inside the
    # polygon, unlike a centroid in some shapes.
    point = polygon.representative_point()

    zcta_points.setdefault(
        zip_code,
        (float(point.y), float(point.x))
    )

print("\nUsable ZCTA coordinate anchors:",
      len(zcta_points))

# --------------------------------------------------
# 4. Match reference areas to historical profiles
# --------------------------------------------------

candidate_rows = []

for area_id in profiles["area_id"].dropna():

    area_id = str(area_id).strip()

    if "_" not in area_id:
        continue

    borough, zip_code = area_id.rsplit(
        "_", 1
    )

    if not is_five_digit_zip(zip_code):
        continue

    if zip_code not in zcta_points:
        continue

    latitude, longitude = zcta_points[
        zip_code
    ]

    candidate_rows.append({
        "area_id": area_id,
        "expected_borough": borough,
        "incident_zip": zip_code,
        "latitude": latitude,
        "longitude": longitude
    })

area_candidates_17_4a = (
    pd.DataFrame(candidate_rows)
    .drop_duplicates("area_id")
    .sort_values(
        ["expected_borough", "area_id"]
    )
    .reset_index(drop=True)
)

print("\n3. MATCHED AREA COUNTS")

display(
    area_candidates_17_4a.groupby(
        "expected_borough"
    ).size().rename(
        "candidate_areas"
    ).reset_index()
)

assert not area_candidates_17_4a.empty

# Select up to four different ZIP areas
# from each borough for backend verification.
sample_anchors = (
    area_candidates_17_4a.groupby(
        "expected_borough",
        group_keys=False
    )
    .head(4)
    .reset_index(drop=True)
)

print("\n4. CANDIDATE NYC TEST ANCHORS")

display(sample_anchors)

# --------------------------------------------------
# 5. Verify each selected area using existing backend
# --------------------------------------------------

from src.ui.service_factory import get_gui_adapter

adapter = get_gui_adapter()

verification_rows = []

for row in sample_anchors.to_dict("records"):

    result = {
        **row,
        "actual_borough": None,
        "geography_status": None,
        "zcta_status": None,
        "kmeans_display_allowed": False,
        "dbscan_status": None,
        "verified": False,
        "error": None
    }

    try:

        response = adapter.evaluate(
            row["latitude"],
            row["longitude"],
            "STREET CONDITION",
            row["incident_zip"]
        )

        geography = response.get(
            "geography"
        ) or {}

        zip_info = response.get(
            "zip_validation"
        ) or {}

        kmeans = response.get(
            "kmeans"
        ) or {}

        dbscan = response.get(
            "dbscan"
        ) or {}

        result["actual_borough"] = (
            geography.get("borough")
        )

        result["geography_status"] = (
            geography.get("status")
        )

        result["zcta_status"] = (
            zip_info.get("status")
        )

        result["kmeans_display_allowed"] = (
            kmeans.get("display_allowed")
            is True
        )

        result["dbscan_status"] = (
            dbscan.get("status")
        )

        result["verified"] = (
            result["actual_borough"]
            == row["expected_borough"]
            and result["zcta_status"]
            == "zcta_consistent"
            and result["kmeans_display_allowed"]
            and kmeans.get("area_id")
            == row["area_id"]
        )

    except Exception as exc:

        result["error"] = (
            type(exc).__name__
            + ": "
            + str(exc)[:150]
        )

    verification_rows.append(result)

anchor_audit_17_4a = pd.DataFrame(
    verification_rows
)

verified_area_anchors_17_4a = (
    anchor_audit_17_4a.loc[
        anchor_audit_17_4a["verified"]
    ].copy().reset_index(drop=True)
)

print("\n5. BACKEND VERIFICATION RESULTS")

display(anchor_audit_17_4a[[
    "area_id",
    "expected_borough",
    "actual_borough",
    "zcta_status",
    "kmeans_display_allowed",
    "dbscan_status",
    "verified",
    "error"
]])

print("\n6. VERIFIED AREA COVERAGE")

verified_by_borough = (
    verified_area_anchors_17_4a.groupby(
        "expected_borough"
    ).size().rename(
        "verified_areas"
    ).reset_index()
)

display(verified_by_borough)

# --------------------------------------------------
# 6. Summarize readiness for a diverse test batch
# --------------------------------------------------

verified_areas = len(
    verified_area_anchors_17_4a
)

verified_boroughs = (
    verified_area_anchors_17_4a[
        "expected_borough"
    ].nunique()
)

print("\n7. READINESS SUMMARY")
print("Historical K-Means areas:", len(profiles))
print("Geographic area candidates:",
      len(area_candidates_17_4a))
print("Anchors checked:", len(sample_anchors))
print("Verified anchors:", verified_areas)
print("Boroughs represented:", verified_boroughs)
print("Available categories:", len(categories_17_4a))

if (
    verified_areas >= 10
    and verified_boroughs >= 3
    and len(categories_17_4a) >= 3
):
    print("\nSTATUS: PASSED")
    print("DIVERSE SYNTHETIC BATCH ANCHORS READY")
    print("HISTORICAL K-MEANS COMPATIBILITY VERIFIED")
    print("NO TRAINING OR MODEL MODIFICATIONS")
else:
    print("\nSTATUS: NEEDS REVIEW")
    print(
        "Additional valid area anchors may be "
        "needed before generating 500 records."
    )
    print(
        "Share the verification table; "
        "do not assume unverified locations "
        "are suitable for training."
    )


 DIVERSE NYC AREA REFERENCE AUDIT

1. REFERENCE INVENTORY
NYC ZCTA polygons: 221
Historical K-Means areas: 238
Supported categories: 15

ZCTA property fields:
[':id', ':version', ':created_at', ':updated_at', 'zcta5', 'arealand', 'areawater', 'centlat', 'centlon', 'intptlat', 'intptlon']

Supported complaint categories:
['BRIDGE CONDITION', 'CURB CONDITION', 'ROOT/SEWER/SIDEWALK CONDITION', 'SEWER', 'SEWER MAINTENANCE', 'SIDEWALK CONDITION', 'STREET CONDITION', 'STREET LIGHT CONDITION', 'STREET SIGN - DAMAGED', 'STREET SIGN - DANGLING', 'STREET SIGN - MISSING', 'TRAFFIC SIGNAL CONDITION', 'WATER DRAINAGE', 'WATER LEAK', 'WATER MAINTENANCE']

2. IDENTIFIER FIELD SCORES
{'zcta5': 221}
Selected ZCTA field: zcta5

Usable ZCTA coordinate anchors: 221

3. MATCHED AREA COUNTS


,expected_borough,candidate_areas
0,BRONX,27
1,BROOKLYN,40
2,MANHATTAN,68
3,QUEENS,67
4,STATEN ISLAND,12



4. CANDIDATE NYC TEST ANCHORS


,area_id,expected_borough,incident_zip,latitude,longitude
0,BRONX_10451,BRONX,10451,40.819421,-73.925290
1,BRONX_10452,BRONX,10452,40.836673,-73.924004
2,BRONX_10453,BRONX,10453,40.852804,-73.912887
3,BRONX_10454,BRONX,10454,40.805335,-73.915894
4,BROOKLYN_11201,BROOKLYN,11201,40.694535,-73.990317
5,BROOKLYN_11203,BROOKLYN,11203,40.649354,-73.933746
6,BROOKLYN_11204,BROOKLYN,11204,40.621153,-73.986236
7,BROOKLYN_11205,BROOKLYN,11205,40.698394,-73.974117
8,MANHATTAN_10001,MANHATTAN,10001,40.751610,-74.004707
9,MANHATTAN_10002,MANHATTAN,10002,40.716824,-73.985563



5. BACKEND VERIFICATION RESULTS


,area_id,expected_borough,actual_borough,zcta_status,kmeans_display_allowed,dbscan_status,verified,error
0,BRONX_10451,BRONX,BRONX,zcta_consistent,True,outside_existing_clusters,True,None
1,BRONX_10452,BRONX,BRONX,zcta_consistent,True,outside_existing_clusters,True,None
2,BRONX_10453,BRONX,BRONX,zcta_consistent,True,outside_existing_clusters,True,None
3,BRONX_10454,BRONX,BRONX,zcta_consistent,True,outside_existing_clusters,True,None
4,BROOKLYN_11201,BROOKLYN,BROOKLYN,zcta_consistent,True,established_cluster,True,None
5,BROOKLYN_11203,BROOKLYN,BROOKLYN,zcta_consistent,True,outside_existing_clusters,True,None
6,BROOKLYN_11204,BROOKLYN,BROOKLYN,zcta_consistent,True,established_cluster,True,None
7,BROOKLYN_11205,BROOKLYN,BROOKLYN,zcta_consistent,True,outside_existing_clusters,True,None
8,MANHATTAN_10001,MANHATTAN,MANHATTAN,zcta_consistent,True,established_cluster,True,None
9,MANHATTAN_10002,MANHATTAN,MANHATTAN,zcta_consistent,True,established_cluster,True,None



6. VERIFIED AREA COVERAGE


,expected_borough,verified_areas
0,BRONX,4
1,BROOKLYN,4
2,MANHATTAN,4
3,QUEENS,4
4,STATEN ISLAND,4



7. READINESS SUMMARY
Historical K-Means areas: 238
Geographic area candidates: 214
Anchors checked: 20
Verified anchors: 20
Boroughs represented: 5
Available categories: 15

STATUS: PASSED
DIVERSE SYNTHETIC BATCH ANCHORS READY
HISTORICAL K-MEANS COMPATIBILITY VERIFIED
NO TRAINING OR MODEL MODIFICATIONS


In [9]:

from pathlib import Path
from shapely.geometry import shape, Point
from shapely.ops import transform as transform_geometry
from pyproj import Transformer
import numpy as np
import pandas as pd
import json
import sys

from src.integration.bulk_complaint_contract import (
    validate_batch,
    read_batch_csv
)

print(" DIVERSE SYNTHETIC COMPLAINT DATA")
print("=" * 72)

root = Path(PROJECT_ROOT).resolve()

output_dir = (
    root / "outputs/tests/phase_17_4b_diverse"
)

csv_path = (
    output_dir / "synthetic_500_diverse_complaints.csv"
)

reference_path = (
    output_dir / "synthetic_area_pattern_reference.csv"
)

assert "verified_area_anchors_17_4a" in globals(), (
    "Run Phase 17.4A first."
)

anchors = verified_area_anchors_17_4a.copy()

assert len(anchors) == 20
assert anchors["expected_borough"].nunique() == 5
assert anchors["verified"].all()

# --------------------------------------------------
# 1. Define deliberately different area patterns
# --------------------------------------------------

# Exactly 25 complaints per area.
# First two patterns are intentionally similar.

patterns = {
    "road_a": {
        "STREET CONDITION": 14,
        "SIDEWALK CONDITION": 4,
        "SEWER": 3,
        "STREET LIGHT CONDITION": 2,
        "TRAFFIC SIGNAL CONDITION": 2
    },
    "road_b": {
        "STREET CONDITION": 13,
        "SIDEWALK CONDITION": 5,
        "SEWER": 3,
        "STREET LIGHT CONDITION": 2,
        "TRAFFIC SIGNAL CONDITION": 2
    },
    "sewer": {
        "SEWER": 15,
        "SEWER MAINTENANCE": 5,
        "WATER DRAINAGE": 2,
        "STREET CONDITION": 2,
        "SIDEWALK CONDITION": 1
    },
    "signals": {
        "STREET LIGHT CONDITION": 12,
        "TRAFFIC SIGNAL CONDITION": 7,
        "STREET SIGN - DAMAGED": 2,
        "STREET CONDITION": 2,
        "SEWER": 2
    }
}

assert all(
    sum(counts.values()) == 25
    for counts in patterns.values()
)

allowed_categories = set(categories_17_4a)

assert all(
    category in allowed_categories
    for counts in patterns.values()
    for category in counts
)

print("\n1. SYNTHETIC PATTERN DESIGN")

display(pd.DataFrame([
    {
        "pattern": name,
        "complaints_per_area": sum(counts.values()),
        "street_condition_pct":
            100 * counts.get("STREET CONDITION", 0) / 25,
        "sewer_pct":
            100 * counts.get("SEWER", 0) / 25,
        "street_light_pct":
            100 * counts.get("STREET LIGHT CONDITION", 0) / 25
    }
    for name, counts in patterns.items()
]))

# --------------------------------------------------
# 2. Load actual NYC ZCTA polygons
# --------------------------------------------------

zcta_path = (
    root / "models/geography"
    / "nyc_zcta_35j5-n34v_snapshot_2026_v1.geojson"
)

with zcta_path.open(encoding="utf-8") as file:
    zcta_data = json.load(file)

polygon_by_zip = {
    str(feature["properties"]["zcta5"]): shape(
        feature["geometry"]
    )
    for feature in zcta_data["features"]
    if feature.get("geometry") is not None
}

to_utm = Transformer.from_crs(
    "EPSG:4326",
    "EPSG:32618",
    always_xy=True
)

to_wgs84 = Transformer.from_crs(
    "EPSG:32618",
    "EPSG:4326",
    always_xy=True
)

rng = np.random.default_rng(1742026)

# --------------------------------------------------
# 3. Generate distinct coordinates within each area
# --------------------------------------------------

def generate_area_points(anchor, count=25):
    zip_code = str(anchor["incident_zip"])

    polygon = polygon_by_zip[zip_code]

    polygon_utm = transform_geometry(
        to_utm.transform,
        polygon
    )

    anchor_x, anchor_y = to_utm.transform(
        float(anchor["longitude"]),
        float(anchor["latitude"])
    )

    # Keep generated points inside the ZCTA,
    # away from its boundary, and reasonably
    # close to the previously verified anchor.
    safe_area = polygon_utm.buffer(-12).intersection(
        Point(anchor_x, anchor_y).buffer(180)
    )

    if safe_area.is_empty:
        raise ValueError(
            f"No safe sampling area for {zip_code}"
        )

    min_x, min_y, max_x, max_y = safe_area.bounds

    locations_utm = []
    locations_wgs84 = []

    for attempt in range(20000):

        x = rng.uniform(min_x, max_x)
        y = rng.uniform(min_y, max_y)

        candidate = Point(x, y)

        if not safe_area.contains(candidate):
            continue

        # Avoid producing practically identical
        # coordinate submissions.
        if any(
            candidate.distance(existing) < 8
            for existing in locations_utm
        ):
            continue

        longitude, latitude = to_wgs84.transform(
            x, y
        )

        locations_utm.append(candidate)

        locations_wgs84.append((
            round(latitude, 8),
            round(longitude, 8)
        ))

        if len(locations_wgs84) == count:
            break

    if len(locations_wgs84) != count:
        raise ValueError(
            f"Only {len(locations_wgs84)} valid "
            f"points found for {zip_code}."
        )

    return locations_wgs84


# --------------------------------------------------
# 4. Assign area patterns across five boroughs
# --------------------------------------------------

generated_records = []
area_metadata = []

pattern_order = [
    "road_a",
    "road_b",
    "sewer",
    "signals"
]

for borough, group in anchors.groupby(
    "expected_borough", sort=True
):

    group = group.sort_values(
        "area_id"
    ).reset_index(drop=True)

    assert len(group) == 4, (
        f"Expected four verified areas: {borough}"
    )

    for position, anchor in enumerate(
        group.to_dict("records")
    ):

        pattern_name = pattern_order[position]
        category_counts = patterns[pattern_name]

        locations = generate_area_points(
            anchor, count=25
        )

        category_list = []

        for category, count in category_counts.items():
            category_list.extend(
                [category] * count
            )

        assert len(category_list) == len(locations)

        # Shuffle category assignment without
        # changing the intentionally designed shares.
        rng.shuffle(category_list)

        area_metadata.append({
            "area_id": anchor["area_id"],
            "expected_borough": borough,
            "incident_zip": anchor["incident_zip"],
            "intended_pattern": pattern_name,
            "planned_complaints": 25,
            "street_condition_pct": (
                100 * category_counts.get(
                    "STREET CONDITION", 0
                ) / 25
            ),
            "sewer_pct": (
                100 * category_counts.get(
                    "SEWER", 0
                ) / 25
            )
        })

        for (latitude, longitude), category in zip(
            locations, category_list
        ):

            generated_records.append({
                "complaint_id": (
                    f"SIM-20261109-"
                    f"{len(generated_records)+1:04d}"
                ),
                "created_date": "2026-11-09",
                "latitude": latitude,
                "longitude": longitude,
                "complaint_category": category,
                "incident_zip": anchor["incident_zip"]
            })

synthetic_500_17_4b = pd.DataFrame(
    generated_records
)

area_reference_17_4b = pd.DataFrame(
    area_metadata
)

# --------------------------------------------------
# 5. Validate the generated CSV input
# --------------------------------------------------

print("\n2. VALIDATING ALL 500 COMPLAINT INPUTS")

validated_17_4b = validate_batch(
    synthetic_500_17_4b,
    allowed_categories
)

checks = []

def check(name, condition):
    checks.append({
        "test": name,
        "passed": bool(condition)
    })

check(
    "Exactly 500 synthetic records",
    len(synthetic_500_17_4b) == 500
)

check(
    "All synthetic complaint IDs are unique",
    synthetic_500_17_4b[
        "complaint_id"
    ].is_unique
)

check(
    "All 500 pass CSV input validation",
    validated_17_4b[
        "validation_status"
    ].eq(
        "ready_for_geographic_validation"
    ).all()
)

check(
    "20 geographic areas represented",
    len(area_reference_17_4b) == 20
)

check(
    "All five NYC boroughs represented",
    area_reference_17_4b[
        "expected_borough"
    ].nunique() == 5
)

check(
    "Each area receives 25 complaints",
    area_reference_17_4b[
        "planned_complaints"
    ].eq(25).all()
)

check(
    "Ten road-heavy areas",
    area_reference_17_4b[
        "intended_pattern"
    ].isin(["road_a", "road_b"]).sum() == 10
)

check(
    "Five sewer-heavy areas",
    area_reference_17_4b[
        "intended_pattern"
    ].eq("sewer").sum() == 5
)

check(
    "Five lighting/signal-heavy areas",
    area_reference_17_4b[
        "intended_pattern"
    ].eq("signals").sum() == 5
)

check(
    "All coordinates are distinct",
    not synthetic_500_17_4b[
        ["latitude", "longitude"]
    ].duplicated().any()
)

check(
    "Multiple complaint categories represented",
    synthetic_500_17_4b[
        "complaint_category"
    ].nunique() >= 7
)

check(
    "Only supported complaint categories used",
    set(
        synthetic_500_17_4b[
            "complaint_category"
        ]
    ).issubset(allowed_categories)
)

check(
    "Road variants have 56% and 52% Street Condition",
    set(
        area_reference_17_4b.loc[
            area_reference_17_4b[
                "intended_pattern"
            ].isin(["road_a", "road_b"]),
            "street_condition_pct"
        ]
    ) == {56.0, 52.0}
)

# --------------------------------------------------
# 6. Independently check ZCTA containment
# --------------------------------------------------

all_inside = True

for row in synthetic_500_17_4b.itertuples(
    index=False
):

    polygon = polygon_by_zip[
        str(row.incident_zip)
    ]

    if not polygon.contains(
        Point(float(row.longitude), float(row.latitude))
    ):
        all_inside = False
        break

check(
    "All generated points lie inside supplied ZCTAs",
    all_inside
)

# --------------------------------------------------
# 7. Display useful summaries
# --------------------------------------------------

print("\n3. COMPLAINT CATEGORY DISTRIBUTION")

display(
    synthetic_500_17_4b.groupby(
        "complaint_category"
    ).size().rename(
        "complaints"
    ).reset_index().sort_values(
        "complaints", ascending=False
    )
)

print("\n4. AREA PATTERN REFERENCE")

display(area_reference_17_4b)

print("\n5. CHECKS")

results_17_4b = pd.DataFrame(checks)

display(results_17_4b)

passed = int(results_17_4b["passed"].sum())
total = len(results_17_4b)

print(f"\nChecks passed: {passed}/{total}")

failures = results_17_4b.loc[
    ~results_17_4b["passed"], "test"
].tolist()

assert not failures, (
    "Phase 17.4B failures: "
    + "; ".join(failures)
)

# --------------------------------------------------
# 8. Save synthetic test files without overwriting
# --------------------------------------------------

csv_content = synthetic_500_17_4b.to_csv(
    index=False,
    lineterminator="\n"
)

reference_content = area_reference_17_4b.to_csv(
    index=False,
    lineterminator="\n"
)

for path, content in [
    (csv_path, csv_content),
    (reference_path, reference_content)
]:
    if path.exists():
        assert path.read_text(
            encoding="utf-8"
        ) == content, (
            f"Existing file differs: {path}. "
            "No overwrite performed."
        )

output_dir.mkdir(
    parents=True,
    exist_ok=True
)

for path, content in [
    (csv_path, csv_content),
    (reference_path, reference_content)
]:
    if not path.exists():
        path.write_text(
            content,
            encoding="utf-8"
        )

# Reopen the saved CSV using our actual CSV reader.
reopened = read_batch_csv(csv_path)

assert len(reopened) == 500

print("\n6. SAVED TEST FILES")
print("Synthetic complaints:", csv_path)
print("Area pattern reference:", reference_path)

print("\nSTATUS: PASSED")
print("DIVERSE SYNTHETIC DATA GENERATED")
print("500 INPUT RECORDS VALIDATED")
print("20 AREAS ACROSS 5 BOROUGHS REPRESENTED")
print("HISTORICAL MODEL ARTIFACTS UNMODIFIED")
print("FULL BACKEND GEOGRAPHIC TEST IS NEXT")


 DIVERSE SYNTHETIC COMPLAINT DATA

1. SYNTHETIC PATTERN DESIGN


,pattern,complaints_per_area,street_condition_pct,sewer_pct,street_light_pct
0,road_a,25,56.0,12.0,8.0
1,road_b,25,52.0,12.0,8.0
2,sewer,25,8.0,60.0,0.0
3,signals,25,8.0,8.0,48.0



2. VALIDATING ALL 500 COMPLAINT INPUTS

3. COMPLAINT CATEGORY DISTRIBUTION


C:\Users\Administrator\AppData\Local\Temp\ipykernel_21452\1493200342.py:152: DeprecationWarning: The 'shapely.ops.transform()' function is deprecated. Use 'shapely.transform()' or 'shapely.transform_coordseq()' instead. See https://shapely.readthedocs.io/en/latest/manual.html#shapely.ops.transform for more details.
  polygon_utm = transform_geometry(
C:\Users\Administrator\AppData\Local\Temp\ipykernel_21452\1493200342.py:152: DeprecationWarning: The 'shapely.ops.transform()' function is deprecated. Use 'shapely.transform()' or 'shapely.transform_coordseq()' instead. See https://shapely.readthedocs.io/en/latest/manual.html#shapely.ops.transform for more details.
  polygon_utm = transform_geometry(
C:\Users\Administrator\AppData\Local\Temp\ipykernel_21452\1493200342.py:152: DeprecationWarning: The 'shapely.ops.transform()' function is deprecated. Use 'shapely.transform()' or 'shapely.transform_coordseq()' instead. See https://shapely.readthedocs.io/en/latest/manual.html#shapely.ops.trans

,complaint_category,complaints
3,STREET CONDITION,155
0,SEWER,115
4,STREET LIGHT CONDITION,80
6,TRAFFIC SIGNAL CONDITION,55
2,SIDEWALK CONDITION,50
1,SEWER MAINTENANCE,25
5,STREET SIGN - DAMAGED,10
7,WATER DRAINAGE,10



4. AREA PATTERN REFERENCE


,area_id,expected_borough,incident_zip,intended_pattern,planned_complaints,street_condition_pct,sewer_pct
0,BRONX_10451,BRONX,10451,road_a,25,56.0,12.0
1,BRONX_10452,BRONX,10452,road_b,25,52.0,12.0
2,BRONX_10453,BRONX,10453,sewer,25,8.0,60.0
3,BRONX_10454,BRONX,10454,signals,25,8.0,8.0
4,BROOKLYN_11201,BROOKLYN,11201,road_a,25,56.0,12.0
5,BROOKLYN_11203,BROOKLYN,11203,road_b,25,52.0,12.0
6,BROOKLYN_11204,BROOKLYN,11204,sewer,25,8.0,60.0
7,BROOKLYN_11205,BROOKLYN,11205,signals,25,8.0,8.0
8,MANHATTAN_10001,MANHATTAN,10001,road_a,25,56.0,12.0
9,MANHATTAN_10002,MANHATTAN,10002,road_b,25,52.0,12.0



5. CHECKS


,test,passed
0,Exactly 500 synthetic records,True
1,All synthetic complaint IDs are unique,True
2,All 500 pass CSV input validation,True
3,20 geographic areas represented,True
4,All five NYC boroughs represented,True
5,Each area receives 25 complaints,True
6,Ten road-heavy areas,True
7,Five sewer-heavy areas,True
8,Five lighting/signal-heavy areas,True
9,All coordinates are distinct,True



Checks passed: 14/14

6. SAVED TEST FILES
Synthetic complaints: D:\Urban-Infrastructure-Pulse-Intelligence\outputs\tests\phase_17_4b_diverse\synthetic_500_diverse_complaints.csv
Area pattern reference: D:\Urban-Infrastructure-Pulse-Intelligence\outputs\tests\phase_17_4b_diverse\synthetic_area_pattern_reference.csv

STATUS: PASSED
DIVERSE SYNTHETIC DATA GENERATED
500 INPUT RECORDS VALIDATED
20 AREAS ACROSS 5 BOROUGHS REPRESENTED
HISTORICAL MODEL ARTIFACTS UNMODIFIED
FULL BACKEND GEOGRAPHIC TEST IS NEXT


In [10]:

from pathlib import Path
from time import perf_counter
import sys
import json
import pandas as pd

from src.integration.bulk_complaint_contract import (
    read_batch_csv,
    validate_batch
)
from src.integration.bulk_complaint_processor import (
    process_complaint_batch
)
from src.integration.bulk_complaint_export import (
    csv_bytes
)
from src.ui.service_factory import get_gui_adapter

print("FULL DIVERSE BATCH VERIFICATION")
print("=" * 72)

root = Path(PROJECT_ROOT).resolve()

input_dir = (
    root / "outputs/tests/phase_17_4b_diverse"
)

csv_path = (
    input_dir / "synthetic_500_diverse_complaints.csv"
)

reference_path = (
    input_dir / "synthetic_area_pattern_reference.csv"
)

taxonomy_path = (
    root / "outputs/tables/dbscan"
    / "dbscan_action_taxonomy_v1.csv"
)

for path in [csv_path, reference_path, taxonomy_path]:
    assert path.is_file(), f"Missing file: {path}"

# --------------------------------------------------
# 1. Load the saved synthetic test data
# --------------------------------------------------

incoming = read_batch_csv(csv_path)

area_reference = pd.read_csv(
    reference_path,
    dtype="string"
)

taxonomy = pd.read_csv(taxonomy_path)

allowed_categories = set(
    taxonomy["complaint_category"]
    .dropna()
    .astype(str)
    .str.strip()
    .str.upper()
)

assert len(incoming) == 500
assert len(area_reference) == 20
assert area_reference["incident_zip"].is_unique
assert incoming["complaint_id"].is_unique

print("\n1. INPUT INVENTORY")
print("Incoming complaints:", len(incoming))
print("Expected areas:", len(area_reference))
print(
    "Distinct locations:",
    len(incoming[[
        "latitude", "longitude"
    ]].drop_duplicates())
)
print(
    "Complaint categories:",
    incoming["complaint_category"].nunique()
)

# --------------------------------------------------
# 2. Confirm input validation before processing
# --------------------------------------------------

input_validation = validate_batch(
    incoming,
    allowed_categories
)

assert input_validation["validation_status"].eq(
    "ready_for_geographic_validation"
).all(), (
    "Synthetic CSV no longer passes input validation."
)

# --------------------------------------------------
# 3. Run all 500 through the historical backend
# --------------------------------------------------

print("\n2. RUNNING FULL BACKEND EVALUATION")

adapter = get_gui_adapter()

start = perf_counter()

diverse_batch_output_17_4c = (
    process_complaint_batch(
        incoming,
        allowed_categories,
        adapter,
        use_cache=True
    )
)

elapsed_seconds = perf_counter() - start

diverse_results_17_4c = (
    diverse_batch_output_17_4c["records"]
)

stats = diverse_batch_output_17_4c["stats"]

print("\nBACKEND RESULTS")

for key, value in stats.items():
    print(f"{key}: {value}")

print(
    "Observed processing duration (seconds):",
    round(elapsed_seconds, 2)
)

# --------------------------------------------------
# 4. Add the expected geographic area reference
# --------------------------------------------------

expected_lookup = area_reference.set_index(
    "incident_zip"
)

audit_17_4c = diverse_results_17_4c.copy()

audit_17_4c["expected_area_id"] = (
    audit_17_4c["incident_zip"].map(
        expected_lookup["area_id"]
    )
)

audit_17_4c["expected_borough"] = (
    audit_17_4c["incident_zip"].map(
        expected_lookup["expected_borough"]
    )
)

audit_17_4c["intended_pattern"] = (
    audit_17_4c["incident_zip"].map(
        expected_lookup["intended_pattern"]
    )
)

# A record is fully verified only when every
# important geographic/K-Means check succeeds.
audit_17_4c["fully_verified"] = (
    audit_17_4c["processing_status"].eq(
        "processed"
    )
    & audit_17_4c["borough"].eq(
        audit_17_4c["expected_borough"]
    )
    & audit_17_4c["zip_validation_status"].eq(
        "zcta_consistent"
    )
    & audit_17_4c[
        "kmeans_display_allowed"
    ].eq(True)
    & audit_17_4c["kmeans_area_id"].eq(
        audit_17_4c["expected_area_id"]
    )
)

# --------------------------------------------------
# 5. Verify every submitted record was preserved
# --------------------------------------------------

checks = []

def check(name, condition):
    checks.append({
        "test": name,
        "passed": bool(condition)
    })

check(
    "500 complaint outcomes returned",
    len(audit_17_4c) == 500
)

check(
    "All original complaint IDs preserved",
    audit_17_4c["complaint_id"].tolist()
    == incoming["complaint_id"].astype(str).tolist()
)

check(
    "Source rows remain unique",
    audit_17_4c["source_row"].is_unique
)

check(
    "No input validation failures",
    stats["input_rejected"] == 0
)

check(
    "No backend processing errors",
    stats["backend_errors"] == 0
)

check(
    "All 500 records have processing outcomes",
    audit_17_4c[
        "processing_status"
    ].notna().all()
)

check(
    "500 unique backend evaluations performed",
    stats["backend_evaluations"] == 500
)

check(
    "500 backend responses retained",
    len(
        diverse_batch_output_17_4c[
            "responses_by_source_row"
        ]
    ) == 500
)

check(
    "Every submitted ZIP has an area reference",
    audit_17_4c[
        "expected_area_id"
    ].notna().all()
)

# --------------------------------------------------
# 6. Verify complete geographic compatibility
# --------------------------------------------------

check(
    "All 500 complaints processed",
    stats["processed"] == 500
)

check(
    "All 500 boroughs match expected boroughs",
    audit_17_4c["borough"].eq(
        audit_17_4c["expected_borough"]
    ).all()
)

check(
    "All 500 ZIPs are ZCTA consistent",
    audit_17_4c[
        "zip_validation_status"
    ].eq("zcta_consistent").all()
)

check(
    "All 500 historical K-Means lookups displayed",
    audit_17_4c[
        "kmeans_display_allowed"
    ].eq(True).all()
)

check(
    "All 500 K-Means area IDs are correct",
    audit_17_4c["kmeans_area_id"].eq(
        audit_17_4c["expected_area_id"]
    ).all()
)

check(
    "All 500 geographic checks fully verified",
    audit_17_4c[
        "fully_verified"
    ].all()
)

check(
    "All 20 areas successfully represented",
    audit_17_4c.loc[
        audit_17_4c["fully_verified"],
        "expected_area_id"
    ].nunique() == 20
)

check(
    "Five boroughs successfully represented",
    audit_17_4c.loc[
        audit_17_4c["fully_verified"],
        "borough"
    ].nunique() == 5
)

# --------------------------------------------------
# 7. Verify category distributions were preserved
# --------------------------------------------------

expected_distribution = (
    incoming.groupby([
        "incident_zip",
        "complaint_category"
    ])
    .size()
    .rename("expected_count")
)

actual_distribution = (
    audit_17_4c.loc[
        audit_17_4c["fully_verified"]
    ]
    .groupby([
        "incident_zip",
        "complaint_category"
    ])
    .size()
    .rename("verified_count")
)

distribution_comparison = (
    pd.concat([
        expected_distribution,
        actual_distribution
    ], axis=1)
    .fillna(0)
    .astype(int)
)

check(
    "All area-category counts preserved",
    distribution_comparison[
        "expected_count"
    ].eq(
        distribution_comparison[
            "verified_count"
        ]
    ).all()
)

# --------------------------------------------------
# 8. Summarize DBSCAN results without assumptions
# --------------------------------------------------

print("\n3. HISTORICAL DBSCAN ASSOCIATIONS")

display(
    audit_17_4c.groupby(
        "dbscan_status",
        dropna=False
    )
    .size()
    .rename("complaints")
    .reset_index()
)

check(
    "Every processed record has a DBSCAN status",
    audit_17_4c.loc[
        audit_17_4c[
            "processing_status"
        ].eq("processed"),
        "dbscan_status"
    ].notna().all()
)

# --------------------------------------------------
# 9. Verify each area's coverage
# --------------------------------------------------

area_coverage_17_4c = (
    audit_17_4c.groupby([
        "expected_area_id",
        "expected_borough",
        "intended_pattern"
    ], dropna=False)
    .agg(
        input_complaints=(
            "source_row", "size"
        ),
        fully_verified=(
            "fully_verified", "sum"
        ),
        historical_hotspot_matches=(
            "dbscan_cluster_id",
            "count"
        )
    )
    .reset_index()
    .sort_values([
        "expected_borough",
        "expected_area_id"
    ])
    .reset_index(drop=True)
)

check(
    "Every intended area contains 25 inputs",
    len(area_coverage_17_4c) == 20
    and area_coverage_17_4c[
        "input_complaints"
    ].eq(25).all()
)

check(
    "Every intended area has 25 verified outputs",
    len(area_coverage_17_4c) == 20
    and area_coverage_17_4c[
        "fully_verified"
    ].eq(25).all()
)

print("\n4. VERIFIED AREA COVERAGE")

display(area_coverage_17_4c)

# --------------------------------------------------
# 10. Show errors and unexpected geography clearly
# --------------------------------------------------

problems = audit_17_4c.loc[
    ~audit_17_4c["fully_verified"]
].copy()

print("\n5. GEOGRAPHIC VERIFICATION ISSUES")
print("Records requiring investigation:", len(problems))

if not problems.empty:
    display(
        problems[[
            "complaint_id",
            "expected_area_id",
            "expected_borough",
            "borough",
            "processing_status",
            "geography_status",
            "zip_validation_status",
            "dbscan_status",
            "kmeans_area_id",
            "processing_error"
        ]].head(20)
    )
else:
    print("No mismatches detected.")

# --------------------------------------------------
# 11. Final test checklist
# --------------------------------------------------

results_17_4c = pd.DataFrame(checks)

print("\n6. TEST RESULTS")

display(results_17_4c)

passed = int(results_17_4c["passed"].sum())
total = len(results_17_4c)

print(f"\nChecks passed: {passed}/{total}")

failed_tests = results_17_4c.loc[
    ~results_17_4c["passed"],
    "test"
].tolist()

# --------------------------------------------------
# 12. Save only if geographic verification passes
# --------------------------------------------------

if failed_tests:

    print("\nSTATUS: NEEDS REVIEW")
    print("Failed checks:")

    for name in failed_tests:
        print(" -", name)

    print(
        "\nNo verified output files saved. "
        "Review the mismatched records before "
        "training a new-batch clustering model."
    )

else:

    output_dir = (
        root / "outputs/tests"
        / "phase_17_4c_backend_verified"
    )

    result_path = (
        output_dir
        / "verified_500_backend_results.csv"
    )

    coverage_path = (
        output_dir / "verified_area_coverage.csv"
    )

    summary_path = (
        output_dir / "verification_summary.json"
    )

    summary = {
        "schema_version":
            "phase_17_4c_verification_v1",
        "dataset": "synthetic_500_diverse",
        "input_records": 500,
        "verified_records": int(
            audit_17_4c["fully_verified"].sum()
        ),
        "verified_areas": 20,
        "boroughs": 5,
        "backend_evaluations": int(
            stats["backend_evaluations"]
        ),
        "historical_hotspot_associations": int(
            audit_17_4c[
                "dbscan_cluster_id"
            ].notna().sum()
        ),
        "tests_passed": passed,
        "tests_total": total,
        "historical_reference_period":
            "2023-2025",
        "note": (
            "Synthetic data only. Historical model "
            "lookups are unchanged. No new clustering "
            "model has been trained."
        )
    }

    content_map = {
        result_path: csv_bytes(audit_17_4c),
        coverage_path: csv_bytes(
            area_coverage_17_4c
        ),
        summary_path: json.dumps(
            summary,
            indent=2
        ).encode("utf-8")
    }

    # Protect existing output files.
    for path, content in content_map.items():
        if path.exists():
            assert path.read_bytes() == content, (
                f"Existing file differs: {path}. "
                "No overwrite performed."
            )

    output_dir.mkdir(
        parents=True,
        exist_ok=True
    )

    for path, content in content_map.items():
        if not path.exists():
            path.write_bytes(content)

    print("\n7. SAVED VERIFIED TEST RESULTS")
    print("Results:", result_path)
    print("Area coverage:", coverage_path)
    print("Summary:", summary_path)

    print("\nSTATUS: PASSED")
    print("FULL BACKEND VERIFIED")
    print("500 UNIQUE COMPLAINT INPUTS EVALUATED")
    print("20 GEOGRAPHIC AREAS VERIFIED")
    print("READY FOR NEW-BATCH K-MEANS")
    print("EXISTING HISTORICAL MODELS UNMODIFIED")


FULL DIVERSE BATCH VERIFICATION

1. INPUT INVENTORY
Incoming complaints: 500
Expected areas: 20
Distinct locations: 500
Complaint categories: 8

2. RUNNING FULL BACKEND EVALUATION

BACKEND RESULTS
total: 500
processed: 500
geographic_review: 0
input_rejected: 0
backend_errors: 0
backend_evaluations: 500
cache_entries: 500
Observed processing duration (seconds): 2.8

3. HISTORICAL DBSCAN ASSOCIATIONS


,dbscan_status,complaints
0,ambiguous_association,9
1,established_cluster,137
2,outside_existing_clusters,354



4. VERIFIED AREA COVERAGE


,expected_area_id,expected_borough,intended_pattern,input_complaints,fully_verified,historical_hotspot_matches
0,BRONX_10451,BRONX,road_a,25,25,6
1,BRONX_10452,BRONX,road_b,25,25,7
2,BRONX_10453,BRONX,sewer,25,25,6
3,BRONX_10454,BRONX,signals,25,25,4
4,BROOKLYN_11201,BROOKLYN,road_a,25,25,21
5,BROOKLYN_11203,BROOKLYN,road_b,25,25,0
6,BROOKLYN_11204,BROOKLYN,sewer,25,25,12
7,BROOKLYN_11205,BROOKLYN,signals,25,25,1
8,MANHATTAN_10001,MANHATTAN,road_a,25,25,5
9,MANHATTAN_10002,MANHATTAN,road_b,25,25,23



5. GEOGRAPHIC VERIFICATION ISSUES
Records requiring investigation: 0
No mismatches detected.

6. TEST RESULTS


,test,passed
0,500 complaint outcomes returned,True
1,All original complaint IDs preserved,True
2,Source rows remain unique,True
3,No input validation failures,True
4,No backend processing errors,True
5,All 500 records have processing outcomes,True
6,500 unique backend evaluations performed,True
7,500 backend responses retained,True
8,Every submitted ZIP has an area reference,True
9,All 500 complaints processed,True



Checks passed: 21/21

7. SAVED VERIFIED TEST RESULTS
Results: D:\Urban-Infrastructure-Pulse-Intelligence\outputs\tests\phase_17_4c_backend_verified\verified_500_backend_results.csv
Area coverage: D:\Urban-Infrastructure-Pulse-Intelligence\outputs\tests\phase_17_4c_backend_verified\verified_area_coverage.csv
Summary: D:\Urban-Infrastructure-Pulse-Intelligence\outputs\tests\phase_17_4c_backend_verified\verification_summary.json

STATUS: PASSED
FULL BACKEND VERIFIED
500 UNIQUE COMPLAINT INPUTS EVALUATED
20 GEOGRAPHIC AREAS VERIFIED
READY FOR NEW-BATCH K-MEANS
EXISTING HISTORICAL MODELS UNMODIFIED


### New-Batch K-Means: Find Similar Areas
We can now start implementing the K-Means functionality shown in your PDF.

We'll keep two separate K-Means analyses:
Historical K-Means	New-batch K-Means
Existing 2023–2025 model	New model fitted to incoming area-level complaints
Describes historical reporting behavior	Groups areas with similar complaint-category percentages
Existing 238 area profiles	20 geographic areas in this experiment
Existing model unchanged	New model, separately tested

In [11]:

from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.metrics import (
    adjusted_rand_score,
    silhouette_score
)

print("NEW-BATCH K-MEANS AREA SIMILARITY")
print("=" * 75)

root = Path(PROJECT_ROOT).resolve()

results_path = (
    root / "outputs/tests"
    / "phase_17_4c_backend_verified"
    / "verified_500_backend_results.csv"
)

taxonomy_path = (
    root / "outputs/tables/dbscan"
    / "dbscan_action_taxonomy_v1.csv"
)

reference_path = (
    root / "outputs/tests"
    / "phase_17_4b_diverse"
    / "synthetic_area_pattern_reference.csv"
)

for path in [
    results_path,
    taxonomy_path,
    reference_path
]:
    assert path.is_file(), f"Missing: {path}"

# --------------------------------------------------
# 1. Load backend-verified new complaints
# --------------------------------------------------

records = pd.read_csv(
    results_path,
    dtype="string",
    keep_default_na=False
)

taxonomy = pd.read_csv(taxonomy_path)

category_names = sorted({
    str(value).strip().upper()
    for value in taxonomy[
        "complaint_category"
    ].dropna()
})

required = {
    "complaint_id",
    "borough",
    "incident_zip",
    "complaint_category",
    "processing_status",
    "zip_validation_status",
    "fully_verified",
    "kmeans_area_id",
    "kmeans_cluster_id",
    "kmeans_reporting_profile",
    "kmeans_display_allowed"
}

assert required.issubset(records.columns)

assert len(records) == 500
assert records["complaint_id"].is_unique
assert len(category_names) == 15

# --------------------------------------------------
# 2. Select geographically verified complaints
# --------------------------------------------------

verified_flag = (
    records["fully_verified"]
    .str.lower()
    .eq("true")
)

eligible = records.loc[
    verified_flag
    & records["processing_status"].eq(
        "processed"
    )
    & records["zip_validation_status"].eq(
        "zcta_consistent"
    )
].copy()

# New-batch geographic groups are derived from
# validated borough and ZIP data, not from the
# old K-Means cluster IDs.
eligible["area_id"] = (
    eligible["borough"].str.strip()
    + "_"
    + eligible["incident_zip"].str.strip()
)

assert len(eligible) == 500

assert eligible[
    "complaint_category"
].isin(category_names).all()

print("\n1. ELIGIBLE COMPLAINTS")
print("Geographically verified:", len(eligible))
print("Unique areas:", eligible["area_id"].nunique())
print("Supported categories:", len(category_names))

# --------------------------------------------------
# 3. Build new area-category count matrix
# --------------------------------------------------

area_category_counts_18_1a = pd.crosstab(
    eligible["area_id"],
    eligible["complaint_category"]
).reindex(
    columns=category_names,
    fill_value=0
)

area_totals = area_category_counts_18_1a.sum(
    axis=1
)

# Minimum sample size for this experiment.
# This threshold is a screening safeguard,
# not a guarantee of statistical reliability.
MIN_COMPLAINTS_PER_AREA = 20

eligible_areas = area_totals.loc[
    area_totals >= MIN_COMPLAINTS_PER_AREA
].index

excluded_areas = area_totals.loc[
    area_totals < MIN_COMPLAINTS_PER_AREA
]

feature_matrix_18_1a = (
    area_category_counts_18_1a
    .loc[eligible_areas]
    .div(
        area_totals.loc[eligible_areas],
        axis=0
    )
    .mul(100)
)

assert not feature_matrix_18_1a.isna().any().any()

print("\n2. AREA-CATEGORY FEATURES")
print("Areas eligible for clustering:", len(eligible_areas))
print("Areas below threshold:", len(excluded_areas))
print("Feature matrix shape:", feature_matrix_18_1a.shape)

display(
    feature_matrix_18_1a.loc[:, [
        "STREET CONDITION",
        "SIDEWALK CONDITION",
        "SEWER",
        "SEWER MAINTENANCE",
        "STREET LIGHT CONDITION",
        "TRAFFIC SIGNAL CONDITION"
    ]].round(1)
)

# --------------------------------------------------
# 4. Train separate new-batch K-Means
# --------------------------------------------------

# K=3 is chosen for the PDF-style experiment.
# It is NOT claimed to be optimal for real data.
NEW_BATCH_K = 3

new_batch_kmeans_18_1a = KMeans(
    n_clusters=NEW_BATCH_K,
    random_state=18,
    n_init=20
)

new_labels = new_batch_kmeans_18_1a.fit_predict(
    feature_matrix_18_1a
)

cluster_centers_18_1a = pd.DataFrame(
    new_batch_kmeans_18_1a.cluster_centers_,
    columns=category_names
)

cluster_centers_18_1a.index.name = (
    "new_batch_cluster"
)

# Describe clusters from their learned
# dominant categories, not the numeric IDs.
dominant_categories = (
    cluster_centers_18_1a.idxmax(axis=1)
)

friendly_names = {
    "STREET CONDITION":
        "Road-heavy reporting",
    "SEWER":
        "Sewer-heavy reporting",
    "STREET LIGHT CONDITION":
        "Lighting/signal-heavy reporting"
}

cluster_descriptions = dominant_categories.map(
    lambda category:
        friendly_names.get(
            category,
            f"{category} dominant reporting"
        )
)

print("\n3. NEW-BATCH CLUSTER PROFILES")

cluster_profiles_18_1a = pd.DataFrame({
    "new_batch_cluster":
        cluster_centers_18_1a.index,
    "description":
        cluster_descriptions.values,
    "dominant_category":
        dominant_categories.values,
    "street_condition_pct":
        cluster_centers_18_1a[
            "STREET CONDITION"
        ].round(2).values,
    "sewer_pct":
        cluster_centers_18_1a[
            "SEWER"
        ].round(2).values,
    "street_light_pct":
        cluster_centers_18_1a[
            "STREET LIGHT CONDITION"
        ].round(2).values
})

display(cluster_profiles_18_1a)

# --------------------------------------------------
# 5. Build new vs historical K-Means comparison
# --------------------------------------------------

new_cluster_lookup = pd.Series(
    new_labels,
    index=feature_matrix_18_1a.index,
    name="new_batch_cluster"
)

historical_consistency = (
    eligible.groupby("area_id")
    .agg(
        historical_cluster_variants=(
            "kmeans_cluster_id", "nunique"
        ),
        historical_profile_variants=(
            "kmeans_reporting_profile", "nunique"
        )
    )
)

assert historical_consistency[
    "historical_cluster_variants"
].eq(1).all()

assert historical_consistency[
    "historical_profile_variants"
].eq(1).all()

historical_area_context = (
    eligible.groupby("area_id")
    .agg(
        historical_kmeans_cluster=(
            "kmeans_cluster_id", "first"
        ),
        historical_reporting_profile=(
            "kmeans_reporting_profile", "first"
        )
    )
)

area_similarity_18_1a = pd.DataFrame(
    index=feature_matrix_18_1a.index
)

area_similarity_18_1a[
    "incoming_complaints"
] = area_totals.loc[
    feature_matrix_18_1a.index
]

area_similarity_18_1a[
    "new_batch_cluster"
] = new_cluster_lookup

area_similarity_18_1a[
    "new_batch_description"
] = new_cluster_lookup.map(
    cluster_descriptions
)

for category, output_column in [
    ("STREET CONDITION", "road_pct"),
    ("SEWER", "sewer_pct"),
    ("STREET LIGHT CONDITION", "street_light_pct")
]:
    area_similarity_18_1a[output_column] = (
        feature_matrix_18_1a[category]
    )

area_similarity_18_1a = (
    area_similarity_18_1a.join(
        historical_area_context,
        how="left"
    )
    .reset_index()
    .rename(columns={"index": "area_id"})
)

# Historical and new cluster IDs belong to
# different models and are NOT equivalent.

print("\n4. NEW AND HISTORICAL AREA COMPARISON")

display(
    area_similarity_18_1a[[
        "area_id",
        "incoming_complaints",
        "road_pct",
        "sewer_pct",
        "street_light_pct",
        "new_batch_cluster",
        "new_batch_description",
        "historical_kmeans_cluster",
        "historical_reporting_profile"
    ]].sort_values(
        ["new_batch_cluster", "area_id"]
    ).reset_index(drop=True)
)

# --------------------------------------------------
# 6. Test against intended synthetic patterns
# --------------------------------------------------

# These intended labels are loaded ONLY AFTER
# fitting the model. They were not training inputs.
pattern_reference = pd.read_csv(
    reference_path,
    dtype="string"
)

patterns = pattern_reference.set_index(
    "area_id"
)["intended_pattern"]

area_similarity_18_1a[
    "test_pattern"
] = area_similarity_18_1a[
    "area_id"
].map(patterns)

# The road_a and road_b templates are expected
# to form ONE broad road-reporting group.
expected_groups = (
    area_similarity_18_1a[
        "test_pattern"
    ].replace({
        "road_a": "road",
        "road_b": "road"
    })
)

ari_score = adjusted_rand_score(
    expected_groups,
    area_similarity_18_1a["new_batch_cluster"]
)

silhouette = silhouette_score(
    feature_matrix_18_1a,
    new_labels
)

print("\n5. SYNTHETIC EXPERIMENT QUALITY")

print("K:", NEW_BATCH_K)
print("Silhouette score:", round(silhouette, 4))
print("Synthetic pattern agreement (ARI):", round(ari_score, 4))

print("\n6. LEARNED CLUSTER COUNTS")

display(
    area_similarity_18_1a.groupby([
        "new_batch_cluster",
        "new_batch_description"
    ])
    .agg(
        areas=("area_id", "size"),
        incoming_complaints=(
            "incoming_complaints", "sum"
        )
    )
    .reset_index()
)

# --------------------------------------------------
# 7. Verify the new K-Means experiment
# --------------------------------------------------

checks = []

def check(name, condition):
    checks.append({
        "test": name,
        "passed": bool(condition)
    })

check(
    "All 500 new complaints used",
    len(eligible) == 500
)

check(
    "20 areas eligible for analysis",
    len(feature_matrix_18_1a) == 20
)

check(
    "15 supported category features",
    feature_matrix_18_1a.shape[1] == 15
)

check(
    "At least 20 complaints in each area",
    area_totals.loc[
        eligible_areas
    ].ge(20).all()
)

check(
    "Each area has 25 new complaints",
    area_totals.eq(25).all()
)

check(
    "Category percentages total 100 per area",
    np.allclose(
        feature_matrix_18_1a.sum(axis=1),
        100
    )
)

check(
    "Three new K-Means clusters created",
    len(set(new_labels)) == 3
)

check(
    "Road, sewer and lighting cluster types found",
    set(dominant_categories) == {
        "STREET CONDITION",
        "SEWER",
        "STREET LIGHT CONDITION"
    }
)

cluster_sizes = sorted(
    pd.Series(new_labels).value_counts().tolist()
)

check(
    "Synthetic group sizes are 5, 5 and 10 areas",
    cluster_sizes == [5, 5, 10]
)

check(
    "Synthetic intended grouping recovered",
    np.isclose(ari_score, 1.0)
)

check(
    "Clusters show clear synthetic separation",
    silhouette > 0.80
)

check(
    "All 20 historical area profiles preserved",
    area_similarity_18_1a[
        "historical_reporting_profile"
    ].notna().all()
)

# Road variants should be grouped together
# within each borough.
road_pairs_match = True

for borough, group in (
    area_similarity_18_1a.groupby(
        area_similarity_18_1a[
            "area_id"
        ].str.rsplit("_", n=1).str[0]
    )
):
    road_rows = group.loc[
        group["test_pattern"].isin([
            "road_a", "road_b"
        ])
    ]

    if (
        len(road_rows) != 2
        or road_rows[
            "new_batch_cluster"
        ].nunique() != 1
    ):
        road_pairs_match = False

check(
    "Similar road areas grouped together in all boroughs",
    road_pairs_match
)

check(
    "All modeled areas retain unique identifiers",
    area_similarity_18_1a[
        "area_id"
    ].is_unique
)

# --------------------------------------------------
# 8. Final results
# --------------------------------------------------

results_18_1a = pd.DataFrame(checks)

print("\n7. TEST CHECKLIST")

display(results_18_1a)

passed = int(results_18_1a["passed"].sum())
total = len(results_18_1a)

print(f"\nChecks passed: {passed}/{total}")

failures = results_18_1a.loc[
    ~results_18_1a["passed"],
    "test"
].tolist()

if failures:
    print("\nSTATUS: NEEDS REVIEW")
    for name in failures:
        print("FAILED:", name)
else:
    print("\nSTATUS: PASSED")
    print("NEW-BATCH K-MEANS VERIFIED")
    print("SIMILAR AREA REPORTING PATTERNS IDENTIFIED")
    print("HISTORICAL AND NEW K-MEANS KEPT SEPARATE")
    print("NO HISTORICAL MODEL MODIFICATIONS")
    print("MODEL NOT YET DEPLOYED")


NEW-BATCH K-MEANS AREA SIMILARITY

1. ELIGIBLE COMPLAINTS
Geographically verified: 500
Unique areas: 20
Supported categories: 15

2. AREA-CATEGORY FEATURES
Areas eligible for clustering: 20
Areas below threshold: 0
Feature matrix shape: (20, 15)


complaint_category,STREET CONDITION,SIDEWALK CONDITION,SEWER,SEWER MAINTENANCE,STREET LIGHT CONDITION,TRAFFIC SIGNAL CONDITION
area_id,,,,,,
BRONX_10451,56.0,16.0,12.0,0.0,8.0,8.0
BRONX_10452,52.0,20.0,12.0,0.0,8.0,8.0
BRONX_10453,8.0,4.0,60.0,20.0,0.0,0.0
BRONX_10454,8.0,0.0,8.0,0.0,48.0,28.0
BROOKLYN_11201,56.0,16.0,12.0,0.0,8.0,8.0
BROOKLYN_11203,52.0,20.0,12.0,0.0,8.0,8.0
BROOKLYN_11204,8.0,4.0,60.0,20.0,0.0,0.0
BROOKLYN_11205,8.0,0.0,8.0,0.0,48.0,28.0
MANHATTAN_10001,56.0,16.0,12.0,0.0,8.0,8.0



3. NEW-BATCH CLUSTER PROFILES


c:\Users\Administrator\miniconda3\envs\ml\Lib\site-packages\sklearn\cluster\_kmeans.py:1424: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=1.
  warnings.warn(


,new_batch_cluster,description,dominant_category,street_condition_pct,sewer_pct,street_light_pct
0,0,Road-heavy reporting,STREET CONDITION,54.0,12.0,8.0
1,1,Sewer-heavy reporting,SEWER,8.0,60.0,0.0
2,2,Lighting/signal-heavy reporting,STREET LIGHT CONDITION,8.0,8.0,48.0



4. NEW AND HISTORICAL AREA COMPARISON


... (2 outputs left out)

,new_batch_cluster,new_batch_description,areas,incoming_complaints
0,0,Road-heavy reporting,10,250
1,1,Sewer-heavy reporting,5,125
2,2,Lighting/signal-heavy reporting,5,125



7. TEST CHECKLIST


,test,passed
0,All 500 new complaints used,True
1,20 areas eligible for analysis,True
2,15 supported category features,True
3,At least 20 complaints in each area,True
4,Each area has 25 new complaints,True
5,Category percentages total 100 per area,True
6,Three new K-Means clusters created,True
7,"Road, sewer and lighting cluster types found",True
8,"Synthetic group sizes are 5, 5 and 10 areas",True
9,Synthetic intended grouping recovered,True



Checks passed: 14/14

STATUS: PASSED
NEW-BATCH K-MEANS VERIFIED
SIMILAR AREA REPORTING PATTERNS IDENTIFIED
HISTORICAL AND NEW K-MEANS KEPT SEPARATE
NO HISTORICAL MODEL MODIFICATIONS
MODEL NOT YET DEPLOYED


### Build a Reusable New-Batch K-Means Engine

In [12]:

from pathlib import Path
from textwrap import dedent
import sys
import pandas as pd
import numpy as np
from sklearn.metrics import adjusted_rand_score

print(" REUSABLE NEW-BATCH K-MEANS ENGINE")
print("=" * 72)

root = Path(PROJECT_ROOT).resolve()

if str(root) not in sys.path:
    sys.path.insert(0, str(root))

module_path = (
    root / "src/models/new_batch_area_kmeans.py"
)

results_path = (
    root / "outputs/tests"
    / "phase_17_4c_backend_verified"
    / "verified_500_backend_results.csv"
)

reference_path = (
    root / "outputs/tests"
    / "phase_17_4b_diverse"
    / "synthetic_area_pattern_reference.csv"
)

taxonomy_path = (
    root / "outputs/tables/dbscan"
    / "dbscan_action_taxonomy_v1.csv"
)

for path in [results_path, reference_path, taxonomy_path]:
    assert path.is_file(), f"Missing: {path}"

# --------------------------------------------------
# 1. Define reusable K-Means engine
# --------------------------------------------------

module_source = dedent(r'''
"""New-batch area similarity using complaint-category shares.

This module never modifies or retrains historical
K-Means or DBSCAN artifacts.

A cluster identifies similar reporting composition,
not defect severity or operational priority.
"""

import numpy as np
import pandas as pd

from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score


REQUIRED_COLUMNS = {
    "source_row",
    "complaint_id",
    "borough",
    "incident_zip",
    "complaint_category",
    "processing_status",
    "zip_validation_status"
}

VALID_BOROUGHS = {
    "BRONX",
    "BROOKLYN",
    "MANHATTAN",
    "QUEENS",
    "STATEN ISLAND"
}


def analyze_new_batch_areas(
    records,
    supported_categories,
    k=3,
    min_area_size=20,
    random_state=18
):
    """Cluster geographic areas by new complaint shares.

    Input must come from the geographic batch
    processor. Historical labels are context only.
    """

    if not isinstance(records, pd.DataFrame):
        raise TypeError("Expected a DataFrame.")

    missing = REQUIRED_COLUMNS - set(records.columns)

    if missing:
        raise ValueError(
            f"Missing columns: {sorted(missing)}"
        )

    if records["source_row"].duplicated().any():
        raise ValueError("Duplicate source rows.")

    if not isinstance(k, int) or k < 2:
        raise ValueError("k must be an integer >= 2.")

    if (
        not isinstance(min_area_size, int)
        or min_area_size < 1
    ):
        raise ValueError(
            "Minimum area size must be positive."
        )

    categories = sorted({
        str(value).strip().upper()
        for value in supported_categories
        if pd.notna(value)
        and str(value).strip()
    })

    if not categories:
        raise ValueError("No supported categories.")

    # ----------------------------------------------
    # 2. Normalize and screen geographic evidence
    # ----------------------------------------------

    work = records.copy().reset_index(drop=True)

    text_columns = [
        "borough",
        "incident_zip",
        "complaint_category",
        "processing_status",
        "zip_validation_status"
    ]

    for column in text_columns:
        work[column] = (
            work[column]
            .fillna("")
            .astype(str)
            .str.strip()
        )

    work["borough"] = work[
        "borough"
    ].str.upper()

    work["complaint_category"] = work[
        "complaint_category"
    ].str.upper()

    work["area_id"] = pd.NA
    work["eligibility_reason"] = "eligible"

    # Preserve every excluded row with a reason.
    tests = [
        (
            work["processing_status"].ne("processed"),
            "not_processed"
        ),
        (
            work["zip_validation_status"].ne(
                "zcta_consistent"
            ),
            "zip_not_verified"
        ),
        (
            ~work["borough"].isin(VALID_BOROUGHS),
            "invalid_borough"
        ),
        (
            ~work["incident_zip"].str.fullmatch(
                r"\d{5}"
            ),
            "invalid_zip"
        ),
        (
            ~work["complaint_category"].isin(
                categories
            ),
            "unsupported_category"
        )
    ]

    for bad, reason in tests:
        work.loc[
            bad
            & work["eligibility_reason"].eq(
                "eligible"
            ),
            "eligibility_reason"
        ] = reason

    valid = work[
        "eligibility_reason"
    ].eq("eligible")

    work.loc[valid, "area_id"] = (
        work.loc[valid, "borough"]
        + "_"
        + work.loc[valid, "incident_zip"]
    )

    included = work.loc[valid].copy()
    excluded = work.loc[~valid].copy()

    # ----------------------------------------------
    # 3. Create category-count and share matrices
    # ----------------------------------------------

    if included.empty:

        counts = pd.DataFrame(
            index=pd.Index([], name="area_id"),
            columns=categories
        ).fillna(0).astype(int)

    else:

        counts = pd.crosstab(
            included["area_id"],
            included["complaint_category"]
        ).reindex(
            columns=categories,
            fill_value=0
        )

        counts.index.name = "area_id"

    area_sizes = counts.sum(axis=1).astype(int)

    enough = area_sizes.ge(min_area_size)

    features = (
        counts.loc[enough]
        .div(
            area_sizes.loc[enough],
            axis=0
        )
        .mul(100)
    )

    area_summary = pd.DataFrame({
        "area_id": area_sizes.index,
        "incoming_complaints":
            area_sizes.to_numpy(),
        "meets_minimum":
            enough.to_numpy()
    })

    # ----------------------------------------------
    # 4. Add optional historical K-Means context
    # ----------------------------------------------

    area_summary[
        "historical_kmeans_cluster"
    ] = pd.NA

    area_summary[
        "historical_reporting_profile"
    ] = pd.NA

    historical_columns = {
        "kmeans_area_id",
        "kmeans_cluster_id",
        "kmeans_reporting_profile",
        "kmeans_display_allowed"
    }

    if historical_columns.issubset(
        included.columns
    ):

        display_allowed = (
            included["kmeans_display_allowed"]
            .fillna(False)
            .astype(str)
            .str.strip()
            .str.lower()
            .eq("true")
        )

        matching_area = (
            included["kmeans_area_id"]
            .fillna("")
            .astype(str)
            .str.strip()
            .eq(included["area_id"])
        )

        history = included.loc[
            display_allowed & matching_area
        ].copy()

        if not history.empty:

            consistency = (
                history.groupby("area_id")
                .agg(
                    cluster_variants=(
                        "kmeans_cluster_id",
                        "nunique"
                    ),
                    profile_variants=(
                        "kmeans_reporting_profile",
                        "nunique"
                    )
                )
            )

            if (consistency > 1).any().any():
                raise ValueError(
                    "Conflicting historical area profiles."
                )

            lookup = (
                history.drop_duplicates("area_id")
                .set_index("area_id")
            )

            area_summary[
                "historical_kmeans_cluster"
            ] = area_summary["area_id"].map(
                lookup["kmeans_cluster_id"]
            )

            area_summary[
                "historical_reporting_profile"
            ] = area_summary["area_id"].map(
                lookup["kmeans_reporting_profile"]
            )

    # ----------------------------------------------
    # 5. Prepare traceable output and safeguards
    # ----------------------------------------------

    output = {
        "status": None,
        "reason": None,
        "record_eligibility": work[[
            "source_row",
            "complaint_id",
            "area_id",
            "eligibility_reason"
        ]].copy(),
        "area_summary": area_summary,
        "area_category_counts": counts,
        "area_category_percentages": features,
        "cluster_profiles": pd.DataFrame(),
        "model": None,
        "silhouette": None,
        "stats": {
            "total_records": len(work),
            "eligible_records": len(included),
            "excluded_records": len(excluded),
            "eligible_areas": int(enough.sum()),
            "small_sample_areas": int(
                (~enough).sum()
            ),
            "records_in_eligible_areas": int(
                area_sizes.loc[enough].sum()
            ),
            "records_in_small_areas": int(
                area_sizes.loc[~enough].sum()
            )
        }
    }

    if len(features) < k:

        output["status"] = "insufficient_areas"

        output["reason"] = (
            f"Need at least {k} areas with "
            f"{min_area_size}+ complaints."
        )

        return output

    unique_patterns = np.unique(
        features.to_numpy(dtype=float),
        axis=0
    ).shape[0]

    if unique_patterns < k:

        output["status"] = (
            "insufficient_distinct_patterns"
        )

        output["reason"] = (
            f"Need at least {k} different "
            f"area category-share patterns."
        )

        return output

    # ----------------------------------------------
    # 6. Fit only the NEW area-category features
    # ----------------------------------------------

    model = KMeans(
        n_clusters=k,
        random_state=random_state,
        n_init=20
    )

    labels = model.fit_predict(features)

    if len(np.unique(labels)) != k:

        output["status"] = (
            "insufficient_distinct_patterns"
        )

        output["reason"] = (
            "K-Means did not produce the "
            "requested number of clusters."
        )

        return output

    centers = pd.DataFrame(
        model.cluster_centers_,
        columns=categories
    )

    centers.index.name = "new_batch_cluster"

    dominant = centers.idxmax(axis=1)

    readable_names = {
        "STREET CONDITION":
            "Road-heavy reporting",
        "SEWER":
            "Sewer-heavy reporting",
        "STREET LIGHT CONDITION":
            "Lighting/signal-heavy reporting"
    }

    descriptions = dominant.map(
        lambda category: readable_names.get(
            category,
            f"{category.title()}-dominant reporting"
        )
    )

    cluster_profiles = pd.DataFrame({
        "new_batch_cluster": range(k),
        "description": descriptions.to_numpy(),
        "dominant_category": dominant.to_numpy(),
        "areas": (
            pd.Series(labels)
            .value_counts()
            .reindex(range(k), fill_value=0)
            .to_numpy()
        )
    })

    for category in [
        "STREET CONDITION",
        "SEWER",
        "STREET LIGHT CONDITION"
    ]:
        if category in centers:

            cluster_profiles[
                category + "_pct"
            ] = (
                centers[category]
                .round(2)
                .to_numpy()
            )

    assignments = pd.Series(
        labels,
        index=features.index,
        name="new_batch_cluster"
    )

    area_summary["new_batch_cluster"] = (
        pd.to_numeric(
            area_summary["area_id"].map(
                assignments
            ),
            errors="coerce"
        ).astype("Int64")
    )

    area_summary[
        "new_batch_description"
    ] = area_summary[
        "new_batch_cluster"
    ].map(descriptions)

    output.update({
        "status": "clustered",
        "reason": None,
        "model": model,
        "cluster_profiles": cluster_profiles,
        "silhouette": float(
            silhouette_score(features, labels)
        )
    })

    return output
''').lstrip()

compile(
    module_source,
    str(module_path),
    "exec"
)

namespace = {}

exec(
    compile(
        module_source,
        str(module_path),
        "exec"
    ),
    namespace
)

analyze_new_batch_areas = namespace[
    "analyze_new_batch_areas"
]

# --------------------------------------------------
# 7. Load verified incoming data
# --------------------------------------------------

records = pd.read_csv(
    results_path,
    dtype="string",
    keep_default_na=False
)

taxonomy = pd.read_csv(taxonomy_path)

categories = sorted({
    str(value).strip().upper()
    for value in taxonomy[
        "complaint_category"
    ].dropna()
})

print("\n1. REUSABLE ENGINE — FULL BATCH TEST")

full = analyze_new_batch_areas(
    records,
    categories,
    k=3,
    min_area_size=20
)

print("Status:", full["status"])
print("Silhouette:", full["silhouette"])

display(full["cluster_profiles"])

display(full["area_summary"][[
    "area_id",
    "incoming_complaints",
    "new_batch_cluster",
    "new_batch_description",
    "historical_kmeans_cluster",
    "historical_reporting_profile"
]])

# --------------------------------------------------
# 8. Validate against the known synthetic patterns
# --------------------------------------------------

reference = pd.read_csv(
    reference_path,
    dtype="string"
)

pattern_lookup = reference.set_index(
    "area_id"
)["intended_pattern"]

area_table = full["area_summary"].copy()

actual_labels = area_table[
    "new_batch_cluster"
]

expected_labels = (
    area_table["area_id"]
    .map(pattern_lookup)
    .replace({
        "road_a": "road",
        "road_b": "road"
    })
)

ari_score = adjusted_rand_score(
    expected_labels,
    actual_labels
)

print("\n2. SYNTHETIC QUALITY CHECK")
print("Adjusted Rand Index:", round(ari_score, 4))

# --------------------------------------------------
# 9. Test insufficient per-area complaint counts
# --------------------------------------------------

first_area = (
    records.iloc[0]["borough"]
    + "_"
    + records.iloc[0]["incident_zip"]
)

area_id_series = (
    records["borough"]
    + "_"
    + records["incident_zip"]
)

first_area_mask = area_id_series.eq(
    first_area
)

first_three = records.loc[
    first_area_mask
].index[:3]

small_batch = records.loc[
    ~first_area_mask
    | records.index.isin(first_three)
].copy()

small_result = analyze_new_batch_areas(
    small_batch,
    categories
)

print("\n3. SMALL-SAMPLE SAFEGUARD")
print("Status:", small_result["status"])
print("Stats:", small_result["stats"])

# --------------------------------------------------
# 10. Test insufficient area coverage
# --------------------------------------------------

one_area = records.loc[
    first_area_mask
].copy()

one_area_result = analyze_new_batch_areas(
    one_area,
    categories
)

print("\n4. INSUFFICIENT-AREA SAFEGUARD")
print("Status:", one_area_result["status"])
print("Reason:", one_area_result["reason"])

# --------------------------------------------------
# 11. Test insufficient category-pattern diversity
# --------------------------------------------------

road_area_ids = set(
    reference.loc[
        reference["intended_pattern"].isin([
            "road_a", "road_b"
        ]),
        "area_id"
    ]
)

road_only = records.loc[
    area_id_series.isin(road_area_ids)
].copy()

road_only_result = analyze_new_batch_areas(
    road_only,
    categories
)

print("\n5. INSUFFICIENT-PATTERN SAFEGUARD")
print("Status:", road_only_result["status"])
print("Reason:", road_only_result["reason"])

# --------------------------------------------------
# 12. Test exclusion of geographic mismatches
# --------------------------------------------------

mismatch_data = records.copy()

mismatch_data.loc[
    0, "zip_validation_status"
] = "zcta_mismatch"

mismatch_result = analyze_new_batch_areas(
    mismatch_data,
    categories
)

# --------------------------------------------------
# 13. Test deterministic, reproducible results
# --------------------------------------------------

repeat_result = analyze_new_batch_areas(
    records,
    categories,
    k=3,
    min_area_size=20
)

# --------------------------------------------------
# 14. Comprehensive validation checklist
# --------------------------------------------------

checks = []

def check(name, condition):
    checks.append({
        "test": name,
        "passed": bool(condition)
    })

check(
    "All 500 input records accounted for",
    full["stats"]["total_records"] == 500
)

check(
    "500 eligible complaints",
    full["stats"]["eligible_records"] == 500
)

check(
    "20 areas eligible for clustering",
    full["stats"]["eligible_areas"] == 20
)

check(
    "All source complaint rows preserved",
    len(full["record_eligibility"]) == 500
)

check(
    "No complaints excluded in verified batch",
    full["stats"]["excluded_records"] == 0
)

check(
    "Three new-batch clusters created",
    full["status"] == "clustered"
    and len(full["cluster_profiles"]) == 3
)

check(
    "All new area feature percentages total 100",
    np.allclose(
        full[
            "area_category_percentages"
        ].sum(axis=1),
        100
    )
)

check(
    "Intended synthetic groups recovered",
    np.isclose(ari_score, 1.0)
)

check(
    "Synthetic silhouette exceeds 0.80",
    full["silhouette"] > 0.80
)

check(
    "Three broad reporting patterns recovered",
    set(
        full["cluster_profiles"][
            "description"
        ]
    ) == {
        "Road-heavy reporting",
        "Sewer-heavy reporting",
        "Lighting/signal-heavy reporting"
    }
)

check(
    "Expected synthetic cluster sizes recovered",
    sorted(
        full["cluster_profiles"]["areas"].tolist()
    ) == [5, 5, 10]
)

check(
    "All historical area profiles preserved",
    full["area_summary"][
        "historical_reporting_profile"
    ].notna().all()
)

check(
    "Small-sample area excluded from clustering",
    small_result["status"] == "clustered"
    and small_result["stats"][
        "small_sample_areas"
    ] == 1
    and small_result["stats"][
        "records_in_small_areas"
    ] == 3
)

check(
    "Single-area input does not fit K-Means",
    one_area_result["status"]
    == "insufficient_areas"
    and one_area_result["model"] is None
)

check(
    "Insufficient pattern diversity does not fit K-Means",
    road_only_result["status"]
    == "insufficient_distinct_patterns"
    and road_only_result["model"] is None
)

check(
    "ZIP-mismatched complaint excluded with reason",
    mismatch_result["stats"][
        "excluded_records"
    ] == 1
    and mismatch_result[
        "record_eligibility"
    ].iloc[0]["eligibility_reason"]
    == "zip_not_verified"
)

check(
    "Repeated analysis is reproducible",
    full["area_summary"][
        "new_batch_cluster"
    ].tolist()
    == repeat_result["area_summary"][
        "new_batch_cluster"
    ].tolist()
)

check(
    "Existing historical models not fitted by new engine",
    full["model"].n_features_in_
    == len(categories)
)

# --------------------------------------------------
# 15. Display validation results
# --------------------------------------------------

results_18_1b = pd.DataFrame(checks)

print("\n6. VALIDATION")

display(results_18_1b)

passed = int(results_18_1b["passed"].sum())
total = len(results_18_1b)

print(f"\nChecks passed: {passed}/{total}")

failures = results_18_1b.loc[
    ~results_18_1b["passed"],
    "test"
].tolist()

if failures:

    print("\nSTATUS: NEEDS REVIEW")

    for failure in failures:
        print("FAILED:", failure)

    print("No new module saved.")

else:

    # ----------------------------------------------
    # 16. Save only the validated reusable module
    # ----------------------------------------------

    if module_path.exists():

        assert module_path.read_text(
            encoding="utf-8"
        ) == module_source, (
            "Existing new-batch K-Means module "
            "differs. No overwrite performed."
        )

    else:

        module_path.parent.mkdir(
            parents=True,
            exist_ok=True
        )

        module_path.write_text(
            module_source,
            encoding="utf-8"
        )

    print("\n7. VERIFIED MODULE")
    print(module_path)

    print("\nSTATUS: PASSED")
    print("REUSABLE K-MEANS ENGINE VERIFIED")
    print("SMALL-SAMPLE SAFEGUARDS VERIFIED")
    print("INSUFFICIENT-DIVERSITY SAFEGUARDS VERIFIED")
    print("HISTORICAL MODELS UNMODIFIED")
    print("REAL-DATA VALIDATION STILL REQUIRED")


 REUSABLE NEW-BATCH K-MEANS ENGINE

1. REUSABLE ENGINE — FULL BATCH TEST
Status: clustered
Silhouette: 0.97664289820422


c:\Users\Administrator\miniconda3\envs\ml\Lib\site-packages\sklearn\cluster\_kmeans.py:1424: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=1.
  warnings.warn(


,new_batch_cluster,description,dominant_category,areas,STREET CONDITION_pct,SEWER_pct,STREET LIGHT CONDITION_pct
0,0,Road-heavy reporting,STREET CONDITION,10,54.0,12.0,8.0
1,1,Sewer-heavy reporting,SEWER,5,8.0,60.0,0.0
2,2,Lighting/signal-heavy reporting,STREET LIGHT CONDITION,5,8.0,8.0,48.0


,area_id,incoming_complaints,new_batch_cluster,new_batch_description,historical_kmeans_cluster,historical_reporting_profile
0,BRONX_10451,25,0,Road-heavy reporting,0,Sustained High-Volume Reporting
1,BRONX_10452,25,0,Road-heavy reporting,0,Sustained High-Volume Reporting
2,BRONX_10453,25,1,Sewer-heavy reporting,0,Sustained High-Volume Reporting
3,BRONX_10454,25,2,Lighting/signal-heavy reporting,0,Sustained High-Volume Reporting
4,BROOKLYN_11201,25,0,Road-heavy reporting,0,Sustained High-Volume Reporting
5,BROOKLYN_11203,25,0,Road-heavy reporting,0,Sustained High-Volume Reporting
6,BROOKLYN_11204,25,1,Sewer-heavy reporting,0,Sustained High-Volume Reporting
7,BROOKLYN_11205,25,2,Lighting/signal-heavy reporting,0,Sustained High-Volume Reporting
8,MANHATTAN_10001,25,0,Road-heavy reporting,0,Sustained High-Volume Reporting
9,MANHATTAN_10002,25,0,Road-heavy reporting,0,Sustained High-Volume Reporting



2. SYNTHETIC QUALITY CHECK
Adjusted Rand Index: 1.0


c:\Users\Administrator\miniconda3\envs\ml\Lib\site-packages\sklearn\cluster\_kmeans.py:1424: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=1.
  warnings.warn(



3. SMALL-SAMPLE SAFEGUARD
Status: clustered
Stats: {'total_records': 478, 'eligible_records': 478, 'excluded_records': 0, 'eligible_areas': 19, 'small_sample_areas': 1, 'records_in_eligible_areas': 475, 'records_in_small_areas': 3}

4. INSUFFICIENT-AREA SAFEGUARD
Status: insufficient_areas
Reason: Need at least 3 areas with 20+ complaints.

5. INSUFFICIENT-PATTERN SAFEGUARD
Status: insufficient_distinct_patterns
Reason: Need at least 3 different area category-share patterns.

6. VALIDATION


c:\Users\Administrator\miniconda3\envs\ml\Lib\site-packages\sklearn\cluster\_kmeans.py:1424: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=1.
  warnings.warn(
c:\Users\Administrator\miniconda3\envs\ml\Lib\site-packages\sklearn\cluster\_kmeans.py:1424: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=1.
  warnings.warn(


,test,passed
0,All 500 input records accounted for,True
1,500 eligible complaints,True
2,20 areas eligible for clustering,True
3,All source complaint rows preserved,True
4,No complaints excluded in verified batch,True
5,Three new-batch clusters created,True
6,All new area feature percentages total 100,True
7,Intended synthetic groups recovered,True
8,Synthetic silhouette exceeds 0.80,True
9,Three broad reporting patterns recovered,True



Checks passed: 18/18

7. VERIFIED MODULE
D:\Urban-Infrastructure-Pulse-Intelligence\src\models\new_batch_area_kmeans.py

STATUS: PASSED
REUSABLE K-MEANS ENGINE VERIFIED
SMALL-SAMPLE SAFEGUARDS VERIFIED
INSUFFICIENT-DIVERSITY SAFEGUARDS VERIFIED
HISTORICAL MODELS UNMODIFIED
REAL-DATA VALIDATION STILL REQUIRED


### K Selection and Stability Testing

Should we always use K=3, or should the system evaluate whether another number of groups is better?

Our synthetic dataset deliberately has four exact category-distribution templates, but we want three broader reporting groups. A model with K=4 could separate the two very similar road-heavy templates unnecessarily.

We'll compare K=2, K=3 and K=4, then test how the assignments change when complaint-category counts vary slightly. This will help prevent the assistant from choosing a misleading number of clusters based only on a silhouette score.

In [13]:

from pathlib import Path
from threadpoolctl import threadpool_limits
from sklearn.metrics import adjusted_rand_score
import hashlib
import numpy as np
import pandas as pd

print(" K-SELECTION AND STABILITY TESTS")
print("=" * 75)

root = Path(PROJECT_ROOT).resolve()

module_path = (
    root / "src/models/new_batch_area_kmeans.py"
)

results_path = (
    root / "outputs/tests"
    / "phase_17_4c_backend_verified"
    / "verified_500_backend_results.csv"
)

reference_path = (
    root / "outputs/tests"
    / "phase_17_4b_diverse"
    / "synthetic_area_pattern_reference.csv"
)

taxonomy_path = (
    root / "outputs/tables/dbscan"
    / "dbscan_action_taxonomy_v1.csv"
)

for path in [
    module_path,
    results_path,
    reference_path,
    taxonomy_path
]:
    assert path.is_file(), f"Missing: {path}"

from src.models.new_batch_area_kmeans import (
    analyze_new_batch_areas
)

source_hash_before = hashlib.sha256(
    module_path.read_bytes()
).hexdigest()

records = pd.read_csv(
    results_path,
    dtype="string",
    keep_default_na=False
)

taxonomy = pd.read_csv(taxonomy_path)

categories = sorted({
    str(value).strip().upper()
    for value in taxonomy[
        "complaint_category"
    ].dropna()
})

assert len(records) == 500
assert len(categories) == 15

# --------------------------------------------------
# 1. Compare possible numbers of clusters
# --------------------------------------------------

print("\n1. COMPARING K=2, K=3, K=4 AND K=5")

candidate_results = {}
comparison_rows = []

with threadpool_limits(
    limits=1,
    user_api="openmp"
):

    for k in [2, 3, 4, 5]:

        result = analyze_new_batch_areas(
            records,
            categories,
            k=k,
            min_area_size=20,
            random_state=18
        )

        candidate_results[k] = result

        cluster_sizes = []

        if result["status"] == "clustered":
            cluster_sizes = sorted(
                result["cluster_profiles"][
                    "areas"
                ].astype(int).tolist()
            )

        comparison_rows.append({
            "k": k,
            "status": result["status"],
            "silhouette": (
                round(result["silhouette"], 4)
                if result["silhouette"] is not None
                else np.nan
            ),
            "eligible_areas":
                result["stats"]["eligible_areas"],
            "cluster_sizes": str(cluster_sizes)
        })

k_comparison_18_1c = pd.DataFrame(
    comparison_rows
)

display(k_comparison_18_1c)

# --------------------------------------------------
# 2. Examine the distinct category-share patterns
# --------------------------------------------------

features = candidate_results[3][
    "area_category_percentages"
]

unique_patterns = np.unique(
    features.to_numpy(dtype=float),
    axis=0
).shape[0]

print("\n2. DATASET CHARACTERISTICS")

print("Areas:", len(features))
print("Category features:", features.shape[1])
print("Distinct category-share patterns:", unique_patterns)

# --------------------------------------------------
# 3. Evaluate agreement with test patterns
# --------------------------------------------------

# Intended patterns are used only for evaluation,
# not as inputs to the K-Means models.

reference = pd.read_csv(
    reference_path,
    dtype="string"
)

pattern_lookup = reference.set_index(
    "area_id"
)["intended_pattern"]

def get_assignments(result):
    return (
        result["area_summary"]
        .set_index("area_id")[
            "new_batch_cluster"
        ]
        .dropna()
        .astype(int)
        .sort_index()
    )

baseline_k3 = get_assignments(
    candidate_results[3]
)

baseline_k4 = get_assignments(
    candidate_results[4]
)

expected_four = (
    baseline_k3.index.to_series()
    .map(pattern_lookup)
)

expected_three = expected_four.replace({
    "road_a": "road",
    "road_b": "road"
})

k3_broad_ari = adjusted_rand_score(
    expected_three,
    baseline_k3
)

k4_broad_ari = adjusted_rand_score(
    expected_three,
    baseline_k4
)

k4_four_pattern_ari = adjusted_rand_score(
    expected_four,
    baseline_k4
)

print("\n3. SYNTHETIC PATTERN AGREEMENT")

print(
    "K=3 vs three broad groups:",
    round(k3_broad_ari, 4)
)

print(
    "K=4 vs three broad groups:",
    round(k4_broad_ari, 4)
)

print(
    "K=4 vs four exact templates:",
    round(k4_four_pattern_ari, 4)
)

# --------------------------------------------------
# 4. Check whether K=4 splits similar road areas
# --------------------------------------------------

road_a_ids = pattern_lookup.loc[
    pattern_lookup.eq("road_a")
].index

road_b_ids = pattern_lookup.loc[
    pattern_lookup.eq("road_b")
].index

road_a_clusters = set(
    baseline_k4.loc[road_a_ids].tolist()
)

road_b_clusters = set(
    baseline_k4.loc[road_b_ids].tolist()
)

k4_splits_road_variants = (
    len(road_a_clusters) == 1
    and len(road_b_clusters) == 1
    and road_a_clusters != road_b_clusters
)

print("\n4. SIMILAR ROAD-AREA COMPARISON")
print("K=4 road_a cluster(s):", road_a_clusters)
print("K=4 road_b cluster(s):", road_b_clusters)
print(
    "K=4 separates road variants:",
    k4_splits_road_variants
)

# --------------------------------------------------
# 5. Check K=3 consistency across random seeds
# --------------------------------------------------

print("\n5. RANDOM-SEED STABILITY")

seed_rows = []

with threadpool_limits(
    limits=1,
    user_api="openmp"
):

    for seed in [0, 7, 18, 42, 99]:

        result = analyze_new_batch_areas(
            records,
            categories,
            k=3,
            min_area_size=20,
            random_state=seed
        )

        if result["status"] != "clustered":
            agreement = np.nan
        else:
            labels = get_assignments(result)

            agreement = adjusted_rand_score(
                baseline_k3.loc[labels.index],
                labels
            )

        seed_rows.append({
            "random_state": seed,
            "status": result["status"],
            "agreement_with_baseline": agreement
        })

seed_stability_18_1c = pd.DataFrame(
    seed_rows
)

display(seed_stability_18_1c)

# --------------------------------------------------
# 6. Simulate variation in category reporting
# --------------------------------------------------

# Keep the same 500 records and validated areas,
# but resample complaint categories within each
# area to assess sampling sensitivity.
#
# These are synthetic perturbations, NOT
# additional real complaint records.

def resample_area_categories(frame, seed):

    rng = np.random.default_rng(seed)

    sampled = frame.copy().reset_index(
        drop=True
    )

    grouped_indices = sampled.groupby([
        "borough",
        "incident_zip"
    ]).groups

    for _, indices in grouped_indices.items():

        original_categories = (
            sampled.loc[
                indices,
                "complaint_category"
            ]
            .astype(str)
            .to_numpy()
        )

        sampled.loc[
            indices,
            "complaint_category"
        ] = rng.choice(
            original_categories,
            size=len(indices),
            replace=True
        )

    return sampled


print("\n6. CATEGORY-SAMPLING STABILITY")

bootstrap_rows = []

N_TRIALS = 12

with threadpool_limits(
    limits=1,
    user_api="openmp"
):

    for trial in range(N_TRIALS):

        perturbed = resample_area_categories(
            records,
            seed=18100 + trial
        )

        trial_row = {"trial": trial + 1}

        for k in [3, 4]:

            result = analyze_new_batch_areas(
                perturbed,
                categories,
                k=k,
                min_area_size=20,
                random_state=18
            )

            trial_row[
                f"k{k}_status"
            ] = result["status"]

            if result["status"] == "clustered":

                assignments = get_assignments(
                    result
                )

                reference_labels = (
                    baseline_k3
                    if k == 3
                    else baseline_k4
                )

                trial_row[
                    f"k{k}_agreement"
                ] = adjusted_rand_score(
                    reference_labels.loc[
                        assignments.index
                    ],
                    assignments
                )

            else:

                trial_row[
                    f"k{k}_agreement"
                ] = np.nan

        bootstrap_rows.append(trial_row)

bootstrap_stability_18_1c = pd.DataFrame(
    bootstrap_rows
)

display(bootstrap_stability_18_1c)

# --------------------------------------------------
# 7. Summarize the stability findings
# --------------------------------------------------

stability_summary_18_1c = pd.DataFrame([
    {
        "model": "K=3",
        "median_agreement": (
            bootstrap_stability_18_1c[
                "k3_agreement"
            ].median()
        ),
        "minimum_agreement": (
            bootstrap_stability_18_1c[
                "k3_agreement"
            ].min()
        ),
        "successful_trials": (
            bootstrap_stability_18_1c[
                "k3_status"
            ].eq("clustered").sum()
        )
    },
    {
        "model": "K=4",
        "median_agreement": (
            bootstrap_stability_18_1c[
                "k4_agreement"
            ].median()
        ),
        "minimum_agreement": (
            bootstrap_stability_18_1c[
                "k4_agreement"
            ].min()
        ),
        "successful_trials": (
            bootstrap_stability_18_1c[
                "k4_status"
            ].eq("clustered").sum()
        )
    }
])

print("\n7. STABILITY SUMMARY")

display(
    stability_summary_18_1c.round(4)
)

k3_median = float(
    stability_summary_18_1c.iloc[0][
        "median_agreement"
    ]
)

k4_median = float(
    stability_summary_18_1c.iloc[1][
        "median_agreement"
    ]
)

# --------------------------------------------------
# 8. Validate the selection experiment
# --------------------------------------------------

checks = []

def check(name, condition):
    checks.append({
        "test": name,
        "passed": bool(condition)
    })

check(
    "500 complaint records preserved",
    len(records) == 500
)

check(
    "20 area feature vectors available",
    len(features) == 20
)

check(
    "Four distinct synthetic templates detected",
    unique_patterns == 4
)

for k in [2, 3, 4]:
    check(
        f"K={k} clustering completed",
        candidate_results[k]["status"]
        == "clustered"
    )

check(
    "K=5 rejected due to insufficient patterns",
    candidate_results[5]["status"]
    == "insufficient_distinct_patterns"
)

check(
    "K=3 recovers three intended broad groups",
    np.isclose(k3_broad_ari, 1.0)
)

check(
    "K=4 recovers four exact patterns",
    np.isclose(k4_four_pattern_ari, 1.0)
)

check(
    "K=4 separates similar road variants",
    k4_splits_road_variants
)

check(
    "K=4 has higher synthetic silhouette than K=3",
    candidate_results[4]["silhouette"]
    > candidate_results[3]["silhouette"]
)

check(
    "K=3 has higher broad-group agreement",
    k3_broad_ari > k4_broad_ari
)

check(
    "All five K=3 seeds are reproducible",
    seed_stability_18_1c[
        "agreement_with_baseline"
    ].eq(1.0).all()
)

check(
    "12 category-sampling trials completed",
    len(bootstrap_stability_18_1c)
    == N_TRIALS
)

check(
    "All K=3 sampling trials produced clusters",
    bootstrap_stability_18_1c[
        "k3_status"
    ].eq("clustered").all()
)

check(
    "All K=4 sampling trials produced clusters",
    bootstrap_stability_18_1c[
        "k4_status"
    ].eq("clustered").all()
)

check(
    "K=3 median sampling agreement exceeds 0.90",
    k3_median >= 0.90
)

check(
    "K=3 more stable than K=4 in this experiment",
    k3_median > k4_median
)

check(
    "Historical and new-batch models remain separate",
    candidate_results[3][
        "area_summary"
    ]["historical_reporting_profile"]
    .notna().all()
)

source_hash_after = hashlib.sha256(
    module_path.read_bytes()
).hexdigest()

check(
    "Reusable model source remains unchanged",
    source_hash_before == source_hash_after
)

# --------------------------------------------------
# 9. Report results
# --------------------------------------------------

results_18_1c = pd.DataFrame(checks)

print("\n8. TEST CHECKLIST")

display(results_18_1c)

passed = int(
    results_18_1c["passed"].sum()
)

total = len(results_18_1c)

print(f"\nChecks passed: {passed}/{total}")

failed = results_18_1c.loc[
    ~results_18_1c["passed"],
    "test"
].tolist()

if failed:

    print("\nSTATUS: NEEDS REVIEW")

    for name in failed:
        print("FAILED:", name)

else:

    print("\nSTATUS: PASSED")
    print("K-SELECTION EXPERIMENT VERIFIED")
    print("K=3 BROAD REPORTING GROUPS RECOVERED")
    print("CATEGORY-SAMPLING STABILITY EVALUATED")
    print("MODEL SELECTION REMAINS HUMAN-REVIEWED")
    print("HISTORICAL MODELS UNMODIFIED")
    print("NO PRODUCTION MODEL DEPLOYED")


 K-SELECTION AND STABILITY TESTS

1. COMPARING K=2, K=3, K=4 AND K=5


,k,status,silhouette,eligible_areas,cluster_sizes
0,2,clustered,0.6687,20,"[5, 15]"
1,3,clustered,0.9766,20,"[5, 5, 10]"
2,4,clustered,1.0000,20,"[5, 5, 5, 5]"
3,5,insufficient_distinct_patterns,NaN,20,[]



2. DATASET CHARACTERISTICS
Areas: 20
Category features: 15
Distinct category-share patterns: 4

3. SYNTHETIC PATTERN AGREEMENT
K=3 vs three broad groups: 1.0
K=4 vs three broad groups: 0.678
K=4 vs four exact templates: 1.0

4. SIMILAR ROAD-AREA COMPARISON
K=4 road_a cluster(s): {3}
K=4 road_b cluster(s): {0}
K=4 separates road variants: True

5. RANDOM-SEED STABILITY


,random_state,status,agreement_with_baseline
0,0,clustered,1.0
1,7,clustered,1.0
2,18,clustered,1.0
3,42,clustered,1.0
4,99,clustered,1.0



6. CATEGORY-SAMPLING STABILITY


,trial,k3_status,k3_agreement,k4_status,k4_agreement
0,1,clustered,1.0,clustered,0.692841
1,2,clustered,1.0,clustered,0.633441
2,3,clustered,1.0,clustered,0.670520
3,4,clustered,1.0,clustered,0.581973
4,5,clustered,1.0,clustered,0.633441
5,6,clustered,1.0,clustered,0.633441
6,7,clustered,1.0,clustered,0.634334
7,8,clustered,1.0,clustered,0.614955
8,9,clustered,1.0,clustered,0.607762
9,10,clustered,1.0,clustered,0.634334



7. STABILITY SUMMARY


,model,median_agreement,minimum_agreement,successful_trials
0,K=3,1.0000,1.000,12
1,K=4,0.6339,0.582,12



8. TEST CHECKLIST


,test,passed
0,500 complaint records preserved,True
1,20 area feature vectors available,True
2,Four distinct synthetic templates detected,True
3,K=2 clustering completed,True
4,K=3 clustering completed,True
5,K=4 clustering completed,True
6,K=5 rejected due to insufficient patterns,True
7,K=3 recovers three intended broad groups,True
8,K=4 recovers four exact patterns,True
9,K=4 separates similar road variants,True



Checks passed: 20/20

STATUS: PASSED
K-SELECTION EXPERIMENT VERIFIED
K=3 BROAD REPORTING GROUPS RECOVERED
CATEGORY-SAMPLING STABILITY EVALUATED
MODEL SELECTION REMAINS HUMAN-REVIEWED
HISTORICAL MODELS UNMODIFIED
NO PRODUCTION MODEL DEPLOYED


### Robustness Hardening
Before adding the new K-Means engine to the GUI, we should fix two details in the reusable module:
1. Silhouette safeguard: When the number of qualifying areas equals K, scikit-learn cannot calculate a silhouette score. The engine should return a clear status instead of raising an error.
2. Small-area traceability: When an area has fewer than 20 complaints, its individual records should explicitly say that the area was excluded from clustering.

In [15]:

from pathlib import Path
from textwrap import dedent
from threadpoolctl import threadpool_limits
import ast
import hashlib
import importlib
import sys
import pandas as pd
import numpy as np

print("SAFE K-MEANS REPAIR")
print("=" * 70)

root = Path(PROJECT_ROOT).resolve()

if str(root) not in sys.path:
    sys.path.insert(0, str(root))

module_path = (
    root / "src/models/new_batch_area_kmeans.py"
)

results_path = (
    root / "outputs/tests"
    / "phase_17_4c_backend_verified"
    / "verified_500_backend_results.csv"
)

reference_path = (
    root / "outputs/tests"
    / "phase_17_4b_diverse"
    / "synthetic_area_pattern_reference.csv"
)

taxonomy_path = (
    root / "outputs/tables/dbscan"
    / "dbscan_action_taxonomy_v1.csv"
)

backup_path = (
    root / "outputs/tests/phase_18_1d"
    / "new_batch_area_kmeans_before_hardening.txt"
)

for path in [
    module_path,
    results_path,
    reference_path,
    taxonomy_path
]:
    assert path.is_file(), f"Missing file: {path}"

# --------------------------------------------------
# 1. Read the real module without changing it
# --------------------------------------------------

original_bytes = module_path.read_bytes()
original_source = original_bytes.decode("utf-8")

print("\n1. ORIGINAL MODULE")
print("File:", module_path)
print("Bytes:", len(original_bytes))

tree = ast.parse(original_source)

# --------------------------------------------------
# 2. Locate assignments by Python syntax
#    instead of exact whitespace
# --------------------------------------------------

def find_assignment(variable):
    matches = []

    for node in ast.walk(tree):
        if not isinstance(node, ast.Assign):
            continue

        if any(
            isinstance(target, ast.Name)
            and target.id == variable
            for target in node.targets
        ):
            matches.append(node)

    if len(matches) != 1:
        raise ValueError(
            f"Expected one assignment for "
            f"{variable}, found {len(matches)}. "
            "Module unchanged."
        )

    return matches[0]


enough_node = find_assignment("enough")
patterns_node = find_assignment(
    "unique_patterns"
)

enough_expression = ast.get_source_segment(
    original_source, enough_node
)

patterns_expression = ast.get_source_segment(
    original_source, patterns_node
)

assert "area_sizes" in enough_expression
assert "min_area_size" in enough_expression
assert "np.unique" in patterns_expression

print("\n2. PYTHON STATEMENTS FOUND")
print("Area threshold:", enough_expression)
print("Pattern count:", patterns_expression)

# --------------------------------------------------
# 3. Prepare fixes using syntax locations
# --------------------------------------------------

newline = (
    "\r\n"
    if "\r\n" in original_source
    else "\n"
)

lines = original_source.splitlines(
    keepends=True
)

def indentation_of(node):
    line = lines[node.lineno - 1]
    return line[:len(line) - len(line.lstrip())]


def make_block(body, indentation):
    body_lines = dedent(body).strip(
        "\n"
    ).splitlines()

    return newline + "".join(
        (
            indentation + line + newline
            if line.strip()
            else newline
        )
        for line in body_lines
    )


small_area_fix = """
# Mark records from areas below the minimum
# sample threshold without discarding them.
small_area_mask = (
    valid
    & work["area_id"].isin(
        area_sizes.index[~enough]
    )
)

work.loc[
    small_area_mask,
    "eligibility_reason"
] = "below_minimum_area_size"
"""

silhouette_fix = """
# Silhouette is undefined when the number of
# qualifying areas equals the number of clusters.
if len(features) == k:

    output["status"] = (
        "insufficient_areas_for_validation"
    )

    output["reason"] = (
        f"Need at least {k + 1} qualifying "
        f"areas to evaluate K={k} safely."
    )

    return output
"""

has_small_fix = (
    '"below_minimum_area_size"'
    in original_source
)

has_silhouette_fix = (
    '"insufficient_areas_for_validation"'
    in original_source
)

insertions = []

if not has_small_fix:
    insertions.append((
        enough_node.end_lineno,
        make_block(
            small_area_fix,
            indentation_of(enough_node)
        )
    ))

if not has_silhouette_fix:
    insertions.append((
        patterns_node.lineno - 1,
        make_block(
            silhouette_fix,
            indentation_of(patterns_node)
        )
    ))

# Insert from bottom to top so original line
# numbers remain correct during both insertions.
for position, block in sorted(
    insertions,
    key=lambda item: item[0],
    reverse=True
):
    lines.insert(position, block)

revised_source = "".join(lines)

compile(
    revised_source,
    str(module_path),
    "exec"
)

namespace = {}

exec(
    compile(
        revised_source,
        str(module_path),
        "exec"
    ),
    namespace
)

analyze_fixed = namespace[
    "analyze_new_batch_areas"
]

print("\n3. PATCH PREPARATION")
print("New corrections required:", len(insertions))
print("Corrected module compiles: YES")
print("Original file still unchanged: YES")

# --------------------------------------------------
# 4. Load previously verified synthetic data
# --------------------------------------------------

records = pd.read_csv(
    results_path,
    dtype="string",
    keep_default_na=False
)

reference = pd.read_csv(
    reference_path,
    dtype="string"
)

taxonomy = pd.read_csv(taxonomy_path)

categories = sorted({
    str(x).strip().upper()
    for x in taxonomy[
        "complaint_category"
    ].dropna()
})

assert len(records) == 500

area_ids = (
    records["borough"]
    + "_"
    + records["incident_zip"]
)

checks = []

def check(name, condition):
    checks.append({
        "test": name,
        "passed": bool(condition)
    })

# --------------------------------------------------
# 5. Regression test with all 500 complaints
# --------------------------------------------------

print("\n4. FULL BATCH REGRESSION")

with threadpool_limits(limits=1):
    full = analyze_fixed(
        records,
        categories,
        k=3,
        min_area_size=20
    )

print("Status:", full["status"])
print("Silhouette:", full["silhouette"])

check(
    "500 records retained",
    full["stats"]["total_records"] == 500
)

check(
    "20 areas eligible",
    full["stats"]["eligible_areas"] == 20
)

check(
    "K=3 clustering succeeds",
    full["status"] == "clustered"
)

check(
    "Original silhouette preserved",
    np.isclose(
        full["silhouette"],
        0.97664289820422,
        atol=0.00001
    )
)

check(
    "Expected cluster sizes preserved",
    sorted(
        full["cluster_profiles"][
            "areas"
        ].astype(int).tolist()
    ) == [5, 5, 10]
)

check(
    "Historical K-Means profiles retained",
    full["area_summary"][
        "historical_reporting_profile"
    ].notna().all()
)

# --------------------------------------------------
# 6. Test exactly K qualifying areas
# --------------------------------------------------

print("\n5. EXACTLY-K SAFEGUARD")

selected_areas = [
    reference.loc[
        reference[
            "intended_pattern"
        ].eq(pattern),
        "area_id"
    ].iloc[0]
    for pattern in [
        "road_a",
        "sewer",
        "signals"
    ]
]

three_area_data = records.loc[
    area_ids.isin(selected_areas)
].copy()

with threadpool_limits(limits=1):
    exactly_three = analyze_fixed(
        three_area_data,
        categories,
        k=3
    )

print("Status:", exactly_three["status"])
print("Reason:", exactly_three["reason"])

check(
    "Exactly K areas handled safely",
    exactly_three["status"]
    == "insufficient_areas_for_validation"
)

check(
    "No model fitted for exactly K areas",
    exactly_three["model"] is None
)

check(
    "Three areas retained in the summary",
    len(exactly_three["area_summary"]) == 3
)

# --------------------------------------------------
# 7. Test small-area complaint traceability
# --------------------------------------------------

print("\n6. SMALL-AREA SAFEGUARD")

first_area = area_ids.iloc[0]

first_indices = records.index[
    area_ids.eq(first_area)
]

small_data = records.drop(
    index=first_indices[3:]
).copy()

with threadpool_limits(limits=1):
    small_result = analyze_fixed(
        small_data,
        categories,
        k=3,
        min_area_size=20
    )

small_flagged = small_result[
    "record_eligibility"
].loc[
    lambda df: df[
        "eligibility_reason"
    ].eq("below_minimum_area_size")
]

display(small_flagged)

check(
    "Three small-area records flagged",
    len(small_flagged) == 3
)

check(
    "Small-area records retain correct IDs",
    set(
        small_flagged[
            "complaint_id"
        ].astype(str)
    ) == set(
        records.loc[
            first_indices[:3],
            "complaint_id"
        ].astype(str)
    )
)

check(
    "Remaining 19 areas cluster",
    small_result["status"] == "clustered"
    and small_result["stats"][
        "eligible_areas"
    ] == 19
)

check(
    "Small-area records not silently removed",
    len(
        small_result["record_eligibility"]
    ) == len(small_data)
)

# --------------------------------------------------
# 8. Verify no source modification yet
# --------------------------------------------------

check(
    "Existing module unchanged during testing",
    module_path.read_bytes() == original_bytes
)

# --------------------------------------------------
# 9. Report tests before saving
# --------------------------------------------------

results_18_1d_fix = pd.DataFrame(checks)

print("\n7. REPAIR VALIDATION")

display(results_18_1d_fix)

passed = int(
    results_18_1d_fix["passed"].sum()
)

total = len(results_18_1d_fix)

print(f"\nChecks passed: {passed}/{total}")

failures = results_18_1d_fix.loc[
    ~results_18_1d_fix["passed"],
    "test"
].tolist()

assert not failures, (
    "Repair validation failed: "
    + "; ".join(failures)
)

# --------------------------------------------------
# 10. Back up and save only after tests pass
# --------------------------------------------------

assert module_path.read_bytes() == original_bytes

if revised_source != original_source:

    backup_path.parent.mkdir(
        parents=True,
        exist_ok=True
    )

    if backup_path.exists():
        assert backup_path.read_bytes() == (
            original_bytes
        ), (
            "Existing backup differs. "
            "No overwrite performed."
        )
    else:
        backup_path.write_bytes(
            original_bytes
        )

    temp_path = module_path.with_suffix(
        ".phase18_1d.tmp"
    )

    assert not temp_path.exists(), (
        "Temporary file already exists. "
        "No overwrite performed."
    )

    temp_path.write_bytes(
        revised_source.encode("utf-8")
    )

    assert temp_path.read_bytes() == (
        revised_source.encode("utf-8")
    )

    temp_path.replace(module_path)

# --------------------------------------------------
# 11. Reload the corrected module
# --------------------------------------------------

import src.models.new_batch_area_kmeans as km_module

importlib.invalidate_caches()
importlib.reload(km_module)

assert hasattr(
    km_module,
    "analyze_new_batch_areas"
)

# Confirm the saved import has the new behavior.
saved_check = km_module.analyze_new_batch_areas(
    three_area_data,
    categories,
    k=3
)

assert saved_check["status"] == (
    "insufficient_areas_for_validation"
)

print("\n8. VERIFIED REPAIR")
print("Module:", module_path)

if backup_path.exists():
    print("Backup:", backup_path)

print("\nSTATUS: PASSED")
print("ROBUSTNESS REPAIR VERIFIED")
print("EXACTLY-K SILHOUETTE SAFEGUARD WORKING")
print("SMALL-AREA REASONS PRESERVED")
print("EXISTING HISTORICAL MODELS UNMODIFIED")


SAFE K-MEANS REPAIR

1. ORIGINAL MODULE
File: D:\Urban-Infrastructure-Pulse-Intelligence\src\models\new_batch_area_kmeans.py
Bytes: 11568

2. PYTHON STATEMENTS FOUND
Area threshold: enough = area_sizes.ge(min_area_size)
Pattern count: unique_patterns = np.unique(
        features.to_numpy(dtype=float),
        axis=0
    ).shape[0]

3. PATCH PREPARATION
New corrections required: 2
Corrected module compiles: YES
Original file still unchanged: YES

4. FULL BATCH REGRESSION
Status: clustered
Silhouette: 0.97664289820422

5. EXACTLY-K SAFEGUARD
Status: insufficient_areas_for_validation
Reason: Need at least 4 qualifying areas to evaluate K=3 safely.

6. SMALL-AREA SAFEGUARD


,source_row,complaint_id,area_id,eligibility_reason
0,2,SIM-20261109-0001,BRONX_10451,below_minimum_area_size
1,3,SIM-20261109-0002,BRONX_10451,below_minimum_area_size
2,4,SIM-20261109-0003,BRONX_10451,below_minimum_area_size



7. REPAIR VALIDATION


,test,passed
0,500 records retained,True
1,20 areas eligible,True
2,K=3 clustering succeeds,True
3,Original silhouette preserved,True
4,Expected cluster sizes preserved,True
5,Historical K-Means profiles retained,True
6,Exactly K areas handled safely,True
7,No model fitted for exactly K areas,True
8,Three areas retained in the summary,True
9,Three small-area records flagged,True



Checks passed: 14/14

8. VERIFIED REPAIR
Module: D:\Urban-Infrastructure-Pulse-Intelligence\src\models\new_batch_area_kmeans.py
Backup: D:\Urban-Infrastructure-Pulse-Intelligence\outputs\tests\phase_18_1d\new_batch_area_kmeans_before_hardening.txt

STATUS: PASSED
ROBUSTNESS REPAIR VERIFIED
EXACTLY-K SILHOUETTE SAFEGUARD WORKING
SMALL-AREA REASONS PRESERVED
EXISTING HISTORICAL MODELS UNMODIFIED


### now move to dbscan functionality

In [16]:

from pathlib import Path
import hashlib
import numpy as np
import pandas as pd
from sklearn.cluster import DBSCAN

print(" NEW-BATCH DBSCAN")
print("SAME-CATEGORY SPATIAL CONCENTRATIONS")
print("=" * 72)

root = Path(PROJECT_ROOT).resolve()

input_path = (
    root / "outputs/tests"
    / "phase_17_4c_backend_verified"
    / "verified_500_backend_results.csv"
)

assert input_path.is_file()

original_hash = hashlib.sha256(
    input_path.read_bytes()
).hexdigest()

# --------------------------------------------------
# 1. Load 500 verified incoming complaints
# --------------------------------------------------

records = pd.read_csv(
    input_path,
    dtype="string",
    keep_default_na=False
).reset_index(drop=True)

assert len(records) == 500
assert records["complaint_id"].is_unique
assert records["source_row"].is_unique

assert records["fully_verified"].str.lower().eq(
    "true"
).all()

assert records["processing_status"].eq(
    "processed"
).all()

assert records["zip_validation_status"].eq(
    "zcta_consistent"
).all()

records["lat_num"] = pd.to_numeric(
    records["latitude"],
    errors="coerce"
)

records["lon_num"] = pd.to_numeric(
    records["longitude"],
    errors="coerce"
)

assert np.isfinite(
    records[["lat_num", "lon_num"]]
    .to_numpy(dtype=float)
).all()

assert records[
    ["lat_num", "lon_num"]
].drop_duplicates().shape[0] == 500

print("\n1. DATASET INFORMATION")
print("Complaints:", len(records))
print("Boroughs:", records["borough"].nunique())
print("Categories:",
      records["complaint_category"].nunique())

print("\nHistorical DBSCAN associations:")
print(records["dbscan_status"].value_counts())

# --------------------------------------------------
# 2. Reusable experimental DBSCAN function
# --------------------------------------------------

EARTH_RADIUS_M = 6371008.8

def run_spatial_dbscan(
    frame,
    eps_m,
    min_samples,
    same_category=True
):
    """Identify candidate new-batch concentrations.

    IDs are local to this batch and configuration.
    They are not historical hotspot identifiers.
    """

    frame = frame.reset_index(drop=True).copy()

    candidate_ids = np.full(
        len(frame),
        None,
        dtype=object
    )

    if same_category:

        groups = [
            (category, group.index.to_numpy())
            for category, group in frame.groupby(
                "complaint_category",
                sort=True
            )
        ]

    else:

        groups = [(
            "ALL_CATEGORIES",
            frame.index.to_numpy()
        )]

    for category_number, (
        category, indices
    ) in enumerate(groups):

        if len(indices) < min_samples:
            continue

        coords = frame.loc[
            indices,
            ["lat_num", "lon_num"]
        ].to_numpy(dtype=float)

        coords_rad = np.radians(coords)

        model = DBSCAN(
            eps=eps_m / EARTH_RADIUS_M,
            min_samples=min_samples,
            metric="haversine",
            algorithm="ball_tree"
        )

        labels = model.fit_predict(
            coords_rad
        )

        for cluster_label in sorted(
            set(labels) - {-1}
        ):

            selected = indices[
                labels == cluster_label
            ]

            identifier = (
                f"NB-{category_number:02d}"
                f"-{int(cluster_label):03d}"
            )

            candidate_ids[selected] = identifier

    frame["new_batch_candidate_id"] = (
        candidate_ids
    )

    frame["new_batch_status"] = np.where(
        pd.notna(candidate_ids),
        "candidate_concentration",
        "unclustered"
    )

    return frame


# --------------------------------------------------
# 3. Compare experimental parameters
# --------------------------------------------------

print("\n2. PARAMETER SENSITIVITY AUDIT")

settings = [
    (75, 5),
    (100, 5),
    (125, 5),
    (150, 5),
    (125, 8),
    (150, 8)
]

experiment_rows = []

for eps_m, min_samples in settings:

    result = run_spatial_dbscan(
        records,
        eps_m=eps_m,
        min_samples=min_samples,
        same_category=True
    )

    counts = (
        result["new_batch_candidate_id"]
        .value_counts()
    )

    experiment_rows.append({
        "eps_m": eps_m,
        "min_samples": min_samples,
        "candidate_concentrations": len(counts),
        "clustered_complaints": int(
            result["new_batch_candidate_id"]
            .notna().sum()
        ),
        "unclustered_complaints": int(
            result["new_batch_candidate_id"]
            .isna().sum()
        ),
        "largest_candidate": (
            int(counts.max())
            if len(counts) else 0
        )
    })

parameter_audit_18_2a = pd.DataFrame(
    experiment_rows
)

display(parameter_audit_18_2a)

# --------------------------------------------------
# 4. Select an exploratory configuration
# --------------------------------------------------

# These are testing parameters, NOT validated
# operational thresholds.

EPS_M = 125
MIN_SAMPLES = 5

spatial_results_18_2a = run_spatial_dbscan(
    records,
    eps_m=EPS_M,
    min_samples=MIN_SAMPLES,
    same_category=True
)

print("\n3. SELECTED EXPERIMENT")
print("Radius:", EPS_M, "metres")
print("Minimum reports:", MIN_SAMPLES)

display(
    spatial_results_18_2a[
        "new_batch_status"
    ].value_counts().rename_axis(
        "status"
    ).reset_index(name="complaints")
)

# --------------------------------------------------
# 5. Compare historical and incoming results
# --------------------------------------------------

print("\n4. HISTORICAL VS NEW-BATCH DBSCAN")

comparison_18_2a = pd.crosstab(
    spatial_results_18_2a[
        "dbscan_status"
    ],
    spatial_results_18_2a[
        "new_batch_status"
    ]
)

display(comparison_18_2a)

# --------------------------------------------------
# 6. Summarize local candidate concentrations
# --------------------------------------------------

candidates = spatial_results_18_2a.loc[
    spatial_results_18_2a[
        "new_batch_status"
    ].eq("candidate_concentration")
].copy()

candidates["coordinate_key"] = (
    candidates["latitude"].astype(str)
    + "|"
    + candidates["longitude"].astype(str)
)

candidates["historical_established"] = (
    candidates["dbscan_status"].eq(
        "established_cluster"
    )
)

candidates["historical_outside"] = (
    candidates["dbscan_status"].eq(
        "outside_existing_clusters"
    )
)

candidates["historical_ambiguous"] = (
    candidates["dbscan_status"].eq(
        "ambiguous_association"
    )
)

if candidates.empty:

    candidate_profiles_18_2a = pd.DataFrame(
        columns=[
            "new_batch_candidate_id",
            "complaint_category",
            "incoming_complaints",
            "distinct_locations",
            "boroughs",
            "historical_established",
            "historical_outside",
            "historical_ambiguous",
            "center_latitude",
            "center_longitude"
        ]
    )

else:

    candidate_profiles_18_2a = (
        candidates.groupby(
            "new_batch_candidate_id"
        )
        .agg(
            complaint_category=(
                "complaint_category",
                "first"
            ),
            incoming_complaints=(
                "complaint_id",
                "size"
            ),
            distinct_locations=(
                "coordinate_key",
                "nunique"
            ),
            boroughs=(
                "borough",
                "nunique"
            ),
            historical_established=(
                "historical_established",
                "sum"
            ),
            historical_outside=(
                "historical_outside",
                "sum"
            ),
            historical_ambiguous=(
                "historical_ambiguous",
                "sum"
            ),
            center_latitude=(
                "lat_num",
                "mean"
            ),
            center_longitude=(
                "lon_num",
                "mean"
            )
        )
        .reset_index()
        .sort_values(
            [
                "incoming_complaints",
                "new_batch_candidate_id"
            ],
            ascending=[False, True]
        )
        .reset_index(drop=True)
    )

print("\n5. NEW-BATCH CANDIDATE SUMMARIES")

display(
    candidate_profiles_18_2a.head(25)
)

# --------------------------------------------------
# 7. Describe locations outside historical hotspots
# --------------------------------------------------

print("\n6. CANDIDATES OUTSIDE HISTORICAL HOTSPOTS")

outside_candidates = (
    candidate_profiles_18_2a.loc[
        candidate_profiles_18_2a[
            "historical_outside"
        ].gt(0)
    ]
)

display(
    outside_candidates[[
        "new_batch_candidate_id",
        "complaint_category",
        "incoming_complaints",
        "historical_outside",
        "historical_established",
        "center_latitude",
        "center_longitude"
    ]]
)

# --------------------------------------------------
# 8. Validate all outputs
# --------------------------------------------------

checks = []

def check(name, condition):
    checks.append({
        "test": name,
        "passed": bool(condition)
    })

check(
    "All 500 complaints retained",
    len(spatial_results_18_2a) == 500
)

check(
    "All complaint IDs remain unique",
    spatial_results_18_2a[
        "complaint_id"
    ].is_unique
)

check(
    "Six parameter configurations tested",
    len(parameter_audit_18_2a) == 6
)

check(
    "Every configuration accounts for 500 inputs",
    (
        parameter_audit_18_2a[
            "clustered_complaints"
        ]
        + parameter_audit_18_2a[
            "unclustered_complaints"
        ]
    ).eq(500).all()
)

check(
    "No complaints disappear in selected DBSCAN",
    spatial_results_18_2a[
        "new_batch_status"
    ].value_counts().sum() == 500
)

check(
    "Candidate assignments are traceable",
    spatial_results_18_2a.loc[
        spatial_results_18_2a[
            "new_batch_status"
        ].eq("candidate_concentration"),
        "new_batch_candidate_id"
    ].notna().all()
)

check(
    "Unclustered records have no candidate ID",
    spatial_results_18_2a.loc[
        spatial_results_18_2a[
            "new_batch_status"
        ].eq("unclustered"),
        "new_batch_candidate_id"
    ].isna().all()
)

check(
    "All candidates meet minimum report count",
    candidate_profiles_18_2a[
        "incoming_complaints"
    ].ge(MIN_SAMPLES).all()
)

check(
    "No candidate combines different categories",
    (
        candidates.groupby(
            "new_batch_candidate_id"
        )["complaint_category"]
        .nunique()
        .le(1)
        .all()
    )
)

check(
    "Candidate counts reconcile with records",
    int(
        candidate_profiles_18_2a[
            "incoming_complaints"
        ].sum()
    ) == len(candidates)
)

check(
    "Historical established count preserved",
    spatial_results_18_2a[
        "dbscan_status"
    ].eq("established_cluster").sum() == 137
)

check(
    "Historical ambiguous count preserved",
    spatial_results_18_2a[
        "dbscan_status"
    ].eq("ambiguous_association").sum() == 9
)

check(
    "Historical outside-cluster count preserved",
    spatial_results_18_2a[
        "dbscan_status"
    ].eq("outside_existing_clusters").sum() == 354
)

check(
    "Comparison table accounts for all 500",
    int(comparison_18_2a.to_numpy().sum()) == 500
)

check(
    "Candidate location counts cannot exceed reports",
    (
        candidate_profiles_18_2a[
            "distinct_locations"
        ]
        <= candidate_profiles_18_2a[
            "incoming_complaints"
        ]
    ).all()
)

check(
    "Verified source CSV remains unchanged",
    hashlib.sha256(
        input_path.read_bytes()
    ).hexdigest() == original_hash
)

# --------------------------------------------------
# 9. Final verification
# --------------------------------------------------

results_18_2a = pd.DataFrame(checks)

print("\n7. VALIDATION")

display(results_18_2a)

passed = int(
    results_18_2a["passed"].sum()
)

total = len(results_18_2a)

print(f"\nChecks passed: {passed}/{total}")

failures = results_18_2a.loc[
    ~results_18_2a["passed"],
    "test"
].tolist()

if failures:

    print("\nSTATUS: NEEDS REVIEW")

    for failure in failures:
        print("FAILED:", failure)

else:

    print("\nSTATUS: PASSED")
    print("NEW-BATCH DBSCAN VERIFIED")
    print("SAME-CATEGORY SPATIAL CONCENTRATIONS ANALYZED")
    print("HISTORICAL ASSOCIATIONS PRESERVED")
    print("ALL 500 COMPLAINTS ACCOUNTED FOR")
    print("NO MODEL ARTIFACTS MODIFIED")
    print("NOT YET DEPLOYED")


 NEW-BATCH DBSCAN
SAME-CATEGORY SPATIAL CONCENTRATIONS

1. DATASET INFORMATION
Complaints: 500
Boroughs: 5
Categories: 8

Historical DBSCAN associations:
dbscan_status
outside_existing_clusters    354
established_cluster          137
ambiguous_association          9
Name: count, dtype: int64[pyarrow]

2. PARAMETER SENSITIVITY AUDIT


,eps_m,min_samples,candidate_concentrations,clustered_complaints,unclustered_complaints,largest_candidate
0,75,5,8,45,455,6
1,100,5,23,155,345,11
2,125,5,25,236,264,15
3,150,5,22,278,222,15
4,125,8,7,68,432,14
5,150,8,17,201,299,15



3. SELECTED EXPERIMENT
Radius: 125 metres
Minimum reports: 5


,status,complaints
0,unclustered,264
1,candidate_concentration,236



4. HISTORICAL VS NEW-BATCH DBSCAN


new_batch_status,candidate_concentration,unclustered
dbscan_status,,
ambiguous_association,5,4
established_cluster,65,72
outside_existing_clusters,166,188


... (3 outputs left out)

,new_batch_candidate_id,complaint_category,incoming_complaints,historical_outside,historical_established,center_latitude,center_longitude
0,NB-00-000,SEWER,15,12,3,40.852681,-73.9131
2,NB-00-004,SEWER,15,15,0,40.7561,-73.714045
3,NB-03-010,STREET CONDITION,14,14,0,40.624842,-74.095954
4,NB-03-004,STREET CONDITION,13,13,0,40.649676,-73.933741
5,NB-03-005,STREET CONDITION,13,10,3,40.751816,-74.00489
7,NB-03-003,STREET CONDITION,12,2,10,40.693897,-73.989946
8,NB-04-002,STREET LIGHT CONDITION,12,12,0,40.688507,-74.017779
9,NB-00-001,SEWER,11,5,3,40.620836,-73.985568
10,NB-03-011,STREET CONDITION,11,11,0,40.629499,-74.136945
11,NB-04-003,STREET LIGHT CONDITION,11,11,0,40.747215,-73.702296



7. VALIDATION


,test,passed
0,All 500 complaints retained,True
1,All complaint IDs remain unique,True
2,Six parameter configurations tested,True
3,Every configuration accounts for 500 inputs,True
4,No complaints disappear in selected DBSCAN,True
5,Candidate assignments are traceable,True
6,Unclustered records have no candidate ID,True
7,All candidates meet minimum report count,False
8,No candidate combines different categories,True
9,Candidate counts reconcile with records,True



Checks passed: 15/16

STATUS: NEEDS REVIEW
FAILED: All candidates meet minimum report count


esults are encouraging: 15 of 16 checks passed, and the new-batch DBSCAN detected 25 raw spatial clusters containing 236 simulated complaints.

The failed test is:

All candidates meet minimum report count

Two clusters caused it:
Candidate	Category	Reports
NB-00-002	SEWER	4
NB-03-001	STREET CONDITION	4

Why did this happen?
In DBSCAN, min_samples=5 means a core point must have at least five points in its neighborhood, including itself. It does not strictly guarantee that every final cluster contains five members. Border-point assignments can result in a smaller cluster. 
scikit-learn 1.9.1 documentation+1


So the DBSCAN algorithm is not necessarily wrong. Our test incorrectly assumed that min_samples was also a minimum final cluster size.



This preserves the raw DBSCAN result while preventing four-report groups from being presented as five-report investigation candidates.

Under this adjustment, we expect:
Outcome	Reports
Reportable candidate concentrations	228
Below reporting threshold	8
Unclustered by raw DBSCAN	264
Total	500

These are still synthetic experimental results, not confirmed infrastructure incidents

In [17]:

from pathlib import Path
import hashlib
import numpy as np
import pandas as pd

print(" DBSCAN REPORTING THRESHOLD REPAIR")
print("=" * 72)

root = Path(PROJECT_ROOT).resolve()

input_path = (
    root / "outputs/tests"
    / "phase_17_4c_backend_verified"
    / "verified_500_backend_results.csv"
)

assert input_path.is_file()

assert "run_spatial_dbscan" in globals(), (
    "Run the original Phase 18.2A cell first."
)

source_hash = hashlib.sha256(
    input_path.read_bytes()
).hexdigest()

# --------------------------------------------------
# 1. Load the verified synthetic complaints
# --------------------------------------------------

records = pd.read_csv(
    input_path,
    dtype="string",
    keep_default_na=False
)

records["lat_num"] = pd.to_numeric(
    records["latitude"]
)

records["lon_num"] = pd.to_numeric(
    records["longitude"]
)

assert len(records) == 500
assert records["complaint_id"].is_unique
assert records["fully_verified"].str.lower().eq(
    "true"
).all()

# --------------------------------------------------
# 2. Preserve raw DBSCAN, filter reporting candidates
# --------------------------------------------------

def run_reportable_dbscan(
    frame,
    eps_m,
    min_samples,
    min_report_count=5
):
    """
    Preserve raw DBSCAN memberships.

    A separate minimum group size determines
    which clusters become reportable candidates.
    """

    if min_report_count < 2:
        raise ValueError(
            "Minimum reporting size must be >= 2."
        )

    result = run_spatial_dbscan(
        frame,
        eps_m=eps_m,
        min_samples=min_samples,
        same_category=True
    ).copy()

    # Keep original cluster assignments.
    result["raw_candidate_id"] = (
        result["new_batch_candidate_id"]
        .copy()
    )

    raw_counts = (
        result["raw_candidate_id"]
        .value_counts()
    )

    undersized_ids = set(
        raw_counts.loc[
            raw_counts < min_report_count
        ].index
    )

    undersized = result[
        "raw_candidate_id"
    ].isin(undersized_ids)

    # Suppress these from reported candidates
    # without calling them original DBSCAN noise.
    result.loc[
        undersized,
        "new_batch_candidate_id"
    ] = None

    result["new_batch_status"] = "unclustered"

    result.loc[
        undersized,
        "new_batch_status"
    ] = "below_reporting_threshold"

    result.loc[
        result["new_batch_candidate_id"]
        .notna(),
        "new_batch_status"
    ] = "candidate_concentration"

    return result


# --------------------------------------------------
# 3. Re-evaluate selected configuration
# --------------------------------------------------

EPS_M = 125
MIN_SAMPLES = 5
MIN_REPORT_COUNT = 5

spatial_results_18_2a = run_reportable_dbscan(
    records,
    eps_m=EPS_M,
    min_samples=MIN_SAMPLES,
    min_report_count=MIN_REPORT_COUNT
)

print("\n1. CORRECTED DBSCAN OUTCOMES")

status_counts = (
    spatial_results_18_2a[
        "new_batch_status"
    ].value_counts()
)

display(
    status_counts.rename_axis(
        "status"
    ).reset_index(name="complaints")
)

print("\n2. ORIGINAL VS REPORTABLE CLUSTERS")

raw_counts = (
    spatial_results_18_2a[
        "raw_candidate_id"
    ].value_counts()
)

reportable_counts = (
    spatial_results_18_2a[
        "new_batch_candidate_id"
    ].value_counts()
)

print("Raw DBSCAN clusters:", len(raw_counts))
print("Reportable candidates:", len(reportable_counts))
print(
    "Clusters below reporting threshold:",
    int((raw_counts < MIN_REPORT_COUNT).sum())
)

print(
    "Reports below reporting threshold:",
    int(raw_counts[
        raw_counts < MIN_REPORT_COUNT
    ].sum())
)

# --------------------------------------------------
# 4. Inspect undersized raw DBSCAN clusters
# --------------------------------------------------

below_threshold = (
    spatial_results_18_2a.loc[
        spatial_results_18_2a[
            "new_batch_status"
        ].eq("below_reporting_threshold")
    ].copy()
)

print("\n3. SUPPRESSED REPORTING CANDIDATES")

display(
    below_threshold.groupby([
        "raw_candidate_id",
        "complaint_category"
    ])
    .size()
    .rename("reports")
    .reset_index()
)

# --------------------------------------------------
# 5. Rebuild reportable candidate summaries
# --------------------------------------------------

reportable = spatial_results_18_2a.loc[
    spatial_results_18_2a[
        "new_batch_status"
    ].eq("candidate_concentration")
].copy()

reportable["location_key"] = (
    reportable["latitude"].astype(str)
    + "|"
    + reportable["longitude"].astype(str)
)

reportable["historical_outside"] = (
    reportable["dbscan_status"].eq(
        "outside_existing_clusters"
    )
)

reportable["historical_established"] = (
    reportable["dbscan_status"].eq(
        "established_cluster"
    )
)

reportable["historical_ambiguous"] = (
    reportable["dbscan_status"].eq(
        "ambiguous_association"
    )
)

candidate_profiles_18_2a = (
    reportable.groupby(
        "new_batch_candidate_id"
    )
    .agg(
        complaint_category=(
            "complaint_category", "first"
        ),
        incoming_complaints=(
            "complaint_id", "size"
        ),
        distinct_locations=(
            "location_key", "nunique"
        ),
        historical_outside=(
            "historical_outside", "sum"
        ),
        historical_established=(
            "historical_established", "sum"
        ),
        historical_ambiguous=(
            "historical_ambiguous", "sum"
        ),
        center_latitude=(
            "lat_num", "mean"
        ),
        center_longitude=(
            "lon_num", "mean"
        )
    )
    .reset_index()
    .sort_values(
        [
            "incoming_complaints",
            "new_batch_candidate_id"
        ],
        ascending=[False, True]
    )
    .reset_index(drop=True)
)

candidate_profiles_18_2a[
    "entirely_outside_historical_clusters"
] = (
    candidate_profiles_18_2a[
        "historical_outside"
    ].eq(
        candidate_profiles_18_2a[
            "incoming_complaints"
        ]
    )
)

print("\n4. REPORTABLE CONCENTRATIONS")

display(candidate_profiles_18_2a.head(25))

# --------------------------------------------------
# 6. Preserve historical DBSCAN comparison
# --------------------------------------------------

print("\n5. HISTORICAL VS NEW DBSCAN")

comparison_18_2a = pd.crosstab(
    spatial_results_18_2a[
        "dbscan_status"
    ],
    spatial_results_18_2a[
        "new_batch_status"
    ]
)

display(comparison_18_2a)

# --------------------------------------------------
# 7. Re-evaluate all parameter settings
# --------------------------------------------------

print("\n6. CORRECTED PARAMETER COMPARISON")

settings = [
    (75, 5),
    (100, 5),
    (125, 5),
    (150, 5),
    (125, 8),
    (150, 8)
]

parameter_rows = []

for eps_m, min_samples in settings:

    experiment = run_reportable_dbscan(
        records,
        eps_m=eps_m,
        min_samples=min_samples,
        min_report_count=min_samples
    )

    statuses = experiment[
        "new_batch_status"
    ]

    parameter_rows.append({
        "eps_m": eps_m,
        "min_samples": min_samples,
        "reporting_minimum": min_samples,
        "raw_clusters": experiment[
            "raw_candidate_id"
        ].nunique(),
        "reportable_clusters": experiment[
            "new_batch_candidate_id"
        ].nunique(),
        "reportable_complaints": int(
            statuses.eq(
                "candidate_concentration"
            ).sum()
        ),
        "below_reporting_threshold": int(
            statuses.eq(
                "below_reporting_threshold"
            ).sum()
        ),
        "raw_unclustered": int(
            statuses.eq("unclustered").sum()
        )
    })

parameter_audit_18_2a = pd.DataFrame(
    parameter_rows
)

display(parameter_audit_18_2a)

# --------------------------------------------------
# 8. Verify corrected results
# --------------------------------------------------

checks = []

def check(name, condition):
    checks.append({
        "test": name,
        "passed": bool(condition)
    })

check(
    "All 500 records retained",
    len(spatial_results_18_2a) == 500
)

check(
    "Original complaint IDs preserved",
    spatial_results_18_2a[
        "complaint_id"
    ].tolist() == records[
        "complaint_id"
    ].tolist()
)

check(
    "25 raw DBSCAN clusters preserved",
    len(raw_counts) == 25
)

check(
    "Two undersized clusters identified",
    (raw_counts < 5).sum() == 2
)

check(
    "Eight reports flagged below threshold",
    len(below_threshold) == 8
)

check(
    "23 reportable candidates retained",
    len(candidate_profiles_18_2a) == 23
)

check(
    "228 complaints in reportable candidates",
    len(reportable) == 228
)

check(
    "264 original DBSCAN unclustered reports",
    status_counts.get("unclustered", 0) == 264
)

check(
    "Every reportable candidate has >= 5 reports",
    candidate_profiles_18_2a[
        "incoming_complaints"
    ].ge(5).all()
)

check(
    "Suppressed clusters retain raw IDs",
    below_threshold[
        "raw_candidate_id"
    ].notna().all()
)

check(
    "Suppressed clusters have no reportable IDs",
    below_threshold[
        "new_batch_candidate_id"
    ].isna().all()
)

check(
    "All reportable groups contain one category",
    reportable.groupby(
        "new_batch_candidate_id"
    )["complaint_category"].nunique().eq(1).all()
)

check(
    "All reporting statuses account for 500",
    int(status_counts.sum()) == 500
)

check(
    "137 historical established associations retained",
    spatial_results_18_2a[
        "dbscan_status"
    ].eq("established_cluster").sum() == 137
)

check(
    "Nine historical ambiguous associations retained",
    spatial_results_18_2a[
        "dbscan_status"
    ].eq("ambiguous_association").sum() == 9
)

check(
    "354 historical outside-cluster reports retained",
    spatial_results_18_2a[
        "dbscan_status"
    ].eq("outside_existing_clusters").sum() == 354
)

check(
    "All six parameter experiments reconciled",
    len(parameter_audit_18_2a) == 6
    and (
        parameter_audit_18_2a[
            "reportable_complaints"
        ]
        + parameter_audit_18_2a[
            "below_reporting_threshold"
        ]
        + parameter_audit_18_2a[
            "raw_unclustered"
        ]
    ).eq(500).all()
)

check(
    "Original verified input remains unchanged",
    hashlib.sha256(
        input_path.read_bytes()
    ).hexdigest() == source_hash
)

# --------------------------------------------------
# 9. Final results
# --------------------------------------------------

results_18_2a_fix = pd.DataFrame(checks)

print("\n7. VALIDATION")

display(results_18_2a_fix)

passed = int(
    results_18_2a_fix["passed"].sum()
)

total = len(results_18_2a_fix)

print(f"\nChecks passed: {passed}/{total}")

failures = results_18_2a_fix.loc[
    ~results_18_2a_fix["passed"],
    "test"
].tolist()

if failures:
    print("\nSTATUS: NEEDS REVIEW")

    for failure in failures:
        print("FAILED:", failure)

else:
    print("\nSTATUS: PASSED")
    print("DBSCAN REPORTING FIX VERIFIED")
    print("RAW DBSCAN CLUSTERS PRESERVED")
    print("MINIMUM REPORTABLE SIZE ENFORCED")
    print("HISTORICAL ASSOCIATIONS PRESERVED")
    print("NO MODEL FILES MODIFIED")


 DBSCAN REPORTING THRESHOLD REPAIR

1. CORRECTED DBSCAN OUTCOMES


,status,complaints
0,unclustered,264
1,candidate_concentration,228
2,below_reporting_threshold,8



2. ORIGINAL VS REPORTABLE CLUSTERS
Raw DBSCAN clusters: 25
Reportable candidates: 23
Clusters below reporting threshold: 2
Reports below reporting threshold: 8

3. SUPPRESSED REPORTING CANDIDATES


,raw_candidate_id,complaint_category,reports
0,NB-00-002,SEWER,4
1,NB-03-001,STREET CONDITION,4



4. REPORTABLE CONCENTRATIONS


,new_batch_candidate_id,complaint_category,incoming_complaints,distinct_locations,historical_outside,historical_established,historical_ambiguous,center_latitude,center_longitude,entirely_outside_historical_clusters
0,NB-00-000,SEWER,15,15,12,3,0,40.852681,-73.9131,False
1,NB-00-003,SEWER,15,15,0,14,1,40.731525,-73.990998,False
2,NB-00-004,SEWER,15,15,15,0,0,40.7561,-73.714045,True
3,NB-03-010,STREET CONDITION,14,14,14,0,0,40.624842,-74.095954,True
4,NB-03-004,STREET CONDITION,13,13,13,0,0,40.649676,-73.933741,True
5,NB-03-005,STREET CONDITION,13,13,10,3,0,40.751816,-74.00489,False
6,NB-03-006,STREET CONDITION,13,13,0,13,0,40.717177,-73.985518,False
7,NB-03-003,STREET CONDITION,12,12,2,10,0,40.693897,-73.989946,False
8,NB-04-002,STREET LIGHT CONDITION,12,12,12,0,0,40.688507,-74.017779,True
9,NB-00-001,SEWER,11,11,5,3,3,40.620836,-73.985568,False


... (3 outputs left out)

,eps_m,min_samples,reporting_minimum,raw_clusters,reportable_clusters,reportable_complaints,below_reporting_threshold,raw_unclustered
0,75,5,5,8,8,45,0,455
1,100,5,5,23,22,152,3,345
2,125,5,5,25,23,228,8,264
3,150,5,5,22,22,278,0,222
4,125,8,8,7,7,68,0,432
5,150,8,8,17,17,201,0,299



7. VALIDATION


,test,passed
0,All 500 records retained,True
1,Original complaint IDs preserved,True
2,25 raw DBSCAN clusters preserved,True
3,Two undersized clusters identified,True
4,Eight reports flagged below threshold,True
5,23 reportable candidates retained,True
6,228 complaints in reportable candidates,True
7,264 original DBSCAN unclustered reports,True
8,Every reportable candidate has >= 5 reports,True
9,Suppressed clusters retain raw IDs,True



Checks passed: 18/18

STATUS: PASSED
DBSCAN REPORTING FIX VERIFIED
RAW DBSCAN CLUSTERS PRESERVED
MINIMUM REPORTABLE SIZE ENFORCED
HISTORICAL ASSOCIATIONS PRESERVED
NO MODEL FILES MODIFIED


### REUSABLE NEW-BATCH DBSCAN ENGINE

In [18]:

from pathlib import Path
from textwrap import dedent
from threadpoolctl import threadpool_limits
import sys
import hashlib
import numpy as np
import pandas as pd

print(" REUSABLE NEW-BATCH DBSCAN ENGINE")
print("=" * 72)

root = Path(PROJECT_ROOT).resolve()

if str(root) not in sys.path:
    sys.path.insert(0, str(root))

module_path = (
    root / "src/models/new_batch_dbscan.py"
)

input_path = (
    root / "outputs/tests"
    / "phase_17_4c_backend_verified"
    / "verified_500_backend_results.csv"
)

taxonomy_path = (
    root / "outputs/tables/dbscan"
    / "dbscan_action_taxonomy_v1.csv"
)

assert input_path.is_file()
assert taxonomy_path.is_file()

# --------------------------------------------------
# 1. Create reusable new-batch DBSCAN engine
# --------------------------------------------------

module_source = dedent(r'''
"""New-batch same-category spatial concentration analysis.

This engine never retrains historical DBSCAN.
Candidates are not confirmed infrastructure defects.

Candidate IDs are local to the batch and parameters.
"""

import numpy as np
import pandas as pd
from sklearn.cluster import DBSCAN

EARTH_RADIUS_M = 6371008.8

BOROUGHS = {
    "BRONX",
    "BROOKLYN",
    "MANHATTAN",
    "QUEENS",
    "STATEN ISLAND"
}

REQUIRED = {
    "source_row",
    "complaint_id",
    "latitude",
    "longitude",
    "borough",
    "complaint_category",
    "processing_status",
    "dbscan_status"
}


def analyze_new_batch_spatial(
    records,
    supported_categories,
    eps_m=125,
    min_samples=5,
    min_report_count=5,
    min_distinct_locations=2
):
    """Analyze already processed geographic complaint records.

    ZIP consistency is not required for spatial
    analysis when geographic processing has succeeded.

    Time-window selection must be done upstream.
    """

    if not isinstance(records, pd.DataFrame):
        raise TypeError("Expected pandas DataFrame.")

    if records.columns.duplicated().any():
        raise ValueError("Duplicate input columns.")

    missing = REQUIRED - set(records.columns)

    if missing:
        raise ValueError(
            f"Missing columns: {sorted(missing)}"
        )

    if records["source_row"].duplicated().any():
        raise ValueError("Duplicate source rows.")

    if (
        isinstance(eps_m, bool)
        or not isinstance(eps_m, (int, float))
        or not np.isfinite(eps_m)
        or eps_m <= 0
    ):
        raise ValueError(
            "eps_m must be finite and positive."
        )

    parameters = [
        ("min_samples", min_samples, 2),
        ("min_report_count", min_report_count, 2),
        (
            "min_distinct_locations",
            min_distinct_locations,
            1
        )
    ]

    for name, value, minimum in parameters:
        if (
            isinstance(value, bool)
            or not isinstance(value, int)
            or value < minimum
        ):
            raise ValueError(
                f"{name} must be an integer >= {minimum}."
            )

    categories = {
        str(category).strip().upper()
        for category in supported_categories
        if pd.notna(category)
        and str(category).strip()
    }

    if not categories:
        raise ValueError(
            "Supported categories are empty."
        )

    # ----------------------------------------------
    # 2. Normalize geographic inputs
    # ----------------------------------------------

    work = records.copy().reset_index(drop=True)

    for column in [
        "borough",
        "complaint_category",
        "processing_status",
        "dbscan_status"
    ]:
        work[column] = (
            work[column]
            .fillna("")
            .astype(str)
            .str.strip()
        )

    work["borough"] = work[
        "borough"
    ].str.upper()

    work["complaint_category"] = work[
        "complaint_category"
    ].str.upper()

    work["lat_num"] = pd.to_numeric(
        work["latitude"],
        errors="coerce"
    )

    work["lon_num"] = pd.to_numeric(
        work["longitude"],
        errors="coerce"
    )

    lat = work["lat_num"].to_numpy(
        dtype=float,
        na_value=np.nan
    )

    lon = work["lon_num"].to_numpy(
        dtype=float,
        na_value=np.nan
    )

    invalid_coords = (
        ~np.isfinite(lat)
        | ~np.isfinite(lon)
        | (lat < -90)
        | (lat > 90)
        | (lon < -180)
        | (lon > 180)
    )

    # ----------------------------------------------
    # 3. Preserve all excluded records with reasons
    # ----------------------------------------------

    work["record_eligibility"] = "eligible"

    rules = [
        (
            work["processing_status"].ne(
                "processed"
            ),
            "not_processed"
        ),
        (
            ~work["borough"].isin(BOROUGHS),
            "invalid_borough"
        ),
        (
            pd.Series(invalid_coords),
            "invalid_coordinates"
        ),
        (
            ~work["complaint_category"].isin(
                categories
            ),
            "unsupported_category"
        )
    ]

    for mask, reason in rules:
        work.loc[
            mask
            & work["record_eligibility"].eq(
                "eligible"
            ),
            "record_eligibility"
        ] = reason

    work["raw_candidate_id"] = pd.Series(
        [None] * len(work),
        dtype="object"
    )

    work["new_batch_candidate_id"] = pd.Series(
        [None] * len(work),
        dtype="object"
    )

    work["new_batch_status"] = "excluded_input"

    eligible = work[
        "record_eligibility"
    ].eq("eligible")

    work.loc[
        eligible, "new_batch_status"
    ] = "unclustered"

    # ----------------------------------------------
    # 4. Run DBSCAN separately for each category
    # ----------------------------------------------

    groups = (
        work.loc[eligible]
        .groupby(
            "complaint_category",
            sort=True
        ).groups
    )

    for category_index, (
        category, group_indices
    ) in enumerate(groups.items()):

        indices = np.asarray(
            list(group_indices),
            dtype=int
        )

        if len(indices) < min_samples:
            continue

        coordinates = work.loc[
            indices,
            ["lat_num", "lon_num"]
        ].to_numpy(dtype=float)

        model = DBSCAN(
            eps=float(eps_m) / EARTH_RADIUS_M,
            min_samples=min_samples,
            metric="haversine",
            algorithm="ball_tree"
        )

        labels = model.fit_predict(
            np.radians(coordinates)
        )

        for label in sorted(
            set(labels) - {-1}
        ):

            members = indices[
                labels == label
            ]

            candidate_id = (
                f"NB-{category_index:02d}"
                f"-{int(label):03d}"
            )

            work.loc[
                members,
                "raw_candidate_id"
            ] = candidate_id

    # ----------------------------------------------
    # 5. Apply independent reporting safeguards
    # ----------------------------------------------

    raw_mask = work[
        "raw_candidate_id"
    ].notna()

    raw = work.loc[raw_mask].copy()

    if len(raw):

        raw["location_pair"] = list(zip(
            raw["lat_num"],
            raw["lon_num"]
        ))

        raw_groups = (
            raw.groupby("raw_candidate_id")
            .agg(
                report_count=(
                    "source_row", "size"
                ),
                distinct_locations=(
                    "location_pair", "nunique"
                )
            )
        )

        reportable_ids = set(
            raw_groups.index[
                (
                    raw_groups["report_count"]
                    >= min_report_count
                )
                & (
                    raw_groups["distinct_locations"]
                    >= min_distinct_locations
                )
            ]
        )

    else:
        reportable_ids = set()

    # Raw clusters remain traceable even when
    # they do not meet reporting requirements.

    work.loc[
        raw_mask,
        "new_batch_status"
    ] = "below_reporting_threshold"

    reportable_mask = (
        raw_mask
        & work["raw_candidate_id"].isin(
            reportable_ids
        )
    )

    work.loc[
        reportable_mask,
        "new_batch_status"
    ] = "candidate_concentration"

    work.loc[
        reportable_mask,
        "new_batch_candidate_id"
    ] = work.loc[
        reportable_mask,
        "raw_candidate_id"
    ]

    # ----------------------------------------------
    # 6. Summarize reportable concentrations
    # ----------------------------------------------

    candidate = work.loc[
        reportable_mask
    ].copy()

    profile_columns = [
        "new_batch_candidate_id",
        "complaint_category",
        "incoming_complaints",
        "distinct_locations",
        "boroughs",
        "historical_established",
        "historical_outside",
        "historical_ambiguous",
        "center_latitude",
        "center_longitude",
        "entirely_outside_historical_clusters"
    ]

    if candidate.empty:
        profiles = pd.DataFrame(
            columns=profile_columns
        )

    else:

        candidate["location_pair"] = list(zip(
            candidate["lat_num"],
            candidate["lon_num"]
        ))

        for column, status in [
            (
                "historical_established",
                "established_cluster"
            ),
            (
                "historical_outside",
                "outside_existing_clusters"
            ),
            (
                "historical_ambiguous",
                "ambiguous_association"
            )
        ]:
            candidate[column] = (
                candidate["dbscan_status"].eq(
                    status
                )
            )

        profiles = (
            candidate.groupby(
                "new_batch_candidate_id",
                sort=True
            )
            .agg(
                complaint_category=(
                    "complaint_category",
                    "first"
                ),
                incoming_complaints=(
                    "source_row", "size"
                ),
                distinct_locations=(
                    "location_pair", "nunique"
                ),
                boroughs=(
                    "borough", "nunique"
                ),
                historical_established=(
                    "historical_established", "sum"
                ),
                historical_outside=(
                    "historical_outside", "sum"
                ),
                historical_ambiguous=(
                    "historical_ambiguous", "sum"
                ),
                center_latitude=(
                    "lat_num", "mean"
                ),
                center_longitude=(
                    "lon_num", "mean"
                )
            )
            .reset_index()
        )

        profiles[
            "entirely_outside_historical_clusters"
        ] = profiles[
            "historical_outside"
        ].eq(
            profiles["incoming_complaints"]
        )

        profiles = profiles.sort_values(
            [
                "incoming_complaints",
                "new_batch_candidate_id"
            ],
            ascending=[False, True]
        ).reset_index(drop=True)

    # ----------------------------------------------
    # 7. Account for every original record
    # ----------------------------------------------

    stats = {
        "total_records": len(work),
        "eligible_records": int(
            eligible.sum()
        ),
        "excluded_records": int(
            (~eligible).sum()
        ),
        "raw_clusters": int(
            work["raw_candidate_id"].nunique()
        ),
        "reportable_candidates": len(profiles),
        "candidate_complaints": int(
            work["new_batch_status"].eq(
                "candidate_concentration"
            ).sum()
        ),
        "below_reporting_threshold": int(
            work["new_batch_status"].eq(
                "below_reporting_threshold"
            ).sum()
        ),
        "unclustered": int(
            work["new_batch_status"].eq(
                "unclustered"
            ).sum()
        )
    }

    total_outcomes = sum([
        stats["candidate_complaints"],
        stats["below_reporting_threshold"],
        stats["unclustered"],
        stats["excluded_records"]
    ])

    if total_outcomes != len(work):
        raise AssertionError(
            "Spatial results did not reconcile."
        )

    return {
        "records": work,
        "candidate_profiles": profiles,
        "stats": stats,
        "parameters": {
            "eps_m": eps_m,
            "min_samples": min_samples,
            "min_report_count": min_report_count,
            "min_distinct_locations":
                min_distinct_locations
        }
    }
''').lstrip()

compile(
    module_source,
    str(module_path),
    "exec"
)

namespace = {}

exec(
    compile(
        module_source,
        str(module_path),
        "exec"
    ),
    namespace
)

analyze_new_batch_spatial = namespace[
    "analyze_new_batch_spatial"
]

# --------------------------------------------------
# 2. Load verified synthetic test inputs
# --------------------------------------------------

records = pd.read_csv(
    input_path,
    dtype="string",
    keep_default_na=False
)

taxonomy = pd.read_csv(taxonomy_path)

categories = sorted({
    str(value).strip().upper()
    for value in taxonomy[
        "complaint_category"
    ].dropna()
})

original_records = records.copy(deep=True)

original_hash = hashlib.sha256(
    input_path.read_bytes()
).hexdigest()

# --------------------------------------------------
# 3. Run complete 500-complaint experiment
# --------------------------------------------------

print("\n1. FULL BATCH TEST")

with threadpool_limits(limits=1):

    spatial_engine_output_18_2b = (
        analyze_new_batch_spatial(
            records,
            categories,
            eps_m=125,
            min_samples=5,
            min_report_count=5,
            min_distinct_locations=2
        )
    )

spatial_records_18_2b = (
    spatial_engine_output_18_2b["records"]
)

spatial_profiles_18_2b = (
    spatial_engine_output_18_2b[
        "candidate_profiles"
    ]
)

stats = spatial_engine_output_18_2b["stats"]

print("\nENGINE SUMMARY")

for key, value in stats.items():
    print(f"{key}: {value}")

print("\n2. REPORTABLE CANDIDATES")

display(spatial_profiles_18_2b.head(25))

# --------------------------------------------------
# 4. Test safeguards and output integrity
# --------------------------------------------------

checks = []

def check(name, condition):
    checks.append({
        "test": name,
        "passed": bool(condition)
    })

check(
    "500 records retained",
    stats["total_records"] == 500
)

check(
    "All 500 geographic inputs eligible",
    stats["eligible_records"] == 500
)

check(
    "No records unexpectedly excluded",
    stats["excluded_records"] == 0
)

check(
    "25 raw DBSCAN clusters retained",
    stats["raw_clusters"] == 25
)

check(
    "23 reportable candidates retained",
    stats["reportable_candidates"] == 23
)

check(
    "228 complaints in reportable candidates",
    stats["candidate_complaints"] == 228
)

check(
    "Eight reports below threshold",
    stats["below_reporting_threshold"] == 8
)

check(
    "264 complaints remain unclustered",
    stats["unclustered"] == 264
)

check(
    "Every candidate has at least five reports",
    spatial_profiles_18_2b[
        "incoming_complaints"
    ].ge(5).all()
)

check(
    "Every reported candidate has distinct locations",
    spatial_profiles_18_2b[
        "distinct_locations"
    ].ge(2).all()
)

check(
    "Every candidate belongs to one complaint category",
    spatial_records_18_2b.loc[
        lambda df: df[
            "new_batch_candidate_id"
        ].notna()
    ].groupby(
        "new_batch_candidate_id"
    )["complaint_category"].nunique().eq(1).all()
)

check(
    "All 500 statuses reconciled",
    (
        stats["candidate_complaints"]
        + stats["below_reporting_threshold"]
        + stats["unclustered"]
        + stats["excluded_records"]
    ) == 500
)

# --------------------------------------------------
# 5. Test repeated-coordinate suppression
# --------------------------------------------------

print("\n3. REPEATED-COORDINATE SAFEGUARD")

same_location = records.iloc[:6].copy()

same_location["latitude"] = (
    records.iloc[0]["latitude"]
)

same_location["longitude"] = (
    records.iloc[0]["longitude"]
)

same_location["complaint_category"] = (
    "STREET CONDITION"
)

with threadpool_limits(limits=1):

    repeated_result = analyze_new_batch_spatial(
        same_location,
        categories
    )

print("Repeated-location stats:", repeated_result["stats"])

check(
    "Identical coordinates can form a raw cluster",
    repeated_result["stats"]["raw_clusters"] == 1
)

check(
    "Repeated coordinates alone are not reported",
    repeated_result["stats"][
        "reportable_candidates"
    ] == 0
    and repeated_result["stats"][
        "below_reporting_threshold"
    ] == 6
)

# --------------------------------------------------
# 6. Verify multiple distinct nearby coordinates
# --------------------------------------------------

distinct_nearby = same_location.copy()

base_lat = float(
    same_location.iloc[0]["latitude"]
)

distinct_nearby["latitude"] = [
    base_lat + i * 0.00001
    for i in range(6)
]

with threadpool_limits(limits=1):

    distinct_result = analyze_new_batch_spatial(
        distinct_nearby,
        categories
    )

check(
    "Six nearby distinct locations are reportable",
    distinct_result["stats"][
        "reportable_candidates"
    ] == 1
    and distinct_result["stats"][
        "candidate_complaints"
    ] == 6
)

# --------------------------------------------------
# 7. Geographic review exclusion test
# --------------------------------------------------

geographic_review = records.iloc[:6].copy()

geographic_review[
    "processing_status"
] = "geographic_review_required"

excluded_result = analyze_new_batch_spatial(
    geographic_review,
    categories
)

check(
    "Geographic review records excluded safely",
    excluded_result["stats"][
        "excluded_records"
    ] == 6
    and excluded_result["stats"][
        "reportable_candidates"
    ] == 0
)

check(
    "Excluded records remain individually traceable",
    excluded_result["records"][
        "record_eligibility"
    ].eq("not_processed").all()
)

# --------------------------------------------------
# 8. Missing or mismatched ZIP does not block
#    otherwise valid geographic spatial analysis
# --------------------------------------------------

zip_test = records.copy()

zip_test.loc[
    0, "zip_validation_status"
] = "zcta_mismatch"

with threadpool_limits(limits=1):

    zip_result = analyze_new_batch_spatial(
        zip_test,
        categories
    )

check(
    "ZIP mismatch alone does not suppress valid coordinates",
    zip_result["stats"][
        "eligible_records"
    ] == 500
)

# --------------------------------------------------
# 9. Invalid coordinates handled without crashing
# --------------------------------------------------

bad_coordinates = records.iloc[:2].copy()

bad_coordinates.loc[
    bad_coordinates.index[0],
    "latitude"
] = "not-a-coordinate"

bad_result = analyze_new_batch_spatial(
    bad_coordinates,
    categories
)

check(
    "Invalid coordinates isolated with a reason",
    bad_result["records"].iloc[0][
        "record_eligibility"
    ] == "invalid_coordinates"
)

# --------------------------------------------------
# 10. Empty input should not crash
# --------------------------------------------------

empty_result = analyze_new_batch_spatial(
    records.iloc[:0],
    categories
)

check(
    "Empty input produces safe empty results",
    empty_result["stats"][
        "total_records"
    ] == 0
    and empty_result["candidate_profiles"].empty
)

# --------------------------------------------------
# 11. Verify original data and models untouched
# --------------------------------------------------

check(
    "Input DataFrame remains unchanged",
    records.equals(original_records)
)

check(
    "Verified CSV remains unchanged",
    hashlib.sha256(
        input_path.read_bytes()
    ).hexdigest() == original_hash
)

# --------------------------------------------------
# 12. Display validation results
# --------------------------------------------------

results_18_2b = pd.DataFrame(checks)

print("\n4. VALIDATION")

display(results_18_2b)

passed = int(results_18_2b["passed"].sum())
total = len(results_18_2b)

print(f"\nChecks passed: {passed}/{total}")

failures = results_18_2b.loc[
    ~results_18_2b["passed"],
    "test"
].tolist()

if failures:

    print("\nSTATUS: NEEDS REVIEW")
    print("No module saved.")

    for name in failures:
        print("FAILED:", name)

else:

    # ----------------------------------------------
    # 13. Save new module without overwriting work
    # ----------------------------------------------

    if module_path.exists():

        assert module_path.read_text(
            encoding="utf-8"
        ) == module_source, (
            "Existing new-batch DBSCAN module "
            "differs. No overwrite performed."
        )

    else:

        module_path.parent.mkdir(
            parents=True,
            exist_ok=True
        )

        module_path.write_text(
            module_source,
            encoding="utf-8"
        )

    print("\n5. MODULE CREATED / VERIFIED")
    print(module_path)

    print("\nSTATUS: PASSED")
    print("REUSABLE DBSCAN ENGINE VERIFIED")
    print("MINIMUM REPORTING SAFEGUARDS VERIFIED")
    print("REPEATED-COORDINATE SAFEGUARDS VERIFIED")
    print("HISTORICAL DBSCAN PRESERVED")
    print("NO AUTOMATIC REPAIR PRIORITIES")


 REUSABLE NEW-BATCH DBSCAN ENGINE

1. FULL BATCH TEST

ENGINE SUMMARY
total_records: 500
eligible_records: 500
excluded_records: 0
raw_clusters: 25
reportable_candidates: 23
candidate_complaints: 228
below_reporting_threshold: 8
unclustered: 264

2. REPORTABLE CANDIDATES


,new_batch_candidate_id,complaint_category,incoming_complaints,distinct_locations,boroughs,historical_established,historical_outside,historical_ambiguous,center_latitude,center_longitude,entirely_outside_historical_clusters
0,NB-00-000,SEWER,15,15,1,3,12,0,40.852681,-73.9131,False
1,NB-00-003,SEWER,15,15,1,14,0,1,40.731525,-73.990998,False
2,NB-00-004,SEWER,15,15,1,0,15,0,40.7561,-73.714045,True
3,NB-03-010,STREET CONDITION,14,14,1,0,14,0,40.624842,-74.095954,True
4,NB-03-004,STREET CONDITION,13,13,1,0,13,0,40.649676,-73.933741,True
5,NB-03-005,STREET CONDITION,13,13,1,3,10,0,40.751816,-74.00489,False
6,NB-03-006,STREET CONDITION,13,13,1,13,0,0,40.717177,-73.985518,False
7,NB-03-003,STREET CONDITION,12,12,1,10,2,0,40.693897,-73.989946,False
8,NB-04-002,STREET LIGHT CONDITION,12,12,1,0,12,0,40.688507,-74.017779,True
9,NB-00-001,SEWER,11,11,1,3,5,3,40.620836,-73.985568,False



3. REPEATED-COORDINATE SAFEGUARD
Repeated-location stats: {'total_records': 6, 'eligible_records': 6, 'excluded_records': 0, 'raw_clusters': 1, 'reportable_candidates': 0, 'candidate_complaints': 0, 'below_reporting_threshold': 6, 'unclustered': 0}

4. VALIDATION


,test,passed
0,500 records retained,True
1,All 500 geographic inputs eligible,True
2,No records unexpectedly excluded,True
3,25 raw DBSCAN clusters retained,True
4,23 reportable candidates retained,True
5,228 complaints in reportable candidates,True
6,Eight reports below threshold,True
7,264 complaints remain unclustered,True
8,Every candidate has at least five reports,True
9,Every reported candidate has distinct locations,True



Checks passed: 22/22

5. MODULE CREATED / VERIFIED
D:\Urban-Infrastructure-Pulse-Intelligence\src\models\new_batch_dbscan.py

STATUS: PASSED
REUSABLE DBSCAN ENGINE VERIFIED
MINIMUM REPORTING SAFEGUARDS VERIFIED
REPEATED-COORDINATE SAFEGUARDS VERIFIED
HISTORICAL DBSCAN PRESERVED
NO AUTOMATIC REPAIR PRIORITIES


### Explain Concentrations and Prepare the Map

In [19]:

from pathlib import Path
from threadpoolctl import threadpool_limits
import hashlib
import json
import sys
import numpy as np
import pandas as pd

print(" SPATIAL EXPLANATIONS AND MAP PREPARATION")
print("=" * 75)

root = Path(PROJECT_ROOT).resolve()

if str(root) not in sys.path:
    sys.path.insert(0, str(root))

from src.models.new_batch_dbscan import (
    analyze_new_batch_spatial
)

input_path = (
    root / "outputs/tests"
    / "phase_17_4c_backend_verified"
    / "verified_500_backend_results.csv"
)

taxonomy_path = (
    root / "outputs/tables/dbscan"
    / "dbscan_action_taxonomy_v1.csv"
)

module_path = (
    root / "src/models/new_batch_dbscan.py"
)

output_dir = (
    root / "outputs/tests"
    / "phase_18_2c_spatial_explanations"
)

briefs_path = output_dir / "candidate_briefs.json"
map_path = output_dir / "candidate_map_points.geojson"

for path in [input_path, taxonomy_path, module_path]:
    assert path.is_file(), f"Missing: {path}"

source_hash = hashlib.sha256(
    input_path.read_bytes()
).hexdigest()

module_hash = hashlib.sha256(
    module_path.read_bytes()
).hexdigest()

# --------------------------------------------------
# 1. Reload and analyze the same verified batch
# --------------------------------------------------

source = pd.read_csv(
    input_path,
    dtype="string",
    keep_default_na=False
)

taxonomy = pd.read_csv(taxonomy_path)

categories = sorted({
    str(value).strip().upper()
    for value in taxonomy[
        "complaint_category"
    ].dropna()
})

assert len(source) == 500
assert source["fully_verified"].str.lower().eq(
    "true"
).all()

with threadpool_limits(limits=1):
    output = analyze_new_batch_spatial(
        source,
        categories,
        eps_m=125,
        min_samples=5,
        min_report_count=5,
        min_distinct_locations=2
    )

records = output["records"]
profiles = output["candidate_profiles"]
stats = output["stats"]
parameters = output["parameters"]

print("\n1. ENGINE RECONCILIATION")
print("Total complaints:", stats["total_records"])
print("Raw clusters:", stats["raw_clusters"])
print(
    "Reportable candidates:",
    stats["reportable_candidates"]
)
print(
    "Complaints in candidates:",
    stats["candidate_complaints"]
)

# --------------------------------------------------
# 2. Distance calculation for explanatory locations
# --------------------------------------------------

EARTH_RADIUS_M = 6371008.8

def distance_metres(lat1, lon1, lat2, lon2):
    """Vectorized Haversine distance in metres."""

    lat1 = np.radians(lat1)
    lon1 = np.radians(lon1)
    lat2 = np.radians(lat2)
    lon2 = np.radians(lon2)

    dlat = lat2 - lat1
    dlon = lon2 - lon1

    a = (
        np.sin(dlat / 2) ** 2
        + np.cos(lat1)
        * np.cos(lat2)
        * np.sin(dlon / 2) ** 2
    )

    a = np.clip(a, 0.0, 1.0)

    return 2 * EARTH_RADIUS_M * np.arcsin(
        np.sqrt(a)
    )

# --------------------------------------------------
# 3. Category-specific investigation suggestions
# --------------------------------------------------

guidance = {
    "SEWER": (
        "Review sewer complaint descriptions, "
        "location accuracy and relevant "
        "maintenance history."
    ),
    "STREET CONDITION": (
        "Review street-condition descriptions "
        "to determine whether reports concern "
        "the same roadway issue."
    ),
    "STREET LIGHT CONDITION": (
        "Review lighting complaints and check "
        "whether reports refer to the same "
        "lighting assets or nearby locations."
    ),
    "TRAFFIC SIGNAL CONDITION": (
        "Review traffic-signal complaints and "
        "verify the reported intersections "
        "and equipment."
    )
}

# --------------------------------------------------
# 4. Build explanations and map-ready features
# --------------------------------------------------

reported = records.loc[
    records["new_batch_status"].eq(
        "candidate_concentration"
    )
].copy()

brief_rows = []

geojson = {
    "type": "FeatureCollection",
    "name": "synthetic_new_batch_dbscan_candidates",
    "metadata": {
        "synthetic_data": True,
        "eps_m": float(parameters["eps_m"]),
        "min_samples": int(parameters["min_samples"]),
        "min_report_count": int(
            parameters["min_report_count"]
        ),
        "coordinate_note": (
            "Each point is one submitted complaint "
            "coordinate nearest its candidate's "
            "mean location. It is not a verified "
            "infrastructure asset or defect."
        )
    },
    "features": []
}

for profile in profiles.to_dict(orient="records"):

    candidate_id = str(
        profile["new_batch_candidate_id"]
    )

    group = reported.loc[
        reported["new_batch_candidate_id"].eq(
            candidate_id
        )
    ].copy()

    count = len(group)

    if count != int(profile["incoming_complaints"]):
        raise ValueError(
            f"Count mismatch: {candidate_id}"
        )

    category_values = group[
        "complaint_category"
    ].unique()

    if len(category_values) != 1:
        raise ValueError(
            f"Mixed categories: {candidate_id}"
        )

    category = str(category_values[0])

    coords = group[
        ["lat_num", "lon_num"]
    ].to_numpy(dtype=float)

    center_lat = float(coords[:, 0].mean())
    center_lon = float(coords[:, 1].mean())

    # Find an actual submitted coordinate closest
    # to the average, rather than inventing a
    # street address or asset location.
    distances = distance_metres(
        center_lat,
        center_lon,
        coords[:, 0],
        coords[:, 1]
    )

    representative_index = int(
        np.argmin(distances)
    )

    representative_lat = float(
        coords[representative_index, 0]
    )

    representative_lon = float(
        coords[representative_index, 1]
    )

    max_distance = float(np.max(
        distance_metres(
            representative_lat,
            representative_lon,
            coords[:, 0],
            coords[:, 1]
        )
    ))

    distinct_locations = len(
        np.unique(coords, axis=0)
    )

    boroughs = sorted(
        group["borough"].astype(str).unique()
    )

    zip_areas = sorted({
        value
        for value in group[
            "incident_zip"
        ].astype(str)
        if len(value) == 5 and value.isdigit()
    })

    historical_counts = (
        group["dbscan_status"]
        .value_counts()
    )

    established = int(
        historical_counts.get(
            "established_cluster", 0
        )
    )

    outside = int(
        historical_counts.get(
            "outside_existing_clusters", 0
        )
    )

    ambiguous = int(
        historical_counts.get(
            "ambiguous_association", 0
        )
    )

    unclassified = (
        count - established - outside - ambiguous
    )

    if outside == count:
        relationship = (
            "entirely_outside_historical_clusters"
        )
        relationship_text = (
            "All incoming reports are outside "
            "the saved historical cluster associations."
        )

    elif established == count:
        relationship = (
            "entirely_associated_with_historical_clusters"
        )
        relationship_text = (
            "All incoming reports are associated "
            "with established historical clusters."
        )

    else:
        relationship = (
            "mixed_or_uncertain_historical_associations"
        )
        relationship_text = (
            f"Historical association results: "
            f"{established} established, "
            f"{outside} outside, "
            f"{ambiguous} ambiguous, "
            f"{unclassified} other."
        )

    explanation = (
        f"{count} incoming {category} complaints "
        f"at {distinct_locations} distinct submitted "
        f"coordinates form a candidate spatial "
        f"concentration in this synthetic batch. "
        f"{relationship_text} "
        f"The representative map point "
        f"({representative_lat:.6f}, "
        f"{representative_lon:.6f}) "
        f"is a submitted complaint coordinate, "
        f"not a confirmed defect location."
    )

    action = (
        guidance.get(
            category,
            "Review the complaint descriptions "
            "and their reported locations."
        )
        + " Check for duplicate reports and "
        + "confirm the situation before considering "
        + "a field investigation."
    )

    brief = {
        "candidate_id": candidate_id,
        "complaint_category": category,
        "incoming_complaints": int(count),
        "distinct_locations": int(
            distinct_locations
        ),
        "boroughs": boroughs,
        "incident_zip_areas": zip_areas,
        "representative_latitude": round(
            representative_lat, 7
        ),
        "representative_longitude": round(
            representative_lon, 7
        ),
        "mean_latitude": round(center_lat, 7),
        "mean_longitude": round(center_lon, 7),
        "max_distance_from_representative_m":
            round(max_distance, 1),
        "historical_established": established,
        "historical_outside": outside,
        "historical_ambiguous": ambiguous,
        "historical_unclassified": unclassified,
        "historical_relationship": relationship,
        "explanation": explanation,
        "suggested_review": action,
        "synthetic_data": True
    }

    brief_rows.append(brief)

    geojson["features"].append({
        "type": "Feature",
        "id": candidate_id,
        "geometry": {
            "type": "Point",
            "coordinates": [
                representative_lon,
                representative_lat
            ]
        },
        "properties": {
            "candidate_id": candidate_id,
            "complaint_category": category,
            "incoming_complaints": int(count),
            "distinct_locations": int(
                distinct_locations
            ),
            "historical_relationship":
                relationship,
            "historical_outside": outside,
            "historical_established": established,
            "historical_ambiguous": ambiguous,
            "synthetic_data": True
        }
    })

briefs_18_2c = pd.DataFrame(brief_rows)

# --------------------------------------------------
# 5. Preview spatial intelligence explanations
# --------------------------------------------------

print("\n2. MAP-READY CANDIDATE SUMMARY")

display(
    briefs_18_2c[[
        "candidate_id",
        "complaint_category",
        "incoming_complaints",
        "distinct_locations",
        "historical_relationship",
        "representative_latitude",
        "representative_longitude"
    ]].head(15)
)

print("\n3. HISTORICAL RELATIONSHIP SUMMARY")

display(
    briefs_18_2c[
        "historical_relationship"
    ].value_counts().rename_axis(
        "relationship"
    ).reset_index(name="candidates")
)

print("\n4. SAMPLE INVESTIGATION BRIEF")

example = briefs_18_2c.loc[
    briefs_18_2c["candidate_id"].eq(
        "NB-00-004"
    )
]

if not example.empty:
    sample = example.iloc[0]
    print(sample["explanation"])
    print("\nSuggested review:")
    print(sample["suggested_review"])

# --------------------------------------------------
# 6. Independence from historical DBSCAN
# --------------------------------------------------

# Change historical labels only.
# New cluster memberships must remain identical.

historical_blinded = source.copy()

historical_blinded["dbscan_status"] = (
    "outside_existing_clusters"
)

with threadpool_limits(limits=1):
    blinded_output = analyze_new_batch_spatial(
        historical_blinded,
        categories,
        eps_m=125,
        min_samples=5,
        min_report_count=5,
        min_distinct_locations=2
    )

independent_assignments = (
    records["new_batch_candidate_id"]
    .fillna("NONE")
    .astype(str)
    .tolist()
    ==
    blinded_output["records"][
        "new_batch_candidate_id"
    ]
    .fillna("NONE")
    .astype(str)
    .tolist()
)

# --------------------------------------------------
# 7. Validation checks
# --------------------------------------------------

checks = []

def check(name, condition):
    checks.append({
        "test": name,
        "passed": bool(condition)
    })

check(
    "500 incoming complaints retained",
    stats["total_records"] == 500
)

check(
    "23 reportable candidate concentrations",
    len(briefs_18_2c) == 23
)

check(
    "228 complaints represented in briefs",
    int(
        briefs_18_2c[
            "incoming_complaints"
        ].sum()
    ) == 228
)

check(
    "Every candidate has a unique ID",
    briefs_18_2c["candidate_id"].is_unique
)

check(
    "Each brief matches one reportable candidate",
    set(briefs_18_2c["candidate_id"])
    == set(
        profiles["new_batch_candidate_id"]
    )
)

check(
    "All reported candidate counts reconcile",
    all(
        int(
            reported["new_batch_candidate_id"]
            .eq(row["candidate_id"]).sum()
        ) == row["incoming_complaints"]
        for row in brief_rows
    )
)

check(
    "Each candidate has a single complaint category",
    reported.groupby(
        "new_batch_candidate_id"
    )["complaint_category"].nunique().eq(1).all()
)

check(
    "Every brief has a representative location",
    briefs_18_2c[[
        "representative_latitude",
        "representative_longitude"
    ]].notna().all().all()
)

check(
    "Representative coordinates are submitted locations",
    all(
        any(
            np.isclose(
                group["lat_num"].astype(float),
                row["representative_latitude"],
                atol=0.0000001
            )
            &
            np.isclose(
                group["lon_num"].astype(float),
                row["representative_longitude"],
                atol=0.0000001
            )
        )
        for row in brief_rows
        for group in [
            reported.loc[
                reported["new_batch_candidate_id"]
                .eq(row["candidate_id"])
            ]
        ]
    )
)

check(
    "Historical outcomes reconcile in every brief",
    all(
        row["historical_established"]
        + row["historical_outside"]
        + row["historical_ambiguous"]
        + row["historical_unclassified"]
        == row["incoming_complaints"]
        for row in brief_rows
    )
)

check(
    "New clusters independent of historical labels",
    independent_assignments
)

check(
    "Sewer example has 15 outside historical clusters",
    not example.empty
    and int(
        example.iloc[0]["incoming_complaints"]
    ) == 15
    and int(
        example.iloc[0]["historical_outside"]
    ) == 15
)

check(
    "Map contains 23 candidate point features",
    len(geojson["features"]) == 23
)

check(
    "All map geometries are valid Point structures",
    all(
        feature["geometry"]["type"] == "Point"
        and len(
            feature["geometry"]["coordinates"]
        ) == 2
        for feature in geojson["features"]
    )
)

check(
    "GeoJSON uses longitude-latitude coordinate order",
    all(
        -180 <= feature["geometry"][
            "coordinates"
        ][0] <= 180
        and -90 <= feature["geometry"][
            "coordinates"
        ][1] <= 90
        for feature in geojson["features"]
    )
)

check(
    "All map features explicitly identify synthetic data",
    all(
        feature["properties"]["synthetic_data"]
        for feature in geojson["features"]
    )
)

check(
    "Every brief contains a suggested review",
    briefs_18_2c[
        "suggested_review"
    ].str.len().gt(20).all()
)

check(
    "DBSCAN parameter metadata preserved",
    geojson["metadata"]["eps_m"] == 125
    and geojson["metadata"]["min_samples"] == 5
)

check(
    "All representative distances are finite",
    np.isfinite(
        briefs_18_2c[
            "max_distance_from_representative_m"
        ].to_numpy(dtype=float)
    ).all()
)

check(
    "Original source CSV unchanged",
    hashlib.sha256(
        input_path.read_bytes()
    ).hexdigest() == source_hash
)

check(
    "Existing DBSCAN engine unchanged",
    hashlib.sha256(
        module_path.read_bytes()
    ).hexdigest() == module_hash
)

# Verify JSON serialization before writing.
geojson_text = json.dumps(
    geojson,
    indent=2,
    allow_nan=False
) + "\n"

brief_document = {
    "synthetic_data": True,
    "model": "new_batch_same_category_dbscan",
    "parameters": {
        "eps_m": 125,
        "min_samples": 5,
        "min_report_count": 5,
        "min_distinct_locations": 2
    },
    "limitations": [
        "Candidates are not confirmed defects.",
        "Historical association is not proof of onset.",
        "Representative points are not street addresses.",
        "The test batch is synthetic.",
        "Production parameter selection remains pending."
    ],
    "candidate_briefs": brief_rows
}

briefs_text = json.dumps(
    brief_document,
    indent=2,
    allow_nan=False
) + "\n"

check(
    "Both output files are valid JSON",
    len(json.loads(geojson_text)["features"]) == 23
    and len(
        json.loads(briefs_text)["candidate_briefs"]
    ) == 23
)

# --------------------------------------------------
# 8. Save only after successful validation
# --------------------------------------------------

results_18_2c = pd.DataFrame(checks)

print("\n5. VALIDATION")

display(results_18_2c)

passed = int(results_18_2c["passed"].sum())
total = len(results_18_2c)

print(f"\nChecks passed: {passed}/{total}")

failed = results_18_2c.loc[
    ~results_18_2c["passed"],
    "test"
].tolist()

if failed:

    print("\nSTATUS: NEEDS REVIEW")
    print("No output files saved.")

    for failure in failed:
        print("FAILED:", failure)

else:

    outputs_to_save = {
        briefs_path: briefs_text,
        map_path: geojson_text
    }

    # Check ALL existing files before
    # writing either new output.
    for path, content in outputs_to_save.items():

        if path.exists():
            assert path.read_text(
                encoding="utf-8"
            ) == content, (
                f"Existing output differs: {path}. "
                "No files overwritten."
            )

    output_dir.mkdir(
        parents=True,
        exist_ok=True
    )

    for path, content in outputs_to_save.items():

        if not path.exists():

            path.write_text(
                content,
                encoding="utf-8"
            )

    print("\n6. OUTPUT FILES")

    print("Investigation briefs:", briefs_path)
    print("Map-ready GeoJSON:", map_path)

    print("\nSTATUS: PASSED")
    print("SPATIAL EXPLANATIONS VERIFIED")
    print("23 SYNTHETIC CANDIDATE BRIEFS PREPARED")
    print("MAP-READY GEOJSON PREPARED")
    print("NEW CLUSTERS INDEPENDENT OF HISTORICAL LABELS")
    print("NO AUTOMATIC INFRASTRUCTURE RISK CLAIMS")
    print("HISTORICAL MODEL ARTIFACTS UNMODIFIED")


 SPATIAL EXPLANATIONS AND MAP PREPARATION

1. ENGINE RECONCILIATION
Total complaints: 500
Raw clusters: 25
Reportable candidates: 23
Complaints in candidates: 228

2. MAP-READY CANDIDATE SUMMARY


,candidate_id,complaint_category,incoming_complaints,distinct_locations,historical_relationship,representative_latitude,representative_longitude
0,NB-00-000,SEWER,15,15,mixed_or_uncertain_historical_associations,40.853080,-73.913402
1,NB-00-003,SEWER,15,15,mixed_or_uncertain_historical_associations,40.731104,-73.991363
2,NB-00-004,SEWER,15,15,entirely_outside_historical_clusters,40.755820,-73.713806
3,NB-03-010,STREET CONDITION,14,14,entirely_outside_historical_clusters,40.624780,-74.096207
4,NB-03-004,STREET CONDITION,13,13,entirely_outside_historical_clusters,40.649627,-73.934153
5,NB-03-005,STREET CONDITION,13,13,mixed_or_uncertain_historical_associations,40.751977,-74.004471
6,NB-03-006,STREET CONDITION,13,13,entirely_associated_with_historical_clusters,40.717378,-73.985058
7,NB-03-003,STREET CONDITION,12,12,mixed_or_uncertain_historical_associations,40.693562,-73.989659
8,NB-04-002,STREET LIGHT CONDITION,12,12,entirely_outside_historical_clusters,40.688756,-74.016905
9,NB-00-001,SEWER,11,11,mixed_or_uncertain_historical_associations,40.620924,-73.985241



3. HISTORICAL RELATIONSHIP SUMMARY


,relationship,candidates
0,mixed_or_uncertain_historical_associations,12
1,entirely_outside_historical_clusters,10
2,entirely_associated_with_historical_clusters,1



4. SAMPLE INVESTIGATION BRIEF
15 incoming SEWER complaints at 15 distinct submitted coordinates form a candidate spatial concentration in this synthetic batch. All incoming reports are outside the saved historical cluster associations. The representative map point (40.755820, -73.713806) is a submitted complaint coordinate, not a confirmed defect location.

Suggested review:
Review sewer complaint descriptions, location accuracy and relevant maintenance history. Check for duplicate reports and confirm the situation before considering a field investigation.

5. VALIDATION


,test,passed
0,500 incoming complaints retained,True
1,23 reportable candidate concentrations,True
2,228 complaints represented in briefs,True
3,Every candidate has a unique ID,True
4,Each brief matches one reportable candidate,True
5,All reported candidate counts reconcile,True
6,Each candidate has a single complaint category,True
7,Every brief has a representative location,True
8,Representative coordinates are submitted locat...,True
9,Historical outcomes reconcile in every brief,True



Checks passed: 22/22

6. OUTPUT FILES
Investigation briefs: D:\Urban-Infrastructure-Pulse-Intelligence\outputs\tests\phase_18_2c_spatial_explanations\candidate_briefs.json
Map-ready GeoJSON: D:\Urban-Infrastructure-Pulse-Intelligence\outputs\tests\phase_18_2c_spatial_explanations\candidate_map_points.geojson

STATUS: PASSED
SPATIAL EXPLANATIONS VERIFIED
23 SYNTHETIC CANDIDATE BRIEFS PREPARED
MAP-READY GEOJSON PREPARED
NEW CLUSTERS INDEPENDENT OF HISTORICAL LABELS
NO AUTOMATIC INFRASTRUCTURE RISK CLAIMS
HISTORICAL MODEL ARTIFACTS UNMODIFIED


### COMBINED BATCH INTELLIGENCE

In [20]:

from pathlib import Path
from threadpoolctl import threadpool_limits
import hashlib
import json
import sys

import pandas as pd

print(" COMBINED BATCH INTELLIGENCE")
print("=" * 72)

root = Path(PROJECT_ROOT).resolve()

if str(root) not in sys.path:
    sys.path.insert(0, str(root))

from src.models.new_batch_area_kmeans import (
    analyze_new_batch_areas
)
from src.models.new_batch_dbscan import (
    analyze_new_batch_spatial
)

input_path = (
    root / "outputs/tests"
    / "phase_17_4c_backend_verified"
    / "verified_500_backend_results.csv"
)

taxonomy_path = (
    root / "outputs/tables/dbscan"
    / "dbscan_action_taxonomy_v1.csv"
)

briefs_path = (
    root / "outputs/tests"
    / "phase_18_2c_spatial_explanations"
    / "candidate_briefs.json"
)

map_path = (
    root / "outputs/tests"
    / "phase_18_2c_spatial_explanations"
    / "candidate_map_points.geojson"
)

kmeans_module = (
    root / "src/models/new_batch_area_kmeans.py"
)

dbscan_module = (
    root / "src/models/new_batch_dbscan.py"
)

protected_files = [
    input_path,
    taxonomy_path,
    briefs_path,
    map_path,
    kmeans_module,
    dbscan_module
]

for path in protected_files:
    assert path.is_file(), f"Missing: {path}"

original_hashes = {
    str(path): hashlib.sha256(
        path.read_bytes()
    ).hexdigest()
    for path in protected_files
}

# --------------------------------------------------
# 1. Load existing verified evidence
# --------------------------------------------------

source = pd.read_csv(
    input_path,
    dtype="string",
    keep_default_na=False
)

taxonomy = pd.read_csv(taxonomy_path)

categories = sorted({
    str(x).strip().upper()
    for x in taxonomy[
        "complaint_category"
    ].dropna()
})

with briefs_path.open(
    encoding="utf-8"
) as file:
    brief_document = json.load(file)

with map_path.open(
    encoding="utf-8"
) as file:
    candidate_map = json.load(file)

existing_briefs = pd.DataFrame(
    brief_document["candidate_briefs"]
)

assert len(source) == 500
assert source["source_row"].is_unique
assert source["complaint_id"].is_unique

# --------------------------------------------------
# 2. Run both independent new-batch models
# --------------------------------------------------

print("\n1. RUNNING INDEPENDENT NEW-BATCH MODELS")

with threadpool_limits(limits=1):

    kmeans_output = analyze_new_batch_areas(
        source,
        categories,
        k=3,
        min_area_size=20,
        random_state=18
    )

    dbscan_output = analyze_new_batch_spatial(
        source,
        categories,
        eps_m=125,
        min_samples=5,
        min_report_count=5,
        min_distinct_locations=2
    )

assert kmeans_output["status"] == "clustered"

print(
    "New K-Means areas:",
    kmeans_output["stats"]["eligible_areas"]
)
print(
    "New K-Means clusters:",
    len(kmeans_output["cluster_profiles"])
)
print(
    "New DBSCAN candidates:",
    dbscan_output["stats"]["reportable_candidates"]
)

# --------------------------------------------------
# 3. Prepare K-Means area assignments
# --------------------------------------------------

kmeans_records = (
    kmeans_output["record_eligibility"][[
        "source_row",
        "area_id",
        "eligibility_reason"
    ]]
    .rename(columns={
        "area_id": "new_kmeans_area_id",
        "eligibility_reason":
            "new_kmeans_eligibility"
    })
)

kmeans_areas = (
    kmeans_output["area_summary"][[
        "area_id",
        "incoming_complaints",
        "meets_minimum",
        "new_batch_cluster",
        "new_batch_description"
    ]]
    .rename(columns={
        "area_id": "new_kmeans_area_id",
        "incoming_complaints":
            "kmeans_area_complaint_count",
        "meets_minimum":
            "new_kmeans_area_eligible",
        "new_batch_cluster":
            "new_kmeans_cluster",
        "new_batch_description":
            "new_kmeans_description"
    })
)

# --------------------------------------------------
# 4. Prepare DBSCAN complaint assignments
# --------------------------------------------------

dbscan_records = (
    dbscan_output["records"][[
        "source_row",
        "record_eligibility",
        "new_batch_status",
        "raw_candidate_id",
        "new_batch_candidate_id"
    ]]
    .rename(columns={
        "record_eligibility":
            "new_dbscan_eligibility",
        "new_batch_status":
            "new_dbscan_status",
        "raw_candidate_id":
            "new_dbscan_raw_candidate_id",
        "new_batch_candidate_id":
            "new_dbscan_candidate_id"
    })
)

# --------------------------------------------------
# 5. Combine by original complaint identity
# --------------------------------------------------

# source_row is the explicit join key.
# Never join on cluster number: historical and
# new clusters use unrelated ID systems.

fusion_records_18_3a = (
    source
    .merge(
        kmeans_records,
        on="source_row",
        how="left",
        validate="one_to_one",
        sort=False
    )
    .merge(
        kmeans_areas,
        on="new_kmeans_area_id",
        how="left",
        validate="many_to_one",
        sort=False
    )
    .merge(
        dbscan_records,
        on="source_row",
        how="left",
        validate="one_to_one",
        sort=False
    )
)

fused = fusion_records_18_3a

fused["is_new_spatial_candidate"] = (
    fused["new_dbscan_status"].eq(
        "candidate_concentration"
    )
)

fused["is_historical_established"] = (
    fused["dbscan_status"].eq(
        "established_cluster"
    )
)

fused["is_historical_outside"] = (
    fused["dbscan_status"].eq(
        "outside_existing_clusters"
    )
)

print("\n2. INTEGRATED COMPLAINT COVERAGE")

print("Input complaints:", len(source))
print("Integrated complaints:", len(fused))

print(
    "New K-Means area assignments:",
    fused["new_kmeans_cluster"].notna().sum()
)

print(
    "New DBSCAN candidate complaints:",
    fused["is_new_spatial_candidate"].sum()
)

# --------------------------------------------------
# 6. Summarize intelligence by new K-Means group
# --------------------------------------------------

area_pattern_summary_18_3a = (
    fused.groupby(
        "new_kmeans_description",
        dropna=False
    )
    .agg(
        geographic_areas=(
            "new_kmeans_area_id",
            "nunique"
        ),
        incoming_complaints=(
            "complaint_id",
            "size"
        ),
        spatial_candidate_complaints=(
            "is_new_spatial_candidate",
            "sum"
        ),
        historically_established=(
            "is_historical_established",
            "sum"
        ),
        outside_historical_hotspots=(
            "is_historical_outside",
            "sum"
        )
    )
    .reset_index()
    .sort_values(
        "incoming_complaints",
        ascending=False
    )
)

print("\n3. K-MEANS AND DBSCAN COMBINED SUMMARY")

display(area_pattern_summary_18_3a)

# --------------------------------------------------
# 7. Summarize evidence for all 20 areas
# --------------------------------------------------

area_intelligence_18_3a = (
    fused.groupby(
        [
            "new_kmeans_area_id",
            "new_kmeans_description"
        ],
        dropna=False
    )
    .agg(
        incoming_complaints=(
            "complaint_id",
            "size"
        ),
        new_spatial_candidate_complaints=(
            "is_new_spatial_candidate",
            "sum"
        ),
        distinct_spatial_candidates=(
            "new_dbscan_candidate_id",
            "nunique"
        ),
        historical_established_complaints=(
            "is_historical_established",
            "sum"
        ),
        historical_outside_complaints=(
            "is_historical_outside",
            "sum"
        )
    )
    .reset_index()
    .sort_values("new_kmeans_area_id")
)

print("\n4. AREA-LEVEL INTELLIGENCE")

display(area_intelligence_18_3a)

# --------------------------------------------------
# 8. Connect candidate briefs to area patterns
# --------------------------------------------------

candidate_rows = fused.loc[
    fused["is_new_spatial_candidate"]
].copy()

def joined_values(series):
    return " | ".join(
        sorted({
            str(value)
            for value in series.dropna()
            if str(value).strip()
        })
    )

candidate_area_context = (
    candidate_rows.groupby(
        "new_dbscan_candidate_id"
    )
    .agg(
        kmeans_area_count=(
            "new_kmeans_area_id",
            "nunique"
        ),
        kmeans_pattern_count=(
            "new_kmeans_description",
            "nunique"
        ),
        linked_area_patterns=(
            "new_kmeans_description",
            joined_values
        ),
        linked_area_ids=(
            "new_kmeans_area_id",
            joined_values
        )
    )
    .reset_index()
    .rename(columns={
        "new_dbscan_candidate_id":
            "candidate_id"
    })
)

integrated_candidates_18_3a = (
    existing_briefs.merge(
        candidate_area_context,
        on="candidate_id",
        how="left",
        validate="one_to_one"
    )
)

print("\n5. INTEGRATED CANDIDATE EXPLANATIONS")

display(
    integrated_candidates_18_3a[[
        "candidate_id",
        "complaint_category",
        "incoming_complaints",
        "historical_relationship",
        "kmeans_area_count",
        "kmeans_pattern_count",
        "linked_area_patterns"
    ]].head(15)
)

# --------------------------------------------------
# 9. Show a combined example
# --------------------------------------------------

print("\n6. EXAMPLE — SYNTHETIC SEWER CONCENTRATION")

example = integrated_candidates_18_3a.loc[
    integrated_candidates_18_3a[
        "candidate_id"
    ].eq("NB-00-004")
]

if not example.empty:

    row = example.iloc[0]

    print("Candidate:", row["candidate_id"])
    print("Category:", row["complaint_category"])
    print(
        "Concentrated complaints:",
        row["incoming_complaints"]
    )
    print(
        "Historical context:",
        row["historical_relationship"]
    )
    print(
        "Area reporting pattern:",
        row["linked_area_patterns"]
    )
    print("\nSuggested investigation:")
    print(row["suggested_review"])

# --------------------------------------------------
# 10. Inspect incoming complaint dates
# --------------------------------------------------

print("\n7. INCOMING DATE AUDIT")

if "created_date" in source.columns:

    date_text = (
        source["created_date"]
        .fillna("")
        .astype(str)
        .str.strip()
        .str[:10]
    )

    parsed_days = pd.to_datetime(
        date_text,
        format="%Y-%m-%d",
        errors="coerce"
    )

    day_labels = (
        parsed_days
        .dt.strftime("%Y-%m-%d")
        .fillna("INVALID_OR_MISSING")
    )

    day_audit_18_3a = (
        day_labels
        .value_counts()
        .rename_axis("created_day")
        .reset_index(name="complaints")
    )

    single_valid_day = (
        parsed_days.notna().all()
        and parsed_days.nunique() == 1
    )

else:

    day_audit_18_3a = pd.DataFrame([{
        "created_day": "COLUMN_MISSING",
        "complaints": len(source)
    }])

    single_valid_day = False

display(day_audit_18_3a)

print(
    "Single valid reporting day:",
    single_valid_day
)

print(
    "NOTE: This audit does not yet enforce "
    "a production daily ingestion window."
)

# --------------------------------------------------
# 11. Verify new K-Means ignores historical labels
# --------------------------------------------------

history_blinded = source.copy()

for column in [
    "kmeans_area_id",
    "kmeans_cluster_id",
    "kmeans_reporting_profile"
]:
    if column in history_blinded.columns:
        history_blinded[column] = (
            "HISTORICAL_LABEL_HIDDEN"
        )

if "kmeans_display_allowed" in (
    history_blinded.columns
):
    history_blinded[
        "kmeans_display_allowed"
    ] = "False"

with threadpool_limits(limits=1):

    blinded_kmeans = analyze_new_batch_areas(
        history_blinded,
        categories,
        k=3,
        min_area_size=20,
        random_state=18
    )

original_assignments = (
    kmeans_output["area_summary"]
    .set_index("area_id")[
        "new_batch_cluster"
    ]
    .sort_index()
)

blinded_assignments = (
    blinded_kmeans["area_summary"]
    .set_index("area_id")[
        "new_batch_cluster"
    ]
    .sort_index()
)

kmeans_independent = (
    original_assignments.equals(
        blinded_assignments
    )
)

print("\n8. MODEL INDEPENDENCE")

print(
    "New K-Means independent of historical labels:",
    kmeans_independent
)

# --------------------------------------------------
# 12. Verify reconciliation and output integrity
# --------------------------------------------------

checks = []

def check(name, condition):
    checks.append({
        "test": name,
        "passed": bool(condition)
    })

check(
    "All 500 original complaints retained",
    len(fused) == 500
)

check(
    "Every source row appears exactly once",
    fused["source_row"].is_unique
    and set(fused["source_row"])
    == set(source["source_row"])
)

check(
    "Complaint identities preserved in original order",
    fused["complaint_id"].tolist()
    == source["complaint_id"].tolist()
)

check(
    "All 500 new K-Means assignments present",
    fused["new_kmeans_cluster"].notna().sum()
    == 500
)

check(
    "Exactly 20 new-batch geographic areas",
    fused["new_kmeans_area_id"].nunique() == 20
)

check(
    "Three new K-Means groups retained",
    fused["new_kmeans_cluster"].nunique() == 3
)

check(
    "New K-Means groups contain 10, 5, 5 areas",
    sorted(
        area_pattern_summary_18_3a[
            "geographic_areas"
        ].astype(int).tolist()
    ) == [5, 5, 10]
)

check(
    "Each area receives only one new K-Means label",
    fused.groupby(
        "new_kmeans_area_id"
    )["new_kmeans_cluster"].nunique().eq(1).all()
)

check(
    "All 20 area complaint totals reconcile",
    len(area_intelligence_18_3a) == 20
    and int(
        area_intelligence_18_3a[
            "incoming_complaints"
        ].sum()
    ) == 500
)

check(
    "228 candidate complaints preserved",
    int(
        fused["is_new_spatial_candidate"].sum()
    ) == 228
)

check(
    "Eight reports remain below threshold",
    fused["new_dbscan_status"].eq(
        "below_reporting_threshold"
    ).sum() == 8
)

check(
    "264 reports remain unclustered",
    fused["new_dbscan_status"].eq(
        "unclustered"
    ).sum() == 264
)

check(
    "All 23 DBSCAN candidate IDs retained",
    fused[
        "new_dbscan_candidate_id"
    ].nunique() == 23
)

check(
    "Every candidate remains category-specific",
    candidate_rows.groupby(
        "new_dbscan_candidate_id"
    )["complaint_category"].nunique().eq(1).all()
)

check(
    "All 23 existing briefs successfully integrated",
    len(integrated_candidates_18_3a) == 23
    and integrated_candidates_18_3a[
        "kmeans_area_count"
    ].notna().all()
)

model_counts = (
    candidate_rows.groupby(
        "new_dbscan_candidate_id"
    )
    .size()
    .rename("model_count")
    .reset_index()
    .rename(columns={
        "new_dbscan_candidate_id":
            "candidate_id"
    })
)

brief_count_check = (
    integrated_candidates_18_3a.merge(
        model_counts,
        on="candidate_id",
        validate="one_to_one"
    )
)

check(
    "Integrated brief counts match DBSCAN records",
    len(brief_count_check) == 23
    and brief_count_check[
        "incoming_complaints"
    ].astype(int).eq(
        brief_count_check["model_count"]
    ).all()
)

map_ids = {
    feature["properties"]["candidate_id"]
    for feature in candidate_map["features"]
}

check(
    "Map and integrated brief candidates match",
    len(candidate_map["features"]) == 23
    and map_ids == set(
        integrated_candidates_18_3a[
            "candidate_id"
        ]
    )
)

check(
    "Historical established associations preserved",
    fused["dbscan_status"].eq(
        "established_cluster"
    ).sum() == 137
)

check(
    "Historical ambiguous associations preserved",
    fused["dbscan_status"].eq(
        "ambiguous_association"
    ).sum() == 9
)

check(
    "Historical outside-cluster outcomes preserved",
    fused["dbscan_status"].eq(
        "outside_existing_clusters"
    ).sum() == 354
)

check(
    "Original historical K-Means profiles retained",
    fused["kmeans_reporting_profile"].tolist()
    == source["kmeans_reporting_profile"].tolist()
)

check(
    "New K-Means does not use historical labels",
    kmeans_independent
)

check(
    "Date audit accounts for all 500 reports",
    int(
        day_audit_18_3a[
            "complaints"
        ].sum()
    ) == 500
)

check(
    "No protected model or evidence file changed",
    all(
        hashlib.sha256(
            path.read_bytes()
        ).hexdigest() == original_hashes[str(path)]
        for path in protected_files
    )
)

# --------------------------------------------------
# 13. Final validation
# --------------------------------------------------

results_18_3a = pd.DataFrame(checks)

print("\n9. VALIDATION")

display(results_18_3a)

passed = int(results_18_3a["passed"].sum())
total = len(results_18_3a)

print(f"\nChecks passed: {passed}/{total}")

failed = results_18_3a.loc[
    ~results_18_3a["passed"],
    "test"
].tolist()

if failed:

    print("\nSTATUS: NEEDS REVIEW")

    for name in failed:
        print("FAILED:", name)

else:

    print("\nSTATUS: PASSED")
    print("COMBINED INTELLIGENCE VERIFIED")
    print("NEW K-MEANS AND DBSCAN RESULTS INTEGRATED")
    print("HISTORICAL MODEL CONTEXT PRESERVED")
    print("23 SPATIAL BRIEFS LINKED TO AREA PATTERNS")
    print("INCOMING DATE COVERAGE AUDITED")
    print("NO MODEL OR EVIDENCE FILES MODIFIED")
    print("NO AUTOMATIC INFRASTRUCTURE RISK CLAIMS")


 COMBINED BATCH INTELLIGENCE

1. RUNNING INDEPENDENT NEW-BATCH MODELS
New K-Means areas: 20
New K-Means clusters: 3
New DBSCAN candidates: 23

2. INTEGRATED COMPLAINT COVERAGE
Input complaints: 500
Integrated complaints: 500
New K-Means area assignments: 500
New DBSCAN candidate complaints: 228

3. K-MEANS AND DBSCAN COMBINED SUMMARY


,new_kmeans_description,geographic_areas,incoming_complaints,spatial_candidate_complaints,historically_established,outside_historical_hotspots
1,Road-heavy reporting,10,250,113,72,175
0,Lighting/signal-heavy reporting,5,125,54,20,105
2,Sewer-heavy reporting,5,125,61,45,74



4. AREA-LEVEL INTELLIGENCE


,new_kmeans_area_id,new_kmeans_description,incoming_complaints,new_spatial_candidate_complaints,distinct_spatial_candidates,historical_established_complaints,historical_outside_complaints
0,BRONX_10451,Road-heavy reporting,25,8,1,6,19
1,BRONX_10452,Road-heavy reporting,25,8,1,7,18
2,BRONX_10453,Sewer-heavy reporting,25,15,1,6,19
3,BRONX_10454,Lighting/signal-heavy reporting,25,7,1,4,21
4,BROOKLYN_11201,Road-heavy reporting,25,12,1,21,3
5,BROOKLYN_11203,Road-heavy reporting,25,13,1,0,25
6,BROOKLYN_11204,Sewer-heavy reporting,25,11,1,12,8
7,BROOKLYN_11205,Lighting/signal-heavy reporting,25,5,1,1,24
8,MANHATTAN_10001,Road-heavy reporting,25,13,1,5,20
9,MANHATTAN_10002,Road-heavy reporting,25,13,1,23,0



5. INTEGRATED CANDIDATE EXPLANATIONS


,candidate_id,complaint_category,incoming_complaints,historical_relationship,kmeans_area_count,kmeans_pattern_count,linked_area_patterns
0,NB-00-000,SEWER,15,mixed_or_uncertain_historical_associations,1,1,Sewer-heavy reporting
1,NB-00-003,SEWER,15,mixed_or_uncertain_historical_associations,1,1,Sewer-heavy reporting
2,NB-00-004,SEWER,15,entirely_outside_historical_clusters,1,1,Sewer-heavy reporting
3,NB-03-010,STREET CONDITION,14,entirely_outside_historical_clusters,1,1,Road-heavy reporting
4,NB-03-004,STREET CONDITION,13,entirely_outside_historical_clusters,1,1,Road-heavy reporting
5,NB-03-005,STREET CONDITION,13,mixed_or_uncertain_historical_associations,1,1,Road-heavy reporting
6,NB-03-006,STREET CONDITION,13,entirely_associated_with_historical_clusters,1,1,Road-heavy reporting
7,NB-03-003,STREET CONDITION,12,mixed_or_uncertain_historical_associations,1,1,Road-heavy reporting
8,NB-04-002,STREET LIGHT CONDITION,12,entirely_outside_historical_clusters,1,1,Lighting/signal-heavy reporting
9,NB-00-001,SEWER,11,mixed_or_uncertain_historical_associations,1,1,Sewer-heavy reporting



6. EXAMPLE — SYNTHETIC SEWER CONCENTRATION
Candidate: NB-00-004
Category: SEWER
Concentrated complaints: 15
Historical context: entirely_outside_historical_clusters
Area reporting pattern: Sewer-heavy reporting

Suggested investigation:
Review sewer complaint descriptions, location accuracy and relevant maintenance history. Check for duplicate reports and confirm the situation before considering a field investigation.

7. INCOMING DATE AUDIT


,created_day,complaints
0,2026-11-09,500


Single valid reporting day: True
NOTE: This audit does not yet enforce a production daily ingestion window.

8. MODEL INDEPENDENCE
New K-Means independent of historical labels: True

9. VALIDATION


,test,passed
0,All 500 original complaints retained,True
1,Every source row appears exactly once,True
2,Complaint identities preserved in original order,True
3,All 500 new K-Means assignments present,True
4,Exactly 20 new-batch geographic areas,True
5,Three new K-Means groups retained,True
6,"New K-Means groups contain 10, 5, 5 areas",True
7,Each area receives only one new K-Means label,True
8,All 20 area complaint totals reconcile,True
9,228 candidate complaints preserved,True



Checks passed: 24/24

STATUS: PASSED
COMBINED INTELLIGENCE VERIFIED
NEW K-MEANS AND DBSCAN RESULTS INTEGRATED
HISTORICAL MODEL CONTEXT PRESERVED
23 SPATIAL BRIEFS LINKED TO AREA PATTERNS
INCOMING DATE COVERAGE AUDITED
NO MODEL OR EVIDENCE FILES MODIFIED
NO AUTOMATIC INFRASTRUCTURE RISK CLAIMS


### Enforce Daily Complaint Windows
if the next CSV contains 500 reports from November 9 and 100 from November 10, the system should analyze only the selected reporting day, while retaining the other 100 in an auditable exclusion record.

We'll also interpret timestamps using New York's local time zone, rather than your computer's time zone, so an incoming UTC timestamp near midnight is assigned to the correct NYC reporting day.

In [21]:

from pathlib import Path
from textwrap import dedent
from threadpoolctl import threadpool_limits
import hashlib
import sys
import pandas as pd

print(" DAILY BATCH WINDOW ENFORCEMENT")
print("=" * 72)

root = Path(PROJECT_ROOT).resolve()

if str(root) not in sys.path:
    sys.path.insert(0, str(root))

module_path = (
    root / "src/integration/daily_batch_window.py"
)

input_path = (
    root / "outputs/tests"
    / "phase_17_4c_backend_verified"
    / "verified_500_backend_results.csv"
)

taxonomy_path = (
    root / "outputs/tables/dbscan"
    / "dbscan_action_taxonomy_v1.csv"
)

kmeans_path = (
    root / "src/models/new_batch_area_kmeans.py"
)

dbscan_path = (
    root / "src/models/new_batch_dbscan.py"
)

protected = [
    input_path,
    taxonomy_path,
    kmeans_path,
    dbscan_path
]

for path in protected:
    assert path.is_file(), f"Missing: {path}"

original_hashes = {
    path: hashlib.sha256(
        path.read_bytes()
    ).hexdigest()
    for path in protected
}

# --------------------------------------------------
# 1. Define reusable date-window module
# --------------------------------------------------

module_source = dedent(r'''
"""Select one NYC-local reporting day from an incoming batch.

Naive timestamps are interpreted as New York local
time. Explicit timestamp offsets are converted to
New York time.

Ambiguous/nonexistent local DST times are rejected.
All original records remain auditable.
"""

from datetime import date, datetime
import pandas as pd


NYC_TIMEZONE = "America/New_York"


def _parse_nyc_day(value):
    """Return (NYC day, parsing status)."""

    if pd.isna(value):
        return None, "invalid_or_missing_date"

    text = str(value).strip()

    if not text:
        return None, "invalid_or_missing_date"

    parsed = None

    try:
        parsed = datetime.fromisoformat(
            text.replace("Z", "+00:00")
        )
    except ValueError:
        pass

    if parsed is None:

        for fmt in [
            "%m/%d/%Y %I:%M:%S %p",
            "%m/%d/%Y %H:%M:%S"
        ]:
            try:
                parsed = datetime.strptime(
                    text, fmt
                )
                break
            except ValueError:
                continue

    if parsed is None:
        return None, "invalid_or_missing_date"

    try:
        timestamp = pd.Timestamp(parsed)

        if timestamp.tzinfo is None:

            timestamp = timestamp.tz_localize(
                NYC_TIMEZONE,
                ambiguous="NaT",
                nonexistent="NaT"
            )

            parse_status = "assumed_nyc_local"

        else:

            timestamp = timestamp.tz_convert(
                NYC_TIMEZONE
            )

            parse_status = "converted_timezone"

        if pd.isna(timestamp):
            return None, "invalid_or_missing_date"

        return (
            timestamp.date().isoformat(),
            parse_status
        )

    except (ValueError, TypeError, OverflowError):
        return None, "invalid_or_missing_date"


def select_daily_complaints(
    records,
    reporting_day
):
    """Select an explicit NYC reporting day.

    Returns selected records plus a row-level audit.
    Does not mutate the input DataFrame.
    """

    if not isinstance(records, pd.DataFrame):
        raise TypeError("Expected a DataFrame.")

    required = {
        "source_row",
        "complaint_id",
        "created_date"
    }

    missing = required - set(records.columns)

    if missing:
        raise ValueError(
            f"Missing columns: {sorted(missing)}"
        )

    if records["source_row"].duplicated().any():
        raise ValueError(
            "Duplicate source_row values."
        )

    if not isinstance(reporting_day, str):
        raise ValueError(
            "reporting_day must be YYYY-MM-DD."
        )

    try:
        target = date.fromisoformat(
            reporting_day
        ).isoformat()
    except ValueError as exc:
        raise ValueError(
            "Invalid reporting day."
        ) from exc

    if target != reporting_day:
        raise ValueError(
            "Use YYYY-MM-DD reporting day."
        )

    work = records.copy().reset_index(drop=True)

    parsed = [
        _parse_nyc_day(value)
        for value in work["created_date"]
    ]

    work["nyc_reporting_day"] = [
        item[0] for item in parsed
    ]

    work["date_parse_status"] = [
        item[1] for item in parsed
    ]

    work["daily_window_status"] = (
        "outside_selected_day"
    )

    invalid = work["nyc_reporting_day"].isna()

    work.loc[
        invalid,
        "daily_window_status"
    ] = "invalid_or_missing_date"

    selected = work[
        "nyc_reporting_day"
    ].eq(target).fillna(False)

    work.loc[
        selected,
        "daily_window_status"
    ] = "selected_day"

    # Select from the original records, preserving
    # original model input columns and row order.
    selected_records = (
        records.reset_index(drop=True)
        .loc[selected.to_numpy()]
        .copy()
    )

    audit_columns = [
        "source_row",
        "complaint_id",
        "created_date",
        "nyc_reporting_day",
        "date_parse_status",
        "daily_window_status"
    ]

    audit = work[audit_columns].copy()

    stats = {
        "reporting_day": target,
        "timezone": NYC_TIMEZONE,
        "input_records": int(len(work)),
        "selected_records": int(selected.sum()),
        "outside_selected_day": int(
            work["daily_window_status"].eq(
                "outside_selected_day"
            ).sum()
        ),
        "invalid_or_missing_date": int(
            invalid.sum()
        ),
        "naive_local_assumptions": int(
            work["date_parse_status"].eq(
                "assumed_nyc_local"
            ).sum()
        ),
        "timezone_conversions": int(
            work["date_parse_status"].eq(
                "converted_timezone"
            ).sum()
        )
    }

    assert (
        stats["selected_records"]
        + stats["outside_selected_day"]
        + stats["invalid_or_missing_date"]
    ) == stats["input_records"]

    return {
        "selected_records": selected_records,
        "audit": audit,
        "stats": stats
    }
''').lstrip()

compile(
    module_source,
    str(module_path),
    "exec"
)

# Test entirely in memory before saving.
namespace = {}

exec(
    compile(
        module_source,
        str(module_path),
        "exec"
    ),
    namespace
)

select_daily_complaints = namespace[
    "select_daily_complaints"
]

parse_nyc_day = namespace[
    "_parse_nyc_day"
]

# --------------------------------------------------
# 2. Load the verified 500 complaint batch
# --------------------------------------------------

source = pd.read_csv(
    input_path,
    dtype="string",
    keep_default_na=False
)

taxonomy = pd.read_csv(taxonomy_path)

categories = sorted({
    str(value).strip().upper()
    for value in taxonomy[
        "complaint_category"
    ].dropna()
})

assert len(source) == 500

source_snapshot = source.copy(deep=True)

REPORTING_DAY = "2026-11-09"

# --------------------------------------------------
# 3. Verify original single-day batch
# --------------------------------------------------

print("\n1. ORIGINAL DAILY BATCH")

daily_output_18_3b = select_daily_complaints(
    source,
    REPORTING_DAY
)

print(daily_output_18_3b["stats"])

display(
    daily_output_18_3b["audit"][
        "daily_window_status"
    ].value_counts().rename_axis(
        "status"
    ).reset_index(name="complaints")
)

# --------------------------------------------------
# 4. Construct a controlled mixed-day batch
# --------------------------------------------------

mixed_source = source.copy()

# Two complaints explicitly dated November 10.
mixed_source.loc[
    [0, 1], "created_date"
] = "2026-11-10"

# Two invalid timestamps.
mixed_source.loc[
    [2, 3], "created_date"
] = "not-a-valid-date"

# UTC timestamps converted to NYC local dates.
# November 9, 22:30 UTC = November 9 NYC.
mixed_source.loc[
    4, "created_date"
] = "2026-11-09T22:30:00+00:00"

# November 10, 02:30 UTC = November 9 NYC.
mixed_source.loc[
    5, "created_date"
] = "2026-11-10T02:30:00+00:00"

# November 9, 01:00 UTC = November 8 NYC.
mixed_source.loc[
    6, "created_date"
] = "2026-11-09T01:00:00+00:00"

mixed_input_snapshot = mixed_source.copy(
    deep=True
)

mixed_output_18_3b = select_daily_complaints(
    mixed_source,
    REPORTING_DAY
)

mixed_stats = mixed_output_18_3b["stats"]

print("\n2. MIXED-DAY BATCH TEST")
print(mixed_stats)

display(
    mixed_output_18_3b["audit"].head(10)
)

# --------------------------------------------------
# 5. Verify timezone-specific edge cases
# --------------------------------------------------

print("\n3. TIMEZONE EDGE CASES")

timezone_examples = [
    "2026-11-09T12:00:00",
    "2026-11-10T02:30:00+00:00",
    "2026-11-09T01:00:00+00:00",
    "2026-11-09",
    "11/09/2026 10:30:00 AM",
    "2026-11-01T01:30:00"
]

timezone_rows = []

for value in timezone_examples:

    nyc_day, status = parse_nyc_day(value)

    timezone_rows.append({
        "input": value,
        "nyc_day": nyc_day,
        "parse_status": status
    })

timezone_audit_18_3b = pd.DataFrame(
    timezone_rows
)

display(timezone_audit_18_3b)

# --------------------------------------------------
# 6. Run both models on the filtered input
# --------------------------------------------------

from src.models.new_batch_area_kmeans import (
    analyze_new_batch_areas
)

from src.models.new_batch_dbscan import (
    analyze_new_batch_spatial
)

selected_records_18_3b = (
    mixed_output_18_3b["selected_records"]
)

print("\n4. MODEL TEST WITH FILTERED DAILY COMPLAINTS")
print(
    "Selected complaints:",
    len(selected_records_18_3b)
)

with threadpool_limits(limits=1):

    filtered_kmeans_18_3b = (
        analyze_new_batch_areas(
            selected_records_18_3b,
            categories,
            k=3,
            min_area_size=20,
            random_state=18
        )
    )

    filtered_dbscan_18_3b = (
        analyze_new_batch_spatial(
            selected_records_18_3b,
            categories,
            eps_m=125,
            min_samples=5,
            min_report_count=5,
            min_distinct_locations=2
        )
    )

print(
    "K-Means status:",
    filtered_kmeans_18_3b["status"]
)

print(
    "DBSCAN total processed:",
    filtered_dbscan_18_3b["stats"][
        "total_records"
    ]
)

print(
    "DBSCAN candidates:",
    filtered_dbscan_18_3b["stats"][
        "reportable_candidates"
    ]
)

# --------------------------------------------------
# 7. Test invalid input safeguards
# --------------------------------------------------

invalid_target_rejected = False

try:
    select_daily_complaints(
        source,
        "2026-02-30"
    )
except ValueError:
    invalid_target_rejected = True

duplicated_rows = pd.concat([
    source.iloc[:2],
    source.iloc[[0]]
], ignore_index=True)

duplicate_rejected = False

try:
    select_daily_complaints(
        duplicated_rows,
        REPORTING_DAY
    )
except ValueError:
    duplicate_rejected = True

empty_output = select_daily_complaints(
    source.iloc[:0],
    REPORTING_DAY
)

# --------------------------------------------------
# 8. Final validation checklist
# --------------------------------------------------

checks = []

def check(name, condition):
    checks.append({
        "test": name,
        "passed": bool(condition)
    })

check(
    "Original 500 reports retained",
    daily_output_18_3b["stats"][
        "input_records"
    ] == 500
)

check(
    "Original 500 reports selected for correct day",
    daily_output_18_3b["stats"][
        "selected_records"
    ] == 500
)

check(
    "Original batch contains no date exclusions",
    daily_output_18_3b["stats"][
        "outside_selected_day"
    ] == 0
    and daily_output_18_3b["stats"][
        "invalid_or_missing_date"
    ] == 0
)

check(
    "Original audit preserves source identities",
    daily_output_18_3b["audit"][
        "source_row"
    ].tolist()
    == source["source_row"].tolist()
)

check(
    "Mixed batch preserves 500 original records",
    mixed_stats["input_records"] == 500
)

check(
    "Mixed batch selects 495 NYC-day complaints",
    mixed_stats["selected_records"] == 495
)

check(
    "Three other-day complaints excluded",
    mixed_stats["outside_selected_day"] == 3
)

check(
    "Two invalid dates excluded with reasons",
    mixed_stats["invalid_or_missing_date"] == 2
)

check(
    "UTC November 10 timestamp maps to NYC November 9",
    mixed_output_18_3b["audit"].iloc[5][
        "nyc_reporting_day"
    ] == "2026-11-09"
)

check(
    "UTC November 9 early timestamp maps to NYC November 8",
    mixed_output_18_3b["audit"].iloc[6][
        "nyc_reporting_day"
    ] == "2026-11-08"
)

check(
    "Both explicit UTC conversions recorded",
    mixed_stats["timezone_conversions"] == 3
)

check(
    "Ambiguous DST timestamp rejected safely",
    timezone_audit_18_3b.iloc[5][
        "parse_status"
    ] == "invalid_or_missing_date"
)

check(
    "Selected batch has no mixed NYC reporting days",
    mixed_output_18_3b["audit"].loc[
        lambda df: df[
            "daily_window_status"
        ].eq("selected_day"),
        "nyc_reporting_day"
    ].eq(REPORTING_DAY).all()
)

check(
    "Selected records keep original source IDs",
    set(selected_records_18_3b["source_row"])
    == set(
        mixed_output_18_3b["audit"].loc[
            lambda df: df[
                "daily_window_status"
            ].eq("selected_day"),
            "source_row"
        ]
    )
)

check(
    "K-Means receives only selected 495 records",
    filtered_kmeans_18_3b["stats"][
        "total_records"
    ] == 495
)

check(
    "Filtered K-Means still identifies 3 clusters",
    filtered_kmeans_18_3b["status"]
    == "clustered"
    and len(
        filtered_kmeans_18_3b[
            "cluster_profiles"
        ]
    ) == 3
)

check(
    "DBSCAN receives only selected 495 records",
    filtered_dbscan_18_3b["stats"][
        "total_records"
    ] == 495
)

check(
    "Filtered DBSCAN statuses reconcile",
    sum([
        filtered_dbscan_18_3b["stats"][
            key
        ]
        for key in [
            "candidate_complaints",
            "below_reporting_threshold",
            "unclustered",
            "excluded_records"
        ]
    ]) == 495
)

check(
    "Invalid requested reporting day rejected",
    invalid_target_rejected
)

check(
    "Duplicate source rows rejected",
    duplicate_rejected
)

check(
    "Empty batch handled safely",
    empty_output["stats"][
        "input_records"
    ] == 0
    and empty_output[
        "selected_records"
    ].empty
)

check(
    "Original input DataFrame unchanged",
    source.equals(source_snapshot)
)

check(
    "Mixed-day test input unchanged",
    mixed_source.equals(
        mixed_input_snapshot
    )
)

check(
    "All protected files remain unchanged",
    all(
        hashlib.sha256(
            path.read_bytes()
        ).hexdigest() == original_hashes[path]
        for path in protected
    )
)

results_18_3b = pd.DataFrame(checks)

print("\n5. VALIDATION")

display(results_18_3b)

passed = int(results_18_3b["passed"].sum())
total = len(results_18_3b)

print(f"\nChecks passed: {passed}/{total}")

failures = results_18_3b.loc[
    ~results_18_3b["passed"], "test"
].tolist()

# --------------------------------------------------
# 9. Save only after all tests pass
# --------------------------------------------------

if failures:

    print("\nSTATUS: NEEDS REVIEW")
    print("No new module saved.")

    for failure in failures:
        print("FAILED:", failure)

else:

    if module_path.exists():

        assert module_path.read_text(
            encoding="utf-8"
        ) == module_source, (
            "An existing daily-window module differs. "
            "No overwrite performed."
        )

    else:

        module_path.parent.mkdir(
            parents=True,
            exist_ok=True
        )

        module_path.write_text(
            module_source,
            encoding="utf-8"
        )

    print("\n6. VERIFIED REUSABLE MODULE")
    print(module_path)

    print("\nSTATUS: PASSED")
    print("DAILY WINDOW VERIFIED")
    print("NYC LOCAL REPORTING DAY ENFORCED")
    print("CROSS-DAY RECORDS EXCLUDED FROM MODELS")
    print("INVALID DATES RETAINED IN AUDIT")
    print("K-MEANS AND DBSCAN FILTERED CORRECTLY")
    print("HISTORICAL MODELS UNMODIFIED")


 DAILY BATCH WINDOW ENFORCEMENT

1. ORIGINAL DAILY BATCH
{'reporting_day': '2026-11-09', 'timezone': 'America/New_York', 'input_records': 500, 'selected_records': 500, 'outside_selected_day': 0, 'invalid_or_missing_date': 0, 'naive_local_assumptions': 500, 'timezone_conversions': 0}


,status,complaints
0,selected_day,500



2. MIXED-DAY BATCH TEST
{'reporting_day': '2026-11-09', 'timezone': 'America/New_York', 'input_records': 500, 'selected_records': 495, 'outside_selected_day': 3, 'invalid_or_missing_date': 2, 'naive_local_assumptions': 495, 'timezone_conversions': 3}


,source_row,complaint_id,created_date,nyc_reporting_day,date_parse_status,daily_window_status
0,2,SIM-20261109-0001,2026-11-10,2026-11-10,assumed_nyc_local,outside_selected_day
1,3,SIM-20261109-0002,2026-11-10,2026-11-10,assumed_nyc_local,outside_selected_day
2,4,SIM-20261109-0003,not-a-valid-date,NaN,invalid_or_missing_date,invalid_or_missing_date
3,5,SIM-20261109-0004,not-a-valid-date,NaN,invalid_or_missing_date,invalid_or_missing_date
4,6,SIM-20261109-0005,2026-11-09T22:30:00+00:00,2026-11-09,converted_timezone,selected_day
5,7,SIM-20261109-0006,2026-11-10T02:30:00+00:00,2026-11-09,converted_timezone,selected_day
6,8,SIM-20261109-0007,2026-11-09T01:00:00+00:00,2026-11-08,converted_timezone,outside_selected_day
7,9,SIM-20261109-0008,2026-11-09,2026-11-09,assumed_nyc_local,selected_day
8,10,SIM-20261109-0009,2026-11-09,2026-11-09,assumed_nyc_local,selected_day
9,11,SIM-20261109-0010,2026-11-09,2026-11-09,assumed_nyc_local,selected_day



3. TIMEZONE EDGE CASES


,input,nyc_day,parse_status
0,2026-11-09T12:00:00,2026-11-09,assumed_nyc_local
1,2026-11-10T02:30:00+00:00,2026-11-09,converted_timezone
2,2026-11-09T01:00:00+00:00,2026-11-08,converted_timezone
3,2026-11-09,2026-11-09,assumed_nyc_local
4,11/09/2026 10:30:00 AM,2026-11-09,assumed_nyc_local
5,2026-11-01T01:30:00,NaN,invalid_or_missing_date



4. MODEL TEST WITH FILTERED DAILY COMPLAINTS
Selected complaints: 495
K-Means status: clustered
DBSCAN total processed: 495
DBSCAN candidates: 23

5. VALIDATION


,test,passed
0,Original 500 reports retained,True
1,Original 500 reports selected for correct day,True
2,Original batch contains no date exclusions,True
3,Original audit preserves source identities,True
4,Mixed batch preserves 500 original records,True
5,Mixed batch selects 495 NYC-day complaints,True
6,Three other-day complaints excluded,True
7,Two invalid dates excluded with reasons,True
8,UTC November 10 timestamp maps to NYC November 9,True
9,UTC November 9 early timestamp maps to NYC Nov...,True



Checks passed: 24/24

6. VERIFIED REUSABLE MODULE
D:\Urban-Infrastructure-Pulse-Intelligence\src\integration\daily_batch_window.py

STATUS: PASSED
DAILY WINDOW VERIFIED
NYC LOCAL REPORTING DAY ENFORCED
CROSS-DAY RECORDS EXCLUDED FROM MODELS
INVALID DATES RETAINED IN AUDIT
K-MEANS AND DBSCAN FILTERED CORRECTLY
HISTORICAL MODELS UNMODIFIED


In [22]:

from pathlib import Path
from textwrap import dedent
from threadpoolctl import threadpool_limits
import hashlib
import sys
import pandas as pd

print(" REUSABLE COMBINED INTELLIGENCE PIPELINE")
print("=" * 75)

root = Path(PROJECT_ROOT).resolve()

if str(root) not in sys.path:
    sys.path.insert(0, str(root))

module_path = (
    root / "src/integration/combined_batch_intelligence.py"
)

window_path = (
    root / "src/integration/daily_batch_window.py"
)

kmeans_path = (
    root / "src/models/new_batch_area_kmeans.py"
)

dbscan_path = (
    root / "src/models/new_batch_dbscan.py"
)

input_path = (
    root / "outputs/tests"
    / "phase_17_4c_backend_verified"
    / "verified_500_backend_results.csv"
)

taxonomy_path = (
    root / "outputs/tables/dbscan"
    / "dbscan_action_taxonomy_v1.csv"
)

protected = [
    window_path,
    kmeans_path,
    dbscan_path,
    input_path,
    taxonomy_path
]

# Check exact daily-window filename, because the
# previous output displayed an unusual path.
if not window_path.is_file():
    print("Expected:", window_path)
    print(
        "Available daily-window files:",
        [
            repr(p.name)
            for p in window_path.parent.glob(
                "daily_batch*"
            )
        ]
    )
    raise FileNotFoundError(
        "Daily-window module filename needs review."
    )

for path in protected:
    assert path.is_file(), f"Missing: {path}"

protected_hashes = {
    path: hashlib.sha256(
        path.read_bytes()
    ).hexdigest()
    for path in protected
}

# --------------------------------------------------
# 1. Define reusable combined-intelligence module
# --------------------------------------------------

module_source = dedent(r'''
"""Combine daily new-batch and historical complaint context.

Historical model results are used as existing
record-level context, never as new model inputs.

No infrastructure defects or priorities are inferred.
"""

import pandas as pd

from src.integration.daily_batch_window import (
    select_daily_complaints
)

from src.models.new_batch_area_kmeans import (
    analyze_new_batch_areas
)

from src.models.new_batch_dbscan import (
    analyze_new_batch_spatial
)


def _joined_unique(values):
    return " | ".join(
        sorted({
            str(value)
            for value in values.dropna()
            if str(value).strip()
        })
    )


def analyze_combined_daily_batch(
    records,
    supported_categories,
    reporting_day,
    k=3,
    min_area_size=20,
    eps_m=125,
    min_samples=5,
    min_report_count=5,
    min_distinct_locations=2,
    random_state=18
):
    """Analyze one NYC reporting day and retain full audit."""

    if not isinstance(records, pd.DataFrame):
        raise TypeError("records must be a DataFrame.")

    if records.columns.duplicated().any():
        raise ValueError("Duplicate column names.")

    source = records.copy().reset_index(drop=True)

    # 1. Select one NYC-local day.
    daily = select_daily_complaints(
        source,
        reporting_day
    )

    selected = daily["selected_records"]

    # 2. Fit the two independent new-batch models.
    km = analyze_new_batch_areas(
        selected,
        supported_categories,
        k=k,
        min_area_size=min_area_size,
        random_state=random_state
    )

    spatial = analyze_new_batch_spatial(
        selected,
        supported_categories,
        eps_m=eps_m,
        min_samples=min_samples,
        min_report_count=min_report_count,
        min_distinct_locations=min_distinct_locations
    )

    # 3. Prepare per-complaint K-Means eligibility.
    km_rows = (
        km["record_eligibility"][[
            "source_row",
            "area_id",
            "eligibility_reason"
        ]]
        .rename(columns={
            "area_id": "new_kmeans_area_id",
            "eligibility_reason":
                "new_kmeans_eligibility"
        })
    )

    # 4. Prepare area assignments. These columns
    # may not exist if K-Means could not fit.
    km_areas = km["area_summary"].copy()

    for column in [
        "new_batch_cluster",
        "new_batch_description"
    ]:
        if column not in km_areas.columns:
            km_areas[column] = pd.NA

    km_areas = km_areas[[
        "area_id",
        "incoming_complaints",
        "meets_minimum",
        "new_batch_cluster",
        "new_batch_description"
    ]].rename(columns={
        "area_id": "new_kmeans_area_id",
        "incoming_complaints":
            "new_kmeans_area_complaints",
        "meets_minimum":
            "new_kmeans_area_meets_minimum",
        "new_batch_cluster":
            "new_kmeans_cluster",
        "new_batch_description":
            "new_kmeans_description"
    })

    # 5. Prepare per-complaint DBSCAN results.
    spatial_rows = (
        spatial["records"][[
            "source_row",
            "record_eligibility",
            "new_batch_status",
            "raw_candidate_id",
            "new_batch_candidate_id"
        ]]
        .rename(columns={
            "record_eligibility":
                "new_dbscan_eligibility",
            "new_batch_status":
                "new_dbscan_status",
            "raw_candidate_id":
                "new_dbscan_raw_candidate_id",
            "new_batch_candidate_id":
                "new_dbscan_candidate_id"
        })
    )

    date_rows = daily["audit"][[
        "source_row",
        "nyc_reporting_day",
        "date_parse_status",
        "daily_window_status"
    ]]

    # 6. Use source_row, never model cluster IDs,
    # as the join key.
    combined = (
        source
        .merge(
            date_rows,
            on="source_row",
            how="left",
            validate="one_to_one",
            sort=False
        )
        .merge(
            km_rows,
            on="source_row",
            how="left",
            validate="one_to_one",
            sort=False
        )
        .merge(
            km_areas,
            on="new_kmeans_area_id",
            how="left",
            validate="many_to_one",
            sort=False
        )
        .merge(
            spatial_rows,
            on="source_row",
            how="left",
            validate="one_to_one",
            sort=False
        )
    )

    selected_mask = combined[
        "daily_window_status"
    ].eq("selected_day")

    # 7. Preserve excluded date records explicitly.
    for column in [
        "new_kmeans_eligibility",
        "new_dbscan_eligibility",
        "new_dbscan_status"
    ]:
        combined.loc[
            ~selected_mask, column
        ] = "not_in_reporting_day"

    combined["new_kmeans_fit_status"] = (
        "not_in_reporting_day"
    )

    combined.loc[
        selected_mask,
        "new_kmeans_fit_status"
    ] = km["status"]

    # 8. Build fresh candidate context. Do not
    # reuse briefs from a different reporting day.
    candidate_rows = combined.loc[
        combined["new_dbscan_status"].eq(
            "candidate_concentration"
        )
    ].copy()

    context_columns = [
        "new_dbscan_candidate_id",
        "associated_kmeans_areas",
        "linked_kmeans_area_ids",
        "linked_kmeans_patterns"
    ]

    if candidate_rows.empty:

        context = pd.DataFrame(
            columns=context_columns
        )

    else:

        context = (
            candidate_rows.groupby(
                "new_dbscan_candidate_id"
            )
            .agg(
                associated_kmeans_areas=(
                    "new_kmeans_area_id",
                    "nunique"
                ),
                linked_kmeans_area_ids=(
                    "new_kmeans_area_id",
                    _joined_unique
                ),
                linked_kmeans_patterns=(
                    "new_kmeans_description",
                    _joined_unique
                )
            )
            .reset_index()
        )

    candidate_intelligence = (
        spatial["candidate_profiles"]
        .merge(
            context,
            left_on="new_batch_candidate_id",
            right_on="new_dbscan_candidate_id",
            how="left",
            validate="one_to_one"
        )
        .drop(
            columns=["new_dbscan_candidate_id"]
        )
    )

    guidance = {
        "SEWER":
            "Review sewer complaint details and "
            "relevant maintenance history.",
        "STREET CONDITION":
            "Review roadway complaint descriptions "
            "and reported locations.",
        "STREET LIGHT CONDITION":
            "Verify whether reports describe "
            "the same lighting assets.",
        "TRAFFIC SIGNAL CONDITION":
            "Verify reported intersections "
            "and traffic-signal equipment."
    }

    candidate_intelligence[
        "suggested_review"
    ] = (
        candidate_intelligence[
            "complaint_category"
        ]
        .map(guidance)
        .fillna(
            "Review the complaint descriptions "
            "and submitted locations."
        )
        + " Check for duplicate reports and "
        + "confirm evidence before field action."
    )

    # 9. Reconcile full original input coverage.
    if len(combined) != len(source):
        raise AssertionError(
            "Combined output changed record count."
        )

    if (
        combined["source_row"].tolist()
        != source["source_row"].tolist()
    ):
        raise AssertionError(
            "Complaint order or identity changed."
        )

    if (
        spatial["stats"]["total_records"]
        != daily["stats"]["selected_records"]
    ):
        raise AssertionError(
            "DBSCAN did not receive the selected day."
        )

    if (
        km["stats"]["total_records"]
        != daily["stats"]["selected_records"]
    ):
        raise AssertionError(
            "K-Means did not receive the selected day."
        )

    if (
        candidate_intelligence[
            "incoming_complaints"
        ].sum() != spatial["stats"][
            "candidate_complaints"
        ]
    ):
        raise AssertionError(
            "Candidate complaint counts disagree."
        )

    stats = {
        "input_records": len(source),
        "reporting_day": reporting_day,
        "selected_records":
            daily["stats"]["selected_records"],
        "date_exclusions": (
            daily["stats"]["outside_selected_day"]
            + daily["stats"][
                "invalid_or_missing_date"
            ]
        ),
        "kmeans_status": km["status"],
        "kmeans_eligible_areas":
            km["stats"]["eligible_areas"],
        "kmeans_assigned_complaints": int(
            combined[
                "new_kmeans_cluster"
            ].notna().sum()
        ),
        "dbscan_reportable_candidates":
            spatial["stats"][
                "reportable_candidates"
            ],
        "dbscan_candidate_complaints":
            spatial["stats"][
                "candidate_complaints"
            ],
        "dbscan_below_reporting_threshold":
            spatial["stats"][
                "below_reporting_threshold"
            ],
        "dbscan_unclustered":
            spatial["stats"]["unclustered"],
        "dbscan_excluded":
            spatial["stats"]["excluded_records"]
    }

    return {
        "combined_records": combined,
        "selected_records": selected.copy(),
        "date_audit": daily["audit"],
        "area_summary": km["area_summary"],
        "candidate_intelligence":
            candidate_intelligence,
        "kmeans_output": km,
        "dbscan_output": spatial,
        "stats": stats
    }
''').lstrip()

compile(
    module_source,
    str(module_path),
    "exec"
)

# Test in memory before writing a new module.
namespace = {}

exec(
    compile(
        module_source,
        str(module_path),
        "exec"
    ),
    namespace
)

analyze_combined_daily_batch = namespace[
    "analyze_combined_daily_batch"
]

# --------------------------------------------------
# 2. Load existing synthetic test evidence
# --------------------------------------------------

source = pd.read_csv(
    input_path,
    dtype="string",
    keep_default_na=False
)

taxonomy = pd.read_csv(
    taxonomy_path
)

categories = sorted({
    str(value).strip().upper()
    for value in taxonomy[
        "complaint_category"
    ].dropna()
})

source_snapshot = source.copy(deep=True)

DAY = "2026-11-09"

# --------------------------------------------------
# 3. Run complete 500-record integration
# --------------------------------------------------

print("\n1. COMPLETE DAILY INTEGRATION")

with threadpool_limits(limits=1):

    full = analyze_combined_daily_batch(
        source,
        categories,
        DAY
    )

print("\nSUMMARY")

for key, value in full["stats"].items():
    print(f"{key}: {value}")

print("\n2. COMBINED RECORD PREVIEW")

display(
    full["combined_records"][[
        "complaint_id",
        "nyc_reporting_day",
        "new_kmeans_area_id",
        "new_kmeans_description",
        "new_dbscan_status",
        "new_dbscan_candidate_id",
        "dbscan_status"
    ]].head(10)
)

print("\n3. COMBINED CANDIDATE INTELLIGENCE")

display(
    full["candidate_intelligence"][[
        "new_batch_candidate_id",
        "complaint_category",
        "incoming_complaints",
        "historical_outside",
        "associated_kmeans_areas",
        "linked_kmeans_patterns"
    ]].head(15)
)

# --------------------------------------------------
# 4. Mixed-day input with audit preservation
# --------------------------------------------------

print("\n4. MIXED-DAY INTEGRATION")

mixed = source.copy()

mixed.loc[[0, 1], "created_date"] = (
    "2026-11-10"
)

mixed.loc[[2, 3], "created_date"] = (
    "invalid-date"
)

mixed.loc[
    4, "created_date"
] = "2026-11-09T22:30:00+00:00"

mixed.loc[
    5, "created_date"
] = "2026-11-10T02:30:00+00:00"

mixed.loc[
    6, "created_date"
] = "2026-11-09T01:00:00+00:00"

with threadpool_limits(limits=1):

    mixed_result = analyze_combined_daily_batch(
        mixed,
        categories,
        DAY
    )

print(mixed_result["stats"])

display(
    mixed_result["combined_records"][
        "daily_window_status"
    ]
    .value_counts()
    .rename_axis("status")
    .reset_index(name="complaints")
)

# --------------------------------------------------
# 5. Small-day and empty-day safeguards
# --------------------------------------------------

print("\n5. SMALL-BATCH SAFEGUARD")

single_area = (
    source["borough"]
    + "_"
    + source["incident_zip"]
)

first_area = single_area.iloc[0]

small_day = source.loc[
    single_area.eq(first_area)
].copy()

with threadpool_limits(limits=1):

    small_result = analyze_combined_daily_batch(
        small_day,
        categories,
        DAY
    )

print(
    "Small-day K-Means:",
    small_result["stats"]["kmeans_status"]
)

print(
    "Small-day DBSCAN records:",
    small_result["dbscan_output"][
        "stats"
    ]["total_records"]
)

empty_day = analyze_combined_daily_batch(
    source.iloc[:0].copy(),
    categories,
    DAY
)

# --------------------------------------------------
# 6. Check all key integration requirements
# --------------------------------------------------

checks = []

def check(name, condition):
    checks.append({
        "test": name,
        "passed": bool(condition)
    })

full_records = full["combined_records"]
mixed_records = mixed_result["combined_records"]

check(
    "Original 500 records preserved",
    len(full_records) == 500
)

check(
    "All 500 source rows remain unique",
    full_records["source_row"].is_unique
)

check(
    "Original complaint order preserved",
    full_records["complaint_id"].tolist()
    == source["complaint_id"].tolist()
)

check(
    "All 500 belong to selected NYC day",
    full["stats"]["selected_records"] == 500
)

check(
    "All 500 receive new K-Means assignments",
    full["stats"][
        "kmeans_assigned_complaints"
    ] == 500
)

check(
    "K-Means identifies 20 qualifying areas",
    full["stats"]["kmeans_eligible_areas"]
    == 20
)

check(
    "New K-Means model fits three clusters",
    full["stats"]["kmeans_status"]
    == "clustered"
    and len(
        full["kmeans_output"][
            "cluster_profiles"
        ]
    ) == 3
)

check(
    "23 reportable DBSCAN candidates preserved",
    full["stats"][
        "dbscan_reportable_candidates"
    ] == 23
)

check(
    "228 candidate complaints preserved",
    full["stats"][
        "dbscan_candidate_complaints"
    ] == 228
)

check(
    "Eight below reporting threshold",
    full["stats"][
        "dbscan_below_reporting_threshold"
    ] == 8
)

check(
    "264 DBSCAN unclustered complaints",
    full["stats"]["dbscan_unclustered"]
    == 264
)

check(
    "All 23 candidates have combined context",
    len(full["candidate_intelligence"]) == 23
    and full["candidate_intelligence"][
        "linked_kmeans_patterns"
    ].str.len().gt(0).all()
)

check(
    "Historical DBSCAN outcomes preserved",
    full_records["dbscan_status"]
    .value_counts().to_dict()
    == source["dbscan_status"]
    .value_counts().to_dict()
)

check(
    "Historical K-Means outcomes preserved",
    full_records["kmeans_reporting_profile"]
    .tolist()
    == source["kmeans_reporting_profile"]
    .tolist()
)

check(
    "Mixed input still retains all 500 records",
    len(mixed_records) == 500
)

check(
    "Mixed input selects exactly 495 records",
    mixed_result["stats"][
        "selected_records"
    ] == 495
)

check(
    "Five date exclusions remain auditable",
    mixed_result["stats"][
        "date_exclusions"
    ] == 5
)

excluded = mixed_records.loc[
    mixed_records["daily_window_status"].ne(
        "selected_day"
    )
]

check(
    "Other-day complaints have no new model status",
    len(excluded) == 5
    and excluded["new_dbscan_status"].eq(
        "not_in_reporting_day"
    ).all()
    and excluded["new_kmeans_fit_status"].eq(
        "not_in_reporting_day"
    ).all()
)

check(
    "Other-day complaints have no new clusters",
    excluded["new_kmeans_cluster"].isna().all()
    and excluded[
        "new_dbscan_candidate_id"
    ].isna().all()
)

check(
    "Mixed-day K-Means uses only 495 records",
    mixed_result["kmeans_output"]["stats"][
        "total_records"
    ] == 495
)

check(
    "Mixed-day DBSCAN uses only 495 records",
    mixed_result["dbscan_output"]["stats"][
        "total_records"
    ] == 495
)

check(
    "Single-area K-Means safely declines fitting",
    small_result["stats"]["kmeans_status"]
    == "insufficient_areas"
)

check(
    "Single-area DBSCAN still analyzes 25 reports",
    small_result["dbscan_output"]["stats"][
        "total_records"
    ] == 25
)

check(
    "Empty day handled safely",
    empty_day["stats"]["input_records"] == 0
    and empty_day["combined_records"].empty
)

check(
    "Original DataFrame remains unchanged",
    source.equals(source_snapshot)
)

check(
    "All protected files remain unchanged",
    all(
        hashlib.sha256(
            path.read_bytes()
        ).hexdigest() == protected_hashes[path]
        for path in protected
    )
)

# --------------------------------------------------
# 7. Report test results
# --------------------------------------------------

results_18_3c = pd.DataFrame(checks)

print("\n6. VALIDATION")

display(results_18_3c)

passed = int(
    results_18_3c["passed"].sum()
)

total = len(results_18_3c)

print(f"\nChecks passed: {passed}/{total}")

failed = results_18_3c.loc[
    ~results_18_3c["passed"], "test"
].tolist()

# --------------------------------------------------
# 8. Save only the fully validated new module
# --------------------------------------------------

if failed:

    print("\nSTATUS: NEEDS REVIEW")
    print("No combined module saved.")

    for failure in failed:
        print("FAILED:", failure)

else:

    if module_path.exists():

        assert module_path.read_text(
            encoding="utf-8"
        ) == module_source, (
            "Existing combined module differs. "
            "No overwrite performed."
        )

    else:

        module_path.parent.mkdir(
            parents=True,
            exist_ok=True
        )

        module_path.write_text(
            module_source,
            encoding="utf-8"
        )

    print("\n7. VERIFIED MODULE")
    print(module_path)

    print("\nSTATUS: PASSED")
    print("COMBINED ENGINE VERIFIED")
    print("DAILY NYC REPORTING WINDOW ENFORCED")
    print("NEW K-MEANS AND DBSCAN INTEGRATED")
    print("HISTORICAL MODEL CONTEXT PRESERVED")
    print("DATE-EXCLUDED COMPLAINTS RETAINED")
    print("SMALL-DAY SAFEGUARDS VERIFIED")
    print("NO HISTORICAL MODEL MODIFICATIONS")


 REUSABLE COMBINED INTELLIGENCE PIPELINE

1. COMPLETE DAILY INTEGRATION

SUMMARY
input_records: 500
reporting_day: 2026-11-09
selected_records: 500
date_exclusions: 0
kmeans_status: clustered
kmeans_eligible_areas: 20
kmeans_assigned_complaints: 500
dbscan_reportable_candidates: 23
dbscan_candidate_complaints: 228
dbscan_below_reporting_threshold: 8
dbscan_unclustered: 264
dbscan_excluded: 0

2. COMBINED RECORD PREVIEW


,complaint_id,nyc_reporting_day,new_kmeans_area_id,new_kmeans_description,new_dbscan_status,new_dbscan_candidate_id,dbscan_status
0,SIM-20261109-0001,2026-11-09,BRONX_10451,Road-heavy reporting,unclustered,None,outside_existing_clusters
1,SIM-20261109-0002,2026-11-09,BRONX_10451,Road-heavy reporting,candidate_concentration,NB-03-000,outside_existing_clusters
2,SIM-20261109-0003,2026-11-09,BRONX_10451,Road-heavy reporting,unclustered,None,established_cluster
3,SIM-20261109-0004,2026-11-09,BRONX_10451,Road-heavy reporting,unclustered,None,outside_existing_clusters
4,SIM-20261109-0005,2026-11-09,BRONX_10451,Road-heavy reporting,candidate_concentration,NB-03-000,established_cluster
5,SIM-20261109-0006,2026-11-09,BRONX_10451,Road-heavy reporting,candidate_concentration,NB-03-000,established_cluster
6,SIM-20261109-0007,2026-11-09,BRONX_10451,Road-heavy reporting,candidate_concentration,NB-03-000,outside_existing_clusters
7,SIM-20261109-0008,2026-11-09,BRONX_10451,Road-heavy reporting,candidate_concentration,NB-03-000,outside_existing_clusters
8,SIM-20261109-0009,2026-11-09,BRONX_10451,Road-heavy reporting,unclustered,None,established_cluster
9,SIM-20261109-0010,2026-11-09,BRONX_10451,Road-heavy reporting,below_reporting_threshold,None,outside_existing_clusters



3. COMBINED CANDIDATE INTELLIGENCE


,new_batch_candidate_id,complaint_category,incoming_complaints,historical_outside,associated_kmeans_areas,linked_kmeans_patterns
0,NB-00-000,SEWER,15,12,1,Sewer-heavy reporting
1,NB-00-003,SEWER,15,0,1,Sewer-heavy reporting
2,NB-00-004,SEWER,15,15,1,Sewer-heavy reporting
3,NB-03-010,STREET CONDITION,14,14,1,Road-heavy reporting
4,NB-03-004,STREET CONDITION,13,13,1,Road-heavy reporting
5,NB-03-005,STREET CONDITION,13,10,1,Road-heavy reporting
6,NB-03-006,STREET CONDITION,13,0,1,Road-heavy reporting
7,NB-03-003,STREET CONDITION,12,2,1,Road-heavy reporting
8,NB-04-002,STREET LIGHT CONDITION,12,12,1,Lighting/signal-heavy reporting
9,NB-00-001,SEWER,11,5,1,Sewer-heavy reporting



4. MIXED-DAY INTEGRATION
{'input_records': 500, 'reporting_day': '2026-11-09', 'selected_records': 495, 'date_exclusions': 5, 'kmeans_status': 'clustered', 'kmeans_eligible_areas': 20, 'kmeans_assigned_complaints': 495, 'dbscan_reportable_candidates': 23, 'dbscan_candidate_complaints': 226, 'dbscan_below_reporting_threshold': 8, 'dbscan_unclustered': 261, 'dbscan_excluded': 0}


,status,complaints
0,selected_day,495
1,outside_selected_day,3
2,invalid_or_missing_date,2



5. SMALL-BATCH SAFEGUARD
Small-day K-Means: insufficient_areas
Small-day DBSCAN records: 25

6. VALIDATION


,test,passed
0,Original 500 records preserved,True
1,All 500 source rows remain unique,True
2,Original complaint order preserved,True
3,All 500 belong to selected NYC day,True
4,All 500 receive new K-Means assignments,True
5,K-Means identifies 20 qualifying areas,True
6,New K-Means model fits three clusters,True
7,23 reportable DBSCAN candidates preserved,True
8,228 candidate complaints preserved,True
9,Eight below reporting threshold,True



Checks passed: 26/26

7. VERIFIED MODULE
D:\Urban-Infrastructure-Pulse-Intelligence\src\integration\combined_batch_intelligence.py

STATUS: PASSED
COMBINED ENGINE VERIFIED
DAILY NYC REPORTING WINDOW ENFORCED
NEW K-MEANS AND DBSCAN INTEGRATED
HISTORICAL MODEL CONTEXT PRESERVED
DATE-EXCLUDED COMPLAINTS RETAINED
SMALL-DAY SAFEGUARDS VERIFIED
NO HISTORICAL MODEL MODIFICATIONS


## Anomaly input to the bulk intelligence pipeline — integration only

The historical baseline generation, anomaly rule experiments, engine-writing patches and
threshold calibration that formerly appeared here are owned by **Notebook 07**.
Notebook 06 does **not** recreate the anomaly engine or historical snapshot.

The existing deployed module and baseline are required by the integration pipeline. This
cell verifies that those interfaces are present without modifying or retraining them.


In [ ]:
from pathlib import Path
import importlib

# Keep the original project-root setup from the first cell.
root = Path(PROJECT_ROOT).resolve()
engine_path = root / "src/models/new_batch_anomaly_screen.py"
snapshot_path = root / "outputs/tables/anomaly/historical_daily_counts_2023_2025_v1.csv.gz"
manifest_path = root / "outputs/tables/anomaly/historical_daily_counts_2023_2025_v1_manifest.json"
for required in (engine_path, snapshot_path, manifest_path):
    assert required.is_file(), f"Missing deployed anomaly dependency: {required}"

from src.models.new_batch_anomaly_screen import screen_daily_anomalies
from src.integration.anomaly_snapshot_integration import (
    load_anomaly_snapshot, analyze_batch_with_anomaly_context
)
assert callable(screen_daily_anomalies) and callable(analyze_batch_with_anomaly_context)
print("PASS: production anomaly engine imported; development is maintained in Notebook 07")
print("PASS: historical anomaly reference files present")
print("PASS: three-model integration entry point is available")
print("No model, baseline or dashboard files were written")


### Integrate All Three Models

In [ ]:
# Integration smoke test replaces the old one-off code patch.
# Notebook 06 consumes the deployed model; it does not modify the model or integration source.
from pathlib import Path
from threadpoolctl import threadpool_limits
import pandas as pd

root = Path(PROJECT_ROOT).resolve()
source_path = root / "outputs/tests/phase_17_4c_backend_verified/verified_500_backend_results.csv"
snapshot_path = root / "outputs/tables/anomaly/historical_daily_counts_2023_2025_v1.csv.gz"
manifest_path = root / "outputs/tables/anomaly/historical_daily_counts_2023_2025_v1_manifest.json"
for required in (source_path, snapshot_path, manifest_path):
    assert required.is_file(), f"Missing integration test input: {required}"
from src.integration.anomaly_snapshot_integration import (
    load_anomaly_snapshot, analyze_batch_with_anomaly_context
)
baseline = load_anomaly_snapshot(snapshot_path, manifest_path)
processed = pd.read_csv(source_path, dtype="string", keep_default_na=False)
with threadpool_limits(limits=1):
    integrated = analyze_batch_with_anomaly_context(processed, baseline, "2026-11-09")
assert len(integrated["combined_records"]) == len(processed)
assert isinstance(integrated["stats"], dict)
print("PASS: original processed batch retained, combined anomaly/K-Means/DBSCAN integration works")
print("Reporting stats:", integrated["stats"])
print("No production files changed")


In [33]:

from pathlib import Path
from textwrap import dedent
from threadpoolctl import threadpool_limits
import hashlib
import json
import sys
import pandas as pd
import numpy as np

print(" SMART ASSISTANT INVESTIGATION BRIEFS")
print("=" * 75)

root = Path(PROJECT_ROOT).resolve()

if str(root) not in sys.path:
    sys.path.insert(0, str(root))

from src.integration.anomaly_snapshot_integration import (
    load_anomaly_snapshot,
    analyze_batch_with_anomaly_context
)

module_path = (
    root / "src/integration"
    / "smart_assistant_investigation_briefs.py"
)

output_dir = (
    root / "outputs/tests"
    / "phase_19_2d_investigation_briefs"
)

bundle_path = output_dir / "investigation_bundle.json"
map_path = output_dir / "spatial_candidates_map.geojson"

snapshot_path = (
    root / "outputs/tables/anomaly"
    / "historical_daily_counts_2023_2025_v1.csv.gz"
)

manifest_path = (
    root / "outputs/tables/anomaly"
    / "historical_daily_counts_2023_2025_v1_manifest.json"
)

input_path = (
    root / "outputs/tests"
    / "phase_17_4c_backend_verified"
    / "verified_500_backend_results.csv"
)

protected = [
    snapshot_path,
    manifest_path,
    input_path,
    root / "src/integration"
    / "anomaly_snapshot_integration.py",
    root / "src/integration"
    / "combined_batch_intelligence.py",
    root / "src/models/new_batch_area_kmeans.py",
    root / "src/models/new_batch_dbscan.py",
    root / "src/models/new_batch_anomaly_screen.py"
]

for path in protected:
    assert path.is_file(), f"Missing: {path}"

original_hashes = {
    path: hashlib.sha256(
        path.read_bytes()
    ).hexdigest()
    for path in protected
}

# --------------------------------------------------
# 1. Define the reusable explanation module
# --------------------------------------------------

module_source = dedent(r'''
"""Convert three-model output into investigation briefs.

Explanations are descriptive screening evidence,
not confirmed infrastructure faults, risk scores,
statistical significance or repair priorities.
"""

import math
import numpy as np
import pandas as pd


def _unique(values):
    return sorted({
        str(value).strip()
        for value in values.dropna()
        if str(value).strip()
    })


def _historical_relationship(group):

    statuses = group[
        "dbscan_status"
    ].astype(str)

    established = int(
        statuses.eq("established_cluster").sum()
    )

    outside = int(
        statuses.eq(
            "outside_existing_clusters"
        ).sum()
    )

    ambiguous = int(
        statuses.eq(
            "ambiguous_association"
        ).sum()
    )

    other = (
        len(group)
        - established
        - outside
        - ambiguous
    )

    if outside == len(group):
        relationship = (
            "entirely_outside_historical_associations"
        )

    elif established == len(group):
        relationship = (
            "entirely_associated_with_historical_clusters"
        )

    else:
        relationship = (
            "mixed_or_uncertain_historical_associations"
        )

    return {
        "relationship": relationship,
        "established": established,
        "outside": outside,
        "ambiguous": ambiguous,
        "other": other
    }


def _representative_location(group):
    """Return an actual submitted coordinate."""

    coordinates = group[[
        "latitude",
        "longitude"
    ]].apply(
        pd.to_numeric,
        errors="coerce"
    )

    if coordinates.isna().any().any():
        raise ValueError(
            "Invalid spatial candidate coordinates"
        )

    values = coordinates.to_numpy(
        dtype=float
    )

    if not np.isfinite(values).all():
        raise ValueError(
            "Non-finite candidate coordinates"
        )

    center_lat, center_lon = (
        values.mean(axis=0)
    )

    lat = np.radians(values[:, 0])
    lon = np.radians(values[:, 1])

    lat0 = math.radians(
        float(center_lat)
    )

    lon0 = math.radians(
        float(center_lon)
    )

    a = (
        np.sin((lat - lat0) / 2) ** 2
        + np.cos(lat0)
        * np.cos(lat)
        * np.sin((lon - lon0) / 2) ** 2
    )

    distances = (
        2
        * 6371008.8
        * np.arcsin(
            np.sqrt(np.clip(a, 0, 1))
        )
    )

    chosen = values[
        int(np.argmin(distances))
    ]

    distinct = len(
        np.unique(values, axis=0)
    )

    return (
        float(chosen[0]),
        float(chosen[1]),
        int(distinct)
    )


def build_investigation_briefs(
    integrated_result,
    synthetic_data=False
):
    """Build two types of candidate explanation.

    1. Area-category reporting-increase briefs.
    2. Local spatial concentration briefs.

    Also returns record audit and map-ready GeoJSON.
    """

    rows = integrated_result[
        "combined_records"
    ]

    anomaly_candidates = integrated_result[
        "anomaly_candidates"
    ]

    profiles = integrated_result[
        "kmeans_dbscan_output"
    ]["dbscan_output"]["candidate_profiles"]

    reporting_day = str(
        integrated_result[
            "stats"
        ]["reporting_day"]
    )

    required_columns = {
        "source_row",
        "complaint_id",
        "daily_window_status",
        "complaint_category",
        "latitude",
        "longitude",
        "anomaly_area_id",
        "anomaly_category",
        "anomaly_input_status",
        "anomaly_review_status",
        "anomaly_candidate",
        "new_kmeans_description",
        "new_dbscan_status",
        "new_dbscan_candidate_id",
        "dbscan_status"
    }

    missing = (
        required_columns
        - set(rows.columns)
    )

    if missing:
        raise ValueError(
            f"Missing record columns: {sorted(missing)}"
        )

    if rows["source_row"].duplicated().any():
        raise ValueError(
            "Duplicate source_row values"
        )

    if rows["anomaly_candidate"].isna().any():
        raise ValueError(
            "Missing anomaly candidate flags"
        )

    # ----------------------------------------------
    # 2. Area/category anomaly explanations
    # ----------------------------------------------

    anomaly_briefs = []

    for item in anomaly_candidates.itertuples(
        index=False
    ):

        area = str(item.area_id)
        category = str(
            item.complaint_category
        )

        group = rows.loc[
            rows["anomaly_area_id"].eq(area)
            & rows["anomaly_category"].eq(
                category
            )
            & rows["anomaly_input_status"].eq(
                "eligible"
            )
        ]

        count = len(group)

        if (
            count != int(item.incoming_reports)
            or not group[
                "anomaly_candidate"
            ].all()
        ):
            raise ValueError(
                f"Anomaly count mismatch: "
                f"{area}, {category}"
            )

        spatial_count = int(
            group["new_dbscan_status"]
            .eq("candidate_concentration")
            .sum()
        )

        patterns = _unique(
            group["new_kmeans_description"]
        )

        spatial_ids = _unique(
            group["new_dbscan_candidate_id"]
        )

        historical = (
            _historical_relationship(group)
        )

        explanation = (
            f"{count} {category} complaints "
            f"were submitted in {area} on "
            f"{reporting_day}. The count exceeds "
            f"the historical seasonal maximum "
            f"of {int(item.seasonal_daily_max)} "
            f"and all-weekday maximum of "
            f"{int(item.all_weekday_daily_max)}. "
            f"{spatial_count} complaints also "
            f"belong to new-batch spatial "
            f"concentrations. Reporting pattern: "
            f"{', '.join(patterns) if patterns else 'unavailable'}. "
            f"This is an exploratory reporting "
            f"increase, not a confirmed defect."
        )

        anomaly_briefs.append({
            "evidence_type":
                "area_category_reporting_increase",
            "area_id": area,
            "complaint_category": category,
            "reporting_day": reporting_day,
            "incoming_complaints": count,
            "seasonal_historical_max":
                int(item.seasonal_daily_max),
            "all_weekday_historical_max":
                int(item.all_weekday_daily_max),
            "kmeans_reporting_patterns":
                patterns,
            "complaints_in_spatial_candidates":
                spatial_count,
            "linked_spatial_candidate_ids":
                spatial_ids,
            "historical_dbscan_associations":
                historical,
            "screening_status":
                "cross_window_consistent_candidate",
            "explanation": explanation,
            "suggested_review": (
                "Check report descriptions, "
                "duplicate submissions, location "
                "accuracy, category definitions "
                "and recent operational records "
                "before considering an investigation."
            ),
            "synthetic_data": bool(
                synthetic_data
            )
        })

    # ----------------------------------------------
    # 3. Spatial concentration explanations
    # ----------------------------------------------

    spatial_briefs = []
    map_features = []

    for profile in profiles.itertuples(
        index=False
    ):

        candidate_id = str(
            profile.new_batch_candidate_id
        )

        group = rows.loc[
            rows["new_dbscan_status"].eq(
                "candidate_concentration"
            )
            & rows[
                "new_dbscan_candidate_id"
            ].eq(candidate_id)
        ]

        count = len(group)

        if count != int(
            profile.incoming_complaints
        ):
            raise ValueError(
                f"Spatial count mismatch: "
                f"{candidate_id}"
            )

        categories = _unique(
            group["complaint_category"]
        )

        if (
            len(categories) != 1
            or categories[0] != str(
                profile.complaint_category
            )
        ):
            raise ValueError(
                f"Mixed spatial categories: "
                f"{candidate_id}"
            )

        lat, lon, distinct = (
            _representative_location(group)
        )

        if distinct != int(
            profile.distinct_locations
        ):
            raise ValueError(
                "Spatial location count mismatch"
            )

        historical = (
            _historical_relationship(group)
        )

        anomaly_overlap = int(
            group["anomaly_candidate"].sum()
        )

        area_ids = _unique(
            group["anomaly_area_id"]
        )

        patterns = _unique(
            group["new_kmeans_description"]
        )

        linked_anomaly_groups = sorted({
            (
                f"{row.anomaly_area_id} / "
                f"{row.anomaly_category}"
            )
            for row in group.loc[
                group["anomaly_candidate"]
            ].itertuples(index=False)
        })

        explanation = (
            f"{count} {categories[0]} complaints "
            f"at {distinct} distinct submitted "
            f"coordinates form spatial candidate "
            f"{candidate_id}. Historical "
            f"association: {historical['outside']} "
            f"outside, {historical['established']} "
            f"established, "
            f"{historical['ambiguous']} ambiguous "
            f"and {historical['other']} other. "
            f"{anomaly_overlap} members also "
            f"belong to area-category reporting "
            f"increase candidates. The map pin "
            f"is a submitted complaint coordinate, "
            f"not a verified infrastructure fault."
        )

        spatial_briefs.append({
            "evidence_type":
                "new_batch_spatial_concentration",
            "candidate_id": candidate_id,
            "complaint_category":
                categories[0],
            "reporting_day": reporting_day,
            "incoming_complaints": count,
            "distinct_submitted_locations":
                distinct,
            "representative_latitude":
                round(lat, 7),
            "representative_longitude":
                round(lon, 7),
            "area_ids": area_ids,
            "kmeans_reporting_patterns":
                patterns,
            "complaints_in_anomaly_candidate_groups":
                anomaly_overlap,
            "linked_anomaly_area_categories":
                linked_anomaly_groups,
            "historical_dbscan_associations":
                historical,
            "explanation": explanation,
            "suggested_review": (
                "Review nearby report descriptions, "
                "duplicate coordinates and location "
                "accuracy. Verify conditions before "
                "any field action."
            ),
            "synthetic_data": bool(
                synthetic_data
            )
        })

        map_features.append({
            "type": "Feature",
            "id": candidate_id,
            "geometry": {
                "type": "Point",
                "coordinates": [lon, lat]
            },
            "properties": {
                "candidate_id": candidate_id,
                "complaint_category":
                    categories[0],
                "incoming_complaints": count,
                "distinct_locations": distinct,
                "historical_relationship":
                    historical["relationship"],
                "complaints_in_anomaly_candidate_groups":
                    anomaly_overlap,
                "synthetic_data": bool(
                    synthetic_data
                )
            }
        })

    # ----------------------------------------------
    # 4. Traceable status for every original record
    # ----------------------------------------------

    audit_rows = []

    for item in rows.itertuples(
        index=False
    ):

        audit_rows.append({
            "source_row": str(
                item.source_row
            ),
            "complaint_id": str(
                item.complaint_id
            ),
            "daily_window_status": str(
                item.daily_window_status
            ),
            "anomaly_input_status": str(
                item.anomaly_input_status
            ),
            "anomaly_review_status": str(
                item.anomaly_review_status
            ),
            "anomaly_candidate": bool(
                item.anomaly_candidate
            ),
            "new_dbscan_status": str(
                item.new_dbscan_status
            ),
            "new_dbscan_candidate_id": (
                None
                if pd.isna(
                    item.new_dbscan_candidate_id
                )
                else str(
                    item.new_dbscan_candidate_id
                )
            )
        })

    # ----------------------------------------------
    # 5. Reconciliation
    # ----------------------------------------------

    stats = {
        "input_records": len(rows),
        "reporting_day": reporting_day,
        "anomaly_candidate_groups":
            len(anomaly_briefs),
        "anomaly_candidate_complaints":
            int(
                rows["anomaly_candidate"].sum()
            ),
        "spatial_candidate_groups":
            len(spatial_briefs),
        "spatial_candidate_complaints":
            int(
                rows["new_dbscan_status"]
                .eq("candidate_concentration")
                .sum()
            ),
        "spatial_groups_entirely_outside_history":
            sum(
                brief[
                    "historical_dbscan_associations"
                ]["relationship"]
                == "entirely_outside_historical_associations"
                for brief in spatial_briefs
            ),
        "synthetic_data": bool(
            synthetic_data
        )
    }

    if (
        stats["anomaly_candidate_complaints"]
        != sum(
            item["incoming_complaints"]
            for item in anomaly_briefs
        )
    ):
        raise ValueError(
            "Anomaly brief totals do not reconcile"
        )

    if (
        stats["spatial_candidate_complaints"]
        != sum(
            item["incoming_complaints"]
            for item in spatial_briefs
        )
    ):
        raise ValueError(
            "Spatial brief totals do not reconcile"
        )

    return {
        "metadata": {
            "reporting_day": reporting_day,
            "synthetic_data": bool(
                synthetic_data
            ),
            "interpretation": (
                "Exploratory reporting evidence, "
                "not verified infrastructure faults, "
                "risk scores or automatic repair "
                "priorities."
            )
        },
        "stats": stats,
        "anomaly_briefs": anomaly_briefs,
        "spatial_briefs": spatial_briefs,
        "record_audit": audit_rows,
        "map_geojson": {
            "type": "FeatureCollection",
            "name":
                "new_batch_spatial_candidate_locations",
            "features": map_features
        }
    }
''').lstrip()

compile(
    module_source,
    str(module_path),
    "exec"
)

namespace = {}

exec(
    compile(
        module_source,
        str(module_path),
        "exec"
    ),
    namespace
)

build_investigation_briefs = namespace[
    "build_investigation_briefs"
]

# --------------------------------------------------
# 6. Load saved baseline and run existing pipeline
# --------------------------------------------------

print("\n1. LOADING HISTORICAL BASELINE")

baseline = load_anomaly_snapshot(
    snapshot_path,
    manifest_path
)

source = pd.read_csv(
    input_path,
    dtype="string",
    keep_default_na=False
)

source_before = source.copy(deep=True)

with threadpool_limits(limits=1):

    integrated = analyze_batch_with_anomaly_context(
        source,
        baseline,
        "2026-11-09"
    )

fused_before = integrated[
    "combined_records"
].copy(deep=True)

print("\n2. GENERATING INVESTIGATION BRIEFS")

bundle_19_2d = build_investigation_briefs(
    integrated,
    synthetic_data=True
)

print(bundle_19_2d["stats"])

# --------------------------------------------------
# 7. Display anomaly and spatial brief previews
# --------------------------------------------------

anomaly_briefs_19_2d = pd.DataFrame(
    bundle_19_2d["anomaly_briefs"]
)

spatial_briefs_19_2d = pd.DataFrame(
    bundle_19_2d["spatial_briefs"]
)

print("\n3. ANOMALY INVESTIGATION BRIEFS")

display(
    anomaly_briefs_19_2d[[
        "area_id",
        "complaint_category",
        "incoming_complaints",
        "seasonal_historical_max",
        "all_weekday_historical_max",
        "complaints_in_spatial_candidates",
        "kmeans_reporting_patterns"
    ]].head(15)
)

print("\n4. SPATIAL INVESTIGATION BRIEFS")

display(
    spatial_briefs_19_2d[[
        "candidate_id",
        "complaint_category",
        "incoming_complaints",
        "distinct_submitted_locations",
        "complaints_in_anomaly_candidate_groups",
        "area_ids"
    ]].head(23)
)

# --------------------------------------------------
# 8. Important example: spatial != anomaly
# --------------------------------------------------

print("\n5. SPATIAL CONCENTRATION WITH SPARSE HISTORY")

sewer_example = next(
    (
        item for item in bundle_19_2d[
            "spatial_briefs"
        ]
        if item["candidate_id"] == "NB-00-004"
    ),
    None
)

if sewer_example:

    print(sewer_example["explanation"])

    print("\nLinked anomaly groups:")
    print(
        sewer_example[
            "linked_anomaly_area_categories"
        ]
    )

    print("\nSuggested review:")
    print(
        sewer_example["suggested_review"]
    )

# --------------------------------------------------
# 9. Test an empty output and malformed evidence
# --------------------------------------------------

empty_result = dict(integrated)

empty_result["combined_records"] = (
    integrated["combined_records"]
    .iloc[:0].copy()
)

empty_result["anomaly_candidates"] = (
    integrated["anomaly_candidates"]
    .iloc[:0].copy()
)

empty_km_db = dict(
    integrated["kmeans_dbscan_output"]
)

empty_spatial = dict(
    empty_km_db["dbscan_output"]
)

empty_spatial["candidate_profiles"] = (
    empty_spatial["candidate_profiles"]
    .iloc[:0].copy()
)

empty_km_db["dbscan_output"] = empty_spatial

empty_result[
    "kmeans_dbscan_output"
] = empty_km_db

empty_bundle = build_investigation_briefs(
    empty_result,
    synthetic_data=True
)

# Verify corrupt spatial evidence is rejected.
tampered = dict(integrated)

tampered_km_db = dict(
    integrated["kmeans_dbscan_output"]
)

tampered_spatial = dict(
    tampered_km_db["dbscan_output"]
)

tampered_profiles = (
    tampered_spatial["candidate_profiles"]
    .copy(deep=True)
)

tampered_profiles.loc[
    tampered_profiles.index[0],
    "incoming_complaints"
] = 999

tampered_spatial[
    "candidate_profiles"
] = tampered_profiles

tampered_km_db[
    "dbscan_output"
] = tampered_spatial

tampered[
    "kmeans_dbscan_output"
] = tampered_km_db

tamper_rejected = False

try:
    build_investigation_briefs(
        tampered,
        synthetic_data=True
    )
except ValueError:
    tamper_rejected = True

# --------------------------------------------------
# 10. Prepare JSON outputs in memory
# --------------------------------------------------

bundle_text = json.dumps(
    bundle_19_2d,
    indent=2,
    ensure_ascii=False,
    allow_nan=False
) + "\n"

map_text = json.dumps(
    bundle_19_2d["map_geojson"],
    indent=2,
    ensure_ascii=False,
    allow_nan=False
) + "\n"

# --------------------------------------------------
# 11. Validation
# --------------------------------------------------

checks = []

def check(name, condition):
    checks.append({
        "test": name,
        "passed": bool(condition)
    })

stats = bundle_19_2d["stats"]

check(
    "All 500 original records traceable",
    stats["input_records"] == 500
    and len(
        bundle_19_2d["record_audit"]
    ) == 500
)

check(
    "Original complaint identities preserved",
    [
        item["complaint_id"]
        for item in bundle_19_2d["record_audit"]
    ] == source["complaint_id"].astype(str).tolist()
)

check(
    "15 anomaly investigation briefs",
    stats["anomaly_candidate_groups"] == 15
)

check(
    "204 complaints in anomaly groups",
    stats["anomaly_candidate_complaints"] == 204
)

check(
    "23 spatial concentration briefs",
    stats["spatial_candidate_groups"] == 23
)

check(
    "228 complaints in spatial candidates",
    stats["spatial_candidate_complaints"] == 228
)

check(
    "Ten spatial candidates entirely outside historical associations",
    stats[
        "spatial_groups_entirely_outside_history"
    ] == 10
)

check(
    "Anomaly briefs contain both historical maxima",
    all(
        item["incoming_complaints"]
        > item["seasonal_historical_max"]
        and item["incoming_complaints"]
        > item["all_weekday_historical_max"]
        for item in bundle_19_2d["anomaly_briefs"]
    )
)

check(
    "Anomaly briefs include K-Means context",
    all(
        len(item["kmeans_reporting_patterns"]) > 0
        for item in bundle_19_2d["anomaly_briefs"]
    )
)

check(
    "Spatial briefs preserve their unique IDs",
    len({
        item["candidate_id"]
        for item in bundle_19_2d["spatial_briefs"]
    }) == 23
)

check(
    "Each spatial candidate remains category-specific",
    all(
        item["complaint_category"]
        for item in bundle_19_2d["spatial_briefs"]
    )
)

check(
    "Spatial-anomaly overlap never exceeds spatial count",
    all(
        item[
            "complaints_in_anomaly_candidate_groups"
        ] <= item["incoming_complaints"]
        for item in bundle_19_2d["spatial_briefs"]
    )
)

check(
    "Spatial-anomaly overlap never exceeds anomaly count",
    sum(
        item[
            "complaints_in_anomaly_candidate_groups"
        ]
        for item in bundle_19_2d["spatial_briefs"]
    ) <= 204
)

check(
    "NB-00-004 has 15 spatial complaints",
    sewer_example is not None
    and sewer_example["incoming_complaints"] == 15
)

check(
    "NB-00-004 is entirely outside historical associations",
    sewer_example is not None
    and sewer_example[
        "historical_dbscan_associations"
    ]["outside"] == 15
)

check(
    "NB-00-004 does not become an automatic anomaly",
    sewer_example is not None
    and sewer_example[
        "complaints_in_anomaly_candidate_groups"
    ] == 0
)

check(
    "Every brief explicitly identifies synthetic data",
    all(
        item["synthetic_data"]
        for item in (
            bundle_19_2d["anomaly_briefs"]
            + bundle_19_2d["spatial_briefs"]
        )
    )
)

check(
    "Map contains 23 point features",
    len(
        bundle_19_2d[
            "map_geojson"
        ]["features"]
    ) == 23
)

check(
    "Map candidate IDs reconcile with spatial briefs",
    {
        item["id"]
        for item in bundle_19_2d[
            "map_geojson"
        ]["features"]
    } == {
        item["candidate_id"]
        for item in bundle_19_2d["spatial_briefs"]
    }
)

check(
    "Map coordinates are valid longitude-latitude pairs",
    all(
        -180 <= item["geometry"][
            "coordinates"
        ][0] <= 180
        and -90 <= item["geometry"][
            "coordinates"
        ][1] <= 90
        for item in bundle_19_2d[
            "map_geojson"
        ]["features"]
    )
)

check(
    "Empty input produces safe empty briefs",
    len(empty_bundle["anomaly_briefs"]) == 0
    and len(empty_bundle["spatial_briefs"]) == 0
    and len(empty_bundle["record_audit"]) == 0
)

check(
    "Corrupt spatial candidate counts rejected",
    tamper_rejected
)

check(
    "Source DataFrame unchanged",
    source.equals(source_before)
)

check(
    "Integrated DataFrame unchanged",
    integrated["combined_records"].equals(
        fused_before
    )
)

check(
    "Both JSON outputs serialize and reload",
    len(
        json.loads(bundle_text)[
            "anomaly_briefs"
        ]
    ) == 15
    and len(
        json.loads(map_text)["features"]
    ) == 23
)

check(
    "Historical evidence and models unmodified",
    all(
        hashlib.sha256(
            path.read_bytes()
        ).hexdigest() == original_hashes[path]
        for path in protected
    )
)

results_19_2d = pd.DataFrame(checks)

print("\n6. VALIDATION")

display(results_19_2d)

passed = int(results_19_2d["passed"].sum())
total = len(results_19_2d)

print(f"\nChecks passed: {passed}/{total}")

failures = results_19_2d.loc[
    ~results_19_2d["passed"],
    "test"
].tolist()

# --------------------------------------------------
# 12. Save only if every validation passes
# --------------------------------------------------

if failures:

    print("\nSTATUS: NEEDS REVIEW")
    print("No new module or output files saved.")

    for name in failures:
        print("FAILED:", name)

else:

    destinations = {
        module_path: module_source,
        bundle_path: bundle_text,
        map_path: map_text
    }

    # Check every destination before creating any.
    for path, content in destinations.items():

        if path.exists():

            assert path.read_text(
                encoding="utf-8"
            ) == content, (
                f"Existing file differs: {path}. "
                "No overwrite performed."
            )

    for path, content in destinations.items():

        path.parent.mkdir(
            parents=True,
            exist_ok=True
        )

        if not path.exists():

            with path.open(
                "x",
                encoding="utf-8"
            ) as file:
                file.write(content)

    print("\n7. VERIFIED OUTPUT FILES")

    print("Reusable module:", module_path)
    print("Investigation bundle:", bundle_path)
    print("Spatial map GeoJSON:", map_path)

    print("\nSTATUS: PASSED")
    print("INVESTIGATION BRIEFS VERIFIED")
    print("15 ANOMALY EXPLANATIONS PREPARED")
    print("23 SPATIAL EXPLANATIONS PREPARED")
    print("K-MEANS REPORTING CONTEXT PRESERVED")
    print("HISTORICAL HOTSPOT CONTEXT PRESERVED")
    print("500 RECORDS REMAIN TRACEABLE")
    print("MAP-READY SPATIAL POINTS PREPARED")
    print("NO AUTOMATIC REPAIR PRIORITIES")
    print("NO HISTORICAL DATA OR MODELS MODIFIED")


 SMART ASSISTANT INVESTIGATION BRIEFS

1. LOADING HISTORICAL BASELINE

2. GENERATING INVESTIGATION BRIEFS
{'input_records': 500, 'reporting_day': '2026-11-09', 'anomaly_candidate_groups': 15, 'anomaly_candidate_complaints': 204, 'spatial_candidate_groups': 23, 'spatial_candidate_complaints': 228, 'spatial_groups_entirely_outside_history': 10, 'synthetic_data': True}

3. ANOMALY INVESTIGATION BRIEFS


,area_id,complaint_category,incoming_complaints,seasonal_historical_max,all_weekday_historical_max,complaints_in_spatial_candidates,kmeans_reporting_patterns
0,BRONX_10451,STREET CONDITION,14,5,9,8,[Road-heavy reporting]
1,BRONX_10452,STREET CONDITION,13,9,9,8,[Road-heavy reporting]
2,BRONX_10453,SEWER,15,1,7,15,[Sewer-heavy reporting]
3,BRONX_10454,STREET LIGHT CONDITION,12,4,5,7,[Lighting/signal-heavy reporting]
4,BROOKLYN_11201,STREET CONDITION,14,6,12,12,[Road-heavy reporting]
5,BROOKLYN_11204,SEWER,15,7,9,11,[Sewer-heavy reporting]
6,BROOKLYN_11205,STREET LIGHT CONDITION,12,2,9,5,[Lighting/signal-heavy reporting]
7,MANHATTAN_10001,STREET CONDITION,14,4,6,13,[Road-heavy reporting]
8,MANHATTAN_10002,STREET CONDITION,13,4,6,13,[Road-heavy reporting]
9,MANHATTAN_10003,SEWER,15,2,2,15,[Sewer-heavy reporting]



4. SPATIAL INVESTIGATION BRIEFS


,candidate_id,complaint_category,incoming_complaints,distinct_submitted_locations,complaints_in_anomaly_candidate_groups,area_ids
0,NB-00-000,SEWER,15,15,15,[BRONX_10453]
1,NB-00-003,SEWER,15,15,15,[MANHATTAN_10003]
2,NB-00-004,SEWER,15,15,0,[QUEENS_11005]
3,NB-03-010,STREET CONDITION,14,14,14,[STATEN ISLAND_10301]
4,NB-03-004,STREET CONDITION,13,13,0,[BROOKLYN_11203]
5,NB-03-005,STREET CONDITION,13,13,13,[MANHATTAN_10001]
6,NB-03-006,STREET CONDITION,13,13,13,[MANHATTAN_10002]
7,NB-03-003,STREET CONDITION,12,12,12,[BROOKLYN_11201]
8,NB-04-002,STREET LIGHT CONDITION,12,12,0,[MANHATTAN_10004]
9,NB-00-001,SEWER,11,11,11,[BROOKLYN_11204]



5. SPATIAL CONCENTRATION WITH SPARSE HISTORY
15 SEWER complaints at 15 distinct submitted coordinates form spatial candidate NB-00-004. Historical association: 15 outside, 0 established, 0 ambiguous and 0 other. 0 members also belong to area-category reporting increase candidates. The map pin is a submitted complaint coordinate, not a verified infrastructure fault.

Linked anomaly groups:
[]

Suggested review:
Review nearby report descriptions, duplicate coordinates and location accuracy. Verify conditions before any field action.

6. VALIDATION


,test,passed
0,All 500 original records traceable,True
1,Original complaint identities preserved,True
2,15 anomaly investigation briefs,True
3,204 complaints in anomaly groups,True
4,23 spatial concentration briefs,True
5,228 complaints in spatial candidates,True
6,Ten spatial candidates entirely outside histor...,True
7,Anomaly briefs contain both historical maxima,True
8,Anomaly briefs include K-Means context,True
9,Spatial briefs preserve their unique IDs,True



Checks passed: 26/26

7. VERIFIED OUTPUT FILES
Reusable module: D:\Urban-Infrastructure-Pulse-Intelligence\src\integration\smart_assistant_investigation_briefs.py
Investigation bundle: D:\Urban-Infrastructure-Pulse-Intelligence\outputs\tests\phase_19_2d_investigation_briefs\investigation_bundle.json
Spatial map GeoJSON: D:\Urban-Infrastructure-Pulse-Intelligence\outputs\tests\phase_19_2d_investigation_briefs\spatial_candidates_map.geojson

STATUS: PASSED
INVESTIGATION BRIEFS VERIFIED
15 ANOMALY EXPLANATIONS PREPARED
23 SPATIAL EXPLANATIONS PREPARED
K-MEANS REPORTING CONTEXT PRESERVED
HISTORICAL HOTSPOT CONTEXT PRESERVED
500 RECORDS REMAIN TRACEABLE
MAP-READY SPATIAL POINTS PREPARED
NO AUTOMATIC REPAIR PRIORITIES
NO HISTORICAL DATA OR MODELS MODIFIED


### Prepare the Dashboard Data Feed


Before creating the graphical interface, we'll combine the validated results into one dashboard-ready data structure containing area summaries, anomaly cards, spatial concentration cards, map features and record-level evidence.

One especially important feature is avoiding double-counting. The 204 anomaly-associated complaints and 228 spatial-candidate complaints overlap; they must not be displayed as 432 distinct complaints

In [34]:

from pathlib import Path
from textwrap import dedent
from threadpoolctl import threadpool_limits
import copy
import hashlib
import json
import sys
import pandas as pd

print(" DASHBOARD INTELLIGENCE DATA FEED")
print("=" * 75)

root = Path(PROJECT_ROOT).resolve()

if str(root) not in sys.path:
    sys.path.insert(0, str(root))

from src.integration.anomaly_snapshot_integration import (
    load_anomaly_snapshot,
    analyze_batch_with_anomaly_context
)

module_path = (
    root / "src/integration"
    / "dashboard_intelligence_feed.py"
)

output_dir = (
    root / "outputs/tests"
    / "phase_19_2e_dashboard_feed"
)

feed_path = output_dir / "dashboard_feed.json"
map_path = output_dir / "dashboard_map.geojson"

snapshot_path = (
    root / "outputs/tables/anomaly"
    / "historical_daily_counts_2023_2025_v1.csv.gz"
)

manifest_path = (
    root / "outputs/tables/anomaly"
    / "historical_daily_counts_2023_2025_v1_manifest.json"
)

input_path = (
    root / "outputs/tests"
    / "phase_17_4c_backend_verified"
    / "verified_500_backend_results.csv"
)

bundle_path = (
    root / "outputs/tests"
    / "phase_19_2d_investigation_briefs"
    / "investigation_bundle.json"
)

protected = [
    snapshot_path,
    manifest_path,
    input_path,
    bundle_path,
    root / "src/integration"
    / "anomaly_snapshot_integration.py",
    root / "src/integration"
    / "smart_assistant_investigation_briefs.py",
    root / "src/models/new_batch_area_kmeans.py",
    root / "src/models/new_batch_dbscan.py",
    root / "src/models/new_batch_anomaly_screen.py"
]

for path in protected:
    assert path.is_file(), f"Missing: {path}"

hashes_before = {
    path: hashlib.sha256(
        path.read_bytes()
    ).hexdigest()
    for path in protected
}

# --------------------------------------------------
# 1. Reusable dashboard feed module
# --------------------------------------------------

module_source = dedent(r'''
"""Dashboard data contract for three-model intelligence.

All counts describe complaint reports, not verified
infrastructure faults or automatic repair priorities.
"""

import pandas as pd


def _text(value):
    if value is None or pd.isna(value):
        return None
    return str(value)


def _unique(values):
    return sorted({
        str(value).strip()
        for value in values.dropna()
        if str(value).strip()
    })


def build_dashboard_feed(integrated, bundle):
    """Build reusable JSON-compatible dashboard data."""

    records = integrated[
        "combined_records"
    ].copy()

    briefs = bundle["anomaly_briefs"]
    spatial_briefs = bundle["spatial_briefs"]
    geojson = bundle["map_geojson"]

    required = {
        "source_row",
        "complaint_id",
        "daily_window_status",
        "anomaly_area_id",
        "anomaly_category",
        "anomaly_input_status",
        "anomaly_review_status",
        "anomaly_candidate",
        "new_kmeans_description",
        "kmeans_reporting_profile",
        "new_dbscan_status",
        "new_dbscan_candidate_id",
        "dbscan_status"
    }

    missing = required - set(records.columns)

    if missing:
        raise ValueError(
            f"Missing dashboard columns: {sorted(missing)}"
        )

    if records["source_row"].duplicated().any():
        raise ValueError("Duplicate source_row values")

    record_audit = bundle["record_audit"]

    if len(record_audit) != len(records):
        raise ValueError(
            "Brief record audit count mismatch"
        )

    if [
        str(item["source_row"])
        for item in record_audit
    ] != records["source_row"].astype(str).tolist():
        raise ValueError(
            "Brief and integrated source rows differ"
        )

    anomaly_mask = (
        records["anomaly_candidate"]
        .fillna(False)
        .astype(bool)
    )

    spatial_mask = (
        records["new_dbscan_status"]
        .eq("candidate_concentration")
        .fillna(False)
    )

    eligible_mask = (
        records["anomaly_input_status"]
        .eq("eligible")
        .fillna(False)
    )

    both = int(
        (anomaly_mask & spatial_mask).sum()
    )

    anomaly_only = int(
        (anomaly_mask & ~spatial_mask).sum()
    )

    spatial_only = int(
        (spatial_mask & ~anomaly_mask).sum()
    )

    neither = int(
        (~anomaly_mask & ~spatial_mask).sum()
    )

    n_anomaly = int(anomaly_mask.sum())
    n_spatial = int(spatial_mask.sum())

    if n_anomaly != int(
        bundle["stats"]["anomaly_candidate_complaints"]
    ):
        raise ValueError(
            "Anomaly brief complaint total mismatch"
        )

    if n_spatial != int(
        bundle["stats"]["spatial_candidate_complaints"]
    ):
        raise ValueError(
            "Spatial brief complaint total mismatch"
        )

    if len(briefs) != int(
        bundle["stats"]["anomaly_candidate_groups"]
    ):
        raise ValueError(
            "Anomaly brief group count mismatch"
        )

    if len(spatial_briefs) != int(
        bundle["stats"]["spatial_candidate_groups"]
    ):
        raise ValueError(
            "Spatial brief group count mismatch"
        )

    if both != sum(
        int(item[
            "complaints_in_anomaly_candidate_groups"
        ])
        for item in spatial_briefs
    ):
        raise ValueError(
            "Anomaly-spatial overlap mismatch"
        )

    anomaly_keys = {
        (
            str(row.anomaly_area_id),
            str(row.anomaly_category)
        )
        for row in records.loc[
            anomaly_mask
        ].itertuples(index=False)
    }

    brief_keys = {
        (
            str(item["area_id"]),
            str(item["complaint_category"])
        )
        for item in briefs
    }

    if anomaly_keys != brief_keys:
        raise ValueError(
            "Anomaly brief identities do not reconcile"
        )

    candidate_ids = {
        str(value)
        for value in records.loc[
            spatial_mask,
            "new_dbscan_candidate_id"
        ].dropna()
    }

    spatial_ids = {
        str(item["candidate_id"])
        for item in spatial_briefs
    }

    map_features = geojson["features"]

    map_ids = {
        str(item["id"])
        for item in map_features
    }

    if (
        candidate_ids != spatial_ids
        or spatial_ids != map_ids
        or len(map_features) != len(spatial_ids)
    ):
        raise ValueError(
            "Spatial brief and map IDs disagree"
        )

    # ----------------------------------------------
    # Area-level K-Means reporting summaries
    # ----------------------------------------------

    area_rows = records.loc[
        eligible_mask
    ].copy()

    area_cards = []

    for area_id, group in area_rows.groupby(
        "anomaly_area_id",
        sort=True
    ):

        category_counts = (
            group["anomaly_category"]
            .astype(str)
            .value_counts()
        )

        breakdown = [
            {
                "complaint_category": category,
                "incoming_complaints": int(count)
            }
            for category, count in sorted(
                category_counts.items(),
                key=lambda item: (
                    -item[1], item[0]
                )
            )
        ]

        in_spatial = group[
            "new_dbscan_status"
        ].eq("candidate_concentration")

        area_cards.append({
            "area_id": str(area_id),
            "incoming_complaints": int(len(group)),
            "reporting_patterns": _unique(
                group["new_kmeans_description"]
            ),
            "historical_kmeans_profiles": _unique(
                group["kmeans_reporting_profile"]
            ),
            "category_breakdown": breakdown,
            "dominant_category": (
                breakdown[0]["complaint_category"]
                if breakdown else None
            ),
            "anomaly_candidate_complaints": int(
                group["anomaly_candidate"].sum()
            ),
            "spatial_candidate_complaints": int(
                in_spatial.sum()
            ),
            "spatial_candidate_ids": _unique(
                group.loc[
                    in_spatial,
                    "new_dbscan_candidate_id"
                ]
            ),
            "historical_established_complaints": int(
                group["dbscan_status"].eq(
                    "established_cluster"
                ).sum()
            ),
            "historical_outside_complaints": int(
                group["dbscan_status"].eq(
                    "outside_existing_clusters"
                ).sum()
            ),
            "historical_ambiguous_complaints": int(
                group["dbscan_status"].eq(
                    "ambiguous_association"
                ).sum()
            )
        })

    # ----------------------------------------------
    # Record-level dashboard table
    # ----------------------------------------------

    record_rows = []

    for row in records.itertuples(index=False):

        record_rows.append({
            "source_row": _text(row.source_row),
            "complaint_id": _text(row.complaint_id),
            "area_id": _text(row.anomaly_area_id),
            "complaint_category": _text(
                row.anomaly_category
            ),
            "reporting_day_status": _text(
                row.daily_window_status
            ),
            "anomaly_input_status": _text(
                row.anomaly_input_status
            ),
            "new_kmeans_pattern": _text(
                row.new_kmeans_description
            ),
            "historical_kmeans_profile": _text(
                row.kmeans_reporting_profile
            ),
            "new_dbscan_status": _text(
                row.new_dbscan_status
            ),
            "new_dbscan_candidate_id": _text(
                row.new_dbscan_candidate_id
            ),
            "historical_dbscan_status": _text(
                row.dbscan_status
            ),
            "anomaly_review_status": _text(
                row.anomaly_review_status
            ),
            "anomaly_candidate": bool(
                row.anomaly_candidate
            ),
            "spatial_candidate": (
                row.new_dbscan_status
                == "candidate_concentration"
            )
        })

    patterns = sorted({
        pattern
        for area in area_cards
        for pattern in area["reporting_patterns"]
    })

    summary = {
        "input_complaints": int(len(records)),
        "selected_day_complaints": int(
            records["daily_window_status"]
            .eq("selected_day").sum()
        ),
        "anomaly_eligible_complaints": int(
            eligible_mask.sum()
        ),
        "areas_with_eligible_reports":
            len(area_cards),
        "new_kmeans_patterns": len(patterns),
        "spatial_candidate_groups":
            len(spatial_briefs),
        "spatial_candidate_complaints": n_spatial,
        "anomaly_candidate_groups":
            len(briefs),
        "anomaly_candidate_complaints": n_anomaly,
        "complaints_in_both_signals": both,
        "complaints_anomaly_only": anomaly_only,
        "complaints_spatial_only": spatial_only,
        "complaints_in_neither_signal": neither,
        "distinct_complaints_in_either_signal":
            n_anomaly + n_spatial - both,
        "historical_baseline_areas": int(
            integrated["stats"][
                "historical_baseline_areas"
            ]
        )
    }

    if (
        both + anomaly_only
        + spatial_only + neither
        != len(records)
    ):
        raise ValueError(
            "Dashboard report partition mismatch"
        )

    if sum(
        area["incoming_complaints"]
        for area in area_cards
    ) != int(eligible_mask.sum()):
        raise ValueError(
            "Dashboard area totals mismatch"
        )

    return {
        "metadata": {
            "schema_version": "1.0",
            "reporting_day": str(
                integrated["stats"]["reporting_day"]
            ),
            "synthetic_data": bool(
                bundle["metadata"]["synthetic_data"]
            ),
            "evidence_policy": (
                "Screening candidates and reporting "
                "patterns only. No validated defects, "
                "risk scores, statistical significance "
                "claims or automatic repair priorities."
            ),
            "counting_policy": (
                "Anomaly and spatial complaint counts "
                "may overlap. Use distinct_complaints_"
                "in_either_signal for their union."
            )
        },
        "summary": summary,
        "filters": {
            "boroughs": sorted({
                str(area["area_id"]).split(
                    "_", 1
                )[0]
                for area in area_cards
            }),
            "categories": sorted({
                str(value)
                for value in area_rows[
                    "anomaly_category"
                ].dropna().unique()
            }),
            "kmeans_patterns": patterns,
            "anomaly_review_statuses": _unique(
                records["anomaly_review_status"]
            )
        },
        "area_cards": area_cards,
        "anomaly_cards": [
            {
                "id": (
                    f"ANOMALY::{item['area_id']}::"
                    f"{item['complaint_category']}"
                ),
                **item
            }
            for item in briefs
        ],
        "spatial_cards": [
            dict(item)
            for item in spatial_briefs
        ],
        "records": record_rows,
        "map_geojson": geojson
    }
''').lstrip()

compile(
    module_source,
    str(module_path),
    "exec"
)

namespace = {}

exec(
    compile(
        module_source,
        str(module_path),
        "exec"
    ),
    namespace
)

build_dashboard_feed = namespace[
    "build_dashboard_feed"
]

# --------------------------------------------------
# 2. Load saved results and run integrated models
# --------------------------------------------------

print("\n1. LOADING VERIFIED DATA")

baseline = load_anomaly_snapshot(
    snapshot_path,
    manifest_path
)

source = pd.read_csv(
    input_path,
    dtype="string",
    keep_default_na=False
)

source_before = source.copy(deep=True)

with bundle_path.open(
    encoding="utf-8"
) as file:
    bundle = json.load(file)

with threadpool_limits(limits=1):

    integrated = analyze_batch_with_anomaly_context(
        source,
        baseline,
        "2026-11-09"
    )

integrated_before = (
    integrated["combined_records"]
    .copy(deep=True)
)

# --------------------------------------------------
# 3. Generate dashboard feed
# --------------------------------------------------

print("\n2. BUILDING DASHBOARD DATA FEED")

dashboard_feed_19_2e = build_dashboard_feed(
    integrated,
    bundle
)

summary = dashboard_feed_19_2e[
    "summary"
]

for name, value in summary.items():
    print(f"{name}: {value}")

print("\n3. AREA REPORTING PATTERNS")

display(
    pd.DataFrame(
        dashboard_feed_19_2e["area_cards"]
    )[[
        "area_id",
        "incoming_complaints",
        "reporting_patterns",
        "anomaly_candidate_complaints",
        "spatial_candidate_complaints",
        "dominant_category"
    ]]
)

# --------------------------------------------------
# 4. Display distinct complaint accounting
# --------------------------------------------------

print("\n4. COMPLAINT OVERLAP ACCOUNTING")

signal_accounting = pd.DataFrame([
    {
        "complaint_group": "Both signals",
        "distinct_complaints":
            summary["complaints_in_both_signals"]
    },
    {
        "complaint_group": "Anomaly only",
        "distinct_complaints":
            summary["complaints_anomaly_only"]
    },
    {
        "complaint_group": "Spatial only",
        "distinct_complaints":
            summary["complaints_spatial_only"]
    },
    {
        "complaint_group": "Neither signal",
        "distinct_complaints":
            summary["complaints_in_neither_signal"]
    }
])

display(signal_accounting)

# --------------------------------------------------
# 5. Validate dashboard integrity
# --------------------------------------------------

checks = []

def check(name, condition):
    checks.append({
        "test": name,
        "passed": bool(condition)
    })

check(
    "All 500 source records included",
    summary["input_complaints"] == 500
)

check(
    "All 500 records belong to reporting day",
    summary["selected_day_complaints"] == 500
)

check(
    "All 500 anomaly input records eligible",
    summary["anomaly_eligible_complaints"] == 500
)

check(
    "20 geographic area cards created",
    len(
        dashboard_feed_19_2e["area_cards"]
    ) == 20
)

check(
    "Three new K-Means patterns represented",
    summary["new_kmeans_patterns"] == 3
)

check(
    "All area cards reconcile to 500 reports",
    sum(
        item["incoming_complaints"]
        for item in dashboard_feed_19_2e[
            "area_cards"
        ]
    ) == 500
)

check(
    "All 20 synthetic areas contain 25 reports",
    all(
        item["incoming_complaints"] == 25
        for item in dashboard_feed_19_2e[
            "area_cards"
        ]
    )
)

check(
    "23 spatial candidate cards retained",
    summary["spatial_candidate_groups"] == 23
    and len(
        dashboard_feed_19_2e["spatial_cards"]
    ) == 23
)

check(
    "228 complaints belong to spatial candidates",
    summary["spatial_candidate_complaints"] == 228
)

check(
    "15 anomaly candidate cards retained",
    summary["anomaly_candidate_groups"] == 15
    and len(
        dashboard_feed_19_2e["anomaly_cards"]
    ) == 15
)

check(
    "204 complaints belong to anomaly candidates",
    summary["anomaly_candidate_complaints"] == 204
)

check(
    "Signal groups partition all complaints",
    int(
        signal_accounting[
            "distinct_complaints"
        ].sum()
    ) == 500
)

check(
    "Distinct union uses overlap correction",
    summary[
        "distinct_complaints_in_either_signal"
    ]
    == (
        204 + 228
        - summary["complaints_in_both_signals"]
    )
)

check(
    "Overlap matches spatial investigation briefs",
    summary["complaints_in_both_signals"]
    == sum(
        item[
            "complaints_in_anomaly_candidate_groups"
        ]
        for item in bundle["spatial_briefs"]
    )
)

check(
    "All historical area coverage retained",
    summary["historical_baseline_areas"] == 238
)

check(
    "Dashboard has 500 record-level entries",
    len(
        dashboard_feed_19_2e["records"]
    ) == 500
)

check(
    "Record identities remain in original order",
    [
        item["source_row"]
        for item in dashboard_feed_19_2e["records"]
    ] == source["source_row"].astype(str).tolist()
)

check(
    "Dashboard retains 23 map features",
    len(
        dashboard_feed_19_2e[
            "map_geojson"
        ]["features"]
    ) == 23
)

check(
    "Map IDs match the spatial cards",
    {
        feature["id"]
        for feature in dashboard_feed_19_2e[
            "map_geojson"
        ]["features"]
    } == {
        item["candidate_id"]
        for item in dashboard_feed_19_2e[
            "spatial_cards"
        ]
    }
)

check(
    "Dashboard explicitly labels synthetic data",
    dashboard_feed_19_2e[
        "metadata"
    ]["synthetic_data"] is True
)

check(
    "Dashboard retains investigation explanations",
    all(
        item.get("explanation")
        and item.get("suggested_review")
        for item in (
            dashboard_feed_19_2e["anomaly_cards"]
            + dashboard_feed_19_2e[
                "spatial_cards"
            ]
        )
    )
)

# --------------------------------------------------
# 6. Reject inconsistent upstream evidence
# --------------------------------------------------

tampered_bundle = copy.deepcopy(bundle)

tampered_bundle["stats"][
    "anomaly_candidate_complaints"
] = 999

tamper_rejected = False

try:
    build_dashboard_feed(
        integrated,
        tampered_bundle
    )
except ValueError:
    tamper_rejected = True

check(
    "Inconsistent anomaly counts rejected",
    tamper_rejected
)

check(
    "Original source DataFrame unchanged",
    source.equals(source_before)
)

check(
    "Integrated model records unchanged",
    integrated["combined_records"].equals(
        integrated_before
    )
)

check(
    "Protected data and models unchanged",
    all(
        hashlib.sha256(
            path.read_bytes()
        ).hexdigest() == hashes_before[path]
        for path in protected
    )
)

# --------------------------------------------------
# 7. Serialize outputs and validate JSON
# --------------------------------------------------

feed_text = json.dumps(
    dashboard_feed_19_2e,
    indent=2,
    ensure_ascii=False,
    allow_nan=False
) + "\n"

map_text = json.dumps(
    dashboard_feed_19_2e["map_geojson"],
    indent=2,
    ensure_ascii=False,
    allow_nan=False
) + "\n"

check(
    "Dashboard JSON serializes and reloads",
    json.loads(feed_text)["summary"][
        "input_complaints"
    ] == 500
)

check(
    "GeoJSON serializes and reloads",
    len(
        json.loads(map_text)["features"]
    ) == 23
)

results_19_2e = pd.DataFrame(checks)

print("\n5. VALIDATION")

display(results_19_2e)

passed = int(results_19_2e["passed"].sum())
total = len(results_19_2e)

print(f"\nChecks passed: {passed}/{total}")

failed = results_19_2e.loc[
    ~results_19_2e["passed"],
    "test"
].tolist()

# --------------------------------------------------
# 8. Save only after successful validation
# --------------------------------------------------

if failed:

    print("\nSTATUS: NEEDS REVIEW")
    print("No dashboard feed files saved.")

    for name in failed:
        print("FAILED:", name)

else:

    destinations = {
        module_path: module_source,
        feed_path: feed_text,
        map_path: map_text
    }

    # Check all existing targets before writing.
    for path, content in destinations.items():

        if path.exists():
            assert path.read_text(
                encoding="utf-8"
            ) == content, (
                f"Existing output differs: {path}. "
                "No overwrite performed."
            )

    for path, content in destinations.items():

        path.parent.mkdir(
            parents=True,
            exist_ok=True
        )

        if not path.exists():
            with path.open(
                "x",
                encoding="utf-8"
            ) as file:
                file.write(content)

    print("\n6. VERIFIED DASHBOARD OUTPUTS")

    print("Reusable module:", module_path)
    print("Dashboard feed:", feed_path)
    print("Map GeoJSON:", map_path)

    print("\nSTATUS: PASSED")
    print("DASHBOARD FEED VERIFIED")
    print("THREE-MODEL EVIDENCE PRESERVED")
    print("20 AREA CARDS PREPARED")
    print("15 ANOMALY CARDS PREPARED")
    print("23 SPATIAL CARDS PREPARED")
    print("COMPLAINT OVERLAP ACCOUNTED FOR")
    print("500 COMPLAINT RECORDS TRACEABLE")
    print("NO AUTOMATIC REPAIR PRIORITIES")
    print("NO HISTORICAL DATA OR MODELS MODIFIED")


 DASHBOARD INTELLIGENCE DATA FEED

1. LOADING VERIFIED DATA

2. BUILDING DASHBOARD DATA FEED
input_complaints: 500
selected_day_complaints: 500
anomaly_eligible_complaints: 500
areas_with_eligible_reports: 20
new_kmeans_patterns: 3
spatial_candidate_groups: 23
spatial_candidate_complaints: 228
anomaly_candidate_groups: 15
anomaly_candidate_complaints: 204
complaints_in_both_signals: 156
complaints_anomaly_only: 48
complaints_spatial_only: 72
complaints_in_neither_signal: 224
distinct_complaints_in_either_signal: 276
historical_baseline_areas: 238

3. AREA REPORTING PATTERNS


,area_id,incoming_complaints,reporting_patterns,anomaly_candidate_complaints,spatial_candidate_complaints,dominant_category
0,BRONX_10451,25,[Road-heavy reporting],14,8,STREET CONDITION
1,BRONX_10452,25,[Road-heavy reporting],13,8,STREET CONDITION
2,BRONX_10453,25,[Sewer-heavy reporting],15,15,SEWER
3,BRONX_10454,25,[Lighting/signal-heavy reporting],12,7,STREET LIGHT CONDITION
4,BROOKLYN_11201,25,[Road-heavy reporting],14,12,STREET CONDITION
5,BROOKLYN_11203,25,[Road-heavy reporting],0,13,STREET CONDITION
6,BROOKLYN_11204,25,[Sewer-heavy reporting],15,11,SEWER
7,BROOKLYN_11205,25,[Lighting/signal-heavy reporting],12,5,STREET LIGHT CONDITION
8,MANHATTAN_10001,25,[Road-heavy reporting],14,13,STREET CONDITION
9,MANHATTAN_10002,25,[Road-heavy reporting],13,13,STREET CONDITION



4. COMPLAINT OVERLAP ACCOUNTING


,complaint_group,distinct_complaints
0,Both signals,156
1,Anomaly only,48
2,Spatial only,72
3,Neither signal,224



5. VALIDATION


,test,passed
0,All 500 source records included,True
1,All 500 records belong to reporting day,True
2,All 500 anomaly input records eligible,True
3,20 geographic area cards created,True
4,Three new K-Means patterns represented,True
5,All area cards reconcile to 500 reports,True
6,All 20 synthetic areas contain 25 reports,True
7,23 spatial candidate cards retained,True
8,228 complaints belong to spatial candidates,True
9,15 anomaly candidate cards retained,True



Checks passed: 27/27

6. VERIFIED DASHBOARD OUTPUTS
Reusable module: D:\Urban-Infrastructure-Pulse-Intelligence\src\integration\dashboard_intelligence_feed.py
Dashboard feed: D:\Urban-Infrastructure-Pulse-Intelligence\outputs\tests\phase_19_2e_dashboard_feed\dashboard_feed.json
Map GeoJSON: D:\Urban-Infrastructure-Pulse-Intelligence\outputs\tests\phase_19_2e_dashboard_feed\dashboard_map.geojson

STATUS: PASSED
DASHBOARD FEED VERIFIED
THREE-MODEL EVIDENCE PRESERVED
20 AREA CARDS PREPARED
15 ANOMALY CARDS PREPARED
23 SPATIAL CARDS PREPARED
COMPLAINT OVERLAP ACCOUNTED FOR
500 COMPLAINT RECORDS TRACEABLE
NO AUTOMATIC REPAIR PRIORITIES
NO HISTORICAL DATA OR MODELS MODIFIED


In [35]:

from pathlib import Path
from textwrap import dedent
import ast
import hashlib
import importlib.util
import json
import sys
import pandas as pd

print(" STREAMLIT DASHBOARD GUI")
print("=" * 75)

root = Path(PROJECT_ROOT).resolve()

app_path = (
    root / "dashboard"
    / "urban_pulse_dashboard.py"
)

feed_path = (
    root / "outputs/tests"
    / "phase_19_2e_dashboard_feed"
    / "dashboard_feed.json"
)

map_path = (
    root / "outputs/tests"
    / "phase_19_2e_dashboard_feed"
    / "dashboard_map.geojson"
)

protected = [
    feed_path,
    map_path,
    root / "src/integration"
    / "dashboard_intelligence_feed.py",
    root / "src/integration"
    / "anomaly_snapshot_integration.py",
    root / "src/models"
    / "new_batch_area_kmeans.py",
    root / "src/models"
    / "new_batch_dbscan.py",
    root / "src/models"
    / "new_batch_anomaly_screen.py"
]

for path in protected:
    assert path.is_file(), f"Missing: {path}"

hashes_before = {
    path: hashlib.sha256(
        path.read_bytes()
    ).hexdigest()
    for path in protected
}

# --------------------------------------------------
# 1. Build a standalone GUI that reads saved JSON
# --------------------------------------------------

app_source = dedent(r'''
"""Read-only Urban Infrastructure Pulse dashboard.

Launch from the project root:
    python -m streamlit run dashboard/urban_pulse_dashboard.py

This prototype displays saved synthetic demonstration
results. It does not retrain or modify historical models.
"""

import json
from pathlib import Path

import pandas as pd


PROJECT_ROOT = Path(__file__).resolve().parents[1]

FEED_PATH = (
    PROJECT_ROOT
    / "outputs"
    / "tests"
    / "phase_19_2e_dashboard_feed"
    / "dashboard_feed.json"
)


def load_feed(path=FEED_PATH):
    with Path(path).open(
        "r",
        encoding="utf-8"
    ) as file:
        return json.load(file)


def signal_type(row):
    anomaly = bool(
        row.get("anomaly_candidate", False)
    )
    spatial = bool(
        row.get("spatial_candidate", False)
    )

    if anomaly and spatial:
        return "Both signals"

    if anomaly:
        return "Anomaly only"

    if spatial:
        return "Spatial only"

    return "Neither"


def filter_records(
    records,
    borough="All",
    category="All",
    pattern="All",
    signal="All"
):
    selected = []

    for record in records:

        area_id = str(
            record.get("area_id") or ""
        )

        current_borough = (
            area_id.split("_", 1)[0]
        )

        if (
            borough != "All"
            and current_borough != borough
        ):
            continue

        if (
            category != "All"
            and record.get(
                "complaint_category"
            ) != category
        ):
            continue

        if (
            pattern != "All"
            and record.get(
                "new_kmeans_pattern"
            ) != pattern
        ):
            continue

        if (
            signal != "All"
            and signal_type(record) != signal
        ):
            continue

        selected.append(record)

    return selected


def build_filtered_view(feed, records):
    """Link selected reports to saved evidence cards."""

    anomaly_keys = {
        (
            str(item["area_id"]),
            str(item["complaint_category"])
        )
        for item in records
        if item["anomaly_candidate"]
    }

    spatial_ids = {
        str(item["new_dbscan_candidate_id"])
        for item in records
        if (
            item["spatial_candidate"]
            and item.get(
                "new_dbscan_candidate_id"
            ) is not None
        )
    }

    anomaly_cards = [
        item
        for item in feed["anomaly_cards"]
        if (
            str(item["area_id"]),
            str(item["complaint_category"])
        ) in anomaly_keys
    ]

    spatial_cards = [
        item
        for item in feed["spatial_cards"]
        if str(item["candidate_id"]) in spatial_ids
    ]

    map_features = [
        feature
        for feature in feed[
            "map_geojson"
        ]["features"]
        if str(feature["id"]) in spatial_ids
    ]

    both = sum(
        signal_type(item) == "Both signals"
        for item in records
    )

    anomaly_only = sum(
        signal_type(item) == "Anomaly only"
        for item in records
    )

    spatial_only = sum(
        signal_type(item) == "Spatial only"
        for item in records
    )

    neither = sum(
        signal_type(item) == "Neither"
        for item in records
    )

    summary = {
        "selected": len(records),
        "both": both,
        "anomaly_only": anomaly_only,
        "spatial_only": spatial_only,
        "neither": neither,
        "anomaly_reports":
            both + anomaly_only,
        "spatial_reports":
            both + spatial_only,
        "distinct_either":
            both + anomaly_only + spatial_only,
        "anomaly_groups":
            len(anomaly_cards),
        "spatial_groups":
            len(spatial_cards),
        "areas": len({
            str(item["area_id"])
            for item in records
        })
    }

    if sum([
        both,
        anomaly_only,
        spatial_only,
        neither
    ]) != len(records):
        raise ValueError(
            "Selected complaint counts disagree."
        )

    return {
        "records": records,
        "anomaly_cards": anomaly_cards,
        "spatial_cards": spatial_cards,
        "map_features": map_features,
        "summary": summary
    }


def validate_feed(feed):
    """Reject inconsistent dashboard source data."""

    required = {
        "metadata",
        "summary",
        "filters",
        "area_cards",
        "anomaly_cards",
        "spatial_cards",
        "records",
        "map_geojson"
    }

    if not required.issubset(feed):
        raise ValueError(
            "Dashboard feed is missing required sections."
        )

    records = feed["records"]
    summary = feed["summary"]

    if len(records) != summary[
        "input_complaints"
    ]:
        raise ValueError(
            "Input record count mismatch."
        )

    identities = [
        str(item["source_row"])
        for item in records
    ]

    if len(identities) != len(
        set(identities)
    ):
        raise ValueError(
            "Duplicate complaint source rows."
        )

    view = build_filtered_view(
        feed,
        records
    )

    counts = view["summary"]

    expected = {
        "anomaly_candidate_complaints":
            counts["anomaly_reports"],
        "spatial_candidate_complaints":
            counts["spatial_reports"],
        "complaints_in_both_signals":
            counts["both"],
        "complaints_anomaly_only":
            counts["anomaly_only"],
        "complaints_spatial_only":
            counts["spatial_only"],
        "complaints_in_neither_signal":
            counts["neither"],
        "distinct_complaints_in_either_signal":
            counts["distinct_either"]
    }

    for key, value in expected.items():
        if summary[key] != value:
            raise ValueError(
                f"Dashboard count mismatch: {key}"
            )

    if counts["anomaly_groups"] != summary[
        "anomaly_candidate_groups"
    ]:
        raise ValueError(
            "Anomaly group count mismatch."
        )

    if counts["spatial_groups"] != summary[
        "spatial_candidate_groups"
    ]:
        raise ValueError(
            "Spatial group count mismatch."
        )

    return view


def main():
    import streamlit as st

    st.set_page_config(
        page_title="Urban Infrastructure Pulse",
        page_icon="🏙️",
        layout="wide",
        initial_sidebar_state="expanded"
    )

    st.title("Urban Infrastructure Pulse")
    st.caption(
        "Smart infrastructure reporting intelligence"
    )

    if not FEED_PATH.is_file():
        st.error(
            "The validated dashboard feed was not found."
        )
        st.code(str(FEED_PATH))
        st.stop()

    try:
        feed = load_feed()
        validate_feed(feed)
    except (OSError, ValueError, KeyError) as exc:
        st.error(
            f"Invalid dashboard feed: {exc}"
        )
        st.stop()

    if feed["metadata"].get(
        "synthetic_data", False
    ):
        st.warning(
            "Synthetic demonstration data — "
            "not live NYC 311 incidents or "
            "verified infrastructure problems."
        )

    st.caption(
        "Reporting day: "
        + str(feed["metadata"]["reporting_day"])
        + " | Historical comparison: 2023–2025"
    )

    # ----------------------------------------------
    # Interactive filters
    # ----------------------------------------------

    st.sidebar.header("Explore complaints")

    borough = st.sidebar.selectbox(
        "Borough",
        ["All"] + feed["filters"]["boroughs"]
    )

    category = st.sidebar.selectbox(
        "Complaint category",
        ["All"] + feed["filters"]["categories"]
    )

    pattern = st.sidebar.selectbox(
        "K-Means reporting pattern",
        ["All"] + feed["filters"][
            "kmeans_patterns"
        ]
    )

    signal = st.sidebar.selectbox(
        "Investigation signal",
        [
            "All",
            "Both signals",
            "Anomaly only",
            "Spatial only",
            "Neither"
        ]
    )

    st.sidebar.caption(
        "Filters change the displayed report subset. "
        "Historical maxima and investigation "
        "explanations refer to the complete "
        "original reporting-day batch."
    )

    records = filter_records(
        feed["records"],
        borough=borough,
        category=category,
        pattern=pattern,
        signal=signal
    )

    view = build_filtered_view(
        feed, records
    )

    stats = view["summary"]

    # ----------------------------------------------
    # Overview metrics
    # ----------------------------------------------

    m1, m2, m3, m4, m5 = st.columns(5)

    m1.metric(
        "Selected reports",
        stats["selected"]
    )

    m2.metric(
        "Anomaly-associated",
        stats["anomaly_reports"]
    )

    m3.metric(
        "Spatial-associated",
        stats["spatial_reports"]
    )

    m4.metric(
        "Distinct with either",
        stats["distinct_either"]
    )

    m5.metric(
        "Areas represented",
        stats["areas"]
    )

    st.caption(
        "Anomaly and spatial totals overlap. "
        "'Distinct with either' counts each "
        "complaint once."
    )

    tabs = st.tabs([
        "Overview",
        "K-Means Areas",
        "Anomaly Reviews",
        "DBSCAN Map",
        "Complaint Records"
    ])

    # ----------------------------------------------
    # Tab 1: Overview
    # ----------------------------------------------

    with tabs[0]:

        st.subheader("Reporting overview")

        summary_frame = pd.DataFrame([
            {
                "Signal classification":
                    "Both signals",
                "Complaints": stats["both"]
            },
            {
                "Signal classification":
                    "Anomaly only",
                "Complaints":
                    stats["anomaly_only"]
            },
            {
                "Signal classification":
                    "Spatial only",
                "Complaints":
                    stats["spatial_only"]
            },
            {
                "Signal classification":
                    "Neither",
                "Complaints":
                    stats["neither"]
            }
        ])

        left, right = st.columns(2)

        with left:
            st.markdown(
                "**Distinct complaint accounting**"
            )
            st.dataframe(
                summary_frame,
                hide_index=True,
                use_container_width=True
            )

        with right:
            st.markdown(
                "**Complaints by category**"
            )

            if records:
                frame = pd.DataFrame(records)
                st.bar_chart(
                    frame[
                        "complaint_category"
                    ].value_counts()
                )
            else:
                st.info(
                    "No complaints match these filters."
                )

        if records:
            st.markdown(
                "**Complaints by borough–ZIP area**"
            )

            frame = pd.DataFrame(records)

            st.bar_chart(
                frame["area_id"]
                .value_counts()
                .sort_index()
            )

    # ----------------------------------------------
    # Tab 2: K-Means area patterns
    # ----------------------------------------------

    with tabs[1]:

        st.subheader(
            "K-Means area reporting patterns"
        )

        st.caption(
            "Patterns describe relative complaint "
            "category mixtures, not infrastructure "
            "condition or repair urgency."
        )

        if records:
            frame = pd.DataFrame(records)

            area_table = []

            for area_id, group in frame.groupby(
                "area_id", sort=True
            ):

                most_common = (
                    group["complaint_category"]
                    .value_counts()
                    .index[0]
                )

                patterns = sorted({
                    str(value)
                    for value in group[
                        "new_kmeans_pattern"
                    ].dropna()
                })

                area_table.append({
                    "Area": area_id,
                    "Selected reports":
                        len(group),
                    "K-Means pattern":
                        " | ".join(patterns),
                    "Most reported category":
                        most_common,
                    "Anomaly-associated":
                        int(group[
                            "anomaly_candidate"
                        ].sum()),
                    "Spatial-associated":
                        int(group[
                            "spatial_candidate"
                        ].sum())
                })

            st.dataframe(
                pd.DataFrame(area_table),
                hide_index=True,
                use_container_width=True
            )
        else:
            st.info(
                "No area patterns match the filters."
            )

    # ----------------------------------------------
    # Tab 3: Anomaly screening evidence
    # ----------------------------------------------

    with tabs[2]:

        st.subheader(
            "Historical reporting-increase reviews"
        )

        st.caption(
            "These are screening candidates, "
            "not statistically confirmed anomalies."
        )

        if not view["anomaly_cards"]:
            st.info(
                "No anomaly candidate groups match "
                "the selected filters."
            )

        for card in view["anomaly_cards"]:

            matched = sum(
                record["anomaly_candidate"]
                and record["area_id"]
                    == card["area_id"]
                and record["complaint_category"]
                    == card["complaint_category"]
                for record in records
            )

            heading = (
                f"{card['area_id']} — "
                f"{card['complaint_category']} "
                f"({matched} selected reports)"
            )

            with st.expander(heading):

                st.write(
                    card["explanation"]
                )

                st.markdown(
                    "**Historical comparison**"
                )

                c1, c2, c3 = st.columns(3)

                c1.metric(
                    "Full-batch reports",
                    card["incoming_complaints"]
                )

                c2.metric(
                    "Seasonal historical max",
                    card[
                        "seasonal_historical_max"
                    ]
                )

                c3.metric(
                    "All-Monday historical max",
                    card[
                        "all_weekday_historical_max"
                    ]
                )

                st.markdown(
                    "**Suggested verification**"
                )

                st.write(
                    card["suggested_review"]
                )

    # ----------------------------------------------
    # Tab 4: DBSCAN spatial concentrations
    # ----------------------------------------------

    with tabs[3]:

        st.subheader(
            "New-batch spatial concentrations"
        )

        st.caption(
            "Pins represent submitted complaint "
            "coordinates. They are not verified "
            "infrastructure fault locations."
        )

        if not view["map_features"]:
            st.info(
                "No spatial concentrations match "
                "the current filters."
            )

        else:
            points = []

            for feature in view[
                "map_features"
            ]:

                longitude, latitude = (
                    feature["geometry"][
                        "coordinates"
                    ]
                )

                points.append({
                    "latitude": latitude,
                    "longitude": longitude,
                    "candidate_id": feature["id"]
                })

            points_frame = pd.DataFrame(
                points
            )

            st.map(
                points_frame[[
                    "latitude",
                    "longitude"
                ]]
            )

            st.caption(
                "One representative point per "
                "spatial candidate; not every "
                "individual complaint location."
            )

        for card in view["spatial_cards"]:

            matched = sum(
                record["spatial_candidate"]
                and record[
                    "new_dbscan_candidate_id"
                ] == card["candidate_id"]
                for record in records
            )

            with st.expander(
                f"{card['candidate_id']} — "
                f"{card['complaint_category']} "
                f"({matched} selected reports)"
            ):

                st.write(
                    card["explanation"]
                )

                st.write(
                    "**Full-batch spatial reports:**",
                    card["incoming_complaints"]
                )

                st.write(
                    "**Distinct submitted locations:**",
                    card[
                        "distinct_submitted_locations"
                    ]
                )

                st.write(
                    "**Suggested verification:** "
                    + card["suggested_review"]
                )

    # ----------------------------------------------
    # Tab 5: Record-level evidence
    # ----------------------------------------------

    with tabs[4]:

        st.subheader(
            "Complaint-level evidence"
        )

        st.caption(
            "This table retains both new-batch "
            "signals and historical comparison "
            "statuses for traceability."
        )

        if not records:
            st.info(
                "No complaint records match the filters."
            )

        else:
            frame = pd.DataFrame(
                records
            )

            st.dataframe(
                frame,
                hide_index=True,
                use_container_width=True
            )

            st.download_button(
                "Export filtered complaints as CSV",
                data=frame.to_csv(
                    index=False
                ).encode("utf-8"),
                file_name=(
                    "urban_pulse_filtered_complaints.csv"
                ),
                mime="text/csv"
            )

    st.divider()

    st.caption(
        "Urban Infrastructure Pulse Intelligence "
        "| Synthetic-data pilot | "
        "Human verification required before "
        "operational action."
    )


if __name__ == "__main__":
    main()
''').lstrip()

ast.parse(app_source)

compiled_app = compile(
    app_source,
    str(app_path),
    "exec"
)

namespace = {
    "__name__": "_phase20_validation",
    "__file__": str(app_path)
}

exec(compiled_app, namespace)

load_feed = namespace["load_feed"]
validate_feed = namespace["validate_feed"]
filter_records = namespace["filter_records"]
build_filtered_view = namespace[
    "build_filtered_view"
]
signal_type = namespace["signal_type"]

# --------------------------------------------------
# 2. Load the real saved dashboard feed
# --------------------------------------------------

print("\n1. VALIDATING SAVED DASHBOARD DATA")

feed = load_feed(feed_path)
full_view = validate_feed(feed)

print("Reporting day:",
      feed["metadata"]["reporting_day"])

print("Synthetic data:",
      feed["metadata"]["synthetic_data"])

for name, value in full_view["summary"].items():
    print(f"{name}: {value}")

# --------------------------------------------------
# 3. Test interactive filter behavior
# --------------------------------------------------

print("\n2. TESTING DASHBOARD FILTERS")

rows = feed["records"]

bronx_rows = filter_records(
    rows,
    borough="BRONX"
)

bronx_sewer = filter_records(
    rows,
    borough="BRONX",
    category="SEWER"
)

both_rows = filter_records(
    rows,
    signal="Both signals"
)

anomaly_only_rows = filter_records(
    rows,
    signal="Anomaly only"
)

spatial_only_rows = filter_records(
    rows,
    signal="Spatial only"
)

neither_rows = filter_records(
    rows,
    signal="Neither"
)

queens_sewer_rows = [
    row
    for row in rows
    if (
        row["area_id"] == "QUEENS_11005"
        and row["complaint_category"] == "SEWER"
    )
]

queens_sewer_view = build_filtered_view(
    feed,
    queens_sewer_rows
)

road_pattern_rows = filter_records(
    rows,
    pattern="Road-heavy reporting"
)

# --------------------------------------------------
# 4. Validation checklist
# --------------------------------------------------

checks = []

def check(name, condition):
    checks.append({
        "test": name,
        "passed": bool(condition)
    })

summary = full_view["summary"]

check(
    "Dashboard Python source compiles",
    len(app_source) > 0
)

check(
    "All 500 complaint records load",
    summary["selected"] == 500
)

check(
    "15 anomaly groups represented",
    summary["anomaly_groups"] == 15
)

check(
    "23 spatial groups represented",
    summary["spatial_groups"] == 23
)

check(
    "204 anomaly-associated complaints",
    summary["anomaly_reports"] == 204
)

check(
    "228 spatial-associated complaints",
    summary["spatial_reports"] == 228
)

check(
    "156 complaints in both signals",
    summary["both"] == 156
)

check(
    "48 anomaly-only complaints",
    summary["anomaly_only"] == 48
)

check(
    "72 spatial-only complaints",
    summary["spatial_only"] == 72
)

check(
    "224 complaints in neither signal",
    summary["neither"] == 224
)

check(
    "276 distinct complaints in either signal",
    summary["distinct_either"] == 276
)

check(
    "All 20 reporting areas available",
    summary["areas"] == 20
)

check(
    "Borough filter selects 100 Bronx reports",
    len(bronx_rows) == 100
)

check(
    "Combined borough-category filter works",
    len(bronx_sewer) > 0
    and all(
        item["area_id"].startswith(
            "BRONX_"
        )
        and item["complaint_category"]
            == "SEWER"
        for item in bronx_sewer
    )
)

check(
    "Both-signal filter is correct",
    len(both_rows) == 156
)

check(
    "Anomaly-only filter is correct",
    len(anomaly_only_rows) == 48
)

check(
    "Spatial-only filter is correct",
    len(spatial_only_rows) == 72
)

check(
    "Neither-signal filter is correct",
    len(neither_rows) == 224
)

check(
    "All four signal filters partition 500",
    sum(map(len, [
        both_rows,
        anomaly_only_rows,
        spatial_only_rows,
        neither_rows
    ])) == 500
)

check(
    "Road-heavy reporting pattern filter works",
    len(road_pattern_rows) > 0
    and all(
        item["new_kmeans_pattern"]
        == "Road-heavy reporting"
        for item in road_pattern_rows
    )
)

check(
    "Sparse Queens sewer remains spatial-only",
    len(queens_sewer_rows) == 15
    and all(
        item["spatial_candidate"]
        and not item["anomaly_candidate"]
        for item in queens_sewer_rows
    )
)

check(
    "Sparse Queens sewer map candidate retained",
    "NB-00-004" in {
        feature["id"]
        for feature in queens_sewer_view[
            "map_features"
        ]
    }
)

check(
    "Sparse Queens sewer has no anomaly card",
    len(
        queens_sewer_view["anomaly_cards"]
    ) == 0
)

check(
    "All spatial cards have matching map points",
    len(
        full_view["map_features"]
    ) == 23
)

check(
    "Source data explicitly identified as synthetic",
    feed["metadata"]["synthetic_data"] is True
)

# Test an empty filter result.
empty_selection = filter_records(
    rows,
    borough="NONEXISTENT"
)

empty_view = build_filtered_view(
    feed,
    empty_selection
)

check(
    "Empty filter selection handled safely",
    empty_view["summary"]["selected"] == 0
    and not empty_view["anomaly_cards"]
    and not empty_view["map_features"]
)

# Check the original feed was not modified.
check(
    "Saved feed source unchanged",
    all(
        hashlib.sha256(
            path.read_bytes()
        ).hexdigest() == hashes_before[path]
        for path in protected
    )
)

results_20_1 = pd.DataFrame(checks)

print("\n3. VALIDATION")

display(results_20_1)

passed = int(
    results_20_1["passed"].sum()
)
total = len(results_20_1)

print(f"\nChecks passed: {passed}/{total}")

failed = results_20_1.loc[
    ~results_20_1["passed"],
    "test"
].tolist()

# --------------------------------------------------
# 5. Save the GUI only after validation
# --------------------------------------------------

if failed:

    print("\nSTATUS: NEEDS REVIEW")
    print("Dashboard GUI was not saved.")

    for item in failed:
        print("FAILED:", item)

else:

    if app_path.exists():
        assert app_path.read_text(
            encoding="utf-8"
        ) == app_source, (
            "An existing dashboard file differs. "
            "No overwrite performed."
        )
    else:
        app_path.parent.mkdir(
            parents=True,
            exist_ok=True
        )

        with app_path.open(
            "x",
            encoding="utf-8"
        ) as file:
            file.write(app_source)

    streamlit_installed = (
        importlib.util.find_spec(
            "streamlit"
        ) is not None
    )

    print("\n4. GUI FILE VERIFIED")
    print("Dashboard:", app_path)
    print("Streamlit installed:", streamlit_installed)

    print("\nSTATUS: PASSED")
    print("DASHBOARD GUI CREATED")
    print("OVERVIEW METRICS IMPLEMENTED")
    print("BOROUGH AND CATEGORY FILTERS IMPLEMENTED")
    print("K-MEANS PATTERN VIEW IMPLEMENTED")
    print("ANOMALY INVESTIGATION CARDS IMPLEMENTED")
    print("SPATIAL CANDIDATE MAP IMPLEMENTED")
    print("COMPLAINT TABLE AND CSV EXPORT IMPLEMENTED")
    print("HISTORICAL DATA AND MODELS UNMODIFIED")

    if streamlit_installed:
        print("\nREADY TO LAUNCH FROM VS CODE TERMINAL")
        print(
            "python -m streamlit run "
            "dashboard/urban_pulse_dashboard.py"
        )
    else:
        print("\nSTREAMLIT PACKAGE NOT DETECTED")
        print(
            "Install in the active ml environment: "
            "python -m pip install streamlit"
        )


 STREAMLIT DASHBOARD GUI

1. VALIDATING SAVED DASHBOARD DATA
Reporting day: 2026-11-09
Synthetic data: True
selected: 500
both: 156
anomaly_only: 48
spatial_only: 72
neither: 224
anomaly_reports: 204
spatial_reports: 228
distinct_either: 276
anomaly_groups: 15
spatial_groups: 23
areas: 20

2. TESTING DASHBOARD FILTERS

3. VALIDATION


,test,passed
0,Dashboard Python source compiles,True
1,All 500 complaint records load,True
2,15 anomaly groups represented,True
3,23 spatial groups represented,True
4,204 anomaly-associated complaints,True
5,228 spatial-associated complaints,True
6,156 complaints in both signals,True
7,48 anomaly-only complaints,True
8,72 spatial-only complaints,True
9,224 complaints in neither signal,True



Checks passed: 27/27

4. GUI FILE VERIFIED
Dashboard: D:\Urban-Infrastructure-Pulse-Intelligence\dashboard\urban_pulse_dashboard.py
Streamlit installed: True

STATUS: PASSED
DASHBOARD GUI CREATED
OVERVIEW METRICS IMPLEMENTED
BOROUGH AND CATEGORY FILTERS IMPLEMENTED
K-MEANS PATTERN VIEW IMPLEMENTED
ANOMALY INVESTIGATION CARDS IMPLEMENTED
SPATIAL CANDIDATE MAP IMPLEMENTED
COMPLAINT TABLE AND CSV EXPORT IMPLEMENTED
HISTORICAL DATA AND MODELS UNMODIFIED

READY TO LAUNCH FROM VS CODE TERMINAL
python -m streamlit run dashboard/urban_pulse_dashboard.py


In [36]:

from pathlib import Path
import ast
import hashlib
import json
import pandas as pd

print(" LIVE CSV UPLOAD INTEGRATION AUDIT")
print("=" * 75)

root = Path(PROJECT_ROOT).resolve()

files = {
    "bulk_contract":
        root / "src/integration/bulk_complaint_contract.py",
    "bulk_processor":
        root / "src/integration/bulk_complaint_processor.py",
    "daily_window":
        root / "src/integration/daily_batch_window.py",
    "three_model_integration":
        root / "src/integration/anomaly_snapshot_integration.py",
    "investigation_briefs":
        root / "src/integration/smart_assistant_investigation_briefs.py",
    "dashboard_feed_builder":
        root / "src/integration/dashboard_intelligence_feed.py",
    "streamlit_gui":
        root / "dashboard/urban_pulse_dashboard.py",
    "upload_template":
        root / "outputs/templates/new_complaints_template.csv",
    "verified_500":
        root / "outputs/tests/phase_17_4c_backend_verified"
        / "verified_500_backend_results.csv",
    "dashboard_feed":
        root / "outputs/tests/phase_19_2e_dashboard_feed"
        / "dashboard_feed.json",
    "historical_snapshot":
        root / "outputs/tables/anomaly"
        / "historical_daily_counts_2023_2025_v1.csv.gz"
}

existing = {
    name: path
    for name, path in files.items()
    if path.is_file()
}

hashes_before = {
    name: hashlib.sha256(path.read_bytes()).hexdigest()
    for name, path in existing.items()
    if name != "historical_snapshot"
}

# --------------------------------------------------
# 1. File readiness
# --------------------------------------------------

print("\n1. INTEGRATION FILE READINESS")

readiness = pd.DataFrame([
    {
        "component": name,
        "available": path.is_file(),
        "size_kb": round(
            path.stat().st_size / 1024, 1
        ) if path.is_file() else None
    }
    for name, path in files.items()
])

display(readiness)

# --------------------------------------------------
# 2. Discover actual processor API without importing
#    or executing the backend module
# --------------------------------------------------

def inspect_python_api(path):

    source = path.read_text(
        encoding="utf-8-sig"
    )

    tree = ast.parse(
        source,
        filename=str(path)
    )

    entries = []

    for node in tree.body:

        if isinstance(
            node,
            (ast.FunctionDef, ast.AsyncFunctionDef)
        ):

            if node.name.startswith("_"):
                continue

            entries.append({
                "object": node.name,
                "type": "function",
                "signature": (
                    f"{node.name}("
                    f"{ast.unparse(node.args)})"
                ),
                "description": (
                    ast.get_docstring(node) or ""
                ).split("\n")[0][:160]
            })

        elif isinstance(node, ast.ClassDef):

            entries.append({
                "object": node.name,
                "type": "class",
                "signature": f"class {node.name}",
                "description": (
                    ast.get_docstring(node) or ""
                ).split("\n")[0][:160]
            })

            for method in node.body:

                if not isinstance(
                    method,
                    (
                        ast.FunctionDef,
                        ast.AsyncFunctionDef
                    )
                ):
                    continue

                if (
                    method.name.startswith("_")
                    and method.name != "__init__"
                ):
                    continue

                entries.append({
                    "object": (
                        f"{node.name}.{method.name}"
                    ),
                    "type": "method",
                    "signature": (
                        f"{method.name}("
                        f"{ast.unparse(method.args)})"
                    ),
                    "description": (
                        ast.get_docstring(method) or ""
                    ).split("\n")[0][:160]
                })

    return pd.DataFrame(entries), tree


print("\n2. BULK PROCESSOR PUBLIC API")

if files["bulk_processor"].is_file():

    processor_api, processor_tree = (
        inspect_python_api(
            files["bulk_processor"]
        )
    )

    display(processor_api.head(35))

else:

    processor_api = pd.DataFrame()

    print("Bulk processor file not found.")

print("\n3. BULK INPUT CONTRACT PUBLIC API")

if files["bulk_contract"].is_file():

    contract_api, contract_tree = (
        inspect_python_api(
            files["bulk_contract"]
        )
    )

    display(contract_api.head(35))

else:

    contract_api = pd.DataFrame()

    print("Bulk contract file not found.")

# --------------------------------------------------
# 3. Identify important backend constants and
#    validation settings
# --------------------------------------------------

print("\n4. INPUT CONTRACT CONSTANTS")

if files["bulk_contract"].is_file():

    contract_source = files[
        "bulk_contract"
    ].read_text(
        encoding="utf-8-sig"
    )

    constants = []

    for node in contract_tree.body:

        if not isinstance(
            node,
            (ast.Assign, ast.AnnAssign)
        ):
            continue

        if isinstance(node, ast.Assign):
            targets = node.targets
        else:
            targets = [node.target]

        names = [
            target.id
            for target in targets
            if isinstance(target, ast.Name)
        ]

        for name in names:

            if any(
                word in name.upper()
                for word in [
                    "REQUIRED",
                    "COLUMN",
                    "FIELD",
                    "STATUS",
                    "CATEGORY",
                    "SCHEMA",
                    "ALIAS"
                ]
            ):

                value = (
                    ast.get_source_segment(
                        contract_source,
                        node.value
                    ) or ""
                )

                constants.append({
                    "constant": name,
                    "definition": value[:450]
                })

    if constants:
        display(pd.DataFrame(constants))
    else:
        print(
            "No matching top-level constants. "
            "Review public functions above."
        )

# --------------------------------------------------
# 4. Compare raw-upload template with verified
#    processed backend output
# --------------------------------------------------

print("\n5. UPLOAD TEMPLATE AND VERIFIED BACKEND SCHEMAS")

schema_tables = {}

for name in ["upload_template", "verified_500"]:

    path = files[name]

    if not path.is_file():
        continue

    sample = pd.read_csv(
        path,
        nrows=3,
        dtype="string",
        keep_default_na=False
    )

    schema_tables[name] = sample

    print(f"\n{name}")
    print("Columns:", len(sample.columns))
    print(sample.columns.tolist())

    print("Sample records:", len(sample))

    if not sample.empty:
        display(sample.head(2))

if (
    "upload_template" in schema_tables
    and "verified_500" in schema_tables
):

    template_columns = set(
        schema_tables[
            "upload_template"
        ].columns
    )

    backend_columns = set(
        schema_tables[
            "verified_500"
        ].columns
    )

    print("\n6. SCHEMA DIFFERENCES")

    print(
        "Template columns absent from backend output:",
        sorted(template_columns - backend_columns)
    )

    print(
        "Backend-added output columns:",
        sorted(backend_columns - template_columns)
    )

# --------------------------------------------------
# 5. Verify saved dashboard contract
# --------------------------------------------------

print("\n7. DASHBOARD FEED STRUCTURE")

if files["dashboard_feed"].is_file():

    with files["dashboard_feed"].open(
        encoding="utf-8"
    ) as file:
        feed = json.load(file)

    print("Top-level sections:", sorted(feed.keys()))

    print("Reporting day:",
          feed["metadata"].get("reporting_day"))

    print("Synthetic demonstration:",
          feed["metadata"].get("synthetic_data"))

    print("Saved complaints:",
          len(feed.get("records", [])))

    print("Area cards:",
          len(feed.get("area_cards", [])))

    print("Anomaly cards:",
          len(feed.get("anomaly_cards", [])))

    print("Spatial cards:",
          len(feed.get("spatial_cards", [])))

else:

    print("Dashboard feed missing.")

# --------------------------------------------------
# 6. Verify Phase 20.1 GUI is available
# --------------------------------------------------

print("\n8. STREAMLIT GUI READINESS")

gui_path = files["streamlit_gui"]

if gui_path.is_file():

    gui_source = gui_path.read_text(
        encoding="utf-8"
    )

    ast.parse(gui_source)

    print("Dashboard file:", gui_path)

    print(
        "Uses Streamlit:",
        "import streamlit" in gui_source
    )

    print(
        "Contains filter interface:",
        "filter_records" in gui_source
    )

    print(
        "Contains anomaly view:",
        "Anomaly Reviews" in gui_source
    )

    print(
        "Contains DBSCAN map:",
        "DBSCAN Map" in gui_source
    )

else:

    print(
        "GUI file not yet present. Finish that stage before attaching uploads."
    )

# --------------------------------------------------
# 7. Final read-only audit checks
# --------------------------------------------------

checks = []

def check(name, condition):

    checks.append({
        "test": name,
        "passed": bool(condition)
    })

check(
    "Bulk input contract available",
    files["bulk_contract"].is_file()
)

check(
    "Bulk processor available",
    files["bulk_processor"].is_file()
)

check(
    "Bulk processor API discovered",
    len(processor_api) > 0
)

check(
    "Bulk contract API discovered",
    len(contract_api) > 0
)

check(
    "Upload CSV template available",
    files["upload_template"].is_file()
)

check(
    "Verified processed example available",
    files["verified_500"].is_file()
)

check(
    "Three-model integration available",
    files["three_model_integration"].is_file()
)

check(
    "Investigation brief builder available",
    files["investigation_briefs"].is_file()
)

check(
    "Dashboard data builder available",
    files["dashboard_feed_builder"].is_file()
)

check(
    "Phase 20.1 Streamlit GUI available",
    gui_path.is_file()
)

check(
    "Saved dashboard feed available",
    files["dashboard_feed"].is_file()
)

check(
    "Historical snapshot available",
    files["historical_snapshot"].is_file()
)

check(
    "All inspected files unchanged",
    all(
        hashlib.sha256(
            path.read_bytes()
        ).hexdigest() == hashes_before[name]
        for name, path in existing.items()
        if name != "historical_snapshot"
    )
)

results_20_2a = pd.DataFrame(checks)

print("\n9. VALIDATION")

display(results_20_2a)

passed = int(results_20_2a["passed"].sum())
total = len(results_20_2a)

print(f"\nChecks passed: {passed}/{total}")

if passed == total:

    print("\nSTATUS: UPLOAD INTEGRATION CONTRACT READY")
    print("BACKEND AND GUI COMPONENTS LOCATED")

else:

    print("\nSTATUS: INTEGRATION PREREQUISITES NEED REVIEW")

print("NO FILES CREATED OR MODIFIED")
print("NO MODELS RETRAINED")
print("NO COMPLAINTS PROCESSED")


 LIVE CSV UPLOAD INTEGRATION AUDIT

1. INTEGRATION FILE READINESS


,component,available,size_kb
0,bulk_contract,True,5.5
1,bulk_processor,True,6.3
2,daily_window,True,5.2
3,three_model_integration,True,16.7
4,investigation_briefs,True,15.9
5,dashboard_feed_builder,True,12.0
6,streamlit_gui,True,19.7
7,upload_template,True,0.2
8,verified_500,True,165.4
9,dashboard_feed,True,422.8



2. BULK PROCESSOR PUBLIC API


,object,type,signature,description
0,process_complaint_batch,function,"process_complaint_batch(frame, supported_categ...",



3. BULK INPUT CONTRACT PUBLIC API


,object,type,signature,description
0,read_batch_csv,function,read_batch_csv(path_or_file),Read all fields as text to preserve ZIP zeros.
1,validate_batch,function,"validate_batch(frame, supported_categories)",Return one result for every submitted row.


... (5 outputs left out)

,source_row,complaint_id,created_date,latitude,longitude,complaint_category,incident_zip,validation_status,validation_errors,processing_status,...,kmeans_display_allowed,featured_count,supporting_count,limited_evidence_count,warning_count,processing_error,expected_area_id,expected_borough,intended_pattern,fully_verified
0,2,SIM-20261109-0001,2026-11-09,40.82080705,-73.92516448,SIDEWALK CONDITION,10451,ready_for_geographic_validation,,processed,...,True,0,0,0,12,,BRONX_10451,BRONX,road_a,True
1,3,SIM-20261109-0002,2026-11-09,40.82051049,-73.92569415,STREET CONDITION,10451,ready_for_geographic_validation,,processed,...,True,0,0,0,12,,BRONX_10451,BRONX,road_a,True



6. SCHEMA DIFFERENCES
Template columns absent from backend output: []
Backend-added output columns: ['borough', 'dbscan_cluster_id', 'dbscan_status', 'expected_area_id', 'expected_borough', 'featured_count', 'fully_verified', 'geography_status', 'intended_pattern', 'kmeans_area_id', 'kmeans_cluster_id', 'kmeans_display_allowed', 'kmeans_reporting_profile', 'kmeans_status', 'limited_evidence_count', 'processing_error', 'processing_status', 'source_row', 'supporting_count', 'system_status', 'validation_errors', 'validation_status', 'warning_count', 'zip_validation_status']

7. DASHBOARD FEED STRUCTURE
Top-level sections: ['anomaly_cards', 'area_cards', 'filters', 'map_geojson', 'metadata', 'records', 'spatial_cards', 'summary']
Reporting day: 2026-11-09
Synthetic demonstration: True
Saved complaints: 500
Area cards: 20
Anomaly cards: 15
Spatial cards: 23

8. STREAMLIT GUI READINESS
Dashboard file: D:\Urban-Infrastructure-Pulse-Intelligence\dashboard\urban_pulse_dashboard.py
Uses Streaml

,test,passed
0,Bulk input contract available,True
1,Bulk processor available,True
2,Bulk processor API discovered,True
3,Bulk contract API discovered,True
4,Upload CSV template available,True
5,Verified processed example available,True
6,Three-model integration available,True
7,Investigation brief builder available,True
8,Dashboard data builder available,True
9,Phase 20.1 Streamlit GUI available,True



Checks passed: 13/13

STATUS: UPLOAD INTEGRATION CONTRACT READY
BACKEND AND GUI COMPONENTS LOCATED
NO FILES CREATED OR MODIFIED
NO MODELS RETRAINED
NO COMPLAINTS PROCESSED


In [37]:

from pathlib import Path
from textwrap import dedent
from io import BytesIO
import ast
import hashlib
import inspect
import sys
import pandas as pd

print(" STREAMLIT CSV UPLOAD VALIDATION")
print("=" * 75)

root = Path(PROJECT_ROOT).resolve()

if str(root) not in sys.path:
    sys.path.insert(0, str(root))

from src.integration.bulk_complaint_contract import (
    INPUT_COLUMNS,
    REQUIRED_COLUMNS,
    read_batch_csv,
    validate_batch
)

from src.integration.bulk_complaint_processor import (
    process_complaint_batch
)

adapter_path = (
    root / "src/integration/upload_preflight.py"
)

page_path = (
    root / "dashboard/pages/1_Upload_Complaints.py"
)

template_path = (
    root / "outputs/templates/new_complaints_template.csv"
)

verified_path = (
    root / "outputs/tests/phase_17_4c_backend_verified"
    / "verified_500_backend_results.csv"
)

taxonomy_path = (
    root / "outputs/tables/dbscan"
    / "dbscan_action_taxonomy_v1.csv"
)

dashboard_path = (
    root / "dashboard/urban_pulse_dashboard.py"
)

protected = [
    template_path,
    verified_path,
    taxonomy_path,
    dashboard_path,
    root / "src/integration/bulk_complaint_contract.py",
    root / "src/integration/bulk_complaint_processor.py"
]

for path in protected:
    assert path.is_file(), f"Missing: {path}"

hashes_before = {
    str(path): hashlib.sha256(
        path.read_bytes()
    ).hexdigest()
    for path in protected
}

categories = sorted({
    str(value).strip().upper()
    for value in pd.read_csv(taxonomy_path)[
        "complaint_category"
    ].dropna()
})

assert len(categories) == 15

print("\n1. VERIFIED UPLOAD CONTRACT")
print("Input columns:", list(INPUT_COLUMNS))
print("Required columns:", list(REQUIRED_COLUMNS))
print("Supported categories:", len(categories))

# --------------------------------------------------
# 1. Reusable CSV-upload preflight module
# --------------------------------------------------

adapter_source = dedent(r'''
"""Safe in-memory upload validation for NYC 311 batches.

Uses the existing bulk complaint contract.
This module does not run models or modify files.
"""

from io import BytesIO

import pandas as pd

from src.integration.bulk_complaint_contract import (
    REQUIRED_COLUMNS,
    read_batch_csv,
    validate_batch
)


MAX_UPLOAD_BYTES = 10 * 1024 * 1024
MAX_UPLOAD_ROWS = 5000


def _find_validation_frame(value, expected_rows):
    """Find row-level results in the contract output."""

    if isinstance(value, pd.DataFrame):
        if len(value) == expected_rows:
            return value.copy()
        return None

    if isinstance(value, dict):
        for item in value.values():
            found = _find_validation_frame(
                item, expected_rows
            )
            if found is not None:
                return found

    if isinstance(value, (tuple, list)):

        if (
            len(value) == expected_rows
            and all(
                isinstance(item, dict)
                for item in value
            )
        ):
            return pd.DataFrame(value)

        for item in value:
            found = _find_validation_frame(
                item, expected_rows
            )
            if found is not None:
                return found

    return None


def run_upload_preflight(
    csv_bytes,
    supported_categories,
    max_bytes=MAX_UPLOAD_BYTES,
    max_rows=MAX_UPLOAD_ROWS
):
    """Read and validate CSV, without processing it."""

    if not isinstance(
        csv_bytes, (bytes, bytearray)
    ):
        raise TypeError(
            "Uploaded CSV must be provided as bytes."
        )

    if len(csv_bytes) == 0:
        raise ValueError("Uploaded CSV is empty.")

    if len(csv_bytes) > max_bytes:
        raise ValueError(
            "Uploaded CSV exceeds the size limit."
        )

    frame = read_batch_csv(
        BytesIO(csv_bytes)
    )

    if not isinstance(frame, pd.DataFrame):
        raise ValueError(
            "CSV reader did not return a DataFrame."
        )

    if frame.columns.duplicated().any():
        raise ValueError(
            "Duplicate CSV column names."
        )

    missing = [
        column
        for column in REQUIRED_COLUMNS
        if column not in frame.columns
    ]

    if missing:
        raise ValueError(
            f"Missing required CSV columns: {missing}"
        )

    if frame.empty:
        raise ValueError(
            "CSV has no complaint rows."
        )

    if len(frame) > max_rows:
        raise ValueError(
            f"Maximum {max_rows} complaints per upload."
        )

    # The established project contract is
    # the sole source of row validation.
    validation = validate_batch(
        frame.copy(deep=True),
        supported_categories
    )

    results = _find_validation_frame(
        validation,
        len(frame)
    )

    if results is None:
        raise ValueError(
            "Could not identify one validation result "
            "per uploaded row. Review the bulk "
            "contract return structure."
        )

    status_column = next(
        (
            column for column in [
                "validation_status",
                "status"
            ]
            if column in results.columns
        ),
        None
    )

    if status_column is not None:
        status_counts = {
            str(key): int(value)
            for key, value in results[
                status_column
            ].astype(str).value_counts().items()
        }
    else:
        status_counts = {}

    return {
        "input_records": frame,
        "validation_results": results,
        "row_count": int(len(frame)),
        "status_column": status_column,
        "status_counts": status_counts,
        "model_processing_performed": False,
        "historical_models_modified": False
    }
''').lstrip()

compile(
    adapter_source,
    str(adapter_path),
    "exec"
)

adapter_namespace = {
    "__name__": "_upload_preflight_validation"
}

exec(
    compile(
        adapter_source,
        str(adapter_path),
        "exec"
    ),
    adapter_namespace
)

run_upload_preflight = adapter_namespace[
    "run_upload_preflight"
]

# --------------------------------------------------
# 2. Create a standalone Streamlit upload page
# --------------------------------------------------

page_source = dedent(r'''
"""CSV upload and validation page for Urban Pulse."""

from pathlib import Path
import sys

import pandas as pd
import streamlit as st


PROJECT_ROOT = Path(__file__).resolve().parents[2]

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.integration.upload_preflight import (
    MAX_UPLOAD_BYTES,
    MAX_UPLOAD_ROWS,
    run_upload_preflight
)


TEMPLATE_PATH = (
    PROJECT_ROOT
    / "outputs/templates/new_complaints_template.csv"
)

TAXONOMY_PATH = (
    PROJECT_ROOT
    / "outputs/tables/dbscan/dbscan_action_taxonomy_v1.csv"
)


st.set_page_config(
    page_title="Upload Complaints | Urban Pulse",
    page_icon="📄",
    layout="wide"
)

st.title("Upload New Complaints")

st.caption(
    "Phase 20.2 — CSV upload validation"
)

st.info(
    "This page checks uploaded complaint data "
    "using the existing project validation contract. "
    "K-Means, DBSCAN and anomaly screening are "
    "not run during this preflight stage."
)

if not TEMPLATE_PATH.is_file():
    st.error(
        "The upload template is missing."
    )
    st.stop()

if not TAXONOMY_PATH.is_file():
    st.error(
        "The category taxonomy is missing."
    )
    st.stop()

categories = sorted({
    str(value).strip().upper()
    for value in pd.read_csv(
        TAXONOMY_PATH
    )["complaint_category"].dropna()
})

st.download_button(
    "Download CSV Template",
    data=TEMPLATE_PATH.read_bytes(),
    file_name="new_complaints_template.csv",
    mime="text/csv"
)

st.markdown(
    "Upload a CSV containing complaint IDs, "
    "timestamps, coordinates and categories."
)

st.caption(
    f"Maximum file size: "
    f"{MAX_UPLOAD_BYTES // (1024 * 1024)} MB. "
    f"Maximum complaint rows: {MAX_UPLOAD_ROWS:,}."
)

uploaded = st.file_uploader(
    "Choose a new complaints CSV",
    type=["csv"],
    accept_multiple_files=False
)

if uploaded is not None:

    st.write(
        "**Uploaded file:**",
        uploaded.name
    )

    if st.button(
        "Validate Uploaded Complaints",
        type="primary"
    ):

        try:
            result = run_upload_preflight(
                uploaded.getvalue(),
                categories
            )

        except Exception as exc:
            st.error(
                "Upload validation could not "
                f"be completed: {exc}"
            )

        else:
            st.success(
                "CSV validation finished. "
                "No models were executed."
            )

            st.metric(
                "Submitted complaint rows",
                result["row_count"]
            )

            if result["status_counts"]:

                st.subheader(
                    "Validation status breakdown"
                )

                counts = pd.DataFrame([
                    {
                        "Validation status": key,
                        "Complaints": value
                    }
                    for key, value
                    in result[
                        "status_counts"
                    ].items()
                ])

                st.dataframe(
                    counts,
                    hide_index=True,
                    use_container_width=True
                )

            st.subheader(
                "Row-level validation results"
            )

            st.dataframe(
                result["validation_results"],
                hide_index=True,
                use_container_width=True
            )

            csv_output = (
                result["validation_results"]
                .to_csv(index=False)
                .encode("utf-8")
            )

            st.download_button(
                "Download Validation Results",
                data=csv_output,
                file_name=(
                    "upload_validation_results.csv"
                ),
                mime="text/csv"
            )

            st.warning(
                "Validation is not geographic model "
                "processing. These complaints have not "
                "yet been assigned K-Means patterns, "
                "DBSCAN concentrations or anomaly flags."
            )

st.divider()

st.caption(
    "Urban Infrastructure Pulse Intelligence "
    "| Upload validation only | "
    "No historical models modified"
)
''').lstrip()

ast.parse(page_source)

# --------------------------------------------------
# 3. Test actual input through existing contract
# --------------------------------------------------

print("\n2. TESTING REAL CSV UPLOAD CONTRACT")

template_result = run_upload_preflight(
    template_path.read_bytes(),
    categories
)

print(
    "Template records:",
    template_result["row_count"]
)

print(
    "Template validation statuses:",
    template_result["status_counts"]
)

display(
    template_result[
        "validation_results"
    ].head()
)

# Reconstruct the six-column raw input from
# previously verified synthetic backend records.
verified = pd.read_csv(
    verified_path,
    dtype="string",
    keep_default_na=False
)

raw_500 = verified.loc[
    :, list(INPUT_COLUMNS)
].copy()

raw_500_snapshot = raw_500.copy(deep=True)

csv_500 = raw_500.to_csv(
    index=False
).encode("utf-8")

batch_result = run_upload_preflight(
    csv_500,
    categories
)

print("\n3. TESTING 500-COMPLAINT UPLOAD")

print("Rows read:", batch_result["row_count"])

print(
    "Validation statuses:",
    batch_result["status_counts"]
)

display(
    batch_result[
        "validation_results"
    ].head(5)
)

# --------------------------------------------------
# 4. Test invalid and missing-file safeguards
# --------------------------------------------------

def is_rejected(data):
    try:
        run_upload_preflight(
            data,
            categories
        )
    except (
        ValueError,
        TypeError,
        pd.errors.EmptyDataError
    ):
        return True
    return False

empty_rejected = is_rejected(b"")

missing_column_bytes = (
    b"complaint_id,created_date\n"
    b"BAD-001,2026-11-09\n"
)

missing_columns_rejected = is_rejected(
    missing_column_bytes
)

oversized_rejected = is_rejected(
    b"x" * (10 * 1024 * 1024 + 1)
)

# --------------------------------------------------
# 5. Discover exact processor signature
# --------------------------------------------------

print("\n4. COMPLETE BULK PROCESSOR SIGNATURE")

processor_signature = inspect.signature(
    process_complaint_batch
)

print(
    "process_complaint_batch",
    processor_signature
)

parameters = pd.DataFrame([
    {
        "parameter": name,
        "kind": str(parameter.kind),
        "required": (
            parameter.default
            is inspect.Parameter.empty
        ),
        "default": (
            None
            if parameter.default
            is inspect.Parameter.empty
            else repr(parameter.default)[:150]
        )
    }
    for name, parameter
    in processor_signature.parameters.items()
])

display(parameters)

# --------------------------------------------------
# 6. Validate before saving new files
# --------------------------------------------------

checks = []

def check(name, condition):
    checks.append({
        "test": name,
        "passed": bool(condition)
    })

check(
    "Upload adapter compiles",
    bool(adapter_source)
)

check(
    "Streamlit page compiles",
    bool(page_source)
)

check(
    "Project upload template accepted",
    template_result["row_count"] == 2
)

check(
    "Template has row-level validation results",
    len(
        template_result[
            "validation_results"
        ]
    ) == 2
)

check(
    "Verified synthetic upload has 500 rows",
    batch_result["row_count"] == 500
)

check(
    "500 rows receive validation results",
    len(
        batch_result[
            "validation_results"
        ]
    ) == 500
)

check(
    "Required CSV fields preserved",
    set(REQUIRED_COLUMNS).issubset(
        batch_result[
            "input_records"
        ].columns
    )
)

check(
    "Original uploaded text preserved",
    batch_result[
        "input_records"
    ]["complaint_id"].astype(str).tolist()
    == raw_500["complaint_id"].astype(str).tolist()
)

check(
    "Original ZIP text preserved",
    batch_result[
        "input_records"
    ]["incident_zip"].astype(str).tolist()
    == raw_500["incident_zip"].astype(str).tolist()
)

check(
    "Empty upload rejected",
    empty_rejected
)

check(
    "Missing mandatory columns rejected",
    missing_columns_rejected
)

check(
    "Oversized CSV rejected",
    oversized_rejected
)

check(
    "Upload validation does not run models",
    batch_result[
        "model_processing_performed"
    ] is False
)

check(
    "Existing input DataFrame unchanged",
    raw_500.equals(raw_500_snapshot)
)

check(
    "Backend processor signature identified",
    "frame" in processor_signature.parameters
)

check(
    "Historical model artifacts untouched",
    all(
        hashlib.sha256(
            path.read_bytes()
        ).hexdigest()
        == hashes_before[str(path)]
        for path in protected
    )
)

results_20_2b = pd.DataFrame(checks)

print("\n5. VALIDATION")

display(results_20_2b)

passed = int(
    results_20_2b["passed"].sum()
)

total = len(results_20_2b)

print(f"\nChecks passed: {passed}/{total}")

failures = results_20_2b.loc[
    ~results_20_2b["passed"],
    "test"
].tolist()

# --------------------------------------------------
# 7. Save only after all tests pass
# --------------------------------------------------

if failures:

    print("\nSTATUS: NEEDS REVIEW")
    print("No upload GUI files saved.")

    for failure in failures:
        print("FAILED:", failure)

else:

    destinations = {
        adapter_path: adapter_source,
        page_path: page_source
    }

    for path, content in destinations.items():

        if path.exists():
            assert path.read_text(
                encoding="utf-8"
            ) == content, (
                f"Existing file differs: {path}. "
                "No overwrite performed."
            )

    for path, content in destinations.items():

        path.parent.mkdir(
            parents=True,
            exist_ok=True
        )

        if not path.exists():
            with path.open(
                "x",
                encoding="utf-8"
            ) as file:
                file.write(content)

    print("\n6. VERIFIED UPLOAD GUI FILES")

    print("Upload adapter:", adapter_path)
    print("Streamlit upload page:", page_path)

    print("\nSTATUS: PASSED")
    print("CSV UPLOAD VALIDATION READY")
    print("EXISTING BULK CONTRACT REUSED")
    print("UPLOAD SIZE AND COLUMN SAFEGUARDS VERIFIED")
    print("500-ROW INPUT CONTRACT TESTED")
    print("NO HISTORICAL DATA OR MODELS MODIFIED")
    print("LIVE MODEL PROCESSING NOT YET ENABLED")


 STREAMLIT CSV UPLOAD VALIDATION

1. VERIFIED UPLOAD CONTRACT
Input columns: ['complaint_id', 'created_date', 'latitude', 'longitude', 'complaint_category', 'incident_zip']
Required columns: ['complaint_id', 'created_date', 'latitude', 'longitude', 'complaint_category']
Supported categories: 15

2. TESTING REAL CSV UPLOAD CONTRACT
Template records: 2
Template validation statuses: {'ready_for_geographic_validation': 2}


,source_row,complaint_id,created_date,latitude,longitude,complaint_category,incident_zip,validation_status,validation_errors
0,2,NEW-001,2026-11-09,40.702106,-74.013110,STREET CONDITION,None,ready_for_geographic_validation,
1,3,NEW-002,2026-11-09,40.751690,-74.004661,STREET CONDITION,10001,ready_for_geographic_validation,



3. TESTING 500-COMPLAINT UPLOAD
Rows read: 500
Validation statuses: {'ready_for_geographic_validation': 500}


,source_row,complaint_id,created_date,latitude,longitude,complaint_category,incident_zip,validation_status,validation_errors
0,2,SIM-20261109-0001,2026-11-09,40.820807,-73.925164,SIDEWALK CONDITION,10451,ready_for_geographic_validation,
1,3,SIM-20261109-0002,2026-11-09,40.820510,-73.925694,STREET CONDITION,10451,ready_for_geographic_validation,
2,4,SIM-20261109-0003,2026-11-09,40.818989,-73.927052,SIDEWALK CONDITION,10451,ready_for_geographic_validation,
3,5,SIM-20261109-0004,2026-11-09,40.819652,-73.923823,TRAFFIC SIGNAL CONDITION,10451,ready_for_geographic_validation,
4,6,SIM-20261109-0005,2026-11-09,40.819394,-73.926800,STREET CONDITION,10451,ready_for_geographic_validation,



4. COMPLETE BULK PROCESSOR SIGNATURE
process_complaint_batch (frame, supported_categories, adapter, use_cache=True)


,parameter,kind,required,default
0,frame,POSITIONAL_OR_KEYWORD,True,NaN
1,supported_categories,POSITIONAL_OR_KEYWORD,True,NaN
2,adapter,POSITIONAL_OR_KEYWORD,True,NaN
3,use_cache,POSITIONAL_OR_KEYWORD,False,True



5. VALIDATION


,test,passed
0,Upload adapter compiles,True
1,Streamlit page compiles,True
2,Project upload template accepted,True
3,Template has row-level validation results,True
4,Verified synthetic upload has 500 rows,True
5,500 rows receive validation results,True
6,Required CSV fields preserved,True
7,Original uploaded text preserved,True
8,Original ZIP text preserved,True
9,Empty upload rejected,True



Checks passed: 16/16

6. VERIFIED UPLOAD GUI FILES
Upload adapter: D:\Urban-Infrastructure-Pulse-Intelligence\src\integration\upload_preflight.py
Streamlit upload page: D:\Urban-Infrastructure-Pulse-Intelligence\dashboard\pages\1_Upload_Complaints.py

STATUS: PASSED
CSV UPLOAD VALIDATION READY
EXISTING BULK CONTRACT REUSED
UPLOAD SIZE AND COLUMN SAFEGUARDS VERIFIED
500-ROW INPUT CONTRACT TESTED
NO HISTORICAL DATA OR MODELS MODIFIED
LIVE MODEL PROCESSING NOT YET ENABLED


In [38]:

import inspect
import ast
import pandas as pd

from src.integration.bulk_complaint_processor import (
    process_complaint_batch
)

from src.integration.bulk_complaint_contract import (
    validate_batch,
    read_batch_csv
)

print(" BACKEND PROCESSING CONTRACT")
print("=" * 70)

# 1. Exact processor signature

signature = inspect.signature(
    process_complaint_batch
)

print("\n1. COMPLETE PROCESSOR SIGNATURE")
print(
    "process_complaint_batch",
    signature
)

parameters = []

for name, parameter in signature.parameters.items():

    parameters.append({
        "parameter": name,
        "required": (
            parameter.default
            is inspect.Parameter.empty
        ),
        "default": (
            "REQUIRED"
            if parameter.default
            is inspect.Parameter.empty
            else repr(parameter.default)
        ),
        "kind": str(parameter.kind)
    })

display(pd.DataFrame(parameters))

# 2. Show the existing implementation

print("\n2. BULK PROCESSOR IMPLEMENTATION")

source_lines, start_line = (
    inspect.getsourcelines(
        process_complaint_batch
    )
)

for offset, line in enumerate(source_lines):
    print(
        f"{start_line + offset:04d}: "
        f"{line}",
        end=""
    )

# 3. Discover return structure without execution

print("\n\n3. PROCESSOR RETURN STATEMENTS")

tree = ast.parse(
    inspect.getsource(
        process_complaint_batch
    )
)

returns = [
    node for node in ast.walk(tree)
    if isinstance(node, ast.Return)
]

for index, node in enumerate(returns, 1):

    print(f"\nReturn statement {index}:")

    try:
        print(
            ast.unparse(node.value)[:2000]
            if node.value is not None
            else "None"
        )
    except Exception:
        print("Could not format return.")

# 4. Confirm related API signatures

print("\n4. EXISTING CSV CONTRACT")

print(
    "read_batch_csv:",
    inspect.signature(read_batch_csv)
)

print(
    "validate_batch:",
    inspect.signature(validate_batch)
)

print("\nSTATUS: BACKEND CONTRACT INSPECTED")
print("NO FILES MODIFIED")
print("NO MODELS EXECUTED")


 BACKEND PROCESSING CONTRACT

1. COMPLETE PROCESSOR SIGNATURE
process_complaint_batch (frame, supported_categories, adapter, use_cache=True)


,parameter,required,default,kind
0,frame,True,REQUIRED,POSITIONAL_OR_KEYWORD
1,supported_categories,True,REQUIRED,POSITIONAL_OR_KEYWORD
2,adapter,True,REQUIRED,POSITIONAL_OR_KEYWORD
3,use_cache,False,True,POSITIONAL_OR_KEYWORD



2. BULK PROCESSOR IMPLEMENTATION
0014: def process_complaint_batch(
0015:     frame,
0016:     supported_categories,
0017:     adapter,
0018:     use_cache=True
0019: ):
0020:     validated = validate_batch(
0021:         frame,
0022:         supported_categories
0023:     )
0024: 
0025:     results = []
0026:     responses_by_source_row = {}
0027:     cache = {}
0028:     backend_evaluations = 0
0029: 
0030:     for record in validated.to_dict("records"):
0031: 
0032:         output = dict(record)
0033: 
0034:         output.update({
0035:             "processing_status": None,
0036:             "system_status": None,
0037:             "borough": None,
0038:             "geography_status": None,
0039:             "zip_validation_status": None,
0040:             "dbscan_status": None,
0041:             "dbscan_cluster_id": None,
0042:             "kmeans_status": None,
0043:             "kmeans_area_id": None,
0044:             "kmeans_cluster_id": None,
0045:             "kmeans_repo

In [39]:

from pathlib import Path
import ast
import inspect
import json
import os
import textwrap

from src.integration.bulk_complaint_processor import (
    process_complaint_batch
)

print("BACKEND ADAPTER DISCOVERY")
print("=" * 72)

root = Path(PROJECT_ROOT).resolve()

# --------------------------------------------------
# 1. Identify exactly how adapter is called
# --------------------------------------------------

source = textwrap.dedent(
    inspect.getsource(process_complaint_batch)
)

tree = ast.parse(source)

function = tree.body[0]

print("\n1. ADAPTER CALLS INSIDE BULK PROCESSOR")

adapter_calls = []

for node in ast.walk(function):

    if not isinstance(node, ast.Call):
        continue

    target = node.func

    is_adapter_call = (
        isinstance(target, ast.Name)
        and target.id == "adapter"
    )

    is_adapter_method = (
        isinstance(target, ast.Attribute)
        and isinstance(target.value, ast.Name)
        and target.value.id == "adapter"
    )

    if is_adapter_call or is_adapter_method:

        adapter_calls.append({
            "line": node.lineno,
            "expression": ast.unparse(node)[:550]
        })

for call in adapter_calls:
    print(
        f"Line {call['line']}: "
        f"{call['expression']}"
    )

if not adapter_calls:
    print(
        "No direct adapter calls found. "
        "Checking other references below."
    )

# --------------------------------------------------
# 2. Print only relevant adapter source lines
# --------------------------------------------------

print("\n2. ADAPTER IMPLEMENTATION CONTEXT")

source_lines = source.splitlines()

relevant_lines = set()

for index, line in enumerate(source_lines):

    if (
        "adapter" in line
        or "backend_evaluations" in line
        or "responses_by_source_row" in line
    ):

        for position in range(
            max(0, index - 3),
            min(len(source_lines), index + 7)
        ):
            relevant_lines.add(position)

for index in sorted(relevant_lines):

    print(
        f"{index + 1:03d}: "
        f"{source_lines[index]}"
    )

# --------------------------------------------------
# 3. Identify processor return fields
# --------------------------------------------------

print("\n3. PROCESSOR RETURN CONTRACT")

return_nodes = [
    node
    for node in ast.walk(function)
    if isinstance(node, ast.Return)
]

for number, node in enumerate(
    return_nodes,
    start=1
):

    value = node.value

    if isinstance(value, ast.Dict):

        keys = []

        for key in value.keys:

            if isinstance(key, ast.Constant):
                keys.append(str(key.value))
            else:
                keys.append(ast.unparse(key))

        print(
            f"Return {number} dictionary keys:",
            keys
        )

    else:

        print(
            f"Return {number}:",
            ast.unparse(value)[:350]
            if value is not None
            else "None"
        )

# --------------------------------------------------
# 4. Search actual project call sites
# --------------------------------------------------

print("\n4. EXISTING PROCESSOR USAGE EXAMPLES")

search_roots = [
    root / "src",
    root / "notebooks",
    root / "dashboard"
]

skip_directories = {
    ".git",
    ".venv",
    "venv",
    "__pycache__",
    ".ipynb_checkpoints",
    "node_modules",
    "site-packages"
}

examples = []
MAX_EXAMPLES = 8

def inspect_code_lines(code, location):

    lines = code.splitlines()

    for index, line in enumerate(lines):

        if "process_complaint_batch(" not in line:
            continue

        # Ignore its function definition.
        if line.lstrip().startswith(
            ("def ", "async def ")
        ):
            continue

        start = max(0, index - 8)
        end = min(
            len(lines),
            index + 16
        )

        excerpt = "\n".join(
            f"{position + 1:03d}: {lines[position]}"
            for position in range(start, end)
        )

        examples.append({
            "location": location,
            "excerpt": excerpt
        })

        if len(examples) >= MAX_EXAMPLES:
            return

for search_root in search_roots:

    if not search_root.is_dir():
        continue

    if len(examples) >= MAX_EXAMPLES:
        break

    for current_dir, directories, filenames in os.walk(
        search_root
    ):

        directories[:] = sorted(
            name for name in directories
            if name not in skip_directories
        )

        for filename in sorted(filenames):

            if len(examples) >= MAX_EXAMPLES:
                break

            path = Path(current_dir) / filename

            if path.suffix not in {
                ".py", ".ipynb"
            }:
                continue

            try:

                if path.suffix == ".py":

                    code = path.read_text(
                        encoding="utf-8-sig"
                    )

                    inspect_code_lines(
                        code,
                        str(path.relative_to(root))
                    )

                else:

                    notebook = json.loads(
                        path.read_text(
                            encoding="utf-8"
                        )
                    )

                    for cell_index, cell in enumerate(
                        notebook.get("cells", [])
                    ):

                        if cell.get(
                            "cell_type"
                        ) != "code":
                            continue

                        content = cell.get(
                            "source", []
                        )

                        code = (
                            "".join(content)
                            if isinstance(content, list)
                            else str(content)
                        )

                        if (
                            "process_complaint_batch("
                            not in code
                        ):
                            continue

                        inspect_code_lines(
                            code,
                            (
                                str(path.relative_to(root))
                                + f" | cell {cell_index}"
                            )
                        )

                        if len(examples) >= MAX_EXAMPLES:
                            break

            except (
                OSError,
                UnicodeError,
                json.JSONDecodeError
            ):
                continue

for number, example in enumerate(
    examples,
    start=1
):

    print(f"\nEXAMPLE {number}")
    print("Location:", example["location"])
    print(example["excerpt"])

if not examples:
    print(
        "No previous processor invocation found "
        "in src, notebooks or dashboard."
    )

# --------------------------------------------------
# 5. Check currently loaded adapter variables
# --------------------------------------------------

print("\n5. POSSIBLE EXISTING NOTEBOOK ADAPTERS")

adapter_variables = []

for name, value in list(globals().items()):

    if name.startswith("_"):
        continue

    if "adapter" not in name.lower():
        continue

    if inspect.ismodule(value):
        continue

    adapter_variables.append({
        "variable": name,
        "python_type": type(value).__name__,
        "callable": callable(value)
    })

if adapter_variables:

    for item in adapter_variables:

        print(
            f"{item['variable']} | "
            f"type={item['python_type']} | "
            f"callable={item['callable']}"
        )

else:

    print(
        "No adapter-named object currently "
        "loaded in notebook memory."
    )

# --------------------------------------------------
# 6. Completion
# --------------------------------------------------

print("\n6. AUDIT SUMMARY")

print(
    "Direct adapter calls:",
    len(adapter_calls)
)

print(
    "Existing processor usage examples:",
    len(examples)
)

print(
    "Adapter-related notebook variables:",
    len(adapter_variables)
)

print("\nSTATUS: ADAPTER DISCOVERY COMPLETED")
print("NO COMPLAINTS PROCESSED")
print("NO FILES MODIFIED")
print("NO MODELS RETRAINED")


BACKEND ADAPTER DISCOVERY

1. ADAPTER CALLS INSIDE BULK PROCESSOR
Line 74: adapter.evaluate(key[0], key[1], key[2], key[3])

2. ADAPTER IMPLEMENTATION CONTEXT
001: def process_complaint_batch(
002:     frame,
003:     supported_categories,
004:     adapter,
005:     use_cache=True
006: ):
007:     validated = validate_batch(
008:         frame,
009:         supported_categories
010:     )
011: 
012:     results = []
013:     responses_by_source_row = {}
014:     cache = {}
015:     backend_evaluations = 0
016: 
017:     for record in validated.to_dict("records"):
018: 
019:         output = dict(record)
020: 
021:         output.update({
069:                 response = cache[key]
070: 
071:             else:
072:                 backend_evaluations += 1
073: 
074:                 response = adapter.evaluate(
075:                     key[0],
076:                     key[1],
077:                     key[2],
078:                     key[3]
079:                 )
080: 
166: 
167:          

In [40]:

from pathlib import Path
import ast
import inspect
import json
import textwrap

from src.integration.bulk_complaint_processor import (
    process_complaint_batch
)

print(" FOCUSED ADAPTER INSPECTION")
print("=" * 65)

root = Path(PROJECT_ROOT).resolve()

# 1. Processor return structure
tree = ast.parse(
    textwrap.dedent(
        inspect.getsource(process_complaint_batch)
    )
)

print("\n1. PROCESSOR RETURN STRUCTURE")

for node in ast.walk(tree):
    if not isinstance(node, ast.Return):
        continue

    if isinstance(node.value, ast.Dict):
        print(
            "Return keys:",
            [
                ast.literal_eval(k)
                if isinstance(k, ast.Constant)
                else ast.unparse(k)
                for k in node.value.keys
            ]
        )
    elif node.value is not None:
        print(
            "Return expression:",
            ast.unparse(node.value)[:350]
        )

# 2. Find actual adapter classes in src
print("\n2. EXISTING ADAPTER CLASSES")

adapter_classes = []

for path in sorted((root / "src").rglob("*.py")):

    if "__pycache__" in path.parts:
        continue

    try:
        source = path.read_text(
            encoding="utf-8-sig"
        )
        module_tree = ast.parse(source)
    except (OSError, UnicodeError, SyntaxError):
        continue

    for cls in ast.walk(module_tree):
        if not isinstance(cls, ast.ClassDef):
            continue

        methods = {
            item.name: item
            for item in cls.body
            if isinstance(
                item,
                (ast.FunctionDef, ast.AsyncFunctionDef)
            )
        }

        if "evaluate" not in methods:
            continue

        constructor = methods.get("__init__")
        evaluator = methods["evaluate"]

        info = {
            "class": cls.name,
            "file": str(path.relative_to(root)),
            "constructor": (
                ast.unparse(constructor.args)
                if constructor else "default"
            ),
            "evaluate": ast.unparse(evaluator.args)
        }

        adapter_classes.append(info)

for info in adapter_classes[:10]:
    print("\nClass:", info["class"])
    print("File:", info["file"])
    print("Constructor:", info["constructor"])
    print("Evaluate:", info["evaluate"])

print(
    "\nAdapter classes discovered:",
    len(adapter_classes)
)

# 3. Existing adapter objects in notebook memory
print("\n3. LOADED NOTEBOOK ADAPTER OBJECTS")

objects_found = 0

for name, obj in list(globals().items()):

    if name.startswith("_") or inspect.isclass(obj):
        continue

    if not callable(
        getattr(type(obj), "evaluate", None)
    ):
        continue

    print("Variable:", name)
    print("Type:", type(obj).__name__)
    print("Module:", type(obj).__module__)

    try:
        print(
            "Evaluate signature:",
            inspect.signature(obj.evaluate)
        )
    except (ValueError, TypeError):
        pass

    objects_found += 1

    if objects_found >= 5:
        break

if objects_found == 0:
    print("No loaded adapter instance found.")

# 4. Find existing processor calls in project code
print("\n4. EXISTING PROCESSOR INVOCATIONS")

call_sites = []

search_paths = (
    list((root / "src").rglob("*.py"))
    + list((root / "notebooks").glob("*.ipynb"))
)

for path in search_paths:

    if "__pycache__" in path.parts:
        continue

    try:
        if path.suffix == ".ipynb":
            notebook = json.loads(
                path.read_text(encoding="utf-8")
            )

            code_units = [
                "".join(cell.get("source", []))
                for cell in notebook.get("cells", [])
                if cell.get("cell_type") == "code"
            ]
        else:
            code_units = [
                path.read_text(encoding="utf-8-sig")
            ]

        for index, code in enumerate(code_units):

            if "process_complaint_batch(" not in code:
                continue

            try:
                code_tree = ast.parse(code)
            except SyntaxError:
                continue

            for node in ast.walk(code_tree):

                if not isinstance(node, ast.Call):
                    continue

                func = node.func

                if not (
                    isinstance(func, ast.Name)
                    and func.id == "process_complaint_batch"
                ):
                    continue

                call_sites.append({
                    "file": str(path.relative_to(root)),
                    "cell": index if path.suffix == ".ipynb"
                            else None,
                    "call": ast.unparse(node)[:650]
                })

    except (OSError, UnicodeError, json.JSONDecodeError):
        continue

for item in call_sites[:6]:

    print("\nFile:", item["file"])

    if item["cell"] is not None:
        print("Notebook cell index:", item["cell"])

    print("Call:", item["call"])

print("\nExisting calls discovered:", len(call_sites))

# 5. Completion
print("\n5. RESULT")
print("STATUS: ADAPTER CONTRACT INSPECTED")
print("NO FILES CREATED OR MODIFIED")
print("NO COMPLAINTS PROCESSED")
print("NO HISTORICAL MODELS MODIFIED")


 FOCUSED ADAPTER INSPECTION

1. PROCESSOR RETURN STRUCTURE
Return keys: ['records', 'responses_by_source_row', 'stats']

2. EXISTING ADAPTER CLASSES

Class: CombinedInfrastructureInsightService
File: src\integration\combined_infrastructure_insight.py
Constructor: self, validated_service, kmeans_lookup
Evaluate: self, latitude, longitude, complaint_category, incident_zip=None

Class: InfrastructureGUIAdapter
File: src\integration\infrastructure_gui_contract.py
Constructor: self, insight_service
Evaluate: self, latitude, longitude, complaint_category, incident_zip=None

Class: NewComplaintInsightService
File: src\integration\new_complaint_insight.py
Constructor: self, predictor, engine, payload_builder
Evaluate: self, latitude, longitude, complaint_category=None

Class: ValidatedNewComplaintService
File: src\integration\validated_new_complaint.py
Constructor: self, insight_service, boundary_path, taxonomy_path, boundary_tolerance_m=10.0
Evaluate: self, latitude, longitude, complaint_cate

In [41]:

from pathlib import Path
import inspect
import json
import re

print(" ADAPTER INITIALIZATION AUDIT")
print("=" * 68)

root = Path(PROJECT_ROOT).resolve()

assert "adapter" in globals(), (
    "The working adapter is not loaded. "
    "Run the notebook setup cell first."
)

# --------------------------------------------------
# 1. Inspect actual adapter dependency hierarchy
# --------------------------------------------------

print("\n1. WORKING ADAPTER DEPENDENCY TREE")

visited = set()
discovered_classes = {}

def inspect_object(obj, label, depth=0):

    if depth > 7 or len(visited) >= 45:
        return

    object_id = id(obj)

    if object_id in visited:
        return

    visited.add(object_id)

    cls = type(obj)
    module = cls.__module__

    prefix = "  " * depth

    print(
        f"{prefix}{label}: "
        f"{module}.{cls.__name__}"
    )

    if not module.startswith("src."):
        return

    discovered_classes[
        (module, cls.__name__)
    ] = cls

    try:
        attributes = vars(obj)
    except TypeError:
        return

    for name, value in attributes.items():

        if name.startswith("__"):
            continue

        value_type = type(value)
        value_module = value_type.__module__

        if value_module.startswith("src."):

            inspect_object(
                value,
                name,
                depth + 1
            )

        elif isinstance(value, Path):

            print(
                f"{prefix}  {name}: "
                f"Path({value})"
            )

        elif isinstance(
            value,
            (str, int, float, bool, type(None))
        ):

            # Show only short non-sensitive
            # configuration values.
            if any(
                term in name.lower()
                for term in [
                    "path",
                    "file",
                    "tolerance",
                    "radius",
                    "threshold"
                ]
            ):

                print(
                    f"{prefix}  {name}: "
                    f"{str(value)[:150]}"
                )

            else:

                print(
                    f"{prefix}  {name}: "
                    f"{value_type.__name__}"
                )

        else:

            shape = getattr(
                value, "shape", None
            )

            detail = (
                f" shape={shape}"
                if shape is not None else ""
            )

            print(
                f"{prefix}  {name}: "
                f"{value_module}."
                f"{value_type.__name__}"
                f"{detail}"
            )

inspect_object(
    adapter,
    "adapter"
)

# --------------------------------------------------
# 2. Inspect constructors of actual dependencies
# --------------------------------------------------

print("\n2. SERVICE CONSTRUCTORS")

for (module, class_name), cls in (
    discovered_classes.items()
):

    print(f"\n{module}.{class_name}")

    try:
        print(
            "Constructor signature:",
            inspect.signature(cls)
        )

        lines = inspect.getsource(
            cls.__init__
        ).splitlines()

        # Constructor only, not full service code.
        for line in lines[:28]:
            print(line)

        if len(lines) > 28:
            print(
                f"... {len(lines) - 28} "
                "additional constructor lines"
            )

    except (
        OSError,
        TypeError,
        ValueError
    ) as error:

        print(
            "Source unavailable:",
            type(error).__name__
        )

# --------------------------------------------------
# 3. Find the notebook's original initialization
# --------------------------------------------------

print("\n3. ORIGINAL NOTEBOOK ADAPTER SETUP")

notebook_path = (
    root / "notebooks"
    / "06_smart_assistant_bulk_intelligence.ipynb"
)

assert notebook_path.is_file()

notebook = json.loads(
    notebook_path.read_text(
        encoding="utf-8"
    )
)

setup_patterns = [
    r"\badapter\s*=",
    r"InfrastructureGUIAdapter\s*\(",
    r"CombinedInfrastructureInsightService\s*\(",
    r"ZCTAAwareCombinedInsightService\s*\(",
    r"ValidatedNewComplaintService\s*\(",
    r"NewComplaintInsightService\s*\(",
    r"ZCTACoordinateValidator\s*\("
]

matches = []

for cell_index, cell in enumerate(
    notebook.get("cells", [])
):

    if cell.get("cell_type") != "code":
        continue

    raw_source = cell.get(
        "source", []
    )

    code = (
        "".join(raw_source)
        if isinstance(raw_source, list)
        else str(raw_source)
    )

    lines = code.splitlines()

    positions = [
        i
        for i, line in enumerate(lines)
        if any(
            re.search(pattern, line)
            for pattern in setup_patterns
        )
    ]

    if not positions:
        continue

    # Prefer original setup cells instead of
    # long downstream processing examples.
    score = sum(
        2 if "InfrastructureGUIAdapter(" in lines[i]
        else 1
        for i in positions
    )

    matches.append(
        (score, cell_index, lines, positions)
    )

matches.sort(
    key=lambda item: (
        -item[0],
        item[1]
    )
)

print(
    "Notebook cells with adapter setup references:",
    len(matches)
)

for score, cell_index, lines, positions in matches[:5]:

    print(
        f"\n--- NOTEBOOK CELL {cell_index} ---"
    )

    selected = set()

    for pos in positions:

        selected.update(
            range(
                max(0, pos - 7),
                min(len(lines), pos + 11)
            )
        )

    # Keep the report small enough to share.
    selected = sorted(selected)[:65]

    for index in selected:

        print(
            f"{index + 1:03d}: "
            f"{lines[index][:190]}"
        )

    if len(selected) == 65:
        print("... excerpt limited to 65 lines")

# --------------------------------------------------
# 4. Show relevant loaded configuration types
# --------------------------------------------------

print("\n4. REUSABLE SETUP VARIABLES")

interesting_names = [
    "adapter",
    "insight_service",
    "combined_service",
    "validated_service",
    "zcta_validator",
    "predictor",
    "engine",
    "payload_builder",
    "kmeans_lookup"
]

for name in interesting_names:

    if name in globals():

        value = globals()[name]

        print(
            name,
            "=>",
            type(value).__module__,
            type(value).__name__
        )

# --------------------------------------------------
# 5. Final status
# --------------------------------------------------

print("\n5. AUDIT RESULT")

print(
    "Working adapter:",
    type(adapter).__name__
)

print(
    "Project service classes discovered:",
    len(discovered_classes)
)

print(
    "Notebook initialization references:",
    len(matches)
)

print("\nSTATUS: ADAPTER INITIALIZATION INSPECTED")
print("NO COMPLAINTS PROCESSED")
print("NO FILES MODIFIED")
print("NO HISTORICAL MODELS MODIFIED")


 ADAPTER INITIALIZATION AUDIT

1. WORKING ADAPTER DEPENDENCY TREE
adapter: src.integration.infrastructure_gui_contract.InfrastructureGUIAdapter
  insight_service: src.integration.zcta_aware_combined_insight.ZCTAAwareCombinedInsightService
    combined_service: src.integration.combined_infrastructure_insight.CombinedInfrastructureInsightService
      validated_service: src.integration.validated_new_complaint.ValidatedNewComplaintService
        insight_service: src.integration.new_complaint_insight.NewComplaintInsightService
          predictor: src.models.dbscan_predictor.DBSCANSpatialPredictor
            model_version: str
            eps_meters: float
            earth_radius_km: 6371.0088
            core_coordinates: numpy.ndarray shape=(40977, 2)
            core_labels: numpy.ndarray shape=(40977,)
            eps_radians: float
            tree: sklearn.neighbors._ball_tree.BallTree
          engine: src.recommendations.hotspot_recommender.HotspotRecommendationEngine
          

In [42]:

from pathlib import Path
from textwrap import dedent
import ast
import json
import sys
import pandas as pd
from threadpoolctl import threadpool_limits

print("LIVE CSV UPLOAD PIPELINE")
print("=" * 72)

root = Path(PROJECT_ROOT).resolve()

if str(root) not in sys.path:
    sys.path.insert(0, str(root))

bridge_path = (
    root / "src/integration"
    / "live_uploaded_batch_pipeline.py"
)

page_path = (
    root / "dashboard/pages"
    / "2_Process_New_Batch.py"
)

verified_path = (
    root / "outputs/tests"
    / "phase_17_4c_backend_verified"
    / "verified_500_backend_results.csv"
)

snapshot_path = (
    root / "outputs/tables/anomaly"
    / "historical_daily_counts_2023_2025_v1.csv.gz"
)

manifest_path = (
    root / "outputs/tables/anomaly"
    / "historical_daily_counts_2023_2025_v1_manifest.json"
)

protected = [
    verified_path,
    snapshot_path,
    manifest_path,
    root / "src/ui/service_factory.py",
    root / "src/integration/bulk_complaint_processor.py",
    root / "src/integration/upload_preflight.py",
    root / "src/integration/anomaly_snapshot_integration.py",
    root / "src/integration/smart_assistant_investigation_briefs.py",
    root / "src/integration/dashboard_intelligence_feed.py",
    root / "dashboard/urban_pulse_dashboard.py",
    root / "dashboard/pages/1_Upload_Complaints.py"
]

for path in protected:
    assert path.is_file(), f"Missing prerequisite: {path}"

signatures_before = {
    str(path): (
        path.stat().st_size,
        path.stat().st_mtime_ns
    )
    for path in protected
}

# --------------------------------------------------
# 1. Build reusable upload-processing bridge
# --------------------------------------------------

bridge_source = dedent(r'''
"""Process uploaded complaints using existing models.

Historical models and source files remain read-only.
Uploaded results remain in memory until exported.
"""

from pathlib import Path

import pandas as pd

from src.integration.upload_preflight import (
    run_upload_preflight
)

from src.integration.bulk_complaint_processor import (
    process_complaint_batch
)

from src.integration.daily_batch_window import (
    select_daily_complaints
)

from src.integration.anomaly_snapshot_integration import (
    load_anomaly_snapshot,
    analyze_batch_with_anomaly_context
)

from src.integration.smart_assistant_investigation_briefs import (
    build_investigation_briefs
)

from src.integration.dashboard_intelligence_feed import (
    build_dashboard_feed
)

from src.ui.service_factory import get_gui_adapter


def load_upload_baseline(project_root):

    root = Path(project_root).resolve()

    return load_anomaly_snapshot(
        root / "outputs/tables/anomaly"
        / "historical_daily_counts_2023_2025_v1.csv.gz",

        root / "outputs/tables/anomaly"
        / "historical_daily_counts_2023_2025_v1_manifest.json"
    )


def build_verified_day_feed(
    processed_records,
    baseline,
    reporting_day,
    synthetic_data=True
):
    """Analyze only geographically verified selected-day rows."""

    if not isinstance(
        processed_records,
        pd.DataFrame
    ):
        raise TypeError(
            "Backend records must be a DataFrame."
        )

    required = {
        "source_row",
        "complaint_id",
        "created_date",
        "borough",
        "incident_zip",
        "complaint_category",
        "latitude",
        "longitude",
        "processing_status",
        "zip_validation_status"
    }

    missing = required - set(
        processed_records.columns
    )

    if missing:
        raise ValueError(
            f"Backend missing columns: {sorted(missing)}"
        )

    if processed_records[
        "source_row"
    ].duplicated().any():
        raise ValueError(
            "Duplicate backend source_row values."
        )

    quality_ok = (
        processed_records[
            "processing_status"
        ].astype(str).eq("processed")
        &
        processed_records[
            "zip_validation_status"
        ].astype(str).eq("zcta_consistent")
    )

    verified = processed_records.loc[
        quality_ok
    ].copy()

    window = select_daily_complaints(
        verified,
        reporting_day
    )

    selected = window[
        "selected_records"
    ].copy()

    details = {
        "backend_records":
            int(len(processed_records)),
        "geographically_verified":
            int(len(verified)),
        "excluded_by_backend_quality":
            int((~quality_ok).sum()),
        "excluded_by_reporting_day":
            int(len(verified) - len(selected)),
        "selected_for_new_models":
            int(len(selected))
    }

    if selected.empty:
        return {
            "feed": None,
            "briefs": None,
            "integrated": None,
            "selection_stats": details,
            "status":
                "no_eligible_reporting_day_records"
        }

    integrated = analyze_batch_with_anomaly_context(
        selected,
        baseline,
        reporting_day
    )

    briefs = build_investigation_briefs(
        integrated,
        synthetic_data=synthetic_data
    )

    feed = build_dashboard_feed(
        integrated,
        briefs
    )

    feed["metadata"]["input_origin"] = (
        "uploaded_csv"
    )

    feed["metadata"]["data_verified_in_field"] = (
        False
    )

    feed["metadata"]["upload_note"] = (
        "Uploaded reports were geographically "
        "screened. No infrastructure fault "
        "has been independently verified."
    )

    if feed["summary"][
        "input_complaints"
    ] != len(selected):
        raise AssertionError(
            "Dashboard record count mismatch."
        )

    if len(
        briefs["record_audit"]
    ) != len(selected):
        raise AssertionError(
            "Investigation audit lost records."
        )

    return {
        "feed": feed,
        "briefs": briefs,
        "integrated": integrated,
        "selection_stats": details,
        "status": "analyzed"
    }


def process_uploaded_csv_bytes(
    csv_bytes,
    reporting_day,
    project_root,
    synthetic_data=True,
    adapter=None,
    baseline=None
):
    """Run validation, backend and three-model analysis.

    The backend audit retains all uploaded rows.
    The dashboard contains only verified rows
    belonging to the selected reporting day.
    """

    if baseline is None:
        baseline = load_upload_baseline(
            project_root
        )

    categories = baseline[
        "supported_categories"
    ]

    preflight = run_upload_preflight(
        csv_bytes,
        categories
    )

    original = preflight[
        "input_records"
    ].copy(deep=True)

    if adapter is None:
        adapter = get_gui_adapter()

    backend = process_complaint_batch(
        original,
        categories,
        adapter,
        use_cache=True
    )

    if (
        not isinstance(backend, dict)
        or "records" not in backend
    ):
        raise ValueError(
            "Unexpected bulk backend response."
        )

    rows = backend["records"]

    if isinstance(rows, pd.DataFrame):
        rows = rows.copy(deep=True)

    elif isinstance(rows, (list, tuple)):
        rows = pd.DataFrame(list(rows))

    else:
        raise TypeError(
            "Unexpected backend records container."
        )

    if len(rows) != len(original):
        raise ValueError(
            "Backend dropped uploaded rows."
        )

    analysis = build_verified_day_feed(
        rows,
        baseline,
        reporting_day,
        synthetic_data=synthetic_data
    )

    return {
        **analysis,
        "uploaded_rows": int(len(original)),
        "backend_records": rows,
        "backend_stats": backend.get(
            "stats", {}
        ),
        "preflight_status_counts":
            preflight["status_counts"]
    }
''').lstrip()

# --------------------------------------------------
# 2. Build Streamlit processing page
# --------------------------------------------------

page_source = dedent(r'''
"""Process uploaded complaints with existing backend."""

from datetime import date
from pathlib import Path
import hashlib
import json
import sys

import pandas as pd
import streamlit as st


ROOT = Path(__file__).resolve().parents[2]

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.integration.live_uploaded_batch_pipeline import (
    process_uploaded_csv_bytes,
    load_upload_baseline
)

from src.ui.service_factory import (
    get_gui_adapter
)

from dashboard.urban_pulse_dashboard import (
    validate_feed,
    filter_records,
    build_filtered_view
)


st.set_page_config(
    page_title="Process New Batch | Urban Pulse",
    page_icon="🏙️",
    layout="wide"
)

st.title("Process New Complaint Batch")

st.caption(
    "Upload CSV → Geographic backend → "
    "K-Means + DBSCAN + Anomaly screening"
)

st.warning(
    "Reporting evidence only. Results do not "
    "confirm infrastructure defects or "
    "determine repair priority."
)


@st.cache_resource(
    show_spinner="Loading historical services..."
)
def existing_adapter():
    return get_gui_adapter()


@st.cache_resource(
    show_spinner="Loading historical daily baseline..."
)
def existing_baseline():
    return load_upload_baseline(ROOT)


uploaded_file = st.file_uploader(
    "Choose complaint CSV",
    type=["csv"],
    key="batch_csv"
)

reporting_day = st.date_input(
    "NYC reporting day",
    value=date.today(),
    key="batch_day"
)

synthetic = st.checkbox(
    "Synthetic / test data",
    value=True,
    help=(
        "Leave selected for demonstrations. "
        "Uploaded real-world reports are not "
        "independently verified."
    )
)

st.caption(
    "Maximum upload: 10 MB and 5,000 rows. "
    "Historical models remain read-only."
)

upload_key = None

if uploaded_file is not None:

    payload = uploaded_file.getvalue()

    upload_key = (
        hashlib.sha256(payload).hexdigest(),
        reporting_day.isoformat(),
        bool(synthetic)
    )

if st.button(
    "Process Uploaded Complaints",
    type="primary",
    disabled=uploaded_file is None
):

    try:

        with st.spinner(
            "Validating, geographically verifying "
            "and analyzing complaints..."
        ):

            result = process_uploaded_csv_bytes(
                payload,
                reporting_day.isoformat(),
                ROOT,
                synthetic_data=synthetic,
                adapter=existing_adapter(),
                baseline=existing_baseline()
            )

        st.session_state[
            "urban_pulse_live_result"
        ] = result

        st.session_state[
            "urban_pulse_live_key"
        ] = upload_key

    except Exception as exc:

        st.session_state.pop(
            "urban_pulse_live_result",
            None
        )

        st.session_state.pop(
            "urban_pulse_live_key",
            None
        )

        st.error(
            f"Batch processing failed: "
            f"{type(exc).__name__}: {exc}"
        )


result = st.session_state.get(
    "urban_pulse_live_result"
)

if (
    result is None
    or st.session_state.get(
        "urban_pulse_live_key"
    ) != upload_key
):

    st.info(
        "Upload a CSV and press "
        "Process Uploaded Complaints."
    )

    st.stop()


st.success(
    "Backend processing completed."
)

st.subheader("Processing audit")

metrics = result["selection_stats"]

columns = st.columns(4)

columns[0].metric(
    "Uploaded",
    result["uploaded_rows"]
)

columns[1].metric(
    "Geographically verified",
    metrics["geographically_verified"]
)

columns[2].metric(
    "Selected reporting day",
    metrics["selected_for_new_models"]
)

columns[3].metric(
    "Excluded",
    (
        metrics["excluded_by_backend_quality"]
        + metrics["excluded_by_reporting_day"]
    )
)

st.caption(
    "All uploaded rows remain available in the "
    "backend audit. Only geographically verified "
    "reports from the selected NYC reporting day "
    "enter the new-batch models."
)

with st.expander(
    "Backend audit — including rejected records"
):

    st.dataframe(
        result["backend_records"],
        use_container_width=True,
        hide_index=True
    )

    st.download_button(
        "Download backend audit CSV",
        result["backend_records"]
        .to_csv(index=False)
        .encode("utf-8"),
        "uploaded_batch_backend_audit.csv",
        "text/csv"
    )


feed = result["feed"]

if feed is None:

    st.info(
        "No geographically verified complaints "
        "matched the selected reporting day. "
        "No model conclusions were generated."
    )

    st.stop()


try:
    validate_feed(feed)

except (ValueError, KeyError, TypeError) as exc:

    st.error(
        f"Dashboard feed validation failed: {exc}"
    )

    st.stop()


st.subheader("Verified-day intelligence")

filters = feed["filters"]

f1, f2, f3 = st.columns(3)

borough = f1.selectbox(
    "Borough",
    ["All"] + filters["boroughs"]
)

category = f2.selectbox(
    "Category",
    ["All"] + filters["categories"]
)

signal = f3.selectbox(
    "Signal",
    [
        "All",
        "Both signals",
        "Anomaly only",
        "Spatial only",
        "Neither"
    ]
)

records = filter_records(
    feed["records"],
    borough=borough,
    category=category,
    signal=signal
)

view = build_filtered_view(
    feed,
    records
)

summary = view["summary"]

a, b, c, d = st.columns(4)

a.metric(
    "Filtered reports",
    summary["selected"]
)

b.metric(
    "Anomaly-associated",
    summary["anomaly_reports"]
)

c.metric(
    "Spatial-associated",
    summary["spatial_reports"]
)

d.metric(
    "Distinct with either",
    summary["distinct_either"]
)

st.caption(
    "Anomaly and spatial totals overlap. "
    "The explanations and area cards refer "
    "to the complete processed reporting-day batch."
)

tabs = st.tabs([
    "Area patterns",
    "Anomaly reviews",
    "Spatial map",
    "Records"
])

with tabs[0]:

    areas = pd.DataFrame(
        feed["area_cards"]
    )

    selected_areas = {
        row["area_id"]
        for row in records
    }

    if not areas.empty:

        st.dataframe(
            areas.loc[
                areas["area_id"].isin(
                    selected_areas
                )
            ],
            hide_index=True,
            use_container_width=True
        )

    else:
        st.info(
            "No area reporting patterns available."
        )


with tabs[1]:

    if not view["anomaly_cards"]:
        st.info(
            "No anomaly candidates match the filters."
        )

    for card in view["anomaly_cards"]:

        with st.expander(
            f"{card['area_id']} — "
            f"{card['complaint_category']}"
        ):

            st.write(
                card["explanation"]
            )

            st.caption(
                card["suggested_review"]
            )


with tabs[2]:

    points = []

    for feature in view["map_features"]:

        longitude, latitude = (
            feature["geometry"]["coordinates"]
        )

        points.append({
            "latitude": latitude,
            "longitude": longitude,
            "candidate_id": feature["id"]
        })

    if points:

        st.map(
            pd.DataFrame(points)[[
                "latitude",
                "longitude"
            ]]
        )

    else:

        st.info(
            "No spatial candidate locations "
            "match the current filters."
        )

    for card in view["spatial_cards"]:

        with st.expander(
            f"{card['candidate_id']} — "
            f"{card['complaint_category']}"
        ):

            st.write(
                card["explanation"]
            )

            st.caption(
                card["suggested_review"]
            )


with tabs[3]:

    if records:

        st.dataframe(
            pd.DataFrame(records),
            hide_index=True,
            use_container_width=True
        )

    else:
        st.info(
            "No complaint records match the filters."
        )


feed_bytes = json.dumps(
    feed,
    indent=2,
    ensure_ascii=False,
    allow_nan=False
).encode("utf-8")

st.download_button(
    "Download dashboard feed JSON",
    feed_bytes,
    "uploaded_batch_dashboard_feed.json",
    "application/json"
)

st.caption(
    "Uploaded input and generated outputs "
    "remain in this Streamlit session unless "
    "explicitly downloaded. Historical models "
    "are not modified."
)
''').lstrip()

ast.parse(bridge_source)
ast.parse(page_source)

print("\n1. PYTHON SOURCE VALIDATION PASSED")

# --------------------------------------------------
# 3. Test bridge in memory before saving
# --------------------------------------------------

namespace = {
    "__name__": "_phase20_2d_validation",
    "__file__": str(bridge_path)
}

exec(
    compile(
        bridge_source,
        str(bridge_path),
        "exec"
    ),
    namespace
)

run_upload = namespace[
    "process_uploaded_csv_bytes"
]

build_verified = namespace[
    "build_verified_day_feed"
]

load_baseline = namespace[
    "load_upload_baseline"
]

from src.integration.bulk_complaint_contract import (
    INPUT_COLUMNS
)

from src.ui.service_factory import (
    get_gui_adapter
)

from dashboard.urban_pulse_dashboard import (
    validate_feed,
    build_filtered_view
)

# Recreate raw CSV from previously verified records.
verified_csv = pd.read_csv(
    verified_path,
    dtype="string",
    keep_default_na=False
)

raw_csv = verified_csv.loc[
    :, list(INPUT_COLUMNS)
].to_csv(
    index=False
).encode("utf-8")

print("\n2. EXECUTING REAL 500-RECORD UPLOAD")

baseline = load_baseline(root)

adapter = get_gui_adapter()

with threadpool_limits(limits=1):

    result = run_upload(
        raw_csv,
        "2026-11-09",
        root,
        synthetic_data=True,
        adapter=adapter,
        baseline=baseline
    )

print("\n3. BACKEND SELECTION SUMMARY")

for name, value in result[
    "selection_stats"
].items():

    print(f"{name}: {value}")

feed = result["feed"]

if feed is not None:

    print("\n4. THREE-MODEL DASHBOARD SUMMARY")

    for name, value in feed[
        "summary"
    ].items():

        print(f"{name}: {value}")

# --------------------------------------------------
# 4. Regression checks
# --------------------------------------------------

checks = []

def check(name, condition):

    checks.append({
        "test": name,
        "passed": bool(condition)
    })

check(
    "Backend bridge compiles",
    len(bridge_source) > 0
)

check(
    "Streamlit page compiles",
    len(page_source) > 0
)

check(
    "500 uploaded records processed",
    result["uploaded_rows"] == 500
)

check(
    "500 backend audit records retained",
    len(
        result["backend_records"]
    ) == 500
)

check(
    "500 geographically verified",
    result["selection_stats"][
        "geographically_verified"
    ] == 500
)

check(
    "500 records selected for reporting day",
    result["selection_stats"][
        "selected_for_new_models"
    ] == 500
)

check(
    "Dashboard feed generated",
    feed is not None
)

if feed is not None:

    validate_feed(feed)

    summary = feed["summary"]

    check(
        "500 dashboard complaint records",
        len(feed["records"]) == 500
    )

    check(
        "20 area cards",
        len(feed["area_cards"]) == 20
    )

    check(
        "Three K-Means reporting patterns",
        summary["new_kmeans_patterns"] == 3
    )

    check(
        "23 DBSCAN spatial candidates",
        summary[
            "spatial_candidate_groups"
        ] == 23
    )

    check(
        "228 spatial candidate complaints",
        summary[
            "spatial_candidate_complaints"
        ] == 228
    )

    check(
        "15 anomaly candidate groups",
        summary[
            "anomaly_candidate_groups"
        ] == 15
    )

    check(
        "204 anomaly-associated complaints",
        summary[
            "anomaly_candidate_complaints"
        ] == 204
    )

    check(
        "156 overlapping complaints",
        summary[
            "complaints_in_both_signals"
        ] == 156
    )

    check(
        "276 distinct complaints with either signal",
        summary[
            "distinct_complaints_in_either_signal"
        ] == 276
    )

    check(
        "23 spatial map features",
        len(
            feed["map_geojson"]["features"]
        ) == 23
    )

    check(
        "Dashboard selection reconciles",
        build_filtered_view(
            feed,
            feed["records"]
        )["summary"]["selected"] == 500
    )

    check(
        "Synthetic upload labeled",
        feed["metadata"][
            "synthetic_data"
        ] is True
    )

    check(
        "Uploaded-data origin documented",
        feed["metadata"][
            "input_origin"
        ] == "uploaded_csv"
    )

# --------------------------------------------------
# 5. Test geographic exclusion without
#    rerunning the full historical backend
# --------------------------------------------------

print("\n5. GEOGRAPHIC EXCLUSION TEST")

rejected = result[
    "backend_records"
].copy(deep=True)

rejected.loc[
    rejected.index[0],
    "zip_validation_status"
] = "review_required"

with threadpool_limits(limits=1):

    partial = build_verified(
        rejected,
        baseline,
        "2026-11-09",
        synthetic_data=True
    )

check(
    "Geographically unverified row excluded",
    partial["selection_stats"][
        "excluded_by_backend_quality"
    ] == 1
)

check(
    "499 verified reports remain",
    partial["selection_stats"][
        "selected_for_new_models"
    ] == 499
)

check(
    "Backend audit retains rejected row",
    len(rejected) == 500
)

# --------------------------------------------------
# 6. Test no matching reporting day
# --------------------------------------------------

print("\n6. NO-MATCHING-DAY SAFEGUARD")

no_day = build_verified(
    result["backend_records"],
    baseline,
    "2026-11-10",
    synthetic_data=True
)

check(
    "No matching day generates no dashboard feed",
    no_day["feed"] is None
)

check(
    "No matching day produces zero model inputs",
    no_day["selection_stats"][
        "selected_for_new_models"
    ] == 0
)

# --------------------------------------------------
# 7. Verify existing files untouched
# --------------------------------------------------

signatures_after = {
    str(path): (
        path.stat().st_size,
        path.stat().st_mtime_ns
    )
    for path in protected
}

check(
    "Existing models, dashboard and data unchanged",
    signatures_before == signatures_after
)

results_20_2d = pd.DataFrame(checks)

print("\n7. VALIDATION")

display(results_20_2d)

passed = int(
    results_20_2d["passed"].sum()
)

total = len(results_20_2d)

print(f"\nChecks passed: {passed}/{total}")

failed = results_20_2d.loc[
    ~results_20_2d["passed"],
    "test"
].tolist()

# --------------------------------------------------
# 8. Save only after successful validation
# --------------------------------------------------

if failed:

    print("\nSTATUS: NEEDS REVIEW")
    print("No new integration files saved.")

    for name in failed:
        print("FAILED:", name)

else:

    destinations = {
        bridge_path: bridge_source,
        page_path: page_source
    }

    for path, content in destinations.items():

        if path.exists():

            assert path.read_text(
                encoding="utf-8"
            ) == content, (
                f"Existing file differs: {path}. "
                "No overwrite performed."
            )

    for path, content in destinations.items():

        path.parent.mkdir(
            parents=True,
            exist_ok=True
        )

        if not path.exists():

            with path.open(
                "x",
                encoding="utf-8"
            ) as file:

                file.write(content)

    print("\n8. VERIFIED INTEGRATION FILES")

    print("Live backend bridge:", bridge_path)
    print("New Streamlit page:", page_path)

    print("\nSTATUS: PASSED")
    print("LIVE UPLOAD PIPELINE VERIFIED")
    print("ORIGINAL SERVICE FACTORY REUSED")
    print("GEOGRAPHIC VALIDATION PRESERVED")
    print("K-MEANS, DBSCAN AND ANOMALY INTEGRATED")
    print("INVALID AND OFF-DAY RECORDS EXCLUDED")
    print("EXISTING DEMONSTRATION DASHBOARD PRESERVED")
    print("NO HISTORICAL MODELS MODIFIED")


LIVE CSV UPLOAD PIPELINE

1. PYTHON SOURCE VALIDATION PASSED

2. EXECUTING REAL 500-RECORD UPLOAD

3. BACKEND SELECTION SUMMARY
backend_records: 500
geographically_verified: 500
excluded_by_backend_quality: 0
excluded_by_reporting_day: 0
selected_for_new_models: 500

4. THREE-MODEL DASHBOARD SUMMARY
input_complaints: 500
selected_day_complaints: 500
anomaly_eligible_complaints: 500
areas_with_eligible_reports: 20
new_kmeans_patterns: 3
spatial_candidate_groups: 23
spatial_candidate_complaints: 228
anomaly_candidate_groups: 15
anomaly_candidate_complaints: 204
complaints_in_both_signals: 156
complaints_anomaly_only: 48
complaints_spatial_only: 72
complaints_in_neither_signal: 224
distinct_complaints_in_either_signal: 276
historical_baseline_areas: 238

5. GEOGRAPHIC EXCLUSION TEST

6. NO-MATCHING-DAY SAFEGUARD

7. VALIDATION


,test,passed
0,Backend bridge compiles,True
1,Streamlit page compiles,True
2,500 uploaded records processed,True
3,500 backend audit records retained,True
4,500 geographically verified,True
5,500 records selected for reporting day,True
6,Dashboard feed generated,True
7,500 dashboard complaint records,True
8,20 area cards,True
9,Three K-Means reporting patterns,True



Checks passed: 26/26

8. VERIFIED INTEGRATION FILES
Live backend bridge: D:\Urban-Infrastructure-Pulse-Intelligence\src\integration\live_uploaded_batch_pipeline.py
New Streamlit page: D:\Urban-Infrastructure-Pulse-Intelligence\dashboard\pages\2_Process_New_Batch.py

STATUS: PASSED
LIVE UPLOAD PIPELINE VERIFIED
ORIGINAL SERVICE FACTORY REUSED
GEOGRAPHIC VALIDATION PRESERVED
K-MEANS, DBSCAN AND ANOMALY INTEGRATED
INVALID AND OFF-DAY RECORDS EXCLUDED
EXISTING DEMONSTRATION DASHBOARD PRESERVED
NO HISTORICAL MODELS MODIFIED


In [43]:

from pathlib import Path
from io import BytesIO
import hashlib
import importlib.util
import pandas as pd

from src.integration.bulk_complaint_contract import (
    INPUT_COLUMNS,
    read_batch_csv
)

print(" STREAMLIT GUI ACCEPTANCE PREPARATION")
print("=" * 72)

root = Path(PROJECT_ROOT).resolve()

verified_path = (
    root / "outputs/tests/phase_17_4c_backend_verified"
    / "verified_500_backend_results.csv"
)

dashboard_path = (
    root / "dashboard/urban_pulse_dashboard.py"
)

processing_page = (
    root / "dashboard/pages/2_Process_New_Batch.py"
)

pipeline_path = (
    root / "src/integration/live_uploaded_batch_pipeline.py"
)

output_dir = (
    root / "outputs/tests/phase_20_2e_gui_acceptance"
)

clean_path = output_dir / "gui_test_clean_500.csv"
mixed_path = output_dir / "gui_test_mixed_500.csv"

protected = [
    verified_path,
    dashboard_path,
    processing_page,
    pipeline_path
]

for path in protected:
    assert path.is_file(), f"Missing: {path}"

original_hashes = {
    str(path): hashlib.sha256(path.read_bytes()).hexdigest()
    for path in protected
}

# 1. Reconstruct the original six-column CSV.
verified = pd.read_csv(
    verified_path,
    dtype="string",
    keep_default_na=False
)

columns = list(INPUT_COLUMNS)

assert len(columns) == 6
assert set(columns).issubset(verified.columns)

clean = verified[columns].copy()

assert len(clean) == 500
assert clean["complaint_id"].nunique() == 500

# 2. Create a controlled invalid/mixed-day test.
mixed = clean.copy(deep=True)

mixed.loc[mixed.index[0], "incident_zip"] = "99999"
mixed.loc[mixed.index[1], "created_date"] = "invalid-date"
mixed.loc[mixed.index[2], "created_date"] = "2026-11-10"

# 3. Serialize without changing original records.
clean_bytes = clean.to_csv(
    index=False
).encode("utf-8")

mixed_bytes = mixed.to_csv(
    index=False
).encode("utf-8")

# 4. Ensure both CSVs can be opened by the
# existing project CSV reader.
clean_read = read_batch_csv(
    BytesIO(clean_bytes)
)

mixed_read = read_batch_csv(
    BytesIO(mixed_bytes)
)

checks = {
    "Clean upload contains 500 rows":
        len(clean_read) == 500,

    "Mixed upload contains 500 rows":
        len(mixed_read) == 500,

    "Exactly six input fields used":
        list(clean_read.columns) == columns,

    "All original complaint IDs retained":
        clean_read["complaint_id"].astype(str).tolist()
        == clean["complaint_id"].astype(str).tolist(),

    "Mixed upload contains invalid ZIP probe":
        (
            mixed_read.iloc[0]["incident_zip"]
            == "99999"
        ),

    "Mixed upload contains invalid date probe":
        (
            mixed_read.iloc[1]["created_date"]
            == "invalid-date"
        ),

    "Mixed upload contains off-day probe":
        (
            mixed_read.iloc[2]["created_date"]
            == "2026-11-10"
        ),

    "Streamlit processing page exists":
        processing_page.is_file(),

    "Existing upload pipeline exists":
        pipeline_path.is_file(),

    "Historical and GUI source files unchanged":
        all(
            hashlib.sha256(path.read_bytes()).hexdigest()
            == original_hashes[str(path)]
            for path in protected
        )
}

results = pd.DataFrame([
    {"test": name, "passed": bool(value)}
    for name, value in checks.items()
])

print("\n1. ACCEPTANCE PREPARATION VALIDATION")
display(results)

passed = int(results["passed"].sum())
total = len(results)

print(f"\nChecks passed: {passed}/{total}")

if passed == total:

    def save_without_overwrite(path, content):
        path.parent.mkdir(
            parents=True,
            exist_ok=True
        )

        if path.exists():
            assert path.read_bytes() == content, (
                f"Existing test file differs: {path}"
            )
        else:
            with path.open("xb") as file:
                file.write(content)

    save_without_overwrite(
        clean_path, clean_bytes
    )

    save_without_overwrite(
        mixed_path, mixed_bytes
    )

    print("\n2. GUI TEST FILES")
    print("Clean 500:", clean_path)
    print("Mixed 500:", mixed_path)

    print(
        "\nStreamlit installed:",
        importlib.util.find_spec("streamlit")
        is not None
    )

    print("\nSTATUS: GUI ACCEPTANCE FILES READY")
    print("NO HISTORICAL DATA OR MODELS MODIFIED")

else:
    print("\nSTATUS: NEEDS REVIEW")
    print("NO TEST FILES SAVED")


 STREAMLIT GUI ACCEPTANCE PREPARATION

1. ACCEPTANCE PREPARATION VALIDATION


,test,passed
0,Clean upload contains 500 rows,True
1,Mixed upload contains 500 rows,True
2,Exactly six input fields used,True
3,All original complaint IDs retained,True
4,Mixed upload contains invalid ZIP probe,True
5,Mixed upload contains invalid date probe,True
6,Mixed upload contains off-day probe,True
7,Streamlit processing page exists,True
8,Existing upload pipeline exists,True
9,Historical and GUI source files unchanged,True



Checks passed: 10/10

2. GUI TEST FILES
Clean 500: D:\Urban-Infrastructure-Pulse-Intelligence\outputs\tests\phase_20_2e_gui_acceptance\gui_test_clean_500.csv
Mixed 500: D:\Urban-Infrastructure-Pulse-Intelligence\outputs\tests\phase_20_2e_gui_acceptance\gui_test_mixed_500.csv

Streamlit installed: True

STATUS: GUI ACCEPTANCE FILES READY
NO HISTORICAL DATA OR MODELS MODIFIED


In [44]:

from pathlib import Path
from textwrap import dedent
import ast
import copy
import hashlib
import json
import sys
import pandas as pd

print(" INVESTIGATION RECOMMENDATIONS")
print("=" * 72)

root = Path(PROJECT_ROOT).resolve()

module_path = (
    root / "src/integration"
    / "investigation_review_queue.py"
)

page_path = (
    root / "dashboard/pages"
    / "3_Investigation_Recommendations.py"
)

feed_path = (
    root / "outputs/tests"
    / "phase_19_2e_dashboard_feed"
    / "dashboard_feed.json"
)

protected = [
    feed_path,
    root / "dashboard/urban_pulse_dashboard.py",
    root / "dashboard/pages/2_Process_New_Batch.py",
    root / "src/integration/live_uploaded_batch_pipeline.py",
    root / "src/integration/dashboard_intelligence_feed.py",
    root / "src/integration/smart_assistant_investigation_briefs.py"
]

for path in protected:
    assert path.is_file(), f"Missing: {path}"

before_hashes = {
    str(path): hashlib.sha256(
        path.read_bytes()
    ).hexdigest()
    for path in protected
}

# --------------------------------------------------
# 1. Build human-readable recommendation logic
# --------------------------------------------------

module_source = dedent(r'''
"""Evidence-led human investigation review queue.

The ranking is a transparent review-order heuristic,
NOT an infrastructure risk or repair-priority score.
"""


def display_location(area_id):
    text = str(area_id)

    if "_" not in text:
        return text

    borough, zip_code = text.rsplit("_", 1)

    return (
        borough.title()
        + " — ZIP "
        + zip_code
    )


def suggested_action(category, spatial=False):

    name = str(category).upper()

    if "TRAFFIC SIGNAL" in name:

        action = (
            "Identify the relevant intersection "
            "from the original complaint details. "
            "Compare reports to determine whether "
            "they describe the same signal issue, "
            "review previous signal maintenance "
            "and service records, and verify "
            "operation at the reported location."
        )

    elif "STREET LIGHT" in name:

        action = (
            "Compare reported locations and any "
            "available pole or fixture identifiers. "
            "Check whether reports describe the "
            "same outage, review recent lighting "
            "maintenance records, and verify "
            "the reported condition."
        )

    elif "SEWER" in name:

        action = (
            "Review the sewer complaint descriptions "
            "for a possible shared blockage, drainage "
            "or infrastructure issue. Compare nearby "
            "locations, examine recent sewer "
            "maintenance and cleaning history, "
            "and verify conditions before field action."
        )

    elif (
        "STREET CONDITION" in name
        or "SIDEWALK" in name
    ):

        action = (
            "Compare report descriptions and "
            "locations to identify repeated damage "
            "or reports about the same road or "
            "sidewalk segment. Review repair and "
            "maintenance history, then consider "
            "a site inspection to verify the issue."
        )

    else:

        action = (
            "Review the original complaints, "
            "compare locations and descriptions "
            "for a possible common cause, "
            "examine recent maintenance records, "
            "and verify the reported conditions."
        )

    if spatial:

        action += (
            " Pay particular attention to the "
            "newly identified spatial grouping; "
            "a concentration does not itself "
            "confirm one shared defect."
        )

    return action


def build_review_queue(feed):
    """Generate ranked, evidence-grounded review cards."""

    required = {
        "metadata",
        "summary",
        "anomaly_cards",
        "spatial_cards",
        "area_cards",
        "map_geojson"
    }

    if not required.issubset(feed):
        raise ValueError(
            "Dashboard feed missing required sections."
        )

    summary = feed["summary"]
    anomalies = feed["anomaly_cards"]
    spatial = feed["spatial_cards"]

    if len(anomalies) != int(
        summary["anomaly_candidate_groups"]
    ):
        raise ValueError(
            "Anomaly group count mismatch."
        )

    if len(spatial) != int(
        summary["spatial_candidate_groups"]
    ):
        raise ValueError(
            "Spatial group count mismatch."
        )

    spatial_ids = [
        str(item["candidate_id"])
        for item in spatial
    ]

    if len(set(spatial_ids)) != len(spatial_ids):
        raise ValueError(
            "Duplicate spatial candidate IDs."
        )

    features = feed["map_geojson"]["features"]

    if set(spatial_ids) != {
        str(item["id"])
        for item in features
    }:
        raise ValueError(
            "Map candidate identities disagree."
        )

    queue = []

    # Area/category historical reporting increases.
    for item in anomalies:

        area_id = str(item["area_id"])

        category = str(
            item["complaint_category"]
        )

        count = int(
            item["incoming_complaints"]
        )

        seasonal = int(
            item["seasonal_historical_max"]
        )

        broad = int(
            item["all_weekday_historical_max"]
        )

        overlap = int(
            item["complaints_in_spatial_candidates"]
        )

        if (
            overlap < 0
            or overlap > count
            or count <= seasonal
            or count <= broad
        ):
            raise ValueError(
                "Invalid anomaly comparison evidence."
            )

        has_spatial = overlap > 0

        if has_spatial:

            why = (
                f"{count} {category.lower()} reports "
                f"were received in {display_location(area_id)}. "
                f"The historical seasonal daily "
                f"maximum was {seasonal}, and the "
                f"all-weekday daily maximum was "
                f"{broad}. {overlap} of the new "
                f"reports also belong to one or "
                f"more spatial concentrations. "
                f"Both reporting-volume and "
                f"spatial screening signals "
                f"support reviewing this area."
            )

        else:

            why = (
                f"{count} {category.lower()} reports "
                f"were received in {display_location(area_id)}, "
                f"compared with historical daily "
                f"maxima of {seasonal} (seasonal) "
                f"and {broad} (all-weekday). "
                f"No new spatial concentration "
                f"is linked to these reports."
            )

        queue.append({
            "kind": "area_category",
            "tier": (
                1 if has_spatial else 2
            ),
            "review_label": (
                "Review first — two screening signals"
                if has_spatial
                else "Review reporting increase"
            ),
            "area_ids": [area_id],
            "location": display_location(
                area_id
            ),
            "category": category,
            "incoming_reports": count,
            "seasonal_max": seasonal,
            "all_weekday_max": broad,
            "historical_excess": (
                count - max(seasonal, broad)
            ),
            "spatial_overlap_reports": overlap,
            "candidate_id": None,
            "linked_spatial_ids": [
                str(value)
                for value in item.get(
                    "linked_spatial_candidate_ids", []
                )
            ],
            "kmeans_patterns": [
                str(value)
                for value in item.get(
                    "kmeans_reporting_patterns", []
                )
            ],
            "why": why,
            "suggested_action":
                suggested_action(
                    category,
                    spatial=has_spatial
                ),
            "evidence_note": (
                "Cross-window consistent historical "
                "screening candidate. Not a "
                "statistically confirmed anomaly."
            )
        })

    # Spatial findings without linked anomaly records.
    for item in spatial:

        count = int(
            item["incoming_complaints"]
        )

        overlap = int(
            item[
                "complaints_in_anomaly_candidate_groups"
            ]
        )

        if overlap < 0 or overlap > count:
            raise ValueError(
                "Invalid spatial overlap count."
            )

        if overlap != 0:
            continue

        candidate_id = str(
            item["candidate_id"]
        )

        category = str(
            item["complaint_category"]
        )

        areas = [
            str(value)
            for value in item["area_ids"]
        ]

        locations = ", ".join(
            display_location(area)
            for area in areas
        )

        distinct = int(
            item["distinct_submitted_locations"]
        )

        historical = item[
            "historical_dbscan_associations"
        ]

        outside = int(
            historical["outside"]
        )

        why = (
            f"{count} {category.lower()} reports "
            f"at {distinct} distinct submitted "
            f"coordinates form spatial candidate "
            f"{candidate_id} in {locations}. "
            f"{outside} reports are outside "
            f"established historical DBSCAN "
            f"associations. No qualifying "
            f"area/category reporting-increase "
            f"candidate is linked to this spatial "
            f"group. The concentration merits "
            f"verification, but is not a "
            f"confirmed infrastructure incident."
        )

        queue.append({
            "kind": "spatial_only",
            "tier": 3,
            "review_label":
                "Review spatial concentration",
            "area_ids": areas,
            "location": locations,
            "category": category,
            "incoming_reports": count,
            "seasonal_max": None,
            "all_weekday_max": None,
            "historical_excess": 0,
            "spatial_overlap_reports": 0,
            "candidate_id": candidate_id,
            "linked_spatial_ids": [
                candidate_id
            ],
            "kmeans_patterns": [
                str(value)
                for value in item.get(
                    "kmeans_reporting_patterns", []
                )
            ],
            "why": why,
            "suggested_action":
                suggested_action(
                    category,
                    spatial=True
                ),
            "evidence_note": (
                "Spatial screening only. "
                "This is not a historical-count "
                "anomaly conclusion."
            )
        })

    # Ranking is for human review, not field urgency.
    queue.sort(
        key=lambda item: (
            item["tier"],
            -item["historical_excess"],
            -item["spatial_overlap_reports"],
            -item["incoming_reports"],
            item["location"],
            item["category"]
        )
    )

    for index, item in enumerate(
        queue,
        start=1
    ):
        item["review_order"] = index

    first = sum(
        item["tier"] == 1
        for item in queue
    )

    historical_only = sum(
        item["tier"] == 2
        for item in queue
    )

    spatial_only = sum(
        item["tier"] == 3
        for item in queue
    )

    # Readable area summaries; never show raw
    # category_breakdown dictionaries as objects.
    area_table = []

    for area in feed["area_cards"]:

        area_table.append({
            "area_id": str(
                area["area_id"]
            ),
            "location": display_location(
                area["area_id"]
            ),
            "reports": int(
                area["incoming_complaints"]
            ),
            "reporting_pattern": ", ".join(
                map(
                    str,
                    area.get(
                        "reporting_patterns", []
                    )
                )
            ),
            "dominant_category": str(
                area["dominant_category"]
            ),
            "anomaly_associated": int(
                area["anomaly_candidate_complaints"]
            ),
            "spatial_associated": int(
                area["spatial_candidate_complaints"]
            )
        })

    return {
        "metadata": {
            "reporting_day": feed[
                "metadata"
            ]["reporting_day"],
            "synthetic_data": bool(
                feed["metadata"].get(
                    "synthetic_data", False
                )
            ),
            "review_policy": (
                "Human screening order only. "
                "Not a risk score, verified defect, "
                "statistical significance claim "
                "or repair dispatch instruction."
            )
        },
        "summary": {
            "reports": int(
                summary["input_complaints"]
            ),
            "distinct_with_signals": int(
                summary[
                    "distinct_complaints_in_either_signal"
                ]
            ),
            "review_first_groups": first,
            "historical_only_groups":
                historical_only,
            "spatial_only_groups": spatial_only,
            "review_items": len(queue),
            "all_spatial_candidates":
                len(spatial)
        },
        "queue": queue,
        "area_table": area_table,
        "spatial_cards": spatial,
        "map_geojson": feed["map_geojson"]
    }
''').lstrip()

# --------------------------------------------------
# 2. Create simplified recommendations GUI page
# --------------------------------------------------

page_source = dedent(r'''
"""Human-readable investigation recommendations page."""

from pathlib import Path
import json
import sys

import pandas as pd
import streamlit as st


ROOT = Path(__file__).resolve().parents[2]

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.integration.investigation_review_queue import (
    build_review_queue
)


DEMO_FEED = (
    ROOT
    / "outputs/tests/phase_19_2e_dashboard_feed"
    / "dashboard_feed.json"
)


st.set_page_config(
    page_title="Investigation Recommendations",
    page_icon="🔎",
    layout="wide"
)

st.title("🔎 Where Should We Investigate First?")

st.caption(
    "Urban Infrastructure Pulse — "
    "explainable investigation recommendations"
)

live = st.session_state.get(
    "urban_pulse_live_result"
)

if (
    isinstance(live, dict)
    and isinstance(live.get("feed"), dict)
):

    feed = live["feed"]

    st.info(
        "Showing your most recently processed "
        "uploaded batch from this browser session."
    )

else:

    if not DEMO_FEED.is_file():

        st.error(
            "No completed upload or demonstration "
            "feed is available."
        )

        st.stop()

    with DEMO_FEED.open(
        encoding="utf-8"
    ) as file:

        feed = json.load(file)

    st.info(
        "Showing the saved 500-complaint "
        "synthetic demonstration. Process a "
        "new batch to view uploaded results."
    )


try:

    result = build_review_queue(
        feed
    )

except (ValueError, KeyError, TypeError) as exc:

    st.error(
        f"Could not build recommendations: {exc}"
    )

    st.stop()


if result["metadata"]["synthetic_data"]:

    st.warning(
        "Synthetic demonstration data. "
        "These are test recommendations, "
        "not confirmed NYC incidents."
    )

else:

    st.warning(
        "Uploaded complaint data is unverified "
        "in the field. Screening results are "
        "not confirmed faults or repair priorities."
    )


st.markdown(
    "**What this page answers:** "
    "Which places have noteworthy reporting "
    "evidence, why should they be reviewed, "
    "and what should investigators check?"
)

st.caption(
    "Review order is not a measure of danger. "
    "It considers historical reporting increases "
    "and new spatial concentrations; emergency "
    "severity requires independent assessment."
)

summary = result["summary"]

m1, m2, m3, m4 = st.columns(4)

m1.metric(
    "Analyzed reports",
    summary["reports"]
)

m2.metric(
    "Review first groups",
    summary["review_first_groups"]
)

m3.metric(
    "Spatial-only reviews",
    summary["spatial_only_groups"]
)

m4.metric(
    "Distinct reports flagged",
    summary["distinct_with_signals"]
)


st.divider()

# Filters only change displayed cards;
# rankings remain based on the complete batch.

queue = result["queue"]

boroughs = sorted({
    str(area).rsplit("_", 1)[0]
    for item in queue
    for area in item["area_ids"]
})

categories = sorted({
    item["category"]
    for item in queue
})

f1, f2 = st.columns(2)

borough = f1.selectbox(
    "Filter borough",
    ["All"] + boroughs,
    key="recommendation_borough"
)

category = f2.selectbox(
    "Filter complaint category",
    ["All"] + categories,
    key="recommendation_category"
)


def is_visible(item):

    if (
        category != "All"
        and item["category"] != category
    ):
        return False

    if borough != "All":

        if not any(
            area.startswith(
                borough + "_"
            )
            for area in item["area_ids"]
        ):
            return False

    return True


visible = [
    item
    for item in queue
    if is_visible(item)
]


def show_recommendation(item):

    with st.container(border=True):

        st.markdown(
            f"### {item['review_order']}. "
            f"{item['location']}"
        )

        st.markdown(
            f"**Complaint type:** {item['category']}"
        )

        st.caption(
            item["review_label"]
        )

        if item["kmeans_patterns"]:

            st.write(
                "**Area reporting pattern:**",
                ", ".join(
                    item["kmeans_patterns"]
                )
            )

        st.markdown("**Why investigate here?**")

        st.write(
            item["why"]
        )

        st.markdown("**Suggested action**")

        st.write(
            item["suggested_action"]
        )

        st.caption(
            "Evidence limitation: "
            + item["evidence_note"]
        )


tabs = st.tabs([
    "Investigate first",
    "Other review findings",
    "Spatial map",
    "Area overview"
])


with tabs[0]:

    st.subheader(
        "Places to review first"
    )

    st.write(
        "These areas show both an elevated "
        "historical reporting-count signal "
        "and new spatial concentration evidence."
    )

    first_items = [
        item for item in visible
        if item["tier"] == 1
    ]

    if not first_items:

        st.info(
            "No two-signal reviews match "
            "the selected filters."
        )

    else:

        for item in first_items[:3]:

            show_recommendation(item)

        remaining = first_items[3:]

        if remaining:

            with st.expander(
                f"View {len(remaining)} "
                "more investigation recommendations"
            ):

                for item in remaining:

                    show_recommendation(item)


with tabs[1]:

    st.subheader(
        "Other findings worth reviewing"
    )

    st.write(
        "These findings are based on a "
        "historical reporting increase alone "
        "or a spatial concentration without "
        "a linked historical-count anomaly."
    )

    other_items = [
        item for item in visible
        if item["tier"] != 1
    ]

    if not other_items:

        st.info(
            "No additional findings match "
            "the selected filters."
        )

    for item in other_items:

        show_recommendation(item)


with tabs[2]:

    st.subheader(
        "New spatial concentration locations"
    )

    st.caption(
        "Map pins are representative submitted "
        "complaint coordinates, not verified "
        "fault sites or intersections."
    )

    matching_spatial_ids = {
        str(item["candidate_id"])
        for item in result["spatial_cards"]

        if (
            category == "All"
            or item["complaint_category"]
                == category
        )

        and (
            borough == "All"
            or any(
                str(area).startswith(
                    borough + "_"
                )
                for area in item["area_ids"]
            )
        )
    }

    points = []

    for feature in result[
        "map_geojson"
    ]["features"]:

        if str(feature["id"]) not in (
            matching_spatial_ids
        ):
            continue

        lon, lat = (
            feature["geometry"][
                "coordinates"
            ]
        )

        points.append({
            "latitude": lat,
            "longitude": lon,
            "candidate_id": feature["id"]
        })

    if points:

        st.map(
            pd.DataFrame(points)[[
                "latitude",
                "longitude"
            ]]
        )

        st.dataframe(
            pd.DataFrame(points),
            hide_index=True,
            use_container_width=True
        )

    else:

        st.info(
            "No candidate map locations "
            "match these filters."
        )


with tabs[3]:

    st.subheader(
        "Area reporting patterns"
    )

    st.write(
        "The following is a simplified "
        "K-Means and model-evidence overview."
    )

    frame = pd.DataFrame(
        result["area_table"]
    )

    if borough != "All":

        frame = frame.loc[
            frame["area_id"].str.startswith(
                borough + "_"
            )
        ]

    if category != "All":

        frame = frame.loc[
            frame["dominant_category"]
            == category
        ]

    st.dataframe(
        frame,
        hide_index=True,
        use_container_width=True
    )


st.divider()

st.subheader("Export investigation recommendations")

export_rows = [
    {
        "review_order": item["review_order"],
        "review_label": item["review_label"],
        "location": item["location"],
        "complaint_category": item["category"],
        "incoming_reports": item["incoming_reports"],
        "why_investigate": item["why"],
        "suggested_action": item["suggested_action"],
        "evidence_note": item["evidence_note"]
    }
    for item in visible
]

if export_rows:

    export_csv = pd.DataFrame(
        export_rows
    ).to_csv(
        index=False
    ).encode("utf-8")

    st.download_button(
        "Download Investigation Recommendations CSV",
        data=export_csv,
        file_name=(
            "urban_pulse_investigation_"
            "recommendations.csv"
        ),
        mime="text/csv"
    )


st.caption(
    "Suggestions are for human verification. "
    "Do not dispatch repairs or claim confirmed "
    "faults based solely on these screens."
)
''').lstrip()

# --------------------------------------------------
# 3. Compile both modules without saving
# --------------------------------------------------

compile(
    module_source,
    str(module_path),
    "exec"
)

ast.parse(page_source)

namespace = {
    "__name__": "_review_queue_validation",
    "__file__": str(module_path)
}

exec(
    compile(
        module_source,
        str(module_path),
        "exec"
    ),
    namespace
)

build_queue = namespace[
    "build_review_queue"
]

# --------------------------------------------------
# 4. Test against saved validated dashboard feed
# --------------------------------------------------

print("\n1. LOADING VERIFIED 500-COMPLAINT FEED")

with feed_path.open(
    encoding="utf-8"
) as file:

    feed = json.load(file)

source_snapshot = copy.deepcopy(feed)

queue_result = build_queue(feed)

print("\n2. RECOMMENDATION SUMMARY")

for key, value in queue_result[
    "summary"
].items():

    print(f"{key}: {value}")

# --------------------------------------------------
# 5. Show top recommendations
# --------------------------------------------------

print("\n3. INVESTIGATE FIRST — PREVIEW")

preview = pd.DataFrame([
    {
        "review_order": item["review_order"],
        "location": item["location"],
        "category": item["category"],
        "incoming_reports": item[
            "incoming_reports"
        ],
        "historical_excess": item[
            "historical_excess"
        ],
        "spatial_overlap": item[
            "spatial_overlap_reports"
        ],
        "review_label": item["review_label"]
    }

    for item in queue_result["queue"]
])

display(
    preview.head(15)
)

print("\n4. EXAMPLE INVESTIGATION RECOMMENDATION")

first = queue_result["queue"][0]

print("PLACE:", first["location"])
print("CATEGORY:", first["category"])

print("\nWHY INVESTIGATE?")
print(first["why"])

print("\nSUGGESTED ACTION")
print(first["suggested_action"])

# --------------------------------------------------
# 6. Verify key evidence and safeguards
# --------------------------------------------------

manhattan = next(
    item
    for item in queue_result["queue"]
    if (
        "MANHATTAN_10003" in item["area_ids"]
        and item["category"] == "SEWER"
    )
)

queens = next(
    item
    for item in queue_result["queue"]
    if item["candidate_id"] == "NB-00-004"
)

tampered = copy.deepcopy(feed)

tampered["summary"][
    "anomaly_candidate_groups"
] = 999

tamper_rejected = False

try:

    build_queue(tampered)

except ValueError:

    tamper_rejected = True

checks = []

def check(name, condition):

    checks.append({
        "test": name,
        "passed": bool(condition)
    })


check(
    "Review queue module compiles",
    bool(module_source)
)

check(
    "Streamlit recommendations page compiles",
    bool(page_source)
)

check(
    "500 reports retained in summary",
    queue_result["summary"]["reports"] == 500
)

check(
    "276 unique complaints flagged",
    queue_result["summary"][
        "distinct_with_signals"
    ] == 276
)

check(
    "15 two-signal reviews prepared",
    queue_result["summary"][
        "review_first_groups"
    ] == 15
)

check(
    "Eight spatial-only reviews prepared",
    queue_result["summary"][
        "spatial_only_groups"
    ] == 8
)

check(
    "23 investigation items created",
    queue_result["summary"][
        "review_items"
    ] == 23
)

check(
    "All 23 spatial candidates preserved",
    queue_result["summary"][
        "all_spatial_candidates"
    ] == 23
)

check(
    "Review order numbers unique and sequential",
    [
        item["review_order"]
        for item in queue_result["queue"]
    ] == list(
        range(
            1,
            len(queue_result["queue"]) + 1
        )
    )
)

check(
    "Manhattan sewer evidence correct",
    manhattan["incoming_reports"] == 15
    and manhattan["seasonal_max"] == 2
    and manhattan["all_weekday_max"] == 2
    and manhattan["spatial_overlap_reports"] == 15
)

check(
    "Queens spatial-only candidate retained",
    queens["tier"] == 3
    and queens["incoming_reports"] == 15
)

check(
    "Every recommendation explains why",
    all(
        item["why"].strip()
        for item in queue_result["queue"]
    )
)

check(
    "Every recommendation suggests an action",
    all(
        item["suggested_action"].strip()
        for item in queue_result["queue"]
    )
)

check(
    "No invented intersection-specific location",
    all(
        item["location"] == namespace[
            "display_location"
        ](item["area_ids"][0])
        for item in queue_result["queue"]
        if len(item["area_ids"]) == 1
    )
)

check(
    "Readable area table contains 20 areas",
    len(
        queue_result["area_table"]
    ) == 20
)

check(
    "Inconsistent upstream counts rejected",
    tamper_rejected
)

check(
    "Source dashboard data unchanged",
    feed == source_snapshot
)

check(
    "Original application and data files unchanged",
    all(
        hashlib.sha256(
            path.read_bytes()
        ).hexdigest()
        == before_hashes[str(path)]
        for path in protected
    )
)

results = pd.DataFrame(checks)

print("\n5. VALIDATION")

display(results)

passed = int(results["passed"].sum())
total = len(results)

print(f"\nChecks passed: {passed}/{total}")

failed = results.loc[
    ~results["passed"],
    "test"
].tolist()

# --------------------------------------------------
# 7. Save only if all checks pass
# --------------------------------------------------

if failed:

    print("\nSTATUS: NEEDS REVIEW")
    print("No new files saved.")

    for name in failed:

        print("FAILED:", name)

else:

    destinations = {
        module_path: module_source,
        page_path: page_source
    }

    for path, content in destinations.items():

        if path.exists():

            assert path.read_text(
                encoding="utf-8"
            ) == content, (
                f"Different file already exists: {path}. "
                "No overwrite performed."
            )

    for path, content in destinations.items():

        path.parent.mkdir(
            parents=True,
            exist_ok=True
        )

        if not path.exists():

            with path.open(
                "x",
                encoding="utf-8"
            ) as file:

                file.write(content)

    print("\n6. VERIFIED FILES")

    print("Review queue module:", module_path)
    print("New Streamlit page:", page_path)

    print("\nSTATUS: PASSED")
    print("INVESTIGATION RECOMMENDATIONS READY")
    print("EVIDENCE-LED HUMAN REVIEW ORDER CREATED")
    print("WHY-INVESTIGATE EXPLANATIONS CREATED")
    print("CATEGORY-SPECIFIC ACTIONS CREATED")
    print("SPATIAL-ONLY REVIEWS PRESERVED")
    print("NO HISTORICAL MODELS MODIFIED")


 INVESTIGATION RECOMMENDATIONS

1. LOADING VERIFIED 500-COMPLAINT FEED

2. RECOMMENDATION SUMMARY
reports: 500
distinct_with_signals: 276
review_first_groups: 15
historical_only_groups: 0
spatial_only_groups: 8
review_items: 23
all_spatial_candidates: 23

3. INVESTIGATE FIRST — PREVIEW


,review_order,location,category,incoming_reports,historical_excess,spatial_overlap,review_label
0,1,Manhattan — ZIP 10003,SEWER,15,13,15,Review first — two screening signals
1,2,Staten Island — ZIP 10303,SEWER,15,12,5,Review first — two screening signals
2,3,Bronx — ZIP 10453,SEWER,15,8,15,Review first — two screening signals
3,4,Manhattan — ZIP 10001,STREET CONDITION,14,8,13,Review first — two screening signals
4,5,Staten Island — ZIP 10302,STREET CONDITION,13,8,11,Review first — two screening signals
5,6,Queens — ZIP 11004,STREET CONDITION,13,8,10,Review first — two screening signals
6,7,Manhattan — ZIP 10002,STREET CONDITION,13,7,13,Review first — two screening signals
7,8,Bronx — ZIP 10454,STREET LIGHT CONDITION,12,7,7,Review first — two screening signals
8,9,Brooklyn — ZIP 11204,SEWER,15,6,11,Review first — two screening signals
9,10,Bronx — ZIP 10451,STREET CONDITION,14,5,8,Review first — two screening signals



4. EXAMPLE INVESTIGATION RECOMMENDATION
PLACE: Manhattan — ZIP 10003
CATEGORY: SEWER

WHY INVESTIGATE?
15 sewer reports were received in Manhattan — ZIP 10003. The historical seasonal daily maximum was 2, and the all-weekday daily maximum was 2. 15 of the new reports also belong to one or more spatial concentrations. Both reporting-volume and spatial screening signals support reviewing this area.

SUGGESTED ACTION
Review the sewer complaint descriptions for a possible shared blockage, drainage or infrastructure issue. Compare nearby locations, examine recent sewer maintenance and cleaning history, and verify conditions before field action. Pay particular attention to the newly identified spatial grouping; a concentration does not itself confirm one shared defect.

5. VALIDATION


,test,passed
0,Review queue module compiles,True
1,Streamlit recommendations page compiles,True
2,500 reports retained in summary,True
3,276 unique complaints flagged,True
4,15 two-signal reviews prepared,True
5,Eight spatial-only reviews prepared,True
6,23 investigation items created,True
7,All 23 spatial candidates preserved,True
8,Review order numbers unique and sequential,True
9,Manhattan sewer evidence correct,True



Checks passed: 18/18

6. VERIFIED FILES
Review queue module: D:\Urban-Infrastructure-Pulse-Intelligence\src\integration\investigation_review_queue.py
New Streamlit page: D:\Urban-Infrastructure-Pulse-Intelligence\dashboard\pages\3_Investigation_Recommendations.py

STATUS: PASSED
INVESTIGATION RECOMMENDATIONS READY
EVIDENCE-LED HUMAN REVIEW ORDER CREATED
WHY-INVESTIGATE EXPLANATIONS CREATED
CATEGORY-SPECIFIC ACTIONS CREATED
SPATIAL-ONLY REVIEWS PRESERVED
NO HISTORICAL MODELS MODIFIED


In [2]:

from pathlib import Path
from textwrap import dedent
import ast
import hashlib
import json
import sys
import pandas as pd
import streamlit as st

print(" RECOMMENDATIONS-FIRST DASHBOARD")
print("=" * 70)

# Project root configuration


PROJECT_ROOT = Path(
    r"D:\Urban-Infrastructure-Pulse-Intelligence"
)

assert PROJECT_ROOT.is_dir(), (
    f"Project directory not found: {PROJECT_ROOT}"
)

root = PROJECT_ROOT.resolve()

print("Project root:", root)

if str(root) not in sys.path:
    sys.path.insert(0, str(root))

launcher_path = (
    root / "dashboard/urban_pulse_app.py"
)

explorer_path = (
    root / "dashboard/pages/4_Model_Explorer.py"
)

original_app = (
    root / "dashboard/urban_pulse_dashboard.py"
)

recommendations_page = (
    root / "dashboard/pages"
    / "3_Investigation_Recommendations.py"
)

upload_page = (
    root / "dashboard/pages/1_Upload_Complaints.py"
)

process_page = (
    root / "dashboard/pages/2_Process_New_Batch.py"
)

review_module = (
    root / "src/integration"
    / "investigation_review_queue.py"
)

demo_feed = (
    root / "outputs/tests"
    / "phase_19_2e_dashboard_feed"
    / "dashboard_feed.json"
)

protected = [
    original_app,
    recommendations_page,
    upload_page,
    process_page,
    review_module,
    demo_feed
]

for path in protected:
    assert path.is_file(), f"Missing: {path}"

hashes_before = {
    str(p): hashlib.sha256(
        p.read_bytes()
    ).hexdigest()
    for p in protected
}

# --------------------------------------------------
# 1. New main application entry point
# --------------------------------------------------

launcher_source = dedent(r'''
"""Urban Infrastructure Pulse — primary application.

Launch:
python -m streamlit run dashboard/urban_pulse_app.py

Investigation recommendations are the default page.
"""

from pathlib import Path

import streamlit as st


HERE = Path(__file__).resolve().parent

st.set_page_config(
    page_title="Urban Infrastructure Pulse",
    page_icon="🏙️",
    layout="wide",
    initial_sidebar_state="expanded"
)

pages = {
    "Investigation Intelligence": [
        st.Page(
            str(
                HERE / "pages"
                / "3_Investigation_Recommendations.py"
            ),
            title="Investigation Recommendations",
            icon="🔎",
            default=True
        ),
        st.Page(
            str(
                HERE / "pages"
                / "4_Model_Explorer.py"
            ),
            title="Model Explorer",
            icon="📊"
        )
    ],

    "Complaint Processing": [
        st.Page(
            str(
                HERE / "pages"
                / "1_Upload_Complaints.py"
            ),
            title="Validate CSV Upload",
            icon="📄"
        ),
        st.Page(
            str(
                HERE / "pages"
                / "2_Process_New_Batch.py"
            ),
            title="Process New Batch",
            icon="⚙️"
        )
    ]
}

selected_page = st.navigation(
    pages,
    position="sidebar"
)

selected_page.run()
''').lstrip()

# --------------------------------------------------
# 2. Preserve original dashboard as Model Explorer
# --------------------------------------------------

explorer_source = dedent(r'''
"""Access the original model-exploration dashboard."""

from pathlib import Path
import sys


ROOT = Path(__file__).resolve().parents[2]

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from dashboard.urban_pulse_dashboard import main


main()
''').lstrip()

ast.parse(launcher_source)
ast.parse(explorer_source)

print("\n1. SOURCE VALIDATION PASSED")
print("Streamlit version:", st.__version__)

# --------------------------------------------------
# 3. Verify existing recommendation data
# --------------------------------------------------

from src.integration.investigation_review_queue import (
    build_review_queue
)

with demo_feed.open(
    encoding="utf-8"
) as file:
    feed = json.load(file)

review = build_review_queue(feed)

print("\n2. CURRENT DEMONSTRATION SUMMARY")

for key, value in review["summary"].items():
    print(f"{key}: {value}")

# --------------------------------------------------
# 4. Validate new navigation design
# --------------------------------------------------

checks = []

def check(name, condition):
    checks.append({
        "test": name,
        "passed": bool(condition)
    })

check(
    "New launcher compiles",
    bool(compile(
        launcher_source,
        str(launcher_path),
        "exec"
    ))
)

check(
    "Model Explorer wrapper compiles",
    bool(compile(
        explorer_source,
        str(explorer_path),
        "exec"
    ))
)

check(
    "Streamlit supports modern navigation",
    hasattr(st, "navigation")
    and hasattr(st, "Page")
)

check(
    "Recommendations are the default page",
    "default=True" in launcher_source
    and "3_Investigation_Recommendations.py"
    in launcher_source
)

check(
    "Original dashboard remains accessible",
    "4_Model_Explorer.py" in launcher_source
    and "from dashboard.urban_pulse_dashboard import main"
    in explorer_source
)

check(
    "CSV validation page retained",
    "1_Upload_Complaints.py"
    in launcher_source
)

check(
    "Live processing page retained",
    "2_Process_New_Batch.py"
    in launcher_source
)

check(
    "500 synthetic demo complaints available",
    review["summary"]["reports"] == 500
)

check(
    "15 review-first groups preserved",
    review["summary"][
        "review_first_groups"
    ] == 15
)

check(
    "Eight spatial-only reviews preserved",
    review["summary"][
        "spatial_only_groups"
    ] == 8
)

check(
    "23 investigation items preserved",
    review["summary"]["review_items"] == 23
)

check(
    "Every review has suggested action",
    all(
        item["suggested_action"].strip()
        for item in review["queue"]
    )
)

check(
    "All original source files unchanged",
    all(
        hashlib.sha256(
            p.read_bytes()
        ).hexdigest() == hashes_before[str(p)]
        for p in protected
    )
)

results = pd.DataFrame(checks)

print("\n3. VALIDATION CHECKLIST")
display(results)

passed = int(results["passed"].sum())
total = len(results)

print(f"\nChecks passed: {passed}/{total}")

failed = results.loc[
    ~results["passed"],
    "test"
].tolist()

# --------------------------------------------------
# 5. Save only after checks pass
# --------------------------------------------------

if failed:

    print("\nSTATUS: NEEDS REVIEW")
    print("No new application files saved.")

    for name in failed:
        print("FAILED:", name)

else:

    destinations = {
        launcher_path: launcher_source,
        explorer_path: explorer_source
    }

    # Refuse to overwrite different existing files.
    for path, content in destinations.items():

        if path.exists():
            assert path.read_text(
                encoding="utf-8"
            ) == content, (
                f"Existing file differs: {path}"
            )

    for path, content in destinations.items():

        path.parent.mkdir(
            parents=True,
            exist_ok=True
        )

        if not path.exists():
            with path.open(
                "x",
                encoding="utf-8"
            ) as file:
                file.write(content)

    print("\n4. NEW APPLICATION FILES")

    print("Main launcher:", launcher_path)
    print("Model Explorer:", explorer_path)

    print("\nSTATUS: PASSED")
    print("RECOMMENDATIONS SET AS DEFAULT PAGE")
    print("ORIGINAL MODEL EXPLORER PRESERVED")
    print("CSV VALIDATION AND PROCESSING RETAINED")
    print("NO HISTORICAL MODELS MODIFIED")

    print("\nNEW LAUNCH COMMAND:")
    print(
        "python -m streamlit run "
        "dashboard/urban_pulse_app.py"
    )


 RECOMMENDATIONS-FIRST DASHBOARD
Project root: D:\Urban-Infrastructure-Pulse-Intelligence

1. SOURCE VALIDATION PASSED
Streamlit version: 1.63.0

2. CURRENT DEMONSTRATION SUMMARY
reports: 500
distinct_with_signals: 276
review_first_groups: 15
historical_only_groups: 0
spatial_only_groups: 8
review_items: 23
all_spatial_candidates: 23

3. VALIDATION CHECKLIST


,test,passed
0,New launcher compiles,True
1,Model Explorer wrapper compiles,True
2,Streamlit supports modern navigation,True
3,Recommendations are the default page,True
4,Original dashboard remains accessible,True
5,CSV validation page retained,True
6,Live processing page retained,True
7,500 synthetic demo complaints available,True
8,15 review-first groups preserved,True
9,Eight spatial-only reviews preserved,True



Checks passed: 13/13

4. NEW APPLICATION FILES
Main launcher: D:\Urban-Infrastructure-Pulse-Intelligence\dashboard\urban_pulse_app.py
Model Explorer: D:\Urban-Infrastructure-Pulse-Intelligence\dashboard\pages\4_Model_Explorer.py

STATUS: PASSED
RECOMMENDATIONS SET AS DEFAULT PAGE
ORIGINAL MODEL EXPLORER PRESERVED
CSV VALIDATION AND PROCESSING RETAINED
NO HISTORICAL MODELS MODIFIED

NEW LAUNCH COMMAND:
python -m streamlit run dashboard/urban_pulse_app.py


In [3]:

from pathlib import Path
import hashlib
import numpy as np
import pandas as pd

print(" HISTORICAL VS CURRENT CATEGORY MIX")
print("=" * 72)

# ==================================================
# 0. PROJECT ROOT — FRESH KERNEL SAFE
# ==================================================

PROJECT_ROOT = Path(
    r"D:\Urban-Infrastructure-Pulse-Intelligence"
)

assert PROJECT_ROOT.is_dir(), (
    f"Project directory not found: {PROJECT_ROOT}"
)

root = PROJECT_ROOT.resolve()

print("\nPROJECT ROOT:")
print(root)

# ==================================================
# 1. INPUT AND OUTPUT PATHS
# ==================================================

history_path = (
    root
    / "data"
    / "processed"
    / "selected_modeling_features.csv"
)

current_path = (
    root
    / "outputs"
    / "tests"
    / "phase_17_4c_backend_verified"
    / "verified_500_backend_results.csv"
)

output_dir = (
    root
    / "outputs"
    / "tables"
    / "historical_current_category_comparison"
)

summary_path = (
    output_dir
    / "area_category_change_summary_v1.csv"
)

detail_path = (
    output_dir
    / "area_category_change_details_v1.csv"
)

for path in [history_path, current_path]:
    assert path.is_file(), (
        f"Required input file missing: {path}"
    )

source_hashes = {
    path: hashlib.sha256(
        path.read_bytes()
    ).hexdigest()
    for path in [history_path, current_path]
}

print("\nINPUT FILES VERIFIED")

print("Historical:", history_path)
print("Current:", current_path)

# ==================================================
# 2. LOAD HISTORICAL K-MEANS FEATURES
# ==================================================

print("\n1. LOADING HISTORICAL FEATURES")

historical = pd.read_csv(
    history_path
)

required_history = {
    "area_id",
    "total_complaints"
}

assert required_history.issubset(
    historical.columns
), (
    "Historical input is missing required "
    "area identification or count columns."
)

share_cols = sorted([
    column
    for column in historical.columns
    if column.startswith("share_")
])

assert len(historical) == 238, (
    "Expected 238 historical geographic areas."
)

assert historical["area_id"].is_unique

assert len(share_cols) == 14, (
    "Expected 14 historical category-share "
    f"features, found {len(share_cols)}."
)

historical[share_cols] = (
    historical[share_cols].apply(
        pd.to_numeric,
        errors="raise"
    )
)

historical_values = historical[
    share_cols
].to_numpy(dtype=float)

assert np.isfinite(
    historical_values
).all(), (
    "Historical category shares contain "
    "invalid numeric values."
)

assert (
    (historical_values >= 0)
    & (historical_values <= 1)
).all(), (
    "Historical shares must be "
    "between 0 and 1."
)

historical_share_sums = (
    historical[share_cols]
    .sum(axis=1)
)

assert np.allclose(
    historical_share_sums,
    1.0,
    atol=0.01
), (
    "Historical category shares do not sum "
    "to approximately 100%. Check feature "
    "definitions before comparing."
)

print("Historical areas:", len(historical))

print(
    "Historical category-share features:",
    len(share_cols)
)

print("\nAvailable historical share columns:")

for column in share_cols:
    print(" -", column)

# ==================================================
# 3. LOAD CURRENT VERIFIED COMPLAINTS
# ==================================================

print("\n2. LOADING CURRENT COMPLAINTS")

incoming = pd.read_csv(
    current_path,
    dtype="string",
    keep_default_na=False
)

required_current = {
    "complaint_id",
    "created_date",
    "borough",
    "incident_zip",
    "complaint_category",
    "processing_status",
    "zip_validation_status"
}

assert required_current.issubset(
    incoming.columns
), (
    "Current input is missing required columns."
)

assert len(incoming) == 500, (
    "Expected 500 synthetic input complaints."
)

# Select previously verified records
# belonging to the test reporting day.

eligible = incoming.loc[
    incoming[
        "processing_status"
    ].eq("processed")
    &
    incoming[
        "zip_validation_status"
    ].eq("zcta_consistent")
    &
    incoming[
        "created_date"
    ].str.slice(0, 10).eq(
        "2026-11-09"
    )
].copy()

assert len(eligible) == 500, (
    "The test requires 500 geographically "
    "verified complaints from 2026-11-09."
)

assert eligible[
    "complaint_id"
].is_unique

zip_text = (
    eligible["incident_zip"]
    .str.strip()
)

assert zip_text.str.fullmatch(
    r"\d{5}"
).all(), (
    "Some incident ZIP codes are invalid."
)

eligible["area_id"] = (
    eligible["borough"]
    .str.strip()
    .str.upper()
    + "_"
    + zip_text
)

eligible["complaint_category"] = (
    eligible["complaint_category"]
    .str.strip()
    .str.upper()
)

area_counts = (
    eligible.groupby("area_id")
    .size()
)

assert len(area_counts) == 20

assert area_counts.sum() == 500

print("Incoming records:", len(incoming))
print("Eligible records:", len(eligible))
print("Current geographic areas:", len(area_counts))

print(
    "Current complaint categories:",
    eligible[
        "complaint_category"
    ].nunique()
)

# ==================================================
# 4. ALIGN HISTORICAL AND CURRENT CATEGORIES
# ==================================================

print("\n3. ALIGNING CATEGORY FEATURES")

category_mapping = {
    column: (
        column.removeprefix("share_")
        .replace("_", " ")
        .upper()
    )
    for column in share_cols
}

assert len(set(
    category_mapping.values()
)) == len(share_cols)

historical_categories = set(
    category_mapping.values()
)

incoming_categories = set(
    eligible[
        "complaint_category"
    ].unique()
)

unmatched_categories = sorted(
    incoming_categories
    - historical_categories
)

if unmatched_categories:
    print(
        "Unmatched incoming categories:",
        unmatched_categories
    )

assert not unmatched_categories, (
    "Some current complaint categories have "
    "no corresponding historical share feature. "
    "Review the category mapping before "
    "performing the comparison."
)

missing_areas = sorted(
    set(area_counts.index)
    - set(historical["area_id"])
)

assert not missing_areas, (
    "Some current geographic areas are absent "
    f"from historical features: {missing_areas}"
)

print(
    "Historical categories:",
    len(historical_categories)
)

print(
    "Current categories matched:",
    len(incoming_categories)
)

print(
    "Current areas matched:",
    len(area_counts)
)

# ==================================================
# 5. BUILD CATEGORY-PERCENTAGE TABLES
# ==================================================

print("\n4. CALCULATING CATEGORY PERCENTAGES")

area_ids = sorted(
    area_counts.index.tolist()
)

categories = sorted(
    historical_categories
)

historical_index = (
    historical
    .set_index("area_id")
    .loc[area_ids]
)

historical_shares = pd.DataFrame(
    {
        category_mapping[column]:
            historical_index[column]
            .to_numpy(dtype=float)

        for column in share_cols
    },
    index=area_ids
).reindex(
    columns=categories
)

current_counts = pd.crosstab(
    eligible["area_id"],
    eligible["complaint_category"]
).reindex(
    index=area_ids,
    columns=categories,
    fill_value=0
)

current_shares = (
    current_counts.div(
        current_counts.sum(axis=1),
        axis=0
    )
)

assert np.allclose(
    historical_shares.sum(axis=1),
    1.0,
    atol=0.01
)

assert np.allclose(
    current_shares.sum(axis=1),
    1.0,
    atol=1e-10
)

# Percentage-point difference:
#
# Positive: category share increased.
# Negative: category share decreased.

difference_pp = (
    current_shares
    - historical_shares
) * 100

# Total variation distance between
# historical and current compositions.
#
# 0 = identical category distribution
# 1 = completely different distribution
#
# This is NOT an anomaly probability.

mix_distance = (
    current_shares
    - historical_shares
).abs().sum(axis=1) / 2

print(
    "Historical share table:",
    historical_shares.shape
)

print(
    "Current share table:",
    current_shares.shape
)

print(
    "Current complaints reconciled:",
    int(current_counts.to_numpy().sum())
)

# ==================================================
# 6. BUILD AREA-LEVEL COMPARISON
# ==================================================

print("\n5. BUILDING AREA COMPARISON")

summary_rows = []

for area_id in area_ids:

    historical_top = (
        historical_shares
        .loc[area_id]
        .idxmax()
    )

    current_top = (
        current_shares
        .loc[area_id]
        .idxmax()
    )

    largest_increase = (
        difference_pp
        .loc[area_id]
        .idxmax()
    )

    summary_rows.append({
        "area_id": area_id,

        "historical_reports": int(
            historical_index.loc[
                area_id,
                "total_complaints"
            ]
        ),

        "incoming_reports": int(
            area_counts.loc[area_id]
        ),

        "historical_top_category":
            historical_top,

        "historical_top_share_pct": round(
            100 * historical_shares.loc[
                area_id,
                historical_top
            ],
            2
        ),

        "current_top_category":
            current_top,

        "current_top_share_pct": round(
            100 * current_shares.loc[
                area_id,
                current_top
            ],
            2
        ),

        "top_category_changed": (
            historical_top != current_top
        ),

        "largest_share_increase_category":
            largest_increase,

        "largest_share_increase_pp": round(
            float(
                difference_pp.loc[
                    area_id,
                    largest_increase
                ]
            ),
            2
        ),

        "category_mix_distance": round(
            float(
                mix_distance.loc[area_id]
            ),
            4
        )
    })

summary = pd.DataFrame(
    summary_rows
)

# ==================================================
# 7. BUILD DETAILED CATEGORY COMPARISON
# ==================================================

detail_rows = []

for area_id in area_ids:

    for category in categories:

        detail_rows.append({
            "area_id": area_id,

            "complaint_category":
                category,

            "historical_share_pct": round(
                100 * historical_shares.loc[
                    area_id,
                    category
                ],
                3
            ),

            "current_share_pct": round(
                100 * current_shares.loc[
                    area_id,
                    category
                ],
                3
            ),

            "share_change_pp": round(
                float(
                    difference_pp.loc[
                        area_id,
                        category
                    ]
                ),
                3
            ),

            "current_reports": int(
                current_counts.loc[
                    area_id,
                    category
                ]
            )
        })

details = pd.DataFrame(
    detail_rows
)

# ==================================================
# 8. DISPLAY RESULTS
# ==================================================

print("\n6. HISTORICAL VS CURRENT AREA PATTERNS")

display(
    summary[[
        "area_id",
        "historical_top_category",
        "current_top_category",
        "top_category_changed",
        "largest_share_increase_category",
        "largest_share_increase_pp",
        "category_mix_distance"
    ]]
    .sort_values(
        "category_mix_distance",
        ascending=False
    )
    .reset_index(drop=True)
)

# ==================================================
# 9. EXAMPLE: MANHATTAN ZIP 10003
# ==================================================

print("\n7. MANHATTAN ZIP 10003 COMPARISON")

example_area = "MANHATTAN_10003"

example = details.loc[
    details["area_id"].eq(
        example_area
    )
].copy()

assert not example.empty

display(
    example.sort_values(
        "share_change_pp",
        ascending=False
    ).reset_index(drop=True)
)

print("\nAREA INTERPRETATION")

selected_summary = summary.loc[
    summary["area_id"].eq(
        example_area
    )
].iloc[0]

print(
    "Historical dominant category:",
    selected_summary[
        "historical_top_category"
    ]
)

print(
    "Current dominant category:",
    selected_summary[
        "current_top_category"
    ]
)

print(
    "Largest category-share increase:",
    selected_summary[
        "largest_share_increase_category"
    ]
)

print(
    "Increase in percentage points:",
    selected_summary[
        "largest_share_increase_pp"
    ]
)

print(
    "Category mix distance:",
    selected_summary[
        "category_mix_distance"
    ]
)

# ==================================================
# 10. VALIDATION CHECKLIST
# ==================================================

print("\n8. VALIDATION CHECKLIST")

checks = {
    "238 historical areas preserved":
        len(historical) == 238,

    "14 historical share features":
        len(share_cols) == 14,

    "500 verified current reports":
        len(eligible) == 500,

    "20 geographic areas matched":
        len(summary) == 20,

    "All incoming categories matched":
        not unmatched_categories,

    "Area totals equal 500":
        summary[
            "incoming_reports"
        ].sum() == 500,

    "Historical shares valid":
        np.allclose(
            historical_shares.sum(axis=1),
            1.0,
            atol=0.01
        ),

    "Current shares valid":
        np.allclose(
            current_shares.sum(axis=1),
            1.0
        ),

    "Current complaint counts reconcile":
        current_counts.to_numpy().sum()
        == 500,

    "20 x 14 detailed comparison rows":
        len(details) == 20 * 14,

    "Category mix distances valid":
        mix_distance.between(
            0, 1.01
        ).all(),

    "Original datasets unchanged":
        all(
            hashlib.sha256(
                path.read_bytes()
            ).hexdigest()
            == source_hashes[path]
            for path in source_hashes
        )
}

validation = pd.DataFrame([
    {
        "test": name,
        "passed": bool(value)
    }
    for name, value in checks.items()
])

display(validation)

passed_count = int(
    validation["passed"].sum()
)

total_checks = len(validation)

print(
    f"\nChecks passed: "
    f"{passed_count}/{total_checks}"
)

# ==================================================
# 11. SAVE ONLY IF ALL CHECKS PASS
# ==================================================

if validation["passed"].all():

    destinations = {
        summary_path:
            summary.to_csv(index=False),

        detail_path:
            details.to_csv(index=False)
    }

    # Check for conflicting existing files
    # before writing any new output.

    for path, content in destinations.items():

        if path.exists():

            assert path.read_text(
                encoding="utf-8"
            ) == content, (
                f"Existing output differs: {path}. "
                "No overwrite performed."
            )

    output_dir.mkdir(
        parents=True,
        exist_ok=True
    )

    for path, content in destinations.items():

        if not path.exists():

            with path.open(
                "x",
                encoding="utf-8"
            ) as file:
                file.write(content)

    print("\n9. OUTPUT FILES SAVED")

    print("Area summary:", summary_path)
    print("Category details:", detail_path)

    print("\nSTATUS: PASSED")
    print("CATEGORY COMPARISON VERIFIED")
    print("238 HISTORICAL AREAS PRESERVED")
    print("20 CURRENT AREAS COMPARED")
    print("HISTORICAL VS CURRENT CATEGORY MIX READY")
    print("PCA AND K-MEANS MODELS UNCHANGED")
    print("DBSCAN AND ANOMALY MODELS UNCHANGED")

else:

    print("\nSTATUS: NEEDS REVIEW")
    print("No comparison files saved.")


 HISTORICAL VS CURRENT CATEGORY MIX

PROJECT ROOT:
D:\Urban-Infrastructure-Pulse-Intelligence

INPUT FILES VERIFIED
Historical: D:\Urban-Infrastructure-Pulse-Intelligence\data\processed\selected_modeling_features.csv
Current: D:\Urban-Infrastructure-Pulse-Intelligence\outputs\tests\phase_17_4c_backend_verified\verified_500_backend_results.csv

1. LOADING HISTORICAL FEATURES
Historical areas: 238
Historical category-share features: 14

Available historical share columns:
 - share_curb_condition
 - share_root_sewer_sidewalk_condition
 - share_sewer
 - share_sewer_maintenance
 - share_sidewalk_condition
 - share_street_condition
 - share_street_light_condition
 - share_street_sign_-_damaged
 - share_street_sign_-_dangling
 - share_street_sign_-_missing
 - share_traffic_signal_condition
 - share_water_drainage
 - share_water_leak
 - share_water_maintenance

2. LOADING CURRENT COMPLAINTS
Incoming records: 500
Eligible records: 500
Current geographic areas: 20
Current complaint categories: 8

,area_id,historical_top_category,current_top_category,top_category_changed,largest_share_increase_category,largest_share_increase_pp,category_mix_distance
0,QUEENS_11005,STREET CONDITION,SEWER,True,SEWER,60.00,0.9200
1,MANHATTAN_10003,STREET CONDITION,SEWER,True,SEWER,53.98,0.8190
2,BRONX_10453,TRAFFIC SIGNAL CONDITION,SEWER,True,SEWER,50.50,0.7847
3,BROOKLYN_11204,TRAFFIC SIGNAL CONDITION,SEWER,True,SEWER,48.17,0.7537
4,STATEN ISLAND_10303,STREET CONDITION,SEWER,True,SEWER,40.17,0.6772
5,QUEENS_11040,STREET CONDITION,STREET LIGHT CONDITION,True,STREET LIGHT CONDITION,36.03,0.6287
6,MANHATTAN_10004,STREET CONDITION,STREET LIGHT CONDITION,True,STREET LIGHT CONDITION,39.57,0.5745
7,STATEN ISLAND_10304,STREET CONDITION,STREET LIGHT CONDITION,True,STREET LIGHT CONDITION,29.19,0.5257
8,BROOKLYN_11205,STREET CONDITION,STREET LIGHT CONDITION,True,STREET LIGHT CONDITION,36.62,0.5100
9,BRONX_10451,TRAFFIC SIGNAL CONDITION,STREET CONDITION,True,STREET CONDITION,29.12,0.4259



7. MANHATTAN ZIP 10003 COMPARISON


,area_id,complaint_category,historical_share_pct,current_share_pct,share_change_pp,current_reports
0,MANHATTAN_10003,SEWER,6.015,60.0,53.985,15
1,MANHATTAN_10003,SEWER MAINTENANCE,0.053,20.0,19.947,5
2,MANHATTAN_10003,WATER DRAINAGE,0.027,8.0,7.973,2
3,MANHATTAN_10003,WATER MAINTENANCE,0.027,0.0,-0.027,0
4,MANHATTAN_10003,WATER LEAK,0.373,0.0,-0.373,0
5,MANHATTAN_10003,STREET SIGN - DANGLING,1.011,0.0,-1.011,0
6,MANHATTAN_10003,ROOT SEWER SIDEWALK CONDITION,1.038,0.0,-1.038,0
7,MANHATTAN_10003,STREET SIGN - MISSING,1.118,0.0,-1.118,0
8,MANHATTAN_10003,STREET SIGN - DAMAGED,2.821,0.0,-2.821,0
9,MANHATTAN_10003,CURB CONDITION,3.434,0.0,-3.434,0



AREA INTERPRETATION
Historical dominant category: STREET CONDITION
Current dominant category: SEWER
Largest category-share increase: SEWER
Increase in percentage points: 53.98
Category mix distance: 0.819

8. VALIDATION CHECKLIST


,test,passed
0,238 historical areas preserved,True
1,14 historical share features,True
2,500 verified current reports,True
3,20 geographic areas matched,True
4,All incoming categories matched,True
5,Area totals equal 500,True
6,Historical shares valid,True
7,Current shares valid,True
8,Current complaint counts reconcile,True
9,20 x 14 detailed comparison rows,True



Checks passed: 12/12

9. OUTPUT FILES SAVED
Area summary: D:\Urban-Infrastructure-Pulse-Intelligence\outputs\tables\historical_current_category_comparison\area_category_change_summary_v1.csv
Category details: D:\Urban-Infrastructure-Pulse-Intelligence\outputs\tables\historical_current_category_comparison\area_category_change_details_v1.csv

STATUS: PASSED
CATEGORY COMPARISON VERIFIED
238 HISTORICAL AREAS PRESERVED
20 CURRENT AREAS COMPARED
HISTORICAL VS CURRENT CATEGORY MIX READY
PCA AND K-MEANS MODELS UNCHANGED
DBSCAN AND ANOMALY MODELS UNCHANGED


In [4]:

from pathlib import Path
import ast

root = Path(
    r"D:\Urban-Infrastructure-Pulse-Intelligence"
)

print(" GUI INTEGRATION INSPECTION")
print("=" * 65)

files = [
    root / "src/integration/live_uploaded_batch_pipeline.py",
    root / "src/integration/dashboard_intelligence_feed.py",
    root / "src/integration/combined_batch_intelligence.py",
]

# Find the Streamlit entry point without
# assuming its exact location.
search_folders = [
    root,
    root / "src/ui",
    root / "apps",
    root / "pages",
    root / "dashboard",
]

for folder in search_folders:
    if not folder.is_dir():
        continue

    for path in folder.glob("*.py"):
        try:
            source = path.read_text(encoding="utf-8")
        except UnicodeError:
            continue

        if (
            "st.file_uploader" in source
            or "st.set_page_config" in source
        ):
            files.append(path)

seen = set()

keywords = [
    "file_uploader",
    "process_complaint_batch",
    "run_uploaded",
    "dashboard_feed",
    "investigation",
    "area_id",
    "complaint_category",
]

for path in files:
    if path in seen:
        continue

    seen.add(path)

    print("\nFILE:", path.relative_to(root))

    if not path.is_file():
        print("NOT FOUND")
        continue

    source = path.read_text(encoding="utf-8")
    lines = source.splitlines()

    tree = ast.parse(source)

    print("\nTop-level functions/classes:")

    for node in tree.body:
        if isinstance(
            node,
            (
                ast.FunctionDef,
                ast.AsyncFunctionDef,
                ast.ClassDef,
            ),
        ):
            print(f"  Line {node.lineno}: {node.name}")

    print("\nRelevant integration references:")

    matches = [
        (i, line.strip())
        for i, line in enumerate(lines, start=1)
        if any(
            keyword in line.lower()
            for keyword in keywords
        )
    ]

    for line_number, content in matches[:12]:
        print(f"  {line_number}: {content[:120]}")

    print("Total matching lines:", len(matches))

print("\nINSPECTION COMPLETED")
print("No source files or models were modified.")


 GUI INTEGRATION INSPECTION

FILE: src\integration\live_uploaded_batch_pipeline.py

Top-level functions/classes:
  Line 39: load_upload_baseline
  Line 52: build_verified_day_feed
  Line 196: process_uploaded_csv_bytes

Relevant integration references:
  16: process_complaint_batch
  28: from src.integration.smart_assistant_investigation_briefs import (
  29: build_investigation_briefs
  33: build_dashboard_feed
  74: "complaint_category",
  149: briefs = build_investigation_briefs(
  154: feed = build_dashboard_feed(
  184: "Investigation audit lost records."
  232: backend = process_complaint_batch(
Total matching lines: 9

FILE: src\integration\dashboard_intelligence_feed.py

Top-level functions/classes:
  Line 10: _text
  Line 16: _unique
  Line 24: build_dashboard_feed

Relevant integration references:
  24: def build_dashboard_feed(integrated, bundle):
  39: "anomaly_area_id",
  153: str(row.anomaly_area_id),
  163: str(item["area_id"]),
  164: str(item["complaint_category"])
  2

In [5]:

from pathlib import Path

PROJECT_ROOT = Path(
    r"D:\Urban-Infrastructure-Pulse-Intelligence"
)

print(" BACKEND TO GUI DATA FLOW")
print("=" * 70)

inspection = {
    "src/integration/live_uploaded_batch_pipeline.py": [
        (140, 195),
        (196, 275),
    ],
    "dashboard/urban_pulse_simple.py": [
        (435, 475),
        (630, 705),
    ],
}

for relative_path, sections in inspection.items():

    path = PROJECT_ROOT / relative_path

    print("\n" + "=" * 70)
    print("FILE:", relative_path)
    print("=" * 70)

    if not path.is_file():
        print("FILE NOT FOUND")
        continue

    lines = path.read_text(
        encoding="utf-8"
    ).splitlines()

    for start, end in sections:

        print(f"\n--- LINES {start}–{end} ---")

        for line_number in range(
            start,
            min(end, len(lines)) + 1
        ):

            print(
                f"{line_number:4d} | "
                f"{lines[line_number - 1]}"
            )

print("\nINSPECTION COMPLETE")
print("No files or models modified.")


 BACKEND TO GUI DATA FLOW

FILE: src/integration/live_uploaded_batch_pipeline.py

--- LINES 140–195 ---
 140 |                 "no_eligible_reporting_day_records"
 141 |         }
 142 | 
 143 |     integrated = analyze_batch_with_anomaly_context(
 144 |         selected,
 145 |         baseline,
 146 |         reporting_day
 147 |     )
 148 | 
 149 |     briefs = build_investigation_briefs(
 150 |         integrated,
 151 |         synthetic_data=synthetic_data
 152 |     )
 153 | 
 154 |     feed = build_dashboard_feed(
 155 |         integrated,
 156 |         briefs
 157 |     )
 158 | 
 159 |     feed["metadata"]["input_origin"] = (
 160 |         "uploaded_csv"
 161 |     )
 162 | 
 163 |     feed["metadata"]["data_verified_in_field"] = (
 164 |         False
 165 |     )
 166 | 
 167 |     feed["metadata"]["upload_note"] = (
 168 |         "Uploaded reports were geographically "
 169 |         "screened. No infrastructure fault "
 170 |         "has been independently verified.

In [6]:

from pathlib import Path
import pandas as pd
import numpy as np

ROOT = Path(
    r"D:\Urban-Infrastructure-Pulse-Intelligence"
)

assert ROOT.is_dir(), "Project root not found"

history_path = ROOT / (
    "data/processed/selected_modeling_features.csv"
)

test_path = ROOT / (
    "outputs/tests/phase_17_4c_backend_verified/"
    "verified_500_backend_results.csv"
)

reference_path = ROOT / (
    "outputs/tables/historical_current_category_comparison/"
    "area_category_change_summary_v1.csv"
)

module_path = ROOT / (
    "src/integration/historical_category_comparison.py"
)

print("REUSABLE HISTORICAL COMPARISON")
print("=" * 70)

for path in [history_path, test_path, reference_path]:
    assert path.is_file(), f"Missing file: {path}"

# =====================================================
# NEW MODULE SOURCE
# =====================================================

module_source = r'''
import numpy as np
import pandas as pd


def compare_historical_current_categories(
    backend_records,
    historical_features,
    reporting_day,
    expected_analyzed=None,
):
    """Compare verified-day complaint mix with historical area mix.

    Returns descriptive comparisons, not anomaly decisions.
    Does not train, update, or save any ML model.
    """

    required = {
        "created_date",
        "processing_status",
        "zip_validation_status",
        "borough",
        "incident_zip",
        "complaint_category",
    }

    missing = required - set(backend_records.columns)

    if missing:
        raise ValueError(
            f"Missing backend columns: {sorted(missing)}"
        )

    records = backend_records.copy(deep=True)
    history = historical_features.copy(deep=True)

    # Use the same verified reporting-day rule
    # used in Phase 21.1.

    selected = records.loc[
        records["processing_status"]
        .astype("string").str.strip().eq("processed")
        &
        records["zip_validation_status"]
        .astype("string").str.strip().eq("zcta_consistent")
        &
        records["created_date"]
        .astype("string").str.slice(0, 10)
        .eq(str(reporting_day))
    ].copy()

    if expected_analyzed is not None:
        if len(selected) != int(expected_analyzed):
            raise ValueError(
                "Historical comparison selected "
                f"{len(selected)} records, but the "
                "main pipeline selected "
                f"{expected_analyzed}. "
                "Check eligibility filters."
            )

    if selected.empty:
        return {
            "status": "no_verified_records",
            "eligible_reports": 0,
            "matched_reports": 0,
            "unmatched_areas": [],
            "categories_without_history": [],
            "summary": pd.DataFrame(),
            "details": pd.DataFrame(),
        }

    # Reconstruct the same borough-ZIP area ID.

    zip_code = (
        selected["incident_zip"]
        .astype("string").str.strip()
    )

    if not zip_code.str.fullmatch(
        r"\d{5}"
    ).fillna(False).all():
        raise ValueError(
            "Verified records contain invalid ZIP codes."
        )

    selected["area_id"] = (
        selected["borough"]
        .astype("string").str.strip().str.upper()
        + "_" + zip_code
    )

    selected["complaint_category"] = (
        selected["complaint_category"]
        .astype("string").str.strip().str.upper()
    )

    if selected[
        "complaint_category"
    ].isna().any():
        raise ValueError("Missing complaint categories.")

    if "area_id" not in history.columns:
        raise ValueError("Missing historical area_id.")

    if history["area_id"].duplicated().any():
        raise ValueError("Duplicate historical areas.")

    history["area_id"] = (
        history["area_id"]
        .astype("string").str.strip().str.upper()
    )

    share_cols = sorted(
        col for col in history.columns
        if col.startswith("share_")
    )

    if not share_cols:
        raise ValueError(
            "No historical category-share features."
        )

    history[share_cols] = (
        history[share_cols].apply(
            pd.to_numeric, errors="raise"
        )
    )

    values = history[
        share_cols
    ].to_numpy(dtype=float)

    if (
        not np.isfinite(values).all()
        or not ((values >= 0) & (values <= 1)).all()
    ):
        raise ValueError("Invalid historical shares.")

    if not np.allclose(
        history[share_cols].sum(axis=1),
        1.0,
        atol=0.01
    ):
        raise ValueError(
            "Historical category shares do not sum to 1."
        )

    # Compare only areas having a historical baseline.

    available = set(history["area_id"])

    unmatched = sorted(
        set(selected["area_id"]) - available
    )

    matched = selected.loc[
        selected["area_id"].isin(available)
    ].copy()

    if matched.empty:
        return {
            "status": "no_historical_area_overlap",
            "eligible_reports": int(len(selected)),
            "matched_reports": 0,
            "unmatched_areas": unmatched,
            "categories_without_history": [],
            "summary": pd.DataFrame(),
            "details": pd.DataFrame(),
        }

    area_counts = matched.groupby(
        "area_id"
    ).size()

    area_ids = sorted(area_counts.index)

    mapping = {
        col: col.removeprefix("share_")
        .replace("_", " ").upper()
        for col in share_cols
    }

    if len(set(mapping.values())) != len(mapping):
        raise ValueError(
            "Duplicate normalized historical categories."
        )

    historical_categories = set(mapping.values())

    current_categories = set(
        matched["complaint_category"].unique()
    )

    # A category missing from the historical
    # feature set is represented as zero share,
    # and separately disclosed in the diagnostics.

    categories_without_history = sorted(
        current_categories - historical_categories
    )

    categories = sorted(
        historical_categories | current_categories
    )

    history_index = (
        history.set_index("area_id").loc[area_ids]
    )

    historical_shares = pd.DataFrame(
        {
            mapping[col]:
                history_index[col].to_numpy(dtype=float)
            for col in share_cols
        },
        index=area_ids
    ).reindex(
        columns=categories,
        fill_value=0.0
    )

    counts = pd.crosstab(
        matched["area_id"],
        matched["complaint_category"]
    ).reindex(
        index=area_ids,
        columns=categories,
        fill_value=0
    )

    current_shares = counts.div(
        counts.sum(axis=1),
        axis=0
    )

    difference = (
        current_shares - historical_shares
    )

    mix_distance = (
        difference.abs().sum(axis=1) / 2
    )

    summary_rows = []
    detail_rows = []

    for area_id in area_ids:

        old_top = historical_shares.loc[
            area_id
        ].idxmax()

        new_top = current_shares.loc[
            area_id
        ].idxmax()

        increase = difference.loc[
            area_id
        ].idxmax()

        summary_rows.append({
            "area_id": area_id,
            "incoming_reports": int(
                area_counts.loc[area_id]
            ),
            "historical_top_category": old_top,
            "historical_top_share_pct": round(
                100 * historical_shares.loc[
                    area_id, old_top
                ], 2
            ),
            "current_top_category": new_top,
            "current_top_share_pct": round(
                100 * current_shares.loc[
                    area_id, new_top
                ], 2
            ),
            "top_category_changed": old_top != new_top,
            "largest_share_increase_category": increase,
            "largest_share_increase_pp": round(
                100 * float(
                    difference.loc[area_id, increase]
                ), 2
            ),
            "category_mix_distance": round(
                float(mix_distance.loc[area_id]), 4
            ),
        })

        for category in categories:

            detail_rows.append({
                "area_id": area_id,
                "complaint_category": category,
                "historical_share_pct": round(
                    100 * historical_shares.loc[
                        area_id, category
                    ], 3
                ),
                "current_share_pct": round(
                    100 * current_shares.loc[
                        area_id, category
                    ], 3
                ),
                "share_change_pp": round(
                    100 * float(
                        difference.loc[
                            area_id, category
                        ]
                    ), 3
                ),
                "current_reports": int(
                    counts.loc[area_id, category]
                ),
            })

    return {
        "status": "compared",
        "eligible_reports": int(len(selected)),
        "matched_reports": int(len(matched)),
        "unmatched_areas": unmatched,
        "categories_without_history":
            categories_without_history,
        "summary": pd.DataFrame(summary_rows),
        "details": pd.DataFrame(detail_rows),
    }
'''

# =====================================================
# TEST THE MODULE IN MEMORY BEFORE SAVING
# =====================================================

namespace = {}

exec(
    compile(
        module_source,
        str(module_path),
        "exec"
    ),
    namespace
)

compare = namespace[
    "compare_historical_current_categories"
]

historical = pd.read_csv(history_path)

test_records = pd.read_csv(
    test_path,
    dtype="string",
    keep_default_na=False
)

result = compare(
    backend_records=test_records,
    historical_features=historical,
    reporting_day="2026-11-09",
    expected_analyzed=500,
)

summary = result["summary"]
details = result["details"]

print("\n1. COMPARISON RESULT")
print("Status:", result["status"])
print("Eligible reports:", result["eligible_reports"])
print("Matched reports:", result["matched_reports"])
print("Areas compared:", len(summary))
print("Category comparison rows:", len(details))
print("Unmatched areas:", result["unmatched_areas"])
print(
    "Categories without historical share features:",
    result["categories_without_history"]
)

# =====================================================
# COMPARE WITH VERIFIED PHASE 21.1 RESULTS
# =====================================================

reference = pd.read_csv(reference_path)

joined = reference.merge(
    summary,
    on="area_id",
    suffixes=("_reference", "_module"),
    validate="one_to_one"
)

assert len(joined) == len(reference) == 20

for col in [
    "historical_top_category",
    "current_top_category",
    "largest_share_increase_category",
    "top_category_changed",
]:
    assert (
        joined[f"{col}_reference"].astype(str)
        == joined[f"{col}_module"].astype(str)
    ).all(), f"Mismatch: {col}"

for col in [
    "largest_share_increase_pp",
    "category_mix_distance",
]:
    assert np.allclose(
        joined[f"{col}_reference"],
        joined[f"{col}_module"],
        atol=0.02
    ), f"Mismatch: {col}"

checks = {
    "500 records selected":
        result["eligible_reports"] == 500,

    "All 500 matched to historical areas":
        result["matched_reports"] == 500,

    "20 geographic areas":
        len(summary) == 20,

    "280 category comparison rows":
        len(details) == 280,

    "No unmatched historical areas":
        not result["unmatched_areas"],

    "No unmatched historical categories":
        not result["categories_without_history"],

    "Dominant categories match Phase 21.1":
        True,

    "Category differences match Phase 21.1":
        True,
}

validation = pd.DataFrame([
    {"test": name, "passed": bool(passed)}
    for name, passed in checks.items()
])

print("\n2. VALIDATION")
display(validation)

assert validation["passed"].all()

print(
    f"Checks passed: "
    f"{validation['passed'].sum()}/{len(validation)}"
)

# =====================================================
# SAVE NEW MODULE ONLY AFTER VALIDATION
# =====================================================

assert module_path.parent.is_dir()

if module_path.exists():
    assert (
        module_path.read_text(encoding="utf-8")
        == module_source
    ), (
        "An existing module differs. "
        "No overwrite performed."
    )
    print("\nExisting module already matches.")

else:
    with module_path.open(
        "x",
        encoding="utf-8"
    ) as file:
        file.write(module_source)

    print("\nNew comparison module saved.")

print("Module:", module_path)

print("\n3. EXAMPLE COMPARISON")

display(
    summary.loc[
        summary["area_id"].eq("MANHATTAN_10003")
    ]
)

print("\nSTATUS: MODULE VALIDATED")
print("Existing ML models remain unchanged.")
print("Existing GUI remains unchanged.")


REUSABLE HISTORICAL COMPARISON

1. COMPARISON RESULT
Status: compared
Eligible reports: 500
Matched reports: 500
Areas compared: 20
Category comparison rows: 280
Unmatched areas: []
Categories without historical share features: []

2. VALIDATION


,test,passed
0,500 records selected,True
1,All 500 matched to historical areas,True
2,20 geographic areas,True
3,280 category comparison rows,True
4,No unmatched historical areas,True
5,No unmatched historical categories,True
6,Dominant categories match Phase 21.1,True
7,Category differences match Phase 21.1,True


Checks passed: 8/8

New comparison module saved.
Module: D:\Urban-Infrastructure-Pulse-Intelligence\src\integration\historical_category_comparison.py

3. EXAMPLE COMPARISON


,area_id,incoming_reports,historical_top_category,historical_top_share_pct,current_top_category,current_top_share_pct,top_category_changed,largest_share_increase_category,largest_share_increase_pp,category_mix_distance
10,MANHATTAN_10003,25,STREET CONDITION,42.29,SEWER,60.0,True,SEWER,53.98,0.819



STATUS: MODULE VALIDATED
Existing ML models remain unchanged.
Existing GUI remains unchanged.


In [8]:

from pathlib import Path
import hashlib

print(" INTEGRATION ERROR DIAGNOSIS")
print("=" * 60)

gui_path = Path(
    r"D:\Urban-Infrastructure-Pulse-Intelligence"
) / "dashboard/urban_pulse_simple.py"

print("\n1. MARKER CHECK")
print("Expected marker:", repr(marker))
print("Marker in original:", original.count(marker))
print("Marker in addition_a:", addition_a.count(marker))
print("Marker in updated:", updated.count(marker))

print("\n2. INTEGRATION CHECK")
print(
    "Comparison calculation:",
    updated.count("historical_comparison = None")
)
print(
    "Session-state integration:",
    updated.count(
        '"historical_comparison": historical_comparison,'
    )
)
print(
    "Historical comparison GUI section:",
    updated.count(
        '"Historical vs current complaint patterns"'
    )
)

print("\n3. SOURCE FILE SAFETY")
current_hash = hashlib.sha256(
    gui_path.read_bytes()
).hexdigest()

print(
    "Original GUI unchanged:",
    current_hash == original_hash
)

print("\nDIAGNOSIS COMPLETE")
print("No files were modified.")


 INTEGRATION ERROR DIAGNOSIS

1. MARKER CHECK
Expected marker: '# : HISTORICAL CATEGORY COMPARISON'
Marker in original: 0
Marker in addition_a: 0
Marker in updated: 0

2. INTEGRATION CHECK
Comparison calculation: 1
Session-state integration: 1
Historical comparison GUI section: 1

3. SOURCE FILE SAFETY
Original GUI unchanged: True

DIAGNOSIS COMPLETE
No files were modified.


In [9]:

from pathlib import Path
import ast
import hashlib
import shutil

ROOT = Path(
    r"D:\Urban-Infrastructure-Pulse-Intelligence"
)

gui_path = ROOT / "dashboard/urban_pulse_simple.py"

backup_path = ROOT / (
    "dashboard/urban_pulse_simple_phase21_backup.py.bak"
)

print(" CORRECTED GUI INTEGRATION")
print("=" * 65)

# --------------------------------------------------
# 1. Verify prepared source exists
# --------------------------------------------------

assert isinstance(original, str)
assert isinstance(updated, str)
assert updated != original

assert gui_path.is_file()

# --------------------------------------------------
# 2. Corrected integration validation
# --------------------------------------------------

checks = {
    "Historical comparison calculation": (
        updated.count(
            "historical_comparison = None"
        ) == 1
    ),

    "Session state integration": (
        updated.count(
            '"historical_comparison": historical_comparison,'
        ) == 1
    ),

    "Historical comparison GUI section": (
        updated.count(
            '"Historical vs current complaint patterns"'
        ) == 1
    ),

    "Existing upload analysis preserved": (
        "process_uploaded_csv_bytes(" in updated
    ),

    "Existing investigation review preserved": (
        "build_review_queue(" in updated
    ),

    "Original GUI unchanged": (
        hashlib.sha256(
            gui_path.read_bytes()
        ).hexdigest()
        == hashlib.sha256(
            original.encode("utf-8")
        ).hexdigest()
    ),
}

for name, passed in checks.items():
    print(
        f'{"PASS" if passed else "FAIL"}: {name}'
    )

assert all(checks.values()), (
    "Integration validation failed. "
    "GUI was not modified."
)

# --------------------------------------------------
# 3. Validate Python syntax
# --------------------------------------------------

ast.parse(updated)

compile(
    updated,
    str(gui_path),
    "exec"
)

print("\nPython syntax: PASSED")

# --------------------------------------------------
# 4. Create original GUI backup
# --------------------------------------------------

if backup_path.exists():

    assert (
        backup_path.read_bytes()
        == gui_path.read_bytes()
    ), (
        "Existing backup differs from the "
        "current GUI. No changes made."
    )

    print("Existing backup verified.")

else:

    shutil.copy2(
        gui_path,
        backup_path
    )

    print("New backup created.")

assert (
    backup_path.read_bytes()
    == gui_path.read_bytes()
)

# --------------------------------------------------
# 5. Save corrected GUI
# --------------------------------------------------

gui_path.write_text(
    updated,
    encoding="utf-8"
)

assert (
    gui_path.read_text(
        encoding="utf-8"
    ) == updated
)

# --------------------------------------------------
# 6. Final validation
# --------------------------------------------------

assert (
    backup_path.read_text(
        encoding="utf-8"
    ) == original
)

print("\n" + "=" * 65)
print("SOURCE INTEGRATION PASSED")
print("=" * 65)

print("\nUpdated GUI:")
print(gui_path)

print("\nOriginal backup:")
print(backup_path)

print("\nHistorical comparison added:")
print("- Dynamic uploaded CSV comparison")
print("- Historical vs current categories")
print("- Category percentage changes")
print("- Area-level reporting pattern comparison")

print("\nUnchanged:")
print("- Historical PCA and K-Means")
print("- Current K-Means")
print("- Historical and current DBSCAN")
print("- Anomaly detection")
print("- Existing backend pipeline")

print("\nNEXT: TEST IN STREAMLIT")


 CORRECTED GUI INTEGRATION
PASS: Historical comparison calculation
PASS: Session state integration
PASS: Historical comparison GUI section
PASS: Existing upload analysis preserved
PASS: Existing investigation review preserved
PASS: Original GUI unchanged

Python syntax: PASSED
New backup created.

SOURCE INTEGRATION PASSED

Updated GUI:
D:\Urban-Infrastructure-Pulse-Intelligence\dashboard\urban_pulse_simple.py

Original backup:
D:\Urban-Infrastructure-Pulse-Intelligence\dashboard\urban_pulse_simple_phase21_backup.py.bak

Historical comparison added:
- Dynamic uploaded CSV comparison
- Historical vs current categories
- Category percentage changes
- Area-level reporting pattern comparison

Unchanged:
- Historical PCA and K-Means
- Current K-Means
- Historical and current DBSCAN
- Anomaly detection
- Existing backend pipeline

NEXT: TEST IN STREAMLIT


### Historical Comparable-Day Complaint Analysis

In [10]:

from pathlib import Path
import pandas as pd
import json

print(" HISTORICAL COMPARABLE-DAY AUDIT")
print("=" * 70)

ROOT = Path(
    r"D:\Urban-Infrastructure-Pulse-Intelligence"
)

assert ROOT.is_dir(), "Project directory not found"

# --------------------------------------------------
# 1. Historical daily-count baseline
# --------------------------------------------------

data_path = ROOT / (
    "outputs/tables/anomaly/"
    "historical_daily_counts_2023_2025_v1.csv.gz"
)

manifest_path = ROOT / (
    "outputs/tables/anomaly/"
    "historical_daily_counts_2023_2025_v1_manifest.json"
)

assert data_path.is_file(), (
    f"Historical baseline not found: {data_path}"
)

print("\n1. HISTORICAL INPUT")
print("File:", data_path)
print("Size (MB):", round(
    data_path.stat().st_size / (1024 * 1024), 2
))

# --------------------------------------------------
# 2. Inspect actual columns
# --------------------------------------------------

sample = pd.read_csv(
    data_path,
    nrows=10
)

print("\n2. ACTUAL CSV COLUMNS")

for col in sample.columns:
    print(f" - {col}: {sample[col].dtype}")

print("\n3. SAMPLE HISTORICAL RECORDS")
display(sample.head(5))

# --------------------------------------------------
# 3. Identify likely comparison fields
# --------------------------------------------------

columns = list(sample.columns)

def find_column(candidates):
    lookup = {
        c.lower(): c for c in columns
    }
    for name in candidates:
        if name.lower() in lookup:
            return lookup[name.lower()]
    return None

date_col = find_column([
    "reporting_date",
    "created_date",
    "complaint_date",
    "date",
    "day",
    "report_date",
])

area_col = find_column([
    "area_id",
    "anomaly_area_id",
    "borough_zip",
])

category_col = find_column([
    "complaint_category",
    "category",
    "complaint_type",
])

count_col = find_column([
    "complaint_count",
    "daily_count",
    "report_count",
    "count",
    "n_reports",
    "reports",
])

print("\n4. DETECTED FIELDS")
print("Date column:", date_col)
print("Area column:", area_col)
print("Category column:", category_col)
print("Count column:", count_col)

# --------------------------------------------------
# 4. Inspect historical dates
# --------------------------------------------------

reporting_day = pd.Timestamp("2026-11-09")

print("\n5. TARGET REPORTING DAY")
print("Date:", reporting_day.date())
print("Weekday:", reporting_day.day_name())
print("Quarter:", reporting_day.quarter)

if date_col is not None:

    dates_only = pd.read_csv(
        data_path,
        usecols=[date_col]
    )

    dates = pd.to_datetime(
        dates_only[date_col],
        errors="coerce"
    ).dropna().dt.normalize().drop_duplicates()

    print("\n6. HISTORICAL DATE COVERAGE")

    print("Earliest day:", dates.min().date())
    print("Latest day:", dates.max().date())
    print("Distinct valid days:", len(dates))

    matching_weekday = dates[
        dates.dt.dayofweek
        == reporting_day.dayofweek
    ]

    matching_season = matching_weekday[
        matching_weekday.dt.quarter
        == reporting_day.quarter
    ]

    print(
        "Historical matching weekdays:",
        len(matching_weekday)
    )

    print(
        "Historical same-quarter weekdays:",
        len(matching_season)
    )

else:
    print(
        "\nDate column not automatically identified."
    )
    print(
        "We will use the displayed schema "
        "to identify the correct field."
    )

# --------------------------------------------------
# 5. Inspect the historical baseline manifest
# --------------------------------------------------

print("\n7. HISTORICAL MANIFEST")

if manifest_path.is_file():

    with manifest_path.open(
        "r",
        encoding="utf-8"
    ) as f:
        manifest = json.load(f)

    print("Manifest found.")
    print("Top-level keys:")

    for key in manifest:
        print(" -", key)

    print("\nManifest preview:")

    preview = json.dumps(
        manifest,
        indent=2,
        default=str
    )

    print(preview[:4500])

else:
    print("Manifest not found at expected path.")
    print("The CSV schema can still be inspected.")

# --------------------------------------------------
# 6. Historical feature compatibility
# --------------------------------------------------

features_path = ROOT / (
    "data/processed/selected_modeling_features.csv"
)

assert features_path.is_file()

features_sample = pd.read_csv(
    features_path,
    nrows=3
)

category_shares = [
    c for c in features_sample.columns
    if c.startswith("share_")
]

print("\n8. EXISTING HISTORICAL FEATURE COMPATIBILITY")
print("Category-share features:", len(category_shares))
print("Historical K-Means features unchanged.")

print("\nSTATUS: AUDIT COMPLETED")
print("No files or models modified.")


 HISTORICAL COMPARABLE-DAY AUDIT

1. HISTORICAL INPUT
File: D:\Urban-Infrastructure-Pulse-Intelligence\outputs\tables\anomaly\historical_daily_counts_2023_2025_v1.csv.gz
Size (MB): 1.4

2. ACTUAL CSV COLUMNS
 - area_id: str
 - complaint_category: str
 - historical_day: str
 - complaint_count: int64

3. SAMPLE HISTORICAL RECORDS


,area_id,complaint_category,historical_day,complaint_count
0,BRONX_10451,CURB CONDITION,2023-01-24,1
1,BRONX_10451,CURB CONDITION,2023-02-01,1
2,BRONX_10451,CURB CONDITION,2023-02-07,1
3,BRONX_10451,CURB CONDITION,2023-02-22,1
4,BRONX_10451,CURB CONDITION,2023-03-02,1



4. DETECTED FIELDS
Date column: None
Area column: area_id
Category column: complaint_category
Count column: complaint_count

5. TARGET REPORTING DAY
Date: 2026-11-09
Weekday: Monday
Quarter: 4

Date column not automatically identified.
We will use the displayed schema to identify the correct field.

7. HISTORICAL MANIFEST
Manifest found.
Top-level keys:
 - snapshot_version
 - source_path
 - source_sha256
 - historical_start
 - historical_end
 - calendar_days
 - source_rows
 - eligible_rows
 - area_count
 - category_count
 - sparse_daily_entries
 - schema
 - zero_count_policy
 - date_policy
 - area_policy
 - intended_use
 - category_count_semantics
 - supported_category_count
 - observed_eligible_category_count
 - supported_categories
 - observed_eligible_categories
 - categories_without_eligible_area_records
 - category_coverage
 - category_coverage_limitation

Manifest preview:
{
  "snapshot_version": 1,
  "source_path": "data/processed/nyc311_infrastructure_cleaned.csv",
  "source_s

In [11]:
from IPython.display import display
from pathlib import Path
import hashlib
import json
import numpy as np
import pandas as pd

ROOT = Path(r"D:\Urban-Infrastructure-Pulse-Intelligence")
assert ROOT.is_dir(), "Project folder not found"

history_path = ROOT / "outputs/tables/anomaly/historical_daily_counts_2023_2025_v1.csv.gz"
manifest_path = ROOT / "outputs/tables/anomaly/historical_daily_counts_2023_2025_v1_manifest.json"
current_path = ROOT / "outputs/tests/phase_17_4c_backend_verified/verified_500_backend_results.csv"
output_dir = ROOT / "outputs/tables/historical_comparable_day"

inputs = [history_path, manifest_path, current_path]
for path in inputs:
    assert path.is_file(), f"Missing file: {path}"
original_hashes = {p: hashlib.sha256(p.read_bytes()).hexdigest() for p in inputs}

with manifest_path.open(encoding="utf-8") as file:
    manifest = json.load(file)

historical = pd.read_csv(history_path)
incoming = pd.read_csv(current_path, dtype="string", keep_default_na=False)

assert historical.columns.tolist() == manifest["schema"]
assert len(historical) == manifest["sparse_daily_entries"]
assert int(historical["complaint_count"].sum()) == manifest["eligible_rows"]
assert historical["area_id"].nunique() == manifest["area_count"]
assert historical["complaint_category"].nunique() == len(manifest["observed_eligible_categories"])
assert not historical.duplicated(["area_id", "complaint_category", "historical_day"]).any()
assert historical["complaint_count"].gt(0).all()
assert set(historical["complaint_category"]) == set(manifest["observed_eligible_categories"])

historical["historical_day"] = pd.to_datetime(
    historical["historical_day"], format="%Y-%m-%d", errors="raise"
)
start = pd.Timestamp(manifest["historical_start"])
end = pd.Timestamp(manifest["historical_end"])
assert historical["historical_day"].between(start, end).all()

target = pd.Timestamp("2026-11-09")
assert target > end
calendar = pd.date_range(start, end, freq="D")
matching_weekdays = calendar[calendar.dayofweek == target.dayofweek]
seasonal_weekdays = matching_weekdays[
    matching_weekdays.quarter == target.quarter
]
assert len(matching_weekdays) == 157
assert len(seasonal_weekdays) == 39

required_current = {
    "created_date", "processing_status", "zip_validation_status",
    "borough", "incident_zip", "complaint_category"
}
assert required_current.issubset(incoming.columns)

eligible = incoming.loc[
    incoming["created_date"].str.slice(0, 10).eq(target.strftime("%Y-%m-%d"))
    & incoming["processing_status"].str.strip().eq("processed")
    & incoming["zip_validation_status"].str.strip().eq("zcta_consistent")
].copy()
assert len(eligible) == 500, "Expected 500 verified records in the clean test batch"
assert eligible["incident_zip"].str.fullmatch(r"\d{5}").all()

eligible["area_id"] = (
    eligible["borough"].str.strip().str.upper()
    + "_" + eligible["incident_zip"].str.strip()
)
eligible["complaint_category"] = eligible["complaint_category"].str.strip().str.upper()
assert eligible["area_id"].nunique() == 20
categories = sorted(manifest["supported_categories"])
observed = set(manifest["observed_eligible_categories"])
assert set(eligible["complaint_category"]).issubset(set(categories))

all_mondays = historical.loc[
    historical["historical_day"].dt.dayofweek.eq(target.dayofweek)
]
windows = {
    "same_quarter_weekdays": (all_mondays.loc[
        all_mondays["historical_day"].dt.quarter.eq(target.quarter)
    ], 39, 10, 30),
    "all_matching_weekdays": (all_mondays, 157, 20, 50),
}

summary_rows = []
detail_rows = []
known_areas = set(historical["area_id"])

for area_id, group in eligible.groupby("area_id", sort=True):
    current_counts = group["complaint_category"].value_counts()
    current_total = len(group)
    current_top = str(current_counts.idxmax())
    uncovered = sorted(set(current_counts.index) - observed)

    for window_name, (window, calendar_days, min_days, min_reports) in windows.items():
        area_history = window.loc[window["area_id"].eq(area_id)]
        historic_counts = area_history.groupby("complaint_category")["complaint_count"].sum()
        historic_total = int(historic_counts.sum())
        active_days = int(area_history["historical_day"].nunique())

        if area_id not in known_areas:
            status = "no_historical_area"
        elif active_days < min_days or historic_total < min_reports:
            status = "insufficient_historical_support"
        elif uncovered:
            status = "category_without_historical_coverage"
        elif current_total < 5:
            status = "small_current_sample"
        else:
            status = "supported"

        hist_top = str(historic_counts.idxmax()) if status == "supported" else None
        diffs = []
        for category in categories:
            current_pct = 100 * int(current_counts.get(category, 0)) / current_total
            hist_pct = (
                100 * int(historic_counts.get(category, 0)) / historic_total
                if status == "supported" and category in observed
                else np.nan
            )
            if status == "supported" and category in observed:
                diffs.append(abs(current_pct - hist_pct))
            detail_rows.append({
                "area_id": area_id,
                "reference_window": window_name,
                "complaint_category": category,
                "comparison_status": status,
                "historical_reports": int(historic_counts.get(category, 0)),
                "current_reports": int(current_counts.get(category, 0)),
                "historical_share_pct": round(hist_pct, 3) if pd.notna(hist_pct) else np.nan,
                "current_share_pct": round(current_pct, 3),
                "share_change_pp": round(current_pct - hist_pct, 3) if pd.notna(hist_pct) else np.nan,
            })

        summary_rows.append({
            "area_id": area_id,
            "reference_window": window_name,
            "comparison_status": status,
            "historical_calendar_days": calendar_days,
            "historical_active_days": active_days,
            "historical_reports": historic_total,
            "current_reports": current_total,
            "historical_top_category": hist_top,
            "current_top_category": current_top,
            "top_category_changed": hist_top != current_top if status == "supported" else None,
            "category_mix_distance": round(sum(diffs) / 200, 4) if status == "supported" else np.nan,
            "uncovered_current_categories": ", ".join(uncovered),
        })

summary = pd.DataFrame(summary_rows)
details = pd.DataFrame(detail_rows)

print("HISTORICAL COMPARABLE WEEKDAYS")
print("Historical Mondays:", len(matching_weekdays))
print("Historical Q4 Mondays:", len(seasonal_weekdays))
print("Current verified reports:", len(eligible))
print("Area/window comparisons:", len(summary))
print("Category-level comparison rows:", len(details))

print("\nSEASONAL MONDAY COMPARISON")
display(summary.loc[
    summary["reference_window"].eq("same_quarter_weekdays"),
    ["area_id", "comparison_status", "historical_active_days", "historical_reports",
     "historical_top_category", "current_top_category", "category_mix_distance"]
].sort_values(["comparison_status", "area_id"]).reset_index(drop=True))

print("\nMANHATTAN_10003 — SEASONAL CATEGORY SHARES")
display(details.loc[
    details["area_id"].eq("MANHATTAN_10003")
    & details["reference_window"].eq("same_quarter_weekdays"),
    ["complaint_category", "historical_share_pct", "current_share_pct", "share_change_pp"]
].sort_values("share_change_pp", ascending=False))

manhattan = summary.set_index(["area_id", "reference_window"]).loc[
    ("MANHATTAN_10003", "same_quarter_weekdays")
]
queens = summary.set_index(["area_id", "reference_window"]).loc[
    ("QUEENS_11005", "same_quarter_weekdays")
]
checks = {
    "Historical snapshot reconciles": len(historical) == 394687 and historical["complaint_count"].sum() == 668440,
    "Historical Mondays = 157": len(matching_weekdays) == 157,
    "Historical Q4 Mondays = 39": len(seasonal_weekdays) == 39,
    "500 verified current complaints": len(eligible) == 500,
    "20 areas × 2 reference windows": len(summary) == 40,
    "20 × 2 × 15 detail rows": len(details) == 600,
    "Manhattan seasonal baseline supported": manhattan["comparison_status"] == "supported",
    "Manhattan 38 active Mondays / 153 reports": int(manhattan["historical_active_days"]) == 38 and int(manhattan["historical_reports"]) == 153,
    "Queens sparse baseline correctly withheld": queens["comparison_status"] == "insufficient_historical_support" and int(queens["historical_reports"]) == 2,
    "Supported distances in [0,1]": summary.loc[summary["comparison_status"].eq("supported"), "category_mix_distance"].between(0,1).all(),
    "All input files unchanged": all(hashlib.sha256(p.read_bytes()).hexdigest() == original_hashes[p] for p in inputs),
}
for test, ok in checks.items():
    print(("PASS" if ok else "FAIL") + " — " + test)
assert all(checks.values()), "Validation failed; nothing saved"

outputs = {
    output_dir / "area_comparable_day_summary_2026_11_09_v1.csv": summary.to_csv(index=False),
    output_dir / "area_comparable_day_details_2026_11_09_v1.csv": details.to_csv(index=False),
}
for path, text in outputs.items():
    if path.exists():
        assert path.read_text(encoding="utf-8") == text, f"Existing output differs, no overwrite: {path}"
output_dir.mkdir(parents=True, exist_ok=True)
for path, text in outputs.items():
    if not path.exists():
        path.write_text(text, encoding="utf-8")
    print("Saved:", path)
print("\nSTATUS: PASSED — ALL EXISTING ML MODELS UNCHANGED")


HISTORICAL COMPARABLE WEEKDAYS
Historical Mondays: 157
Historical Q4 Mondays: 39
Current verified reports: 500
Area/window comparisons: 40
Category-level comparison rows: 600

SEASONAL MONDAY COMPARISON


,area_id,comparison_status,historical_active_days,historical_reports,historical_top_category,current_top_category,category_mix_distance
0,QUEENS_11001,insufficient_historical_support,11,16,NaN,STREET CONDITION,NaN
1,QUEENS_11005,insufficient_historical_support,2,2,NaN,SEWER,NaN
2,QUEENS_11040,insufficient_historical_support,5,5,NaN,STREET LIGHT CONDITION,NaN
3,BRONX_10451,supported,37,141,STREET LIGHT CONDITION,STREET CONDITION,0.5067
4,BRONX_10452,supported,36,134,STREET CONDITION,STREET CONDITION,0.3251
5,BRONX_10453,supported,35,129,STREET LIGHT CONDITION,SEWER,0.8025
6,BRONX_10454,supported,31,116,TRAFFIC SIGNAL CONDITION,STREET LIGHT CONDITION,0.3272
7,BROOKLYN_11201,supported,38,278,STREET CONDITION,STREET CONDITION,0.3635
8,BROOKLYN_11203,supported,38,232,TRAFFIC SIGNAL CONDITION,STREET CONDITION,0.3659
9,BROOKLYN_11204,supported,39,310,TRAFFIC SIGNAL CONDITION,SEWER,0.7348



MANHATTAN_10003 — SEASONAL CATEGORY SHARES


,complaint_category,historical_share_pct,current_share_pct,share_change_pp
303,SEWER,7.190,60.0,52.810
304,SEWER MAINTENANCE,0.000,20.0,20.000
312,WATER DRAINAGE,0.000,8.0,8.000
314,WATER MAINTENANCE,0.000,0.0,0.000
302,ROOT/SEWER/SIDEWALK CONDITION,0.654,0.0,-0.654
313,WATER LEAK,0.654,0.0,-0.654
309,STREET SIGN - DANGLING,2.614,0.0,-2.614
308,STREET SIGN - DAMAGED,3.268,0.0,-3.268
301,CURB CONDITION,3.268,0.0,-3.268
310,STREET SIGN - MISSING,3.922,0.0,-3.922


PASS — Historical snapshot reconciles
PASS — Historical Mondays = 157
PASS — Historical Q4 Mondays = 39
PASS — 500 verified current complaints
PASS — 20 areas × 2 reference windows
PASS — 20 × 2 × 15 detail rows
PASS — Manhattan seasonal baseline supported
PASS — Manhattan 38 active Mondays / 153 reports
PASS — Queens sparse baseline correctly withheld
PASS — Supported distances in [0,1]
PASS — All input files unchanged
Saved: D:\Urban-Infrastructure-Pulse-Intelligence\outputs\tables\historical_comparable_day\area_comparable_day_summary_2026_11_09_v1.csv
Saved: D:\Urban-Infrastructure-Pulse-Intelligence\outputs\tables\historical_comparable_day\area_comparable_day_details_2026_11_09_v1.csv

STATUS: PASSED — ALL EXISTING ML MODELS UNCHANGED


In [15]:

from pathlib import Path
import ast
import sys
import base64
import zlib
import hashlib
import importlib

print("DIRECT MODULE INSTALLATION")
print("=" * 65)

ROOT = Path(
    r"D:\Urban-Infrastructure-Pulse-Intelligence"
)

destination = (
    ROOT / "src/integration/historical_comparable_day.py"
)

assert ROOT.is_dir(), "Project root not found"
assert destination.parent.is_dir(), "Integration folder not found"

# Compact copy of the validated Phase 21.4C module.
# No internet connection or Downloads folder required.

MODULE_DATA = """
eNqdWutu28gV/q+nmBJIQW5lxkmaP0JcII276KJINshmd9GqAndEjqSpySHDIW0rroG+Q9+wT9Jz5sbhTY7tH7Y0nPOdM+d+hg6C
4JLJtOZVw68ZScuiojWXpSDljlBB2iovacYyUrOqrBsu9iSjR3LDmwM5cNmUNU9pTgrapAfYdcPYFTyX8WLxidHsrBT5UYG2DW14
KWLyGahIUWZtzkhWMklE2RCaN6wmH9+9XZK/nb1nVMglufzzT+/eflguqCgLCigZa1iKGEuCbIEZoLAcdpY1yCQP25LWICcVVyAl
SBAEwWKxq8sCBMhzTStjuk0JL/As5D2tKti6WJjvoi2qI6EgUmWXKioAGteqbLFY/PrDh8sff01++vnjxx8/fSYX5G5B4CeQtGDJ
l5bWcIzE6iBYkbug4CKhKeo2MWsvzpdELWuN4tKr8/ulBqJ5nihdglwPAb0cA71GoHsQNGM7Y0uWdGZK9BLd5syBh4rxlqZXTGQA
lZZ1Jldw3PiSNvT7Gk6mZfNgMsrzI4C1opncWlDBd0w2K6tivexcCE+wAivWev07/YfdVmAkliVUgL2/smxFuGjIv8mHUjDQNf4x
+HDotK1rJhp7eL33grxeLiJy9ieS8bRZaZ0GwSfWtLUgzU1JqrLMwXU81zWKIClt2L6sj2cFv/XiACwPrmm4EdAnBd9G3N8mjf7b
ilj7OWTwIUDPQb+0JoaAKMd8ef7y1f/+81/48zrWoJMOMAma1qWUpDkwsmvz3D8RCMzLzIj5GTb4gQqHgwA0ajAgDhyZLVVEIq48
wHFVHiASnuZs4Ae4PSZviWwrtIFKErLNG2SQdUlFAwIGpgDZKEoT1LG1kPGPLy2HwE6MN7r4UptSUD16Rwa/A/D8qi5TJlGwBJFb
iYtfeZVc05xnKtu4Bx3KtqzLdn/AvVykPEMXAiL8jjbPKbhRYl3BEN4bp1PMQKiRnGdEsiYcxFAMWacthIwUNd9ZgJUTpqZcMvIL
zVv2l7ou63AXvDdMLAsbmMSAQSaQStWhQYvug6ivPBt8fe1JSM8FxWN6cQzqqZvBGjDDFWdTqwzOenost5LV1/CY5XzPMaH4+0jg
luvyRi1IiCfJTOoArXuAPf368o/PpDVtvw5V6/Z9k44d6I6zPBur1qEpHStE0NeeoWSQ8z7zAp7Rogp7ec3JBFu4FDTUNBFWKf0x
boAyjMjvBjDmKfp3GEVm14mTBD2+pGjhKFsGgeZyjQoVLZAy9VBwe8D12Ck2mkxH4cNE6DUbd3SjpjcXSH/qBJ96TYU7wQ4TpJdn
tlSCP0ERaEpCr0uekZzRK7pn1i6d74G0xoroKFBiwjSK4Q+vwigGn2Y1aH4HtoAuQJDuLJP+von0kayzI/ijUE9GSacvzI+OxxvN
pNt40gfeWyd29J4uEBeyXN5iAweCdYm6obclpOCjVaBW9hEOOFPmIZtVxzBjrLr4XLfMiZ7D9tBQu4yHrq0eePrV+Wdz8jB/7UwO
kVPJA8hvMKFXVMdRpaoLXQPbicOElUZJAdncF2IqBz1eJEhpRGmla2D7YnXyWM1k0EdztEwWrgNsIhKezdWcXjqGuAAJYyqOp3PB
pWWgWpTnFu05xtUVO0o0/2F8FiOpEXM94qyDvylV1VUZyckwS7PESACFXATP/n72rDh7hidlKKa8CJTkutwPWHuqQN16vKEvh44m
nWA9ojnByMTZrNjxljXQXIlQJcAlpi7Qe56HT3BZ0YBQUuVfScq2kdBodE4LI8qeBdNSjU4U75vw/JGCKDfv5DngSFOKqpTcDHkY
0Z0AGCQnBJBtAfVoHE39Ev+UyDacjEA2xLF/gtYsH6gIk+KUlDZuIEqgbrdb3GeT4dNN52VRaz+XYYfJ0zR84K/jFnCcNK3aVH+j
t+Vluu4E1YvrfsO7ialsjhULAyw7Yh+o+hNLCHoWwiD4AnyEfbE9BDzaqWjtQtCUMvz5fcdj3ETPMTLFDnhYIpYFk5jTPfg34H5N
GwpeJyQYBrKzQd9oLTpfGI2JOG6gg6gxEWctLAJWy85vR2SnfMM9wZ9d8AukH2gSszM1J3ZXJNCVKExy12N5byZdSYIB0LZtSMUr
3cx01CPZ7mPy7sDSK2hN8WZEQlcEOZURpXmWgfriDroXJT1BvBPqCfiuJ09gTLMigSiTa3NI67z9Lt5dMZwvBxhtUVAIv/49QBgB
dcYglnI5ftRHaIW5PUqwdOH29abbcq9jDJwKA8wKtO5Pb6edy9cP4qhnODQrvmEd/DO7ew09fgzazqFj/57mkn1Dyu25BRdV6+UP
LlQIkH/88BEWM5vLOvltE4BlrvO37rkdVB+IG/xsG9A/kCAJ4Dee0St8HeZU1pxhP7X1myXpV92JM+vJyLQ1OBk9wNjf/lSLUDfB
oxjI1DHQotobpf4wAdXkAeFawb+0OLL5fjYB9rjG/md78Wrj0olr2vpeOzfb2LtZUDVTWEoS1X54XQ70azBoXwSXRg94BWVvnoDM
IqztB0A5ljvcQS4uiJtazdpGj5ve3Rhg+JBr/0vs9jgks7IZXjp6Itmuulc35xu7zBPPK5JuzdQZLbhJyskNFxn0NYNblJl73n7B
mBC6L+qJjdPiG5ae8GYl6mFC+4v535ey2+Cl3dlr5nBCIo3pGy2K7L2N+nMlyhuhc7cJGGcKF/DmEkLXCtUwwta1VrkuE4NFHKaR
ekn2kAcr9HcbhrFa2R5Db5DCmV/3WF1MuRDU3eWFRpoJ4msMPbM1jEYYTdlAmOlOAtWhoKKpfRXqAC8Ietxjnt0W9Db0SFqRlhDb
+lahu7IY0omM3UbkzHWfJq6tjrSbJgLv3klovkGYg+JsvKr3BZGa/wfOHfOGFdJPqdpKKpmAWP3XHWuP16ZHoezQXSF4Uii/9777
NQCcGb9Em8ng6czmo3e2n7BiNDu9THPwjTpga4ae/im7ty+GyJdsInBFVxl6QFAiVAkyedyLn9UoRegWDfhhi+ZxwO397hLCY0dC
X8g31pTr0dujzYgRwYo4UM0AwDaBm+iUnFBr292Opxy92L9X1FATQrtIOAVrrZzgq0cYyfqvtYAcrwPH2P0AfjP56ugEV1lg4rME
kFirfMRFspMItjQHizkvtClj6IMuZ6DHWMQepOKuBp8edgZAVKRga77bgWLho5da/fThegrww64xGZ/HakAYzx9kqX3X1xyX5HwQ
OPa4jnp40IfJLb8qxbz04vycfOfJ9Lxv5lN1VgOEE+5PDKyR9PkgGCYpZu2iX/l1urXZexJFGVFUsaBi9HysikfzXE0ynfKRmEL/
LrKQbmXoa/xsoMEJA3lF3KLcTfJ1BWCl6/v0ppoZmUypgt1+pZsmmigJK6eREzR6mk/cNKw/zBB4muiGYu0zcyz66QZlsp77MA/1
DhZ1DmRQ+kCtfVMsyavIvG2CcgJT0sBSvnM9IN+YlecDyGeGXtOlBxwrkqqaoh55EMJNonUvz55wnPto0Mh07ea8Uz7okI92xkc5
lW/sXsemfNf7fpq0/68h3rfTZH0X7rLd1KFm3XiOwuMDRc4PSr/4nWA1oPKWJ4j8zcYbswErvAv0QB6X0l2pnRLY8i34bWIzqwsE
7CWn0m0ERebl+fmS/DF6gihzQRC4dioZX0LgXd+SBPG/Si5Ct3EQ2ffuXntwaehdGJp/Lsr8/wmYuC2085K7kvSnUP/az/TrmmL6
xggk9lrlyI0CE7vN5Z3p4j2W47tGb/SaAoLM5TP1oGYuPv2k42+fuQz16qabqv8P2GCsLA==
"""

expected_sha256 = (
    "48d61e6231b08a53c123ea0ba351c9ab"
    "75889649c8aa0961c6423a9b572d3dc5"
)

# Decode and validate without executing the contents.

module_bytes = zlib.decompress(
    base64.b64decode("".join(MODULE_DATA.split()))
)

actual_sha256 = hashlib.sha256(
    module_bytes
).hexdigest()

assert actual_sha256 == expected_sha256, (
    "Module integrity check failed. "
    "No file has been written."
)

source = module_bytes.decode("utf-8")
tree = ast.parse(source)

function_names = {
    node.name for node in tree.body
    if isinstance(
        node,
        (ast.FunctionDef, ast.AsyncFunctionDef)
    )
}

assert (
    "compare_historical_comparable_weekdays"
    in function_names
), "Expected function not found."

# Do not overwrite an existing, different module.

if destination.exists():
    assert destination.read_bytes() == module_bytes, (
        "A different module already exists. "
        "No overwrite performed."
    )
    print("\nExisting module verified.")
else:
    with destination.open("xb") as file:
        file.write(module_bytes)

    print("\nModule created successfully.")

# Import the installed module.

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

importlib.invalidate_caches()

from src.integration.historical_comparable_day import (
    compare_historical_comparable_weekdays
)

assert Path(
    sys.modules[
        compare_historical_comparable_weekdays.__module__
    ].__file__
).resolve() == destination.resolve()

print("\nMODULE FOUND:", destination.is_file())
print("MODULE IMPORT: PASSED")
print("Function:", compare_historical_comparable_weekdays.__name__)

print("\nSaved at:")
print(destination)

print("\nSTATUS: MODULE READY")
print("Existing ML models and GUI unchanged.")


DIRECT MODULE INSTALLATION

Module created successfully.

MODULE FOUND: True
MODULE IMPORT: PASSED
Function: compare_historical_comparable_weekdays

Saved at:
D:\Urban-Infrastructure-Pulse-Intelligence\src\integration\historical_comparable_day.py

STATUS: MODULE READY
Existing ML models and GUI unchanged.


In [16]:

from pathlib import Path
import sys
import json
import pandas as pd

ROOT = Path(
    r"D:\Urban-Infrastructure-Pulse-Intelligence"
)

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.integration.historical_comparable_day import (
    compare_historical_comparable_weekdays
)

print(" FINAL VALIDATION")
print("=" * 65)

# 1. Load historical daily data
historical = pd.read_csv(
    ROOT / "outputs/tables/anomaly/"
    "historical_daily_counts_2023_2025_v1.csv.gz"
)

# 2. Load historical manifest
manifest_path = ROOT / (
    "outputs/tables/anomaly/"
    "historical_daily_counts_2023_2025_v1_manifest.json"
)

with manifest_path.open(
    "r", encoding="utf-8"
) as file:
    manifest = json.load(file)

# 3. Load verified 500-complaint batch
incoming = pd.read_csv(
    ROOT / "outputs/tests/phase_17_4c_backend_verified/"
    "verified_500_backend_results.csv",
    dtype="string",
    keep_default_na=False
)

# 4. Run historical comparable-day analysis
result = compare_historical_comparable_weekdays(
    backend_records=incoming,
    historical_daily_counts=historical,
    manifest=manifest,
    reporting_day="2026-11-09",
    expected_analyzed=500
)

summary = result["summary"]
details = result["details"]

seasonal = summary.loc[
    summary["reference_window"].eq(
        "same_quarter_weekdays"
    )
]

all_weekdays = summary.loc[
    summary["reference_window"].eq(
        "all_matching_weekdays"
    )
]

# 5. Validate expected results
checks = {
    "Analysis completed":
        result["status"] == "compared",

    "500 complaints analyzed":
        result["eligible_reports"] == 500,

    "40 area-window comparisons":
        len(summary) == 40,

    "600 category comparisons":
        len(details) == 600,

    "17 seasonal areas supported":
        seasonal["comparison_status"].eq(
            "supported"
        ).sum() == 17,

    "18 all-weekday areas supported":
        all_weekdays["comparison_status"].eq(
            "supported"
        ).sum() == 18,
}

validation = pd.DataFrame([
    {
        "Test": name,
        "Passed": bool(passed)
    }
    for name, passed in checks.items()
])

print("\nVALIDATION RESULTS")
display(validation)

passed = int(validation["Passed"].sum())

print(
    f"\nChecks passed: {passed}/{len(checks)}"
)

assert validation["Passed"].all(), (
    "Validation failed. Review results."
)

# 6. Inspect example areas
print("\nMANHATTAN ZIP 10003")
display(
    seasonal.loc[
        seasonal["area_id"].eq(
            "MANHATTAN_10003"
        )
    ]
)

print("\nQUEENS ZIP 11005")
display(
    seasonal.loc[
        seasonal["area_id"].eq(
            "QUEENS_11005"
        )
    ]
)

print("\nSTATUS: VALIDATED")
print("Existing models and GUI unchanged.")


 FINAL VALIDATION

VALIDATION RESULTS


,Test,Passed
0,Analysis completed,True
1,500 complaints analyzed,True
2,40 area-window comparisons,True
3,600 category comparisons,True
4,17 seasonal areas supported,True
5,18 all-weekday areas supported,True



Checks passed: 6/6

MANHATTAN ZIP 10003


,area_id,reference_window,comparison_status,historical_calendar_days,historical_active_days,historical_reports,current_reports,historical_top_category,current_top_category,top_category_changed,category_mix_distance,uncovered_current_categories
20,MANHATTAN_10003,same_quarter_weekdays,supported,39,38,153,25,STREET CONDITION,SEWER,True,0.8081,



QUEENS ZIP 11005


,area_id,reference_window,comparison_status,historical_calendar_days,historical_active_days,historical_reports,current_reports,historical_top_category,current_top_category,top_category_changed,category_mix_distance,uncovered_current_categories
28,QUEENS_11005,same_quarter_weekdays,insufficient_historical_support,39,2,2,25,NaN,SEWER,None,NaN,



STATUS: VALIDATED
Existing models and GUI unchanged.


In [18]:

from pathlib import Path
import ast
import os
import hashlib

print("WINDOWS SAVE FIX")
print("=" * 65)

ROOT = Path(
    r"D:\Urban-Infrastructure-Pulse-Intelligence"
)

gui_path = ROOT / "dashboard/urban_pulse_simple.py"

backup_path = ROOT / (
    "dashboard/urban_pulse_simple_before_phase_21_5.py.bak"
)

temporary_path = ROOT / (
    "dashboard/urban_pulse_simple_phase21_5.tmp"
)

# ==============================================
# 1. Verify previous prepared variables
# ==============================================

assert "original_bytes" in globals(), (
    "Previous integration cell must be run first."
)

assert "updated" in globals(), (
    "Updated GUI code is not available in memory."
)

assert isinstance(updated, str)

print("\n1. CHECKING ORIGINAL GUI")

assert gui_path.is_file()

current_bytes = gui_path.read_bytes()

assert current_bytes == original_bytes, (
    "Original GUI has changed. "
    "No modifications performed."
)

print("Original GUI unchanged: PASSED")

# ==============================================
# 2. Verify backup
# ==============================================

print("\n2. CHECKING BACKUP")

assert backup_path.is_file(), (
    "Original GUI backup not found. "
    "No modifications performed."
)

assert backup_path.read_bytes() == original_bytes, (
    "GUI backup does not match original source."
)

print("Original backup verified: PASSED")

# ==============================================
# 3. Verify integrated code
# ==============================================

print("\n3. VALIDATING NEW GUI CODE")

ast.parse(updated)

compile(
    updated,
    str(gui_path),
    "exec"
)

assert updated.count(
    "# PHASE 21.5: COMPARABLE WEEKDAY COMPARISON"
) == 1

assert updated.count(
    '"comparable_day_comparison": '
    'comparable_day_comparison,'
) == 1

assert updated.count(
    '"Historical comparable-day patterns '
    '(2023–2025)"'
) == 1

assert "build_review_queue(feed)" in updated

print("Python syntax: PASSED")
print("Historical integration: PASSED")
print("Investigation queue preserved: PASSED")

# ==============================================
# 4. Check previous temporary file
# ==============================================

print("\n4. CHECKING TEMPORARY FILE")

def normalize_newlines(value):
    return (
        value.replace("\r\r\n", "\n")
             .replace("\r\n", "\n")
             .replace("\r", "\n")
    )

if temporary_path.exists():

    old_temp = temporary_path.read_bytes().decode(
        "utf-8"
    )

    assert normalize_newlines(
        old_temp
    ) == normalize_newlines(updated), (
        "Temporary file contains unexpected "
        "differences. No modifications performed."
    )

    print("Previous temporary file verified.")

else:
    print("No previous temporary file found.")

# ==============================================
# 5. Save using binary mode
# ==============================================

print("\n5. SAVING CORRECTED GUI")

updated_bytes = updated.encode("utf-8")

# Binary writing avoids Windows newline conversion.

with temporary_path.open("wb") as file:
    file.write(updated_bytes)

assert temporary_path.read_bytes() == updated_bytes

print("Temporary file byte verification: PASSED")

# Check the original and backup again.

assert gui_path.read_bytes() == original_bytes
assert backup_path.read_bytes() == original_bytes

# Atomically replace GUI using validated bytes.

os.replace(
    temporary_path,
    gui_path
)

# ==============================================
# 6. Final verification
# ==============================================

print("\n6. FINAL VERIFICATION")

assert gui_path.read_bytes() == updated_bytes
assert backup_path.read_bytes() == original_bytes

ast.parse(
    gui_path.read_bytes().decode("utf-8")
)

print("Updated GUI saved: PASSED")
print("Original backup preserved: PASSED")
print("Python syntax: PASSED")

print("\n" + "=" * 65)
print("SOURCE INTEGRATION PASSED")
print("=" * 65)

print("\nUpdated GUI:")
print(gui_path)

print("\nBackup:")
print(backup_path)

print("\nAdded:")
print("- Historical long-term comparison")
print("- Historical seasonal weekday comparison")
print("- Historical all-weekday comparison")
print("- Current one-day comparison")
print("- Insufficient-history safeguards")

print("\nExisting ML models unchanged.")

print("\nNEXT: TEST STREAMLIT GUI")


WINDOWS SAVE FIX

1. CHECKING ORIGINAL GUI
Original GUI unchanged: PASSED

2. CHECKING BACKUP
Original backup verified: PASSED

3. VALIDATING NEW GUI CODE
Python syntax: PASSED
Historical integration: PASSED
Investigation queue preserved: PASSED

4. CHECKING TEMPORARY FILE
Previous temporary file verified.

5. SAVING CORRECTED GUI
Temporary file byte verification: PASSED

6. FINAL VERIFICATION
Updated GUI saved: PASSED
Original backup preserved: PASSED
Python syntax: PASSED

SOURCE INTEGRATION PASSED

Updated GUI:
D:\Urban-Infrastructure-Pulse-Intelligence\dashboard\urban_pulse_simple.py

Backup:
D:\Urban-Infrastructure-Pulse-Intelligence\dashboard\urban_pulse_simple_before_phase_21_5.py.bak

Added:
- Historical long-term comparison
- Historical seasonal weekday comparison
- Historical all-weekday comparison
- Current one-day comparison
- Insufficient-history safeguards

Existing ML models unchanged.

NEXT: TEST STREAMLIT GUI


In [19]:

from pathlib import Path
import ast
import os
import hashlib

print("SIMPLIFIED HISTORICAL COMPARISON GUI")
print("=" * 70)

ROOT = Path(
    r"D:\Urban-Infrastructure-Pulse-Intelligence"
)

gui_path = ROOT / "dashboard/urban_pulse_simple.py"

backup_path = ROOT / (
    "dashboard/urban_pulse_simple_before_phase_21_6.py.bak"
)

temp_path = ROOT / (
    "dashboard/urban_pulse_simple_phase21_6.tmp"
)

assert gui_path.is_file(), "Existing GUI not found"

original_bytes = gui_path.read_bytes()
original = original_bytes.decode("utf-8")

# ---------------------------------------------------
# 1. Locate the two existing historical UI sections
# ---------------------------------------------------

start_marker = (
    "    # Historical vs current reporting mix"
)

end_marker = (
    '    with st.expander('
    '"View exclusions and download full processing audit"):'
)

assert original.count(start_marker) == 1, (
    "Historical display section not found uniquely. "
    "No files changed."
)

assert original.count(end_marker) == 1, (
    "Audit display section not found uniquely. "
    "No files changed."
)

start = original.index(start_marker)
end = original.index(end_marker)

assert start < end

old_section = original[start:end]

assert (
    "Historical comparable-day patterns"
    in old_section
)

assert (
    "historical_comparison"
    in old_section
)

new_marker = "# PHASE 21.6 — SIMPLE COMPARISON CARD"

assert new_marker not in original, (
    "Phase 21.6 already installed."
)

# ---------------------------------------------------
# 2. New simplified historical UI
# ---------------------------------------------------

replacement = r'''
    # PHASE 21.6 — SIMPLE COMPARISON CARD
    import pandas as _story_pd

    longterm = current.get("historical_comparison")
    comparable = current.get("comparable_day_comparison")

    st.markdown("### What changed in this area?")

    st.caption(
        "Compare the selected reporting day with "
        "similar weekdays in 2023–2025."
    )

    if (
        not isinstance(comparable, dict)
        or comparable.get("status") != "compared"
        or comparable["summary"].empty
    ):
        st.info(
            "Historical comparable-day information "
            "is not available for this analysis."
        )

    else:
        summary = comparable["summary"]
        details = comparable["details"]

        area_options = sorted(
            summary["area_id"].unique().tolist()
        )

        if (
            st.session_state.get("pulse_story_area")
            not in area_options
        ):
            st.session_state["pulse_story_area"] = (
                area_options[0]
            )

        chosen_area = st.selectbox(
            "Select area",
            area_options,
            key="pulse_story_area",
        )

        area_rows = summary.loc[
            summary["area_id"].eq(chosen_area)
        ]

        def _story_window(name):
            found = area_rows.loc[
                area_rows["reference_window"].eq(name)
            ]
            return (
                found.iloc[0]
                if not found.empty
                else None
            )

        seasonal = _story_window(
            "same_quarter_weekdays"
        )

        weekday = _story_window(
            "all_matching_weekdays"
        )

        # Prefer seasonal history when evidence
        # is sufficient; otherwise use all
        # matching weekdays when supported.

        reference = None
        reference_name = None
        reference_label = None

        if (
            seasonal is not None
            and seasonal["comparison_status"] == "supported"
        ):
            reference = seasonal
            reference_name = "same_quarter_weekdays"
            reference_label = "Similar seasonal weekdays"

        elif (
            weekday is not None
            and weekday["comparison_status"] == "supported"
        ):
            reference = weekday
            reference_name = "all_matching_weekdays"
            reference_label = "Similar weekdays"

        selected_day = (
            seasonal
            if seasonal is not None
            else weekday
        )

        current_category = str(
            selected_day["current_top_category"]
        )

        current_reports = int(
            selected_day["current_reports"]
        )

        current_detail = details.loc[
            details["area_id"].eq(chosen_area)
            &
            details["reference_window"].eq(
                selected_day["reference_window"]
            )
            &
            details["complaint_category"].eq(
                current_category
            )
        ]

        current_percentage = (
            float(
                current_detail.iloc[0]["current_share_pct"]
            )
            if not current_detail.empty
            else None
        )

        current_category_count = (
            int(
                current_detail.iloc[0]["current_reports"]
            )
            if not current_detail.empty
            else None
        )

        # -----------------------------------------
        # Simple visual comparison
        # -----------------------------------------

        with st.container(border=True):

            left, right = st.columns(2)

            with left:
                st.caption(
                    "HISTORICALLY ON SIMILAR DAYS"
                )

                if reference is not None:

                    historical_top = str(
                        reference["historical_top_category"]
                    )

                    historical_top_detail = details.loc[
                        details["area_id"].eq(
                            chosen_area
                        )
                        &
                        details["reference_window"].eq(
                            reference_name
                        )
                        &
                        details["complaint_category"].eq(
                            historical_top
                        )
                    ]

                    historical_top_pct = (
                        float(
                            historical_top_detail.iloc[0][
                                "historical_share_pct"
                            ]
                        )
                        if not historical_top_detail.empty
                        else None
                    )

                    st.markdown(
                        f"**{historical_top.title()}**"
                    )

                    if historical_top_pct is not None:
                        st.metric(
                            "Share of historical reports",
                            f"{historical_top_pct:.1f}%",
                        )

                    st.caption(
                        f'{int(reference["historical_reports"])} '
                        "historical reports across "
                        f'{int(reference["historical_active_days"])} '
                        "active matching weekdays."
                    )

                else:
                    st.markdown(
                        "**Not enough historical data**"
                    )

                    st.caption(
                        "Insufficient historical evidence "
                        "for a comparable-day category pattern."
                    )

            with right:
                st.caption("SELECTED REPORTING DAY")

                st.markdown(
                    f"**{current_category.title()}**"
                )

                if current_percentage is not None:
                    st.metric(
                        "Share of today's reports",
                        f"{current_percentage:.1f}%",
                    )

                st.caption(
                    f"{current_category_count} of "
                    f"{current_reports} verified reports "
                    "belong to this category."
                )

            st.divider()

            # -----------------------------------------
            # Compare the SAME category in both periods
            # -----------------------------------------

            st.markdown("**What changed?**")

            if (
                reference is not None
                and current_percentage is not None
            ):

                old_category_row = details.loc[
                    details["area_id"].eq(chosen_area)
                    &
                    details["reference_window"].eq(
                        reference_name
                    )
                    &
                    details["complaint_category"].eq(
                        current_category
                    )
                ]

                uncovered_raw = reference.get(
                    "uncovered_current_categories",
                    ""
                )

                uncovered_categories = [
                    value.strip()
                    for value in str(
                        uncovered_raw
                    ).replace(",", "|").split("|")
                    if value.strip()
                ]

                category_covered = (
                    current_category
                    not in uncovered_categories
                )

                if (
                    not old_category_row.empty
                    and category_covered
                ):

                    historical_pct = float(
                        old_category_row.iloc[0][
                            "historical_share_pct"
                        ]
                    )

                    difference = (
                        current_percentage
                        - historical_pct
                    )

                    st.write(
                        f"**{current_category.title()} "
                        "complaints**"
                    )

                    st.write(
                        f"Historical: {historical_pct:.1f}%"
                    )

                    st.progress(
                        min(
                            max(historical_pct / 100, 0),
                            1
                        )
                    )

                    st.write(
                        f"Selected day: "
                        f"{current_percentage:.1f}%"
                    )

                    st.progress(
                        min(
                            max(
                                current_percentage / 100,
                                0
                            ),
                            1
                        )
                    )

                    if difference > 0:
                        message = (
                            "a higher share than historically"
                        )

                    elif difference < 0:
                        message = (
                            "a lower share than historically"
                        )

                    else:
                        message = (
                            "the same share as historically"
                        )

                    st.write(
                        f"**{difference:+.1f} percentage "
                        f"points** — {message}."
                    )

                else:
                    st.info(
                        "A reliable category-share "
                        "comparison is not available "
                        "for this complaint category."
                    )

            else:
                st.info(
                    "The current reporting pattern is "
                    "available, but the historical sample "
                    "is too limited for comparison."
                )

            # -----------------------------------------
            # Plain-language suggested action
            # -----------------------------------------

            st.divider()

            st.markdown("**Suggested next check**")

            category_upper = current_category.upper()

            if "STREET LIGHT" in category_upper:
                action = (
                    "Review lighting complaints, check "
                    "reported pole locations, and examine "
                    "recent lighting maintenance records."
                )

            elif "TRAFFIC SIGNAL" in category_upper:
                action = (
                    "Review the reported intersections "
                    "and check traffic-signal maintenance "
                    "history."
                )

            elif (
                "SEWER" in category_upper
                or "DRAINAGE" in category_upper
            ):
                action = (
                    "Check whether sewer or drainage "
                    "complaints concern nearby locations, "
                    "and review relevant maintenance records."
                )

            elif (
                "STREET CONDITION" in category_upper
                or "SIDEWALK" in category_upper
                or "CURB" in category_upper
            ):
                action = (
                    "Review reported street locations "
                    "and descriptions, and check recent "
                    "road or sidewalk maintenance history."
                )

            else:
                action = (
                    "Review complaint descriptions, "
                    "reported locations, and the relevant "
                    "maintenance history."
                )

            st.write(action)

            st.caption(
                "Check the existing DBSCAN spatial "
                "findings and historical-count anomaly "
                "results before deciding whether an "
                "on-site inspection is warranted."
            )

        st.caption(
            "Historical percentages pool complaints "
            "across comparable weekdays in 2023–2025. "
            "The selected-day percentages come from "
            "one uploaded reporting day. A difference "
            "does not confirm an infrastructure fault."
        )

        # -----------------------------------------
        # Technical details hidden by default
        # -----------------------------------------

        with st.expander(
            "View detailed analysis",
            expanded=False,
        ):

            st.markdown(
                "**All historical reference patterns**"
            )

            rows = []

            if (
                isinstance(longterm, dict)
                and longterm.get("status") == "compared"
            ):

                old_area = longterm["summary"].loc[
                    longterm["summary"]["area_id"].eq(
                        chosen_area
                    )
                ]

                if not old_area.empty:

                    old = old_area.iloc[0]

                    rows.append({
                        "Reference": "Long-term 2023–2025",
                        "Dominant category":
                            old["historical_top_category"],
                        "Share": (
                            f'{old["historical_top_share_pct"]:.1f}%'
                        ),
                        "Evidence": "All historical dates",
                    })

            for window_id, label in [
                (
                    "same_quarter_weekdays",
                    "Seasonal matching weekdays",
                ),
                (
                    "all_matching_weekdays",
                    "All matching weekdays",
                ),
            ]:

                item = _story_window(window_id)

                if item is None:
                    continue

                if item["comparison_status"] == "supported":

                    top = str(
                        item["historical_top_category"]
                    )

                    top_row = details.loc[
                        details["area_id"].eq(
                            chosen_area
                        )
                        &
                        details["reference_window"].eq(
                            window_id
                        )
                        &
                        details["complaint_category"].eq(top)
                    ]

                    share = (
                        f'{float(top_row.iloc[0]["historical_share_pct"]):.1f}%'
                        if not top_row.empty
                        else "Unavailable"
                    )

                else:
                    top = "Insufficient evidence"
                    share = "—"

                rows.append({
                    "Reference": label,
                    "Dominant category": top,
                    "Share": share,
                    "Evidence": (
                        f'{int(item["historical_reports"])} '
                        "reports, "
                        f'{int(item["historical_active_days"])} '
                        "active of "
                        f'{int(item["historical_calendar_days"])} '
                        "matching days"
                    ),
                })

            rows.append({
                "Reference": "Selected reporting day",
                "Dominant category": current_category,
                "Share": (
                    f"{current_percentage:.1f}%"
                    if current_percentage is not None
                    else "Unavailable"
                ),
                "Evidence": (
                    f"{current_reports} reports, one day"
                ),
            })

            st.dataframe(
                _story_pd.DataFrame(rows),
                hide_index=True,
                use_container_width=True,
            )

            if reference is not None:

                st.markdown(
                    "**All complaint-category percentages**"
                )

                category_detail = details.loc[
                    details["area_id"].eq(chosen_area)
                    &
                    details["reference_window"].eq(
                        reference_name
                    ),
                    [
                        "complaint_category",
                        "historical_share_pct",
                        "current_share_pct",
                        "share_change_pp",
                        "current_reports",
                    ],
                ].copy()

                category_detail = (
                    category_detail.sort_values(
                        "share_change_pp",
                        ascending=False,
                    )
                )

                st.dataframe(
                    category_detail.rename(columns={
                        "complaint_category":
                            "Complaint category",
                        "historical_share_pct":
                            "Historical %",
                        "current_share_pct":
                            "Selected day %",
                        "share_change_pp":
                            "Change (pp)",
                        "current_reports":
                            "Selected day reports",
                    }),
                    hide_index=True,
                    use_container_width=True,
                )

            else:
                st.info(
                    "Detailed category percentages "
                    "are withheld because historical "
                    "support is insufficient."
                )

'''

# ---------------------------------------------------
# 3. Build and validate updated source in memory
# ---------------------------------------------------

updated = (
    original[:start]
    + replacement
    + original[end:]
)

assert updated.count(new_marker) == 1

assert updated.count(start_marker) == 0

assert updated.count(
    "Historical comparable-day patterns (2023–2025)"
) == 0

assert "process_uploaded_csv_bytes(" in updated

assert "build_review_queue(feed)" in updated

assert (
    '"comparable_day_comparison": '
    "comparable_day_comparison,"
) in updated

assert (
    '"historical_comparison": historical_comparison,'
) in updated

ast.parse(updated)
compile(updated, str(gui_path), "exec")

assert gui_path.read_bytes() == original_bytes

print("\nPRE-SAVE VALIDATION")
print("New simplified card: PASSED")
print("Detailed analysis retained: PASSED")
print("Python syntax: PASSED")
print("Upload pipeline preserved: PASSED")
print("Investigation queue preserved: PASSED")

# ---------------------------------------------------
# 4. Back up original GUI
# ---------------------------------------------------

if backup_path.exists():
    assert backup_path.read_bytes() == original_bytes, (
        "An existing backup differs from current GUI. "
        "No changes made."
    )
else:
    with backup_path.open("xb") as f:
        f.write(original_bytes)

assert backup_path.read_bytes() == original_bytes

# ---------------------------------------------------
# 5. Save with binary mode (Windows-safe)
# ---------------------------------------------------

assert not temp_path.exists(), (
    "Temporary file already exists. "
    "Inspect it before trying again."
)

updated_bytes = updated.encode("utf-8")

with temp_path.open("xb") as f:
    f.write(updated_bytes)

assert temp_path.read_bytes() == updated_bytes

assert gui_path.read_bytes() == original_bytes

os.replace(temp_path, gui_path)

assert gui_path.read_bytes() == updated_bytes
assert backup_path.read_bytes() == original_bytes

print("\n" + "=" * 70)
print("GUI SOURCE UPDATE PASSED")
print("=" * 70)

print("Updated GUI:", gui_path)
print("Backup:", backup_path)

print("\nSimplified interface added:")
print("- One area selector")
print("- Historical versus selected-day cards")
print("- Simple percentage bars")
print("- Plain-language change explanation")
print("- Suggested investigation action")
print("- Technical details hidden by default")

print("\nExisting ML models and rankings unchanged.")
print("\nNEXT: TEST GUI IN STREAMLIT")


SIMPLIFIED HISTORICAL COMPARISON GUI

PRE-SAVE VALIDATION
New simplified card: PASSED
Detailed analysis retained: PASSED
Python syntax: PASSED
Upload pipeline preserved: PASSED
Investigation queue preserved: PASSED

GUI SOURCE UPDATE PASSED
Updated GUI: D:\Urban-Infrastructure-Pulse-Intelligence\dashboard\urban_pulse_simple.py
Backup: D:\Urban-Infrastructure-Pulse-Intelligence\dashboard\urban_pulse_simple_before_phase_21_6.py.bak

Simplified interface added:
- One area selector
- Historical versus selected-day cards
- Simple percentage bars
- Plain-language change explanation
- Suggested investigation action
- Technical details hidden by default

Existing ML models and rankings unchanged.

NEXT: TEST GUI IN STREAMLIT


### final 1

In [20]:

from pathlib import Path
import ast

ROOT = Path(
    r"D:\Urban-Infrastructure-Pulse-Intelligence"
)

gui_path = ROOT / "dashboard/urban_pulse_simple.py"

assert gui_path.is_file()

source = gui_path.read_text(encoding="utf-8")
lines = source.splitlines()
tree = ast.parse(source)

print("INVESTIGATION CARD INSPECTION")
print("=" * 65)

# 1. Inspect the actual investigation card function.
for node in tree.body:
    if isinstance(node, ast.FunctionDef):
        if node.name == "_show_location_review":

            print("\n1. INVESTIGATION CARD FUNCTION")
            print("Lines:", node.lineno, "-", node.end_lineno)

            for i in range(
                node.lineno,
                node.end_lineno + 1
            ):
                print(f"{i:4d} | {lines[i-1]}")

# 2. Find where investigation cards are called.
print("\n2. INVESTIGATION CARD CALLS")

for i, line in enumerate(lines, start=1):
    if "_show_location_review(" in line:
        if line.lstrip().startswith("def "):
            continue

        for j in range(
            max(1, i-5),
            min(len(lines), i+5)+1
        ):
            print(f"{j:4d} | {lines[j-1]}")

# 3. Find the historical comparison UI to remove.
print("\n3. EXISTING HISTORICAL SECTIONS")

markers = [
    "PHASE 21.6",
    "What changed in this area?",
    "Historical comparable-day patterns",
    "Historical vs current complaint patterns",
    "View detailed analysis",
]

for marker in markers:
    matches = [
        i for i, line in enumerate(lines, start=1)
        if marker.lower() in line.lower()
    ]
    print(f"{marker}: {matches}")

# 4. Inspect investigation queue area keys.
queue_path = ROOT / (
    "src/integration/investigation_review_queue.py"
)

assert queue_path.is_file()

queue_tree = ast.parse(
    queue_path.read_text(encoding="utf-8")
)

keys = set()

for node in ast.walk(queue_tree):
    if isinstance(node, ast.Dict):
        for key in node.keys:
            if isinstance(key, ast.Constant):
                if isinstance(key.value, str):
                    keys.add(key.value)

print("\n4. INVESTIGATION QUEUE FIELDS")

for key in sorted(keys):
    if any(term in key.lower() for term in [
        "area", "location", "zip", "kmeans",
        "category", "borough"
    ]):
        print("-", key)

print("\nINSPECTION COMPLETE")
print("No files or models modified.")


INVESTIGATION CARD INSPECTION

1. INVESTIGATION CARD FUNCTION
Lines: 437 - 503
 437 | def _show_location_review(item, feed, weekday_plural):
 438 |     tier = _tier(item)
 439 |     order = int(item["review_order"])
 440 |     issue = str(item["category"]).title()
 441 |     spatial_only = item["kind"] == "spatial_only"
 442 |     patterns = ", ".join(item.get("kmeans_patterns", [])) or "Insufficient area evidence"
 443 |     nearby = int(item.get("spatial_overlap_reports", 0) or 0)
 444 | 
 445 |     # Which screening signals are behind this item (derived from fields already present).
 446 |     dbscan_on = spatial_only or nearby > 0
 447 |     anomaly_on = not spatial_only
 448 |     has_patterns = bool(item.get("kmeans_patterns"))
 449 |     signals = (
 450 |         f'<span class="sig {"on" if dbscan_on else "off"}">Nearby reports · DBSCAN</span>'
 451 |         f'<span class="sig {"on" if anomaly_on else "off"}">Unusual activity · Anomaly</span>'
 452 |         f'<span class="sig

In [21]:

from pathlib import Path
import ast
import os
import pandas as pd

ROOT = Path(r"D:\Urban-Infrastructure-Pulse-Intelligence")

gui_path = ROOT / "dashboard/urban_pulse_simple.py"

backup_path = ROOT / (
    "dashboard/urban_pulse_simple_before_embedded_history.py.bak"
)

temp_path = ROOT / (
    "dashboard/urban_pulse_simple_embedded_history.tmp"
)

print("EMBED HISTORY IN INVESTIGATIONS")
print("=" * 65)

assert gui_path.is_file()

original_bytes = gui_path.read_bytes()
source = original_bytes.decode("utf-8")
source = source.replace("\r\n", "\n")

# --------------------------------------------------
# 1. Remove the separate historical comparison UI
# --------------------------------------------------

start_marker = (
    "    # PHASE 21.6 — SIMPLE COMPARISON CARD"
)

end_marker = (
    '    with st.expander('
    '"View exclusions and download full processing audit"):'
)

assert source.count(start_marker) == 1
assert source.count(end_marker) == 1

start = source.index(start_marker)
end = source.index(end_marker)

assert start < end

source = source[:start] + source[end:]

print("Separate historical comparison removed from prepared source.")

# --------------------------------------------------
# 2. Reusable historical description for one
#    investigation (supports multiple areas)
# --------------------------------------------------

function_anchor = (
    "def _show_location_review("
    "item, feed, weekday_plural):"
)

assert source.count(function_anchor) == 1

helper = '''
def _history_note_for_review(item, comparison):
    """Describe historical routine for linked investigation areas.

    Prefer comparable same-quarter weekdays.
    Fall back to all matching weekdays when supported.
    Never infer an area's history from another area.
    """

    import re

    if (
        not isinstance(comparison, dict)
        or comparison.get("status") != "compared"
    ):
        return "Historical comparison unavailable."

    summary = comparison.get("summary")

    if (
        not isinstance(summary, pd.DataFrame)
        or summary.empty
        or "area_id" not in summary.columns
    ):
        return "Historical comparison unavailable."

    known = set(summary["area_id"].astype(str))
    linked_areas = set()

    def collect(value):
        if isinstance(value, str):
            for candidate in re.split(r"[,;|]", value):
                candidate = candidate.strip().upper()
                if candidate in known:
                    linked_areas.add(candidate)

        elif isinstance(value, (list, tuple, set)):
            for member in value:
                collect(member)

    collect(item.get("area_id"))
    collect(item.get("area_ids"))

    if not linked_areas:
        return (
            "No reliable historical area match "
            "for this investigation."
        )

    descriptions = []

    for area_id in sorted(linked_areas):

        area_rows = summary.loc[
            summary["area_id"].eq(area_id)
        ]

        chosen = None
        reference_label = None

        for window, label in [
            (
                "same_quarter_weekdays",
                "similar-season weekdays",
            ),
            (
                "all_matching_weekdays",
                "similar weekdays",
            ),
        ]:
            candidates = area_rows.loc[
                area_rows["reference_window"].eq(window)
                &
                area_rows["comparison_status"].eq(
                    "supported"
                )
            ]

            if not candidates.empty:
                chosen = candidates.iloc[0]
                reference_label = label
                break

        if chosen is None:
            descriptions.append(
                f"{area_id}: Not enough historical data."
            )
            continue

        category = str(
            chosen["historical_top_category"]
        ).title()

        reports = int(
            chosen["historical_reports"]
        )

        descriptions.append(
            f"{area_id}: {category} "
            f"({reference_label}, 2023–2025; "
            f"{reports} historical reports)"
        )

    return " | ".join(descriptions)


'''

source = source.replace(
    function_anchor,
    helper
    + "def _show_location_review("
      "item, feed, weekday_plural, comparison=None):",
    1
)

# --------------------------------------------------
# 3. Replace K-Means evidence body only
# --------------------------------------------------

old_evidence = '''    evidence = [
        ("Area pattern · K-Means", _html(patterns)),'''

new_evidence = '''    historical_pattern = _history_note_for_review(
        item, comparison
    )

    kmeans_body = (
        "<strong>Current one-day K-Means pattern:</strong> "
        + _html(patterns)
        + "<br><strong>Normally reported historically:</strong> "
        + _html(historical_pattern)
    )

    evidence = [
        ("Area pattern · K-Means + History", kmeans_body),'''

assert source.count(old_evidence) == 1

source = source.replace(
    old_evidence,
    new_evidence,
    1
)

# --------------------------------------------------
# 4. Pass comparison to the investigation card
# --------------------------------------------------

old_call = (
    "        _show_location_review("
    "items[selected], feed, weekday_plural)"
)

new_call = '''        _show_location_review(
            items[selected],
            feed,
            weekday_plural,
            current.get("comparable_day_comparison"),
        )'''

assert source.count(old_call) == 1

source = source.replace(
    old_call,
    new_call,
    1
)

# --------------------------------------------------
# 5. Validate updated code in memory
# --------------------------------------------------

tree = ast.parse(source)

compile(
    source,
    str(gui_path),
    "exec"
)

checks = {
    "Separate historical section removed":
        "What changed in this area?" not in source,

    "Investigation helper present":
        source.count(
            "def _history_note_for_review"
        ) == 1,

    "History embedded beside K-Means":
        source.count(
            "Normally reported historically:"
        ) == 1,

    "Comparison passed into review card":
        source.count(
            'current.get("comparable_day_comparison"),'
        ) == 1,

    "Existing DBSCAN preserved":
        '("Nearby reports · DBSCAN",' in source,

    "Existing anomaly preserved":
        '("Unusual activity · Anomaly",' in source,

    "Existing review queue preserved":
        "build_review_queue(feed)" in source,

    "Existing upload processing preserved":
        "process_uploaded_csv_bytes(" in source,

    "Original GUI unchanged":
        gui_path.read_bytes() == original_bytes,
}

print("\nPRE-SAVE CHECKS")

for name, passed in checks.items():
    print(
        f'{"PASS" if passed else "FAIL"}: {name}'
    )

assert all(checks.values())

# --------------------------------------------------
# 6. Test the helper without running Streamlit
# --------------------------------------------------

helper_node = next(
    node for node in tree.body
    if isinstance(node, ast.FunctionDef)
    and node.name == "_history_note_for_review"
)

scope = {"pd": pd}

exec(
    compile(
        ast.Module(
            body=[helper_node],
            type_ignores=[]
        ),
        str(gui_path),
        "exec"
    ),
    scope
)

test_history = {
    "status": "compared",
    "summary": pd.DataFrame([
        {
            "area_id": "BRONX_10451",
            "reference_window": "same_quarter_weekdays",
            "comparison_status": "supported",
            "historical_top_category":
                "STREET LIGHT CONDITION",
            "historical_reports": 141,
        },
        {
            "area_id": "QUEENS_11005",
            "reference_window": "same_quarter_weekdays",
            "comparison_status":
                "insufficient_historical_support",
            "historical_top_category": None,
            "historical_reports": 2,
        },
    ])
}

describe = scope["_history_note_for_review"]

supported_note = describe(
    {"area_id": "BRONX_10451"},
    test_history
)

sparse_note = describe(
    {"area_id": "QUEENS_11005"},
    test_history
)

assert "Street Light Condition" in supported_note
assert "Not enough historical data" in sparse_note

print("\nHistorical helper tests: PASSED")

# --------------------------------------------------
# 7. Backup and save only after validation
# --------------------------------------------------

if backup_path.exists():
    assert backup_path.read_bytes() == original_bytes, (
        "Existing backup differs. No GUI changes made."
    )
else:
    with backup_path.open("xb") as file:
        file.write(original_bytes)

assert backup_path.read_bytes() == original_bytes

assert not temp_path.exists(), (
    "Temporary file already exists. "
    "No GUI changes made."
)

updated_bytes = source.encode("utf-8")

with temp_path.open("xb") as file:
    file.write(updated_bytes)

assert temp_path.read_bytes() == updated_bytes
assert gui_path.read_bytes() == original_bytes

os.replace(temp_path, gui_path)

assert gui_path.read_bytes() == updated_bytes
assert backup_path.read_bytes() == original_bytes

print("\n" + "=" * 65)
print("GUI SOURCE UPDATE PASSED")
print("=" * 65)

print("\nUpdated:", gui_path)
print("Backup:", backup_path)

print("\nChanges:")
print("- Separate historical section removed")
print("- Current K-Means pattern retained")
print("- Normal historical category added beside K-Means")
print("- Sparse-history safeguards retained")
print("- Existing investigation rankings unchanged")
print("- DBSCAN and anomaly evidence unchanged")

print("\nNEXT: TEST STREAMLIT INVESTIGATION CARDS")


EMBED HISTORY IN INVESTIGATIONS
Separate historical comparison removed from prepared source.

PRE-SAVE CHECKS
PASS: Separate historical section removed
PASS: Investigation helper present
PASS: History embedded beside K-Means
PASS: Comparison passed into review card
PASS: Existing DBSCAN preserved
PASS: Existing anomaly preserved
PASS: Existing review queue preserved
PASS: Existing upload processing preserved
PASS: Original GUI unchanged

Historical helper tests: PASSED

GUI SOURCE UPDATE PASSED

Updated: D:\Urban-Infrastructure-Pulse-Intelligence\dashboard\urban_pulse_simple.py
Backup: D:\Urban-Infrastructure-Pulse-Intelligence\dashboard\urban_pulse_simple_before_embedded_history.py.bak

Changes:
- Separate historical section removed
- Current K-Means pattern retained
- Normal historical category added beside K-Means
- Sparse-history safeguards retained
- Existing investigation rankings unchanged
- DBSCAN and anomaly evidence unchanged

NEXT: TEST STREAMLIT INVESTIGATION CARDS


In [22]:

from pathlib import Path
import ast
import json

ROOT = Path(
    r"D:\Urban-Infrastructure-Pulse-Intelligence"
)

print("NEW-BATCH K-MEANS INSPECTION")
print("=" * 65)

source_files = [
    "src/models/new_batch_area_kmeans.py",
    "src/integration/combined_batch_intelligence.py",
    "src/integration/anomaly_snapshot_integration.py",
]

keywords = [
    "area_table",
    "area_profiles",
    "cluster_label",
    "cluster_name",
    "cluster_id",
    "new_kmeans",
    "kmeans_result",
    "profile_label",
]

for relative in source_files:

    path = ROOT / relative

    print("\nFILE:", relative)

    if not path.is_file():
        print("NOT FOUND")
        continue

    source = path.read_text(encoding="utf-8")
    lines = source.splitlines()

    tree = ast.parse(source)

    print("\nFunctions:")

    for node in tree.body:
        if isinstance(
            node,
            (ast.FunctionDef, ast.ClassDef)
        ):
            print("-", node.name)

    print("\nK-Means assignment references:")

    matches = [
        i
        for i, line in enumerate(lines)
        if any(
            keyword in line.lower()
            for keyword in keywords
        )
    ]

    for i in matches[:18]:
        start = max(0, i - 1)
        end = min(len(lines), i + 3)

        for j in range(start, end):
            print(
                f"{j + 1:4d} | {lines[j]}"
            )

        print("---")

    print("Total references:", len(matches))

# Inspect the existing saved dashboard feed.
feed_path = ROOT / (
    "outputs/tests/phase_19_2e_dashboard_feed/"
    "dashboard_feed.json"
)

print("\n" + "=" * 65)
print("SAVED DASHBOARD FEED")
print("=" * 65)

if feed_path.is_file():

    with feed_path.open(
        "r",
        encoding="utf-8"
    ) as file:
        feed = json.load(file)

    print("Top-level fields:")

    for key, value in feed.items():
        print(
            "-",
            key,
            "=>",
            type(value).__name__
        )

    print("\nK-Means-related fields:")

    for key, value in feed.items():

        if "kmeans" in key.lower():

            print("\nField:", key)

            if isinstance(value, dict):
                print("Keys:", list(value.keys())[:25])

            elif isinstance(value, list):
                print("Number of items:", len(value))
                if value and isinstance(value[0], dict):
                    print(
                        "Example item fields:",
                        list(value[0].keys())[:25]
                    )

else:
    print("Saved feed not found; source inspection is sufficient.")

print("\nSTATUS: INSPECTION COMPLETE")
print("No source files or ML models modified.")


NEW-BATCH K-MEANS INSPECTION

FILE: src/models/new_batch_area_kmeans.py

Functions:
- analyze_new_batch_areas

K-Means assignment references:
 239 |         "kmeans_area_id",
 240 |         "kmeans_cluster_id",
 241 |         "kmeans_reporting_profile",
 242 |         "kmeans_display_allowed"
---
 275 |                     cluster_variants=(
 276 |                         "kmeans_cluster_id",
 277 |                         "nunique"
 278 |                     ),
---
 298 |             ] = area_summary["area_id"].map(
 299 |                 lookup["kmeans_cluster_id"]
 300 |             )
 301 | 
---
Total references: 3

FILE: src/integration/combined_batch_intelligence.py

Functions:
- _joined_unique
- analyze_combined_daily_batch

K-Means assignment references:
  89 |         .rename(columns={
  90 |             "area_id": "new_kmeans_area_id",
  91 |             "eligibility_reason":
  92 |                 "new_kmeans_eligibility"
---
  91 |             "eligibility_reason":
  92 |  

In [23]:

from pathlib import Path
import ast

ROOT = Path(
    r"D:\Urban-Infrastructure-Pulse-Intelligence"
)

print("K-MEANS AREA ASSIGNMENT DATA FLOW")
print("=" * 70)

targets = [
    (
        "src/models/new_batch_area_kmeans.py",
        "analyze_new_batch_areas",
    ),
    (
        "src/integration/combined_batch_intelligence.py",
        "analyze_combined_daily_batch",
    ),
    (
        "src/integration/anomaly_snapshot_integration.py",
        "analyze_batch_with_anomaly_context",
    ),
    (
        "src/integration/live_uploaded_batch_pipeline.py",
        "build_verified_day_feed",
    ),
]

for relative, function_name in targets:

    path = ROOT / relative
    assert path.is_file(), f"Missing: {path}"

    source = path.read_text(encoding="utf-8")
    lines = source.splitlines()
    tree = ast.parse(source)

    functions = [
        node for node in tree.body
        if isinstance(node, ast.FunctionDef)
        and node.name == function_name
    ]

    assert len(functions) == 1

    function = functions[0]

    print("\n" + "=" * 70)
    print("FILE:", relative)
    print("FUNCTION:", function_name)
    print("=" * 70)

    # Show returned dictionary keys and
    # the source lines around the return.

    returns = [
        node for node in ast.walk(function)
        if isinstance(node, ast.Return)
    ]

    print("\nRETURN STRUCTURES")

    for node in returns:

        value = node.value

        if isinstance(value, ast.Dict):

            keys = [
                key.value
                if isinstance(key, ast.Constant)
                else "<dynamic key>"
                for key in value.keys
            ]

            print(
                f"Line {node.lineno}:",
                keys
            )

        else:
            print(
                f"Line {node.lineno}:",
                type(value).__name__
            )

    # Show code around the last return.
    if returns:

        last = max(
            returns,
            key=lambda node: node.lineno
        )

        start = max(
            function.lineno,
            last.lineno - 12
        )

        end = min(
            function.end_lineno,
            last.end_lineno + 3
        )

        print("\nFINAL RETURN CONTEXT")

        for index in range(start, end + 1):
            print(
                f"{index:4d} | "
                f"{lines[index - 1]}"
            )

# --------------------------------------------------
# Inspect K-Means assignment fields
# --------------------------------------------------

path = ROOT / (
    "src/integration/combined_batch_intelligence.py"
)

lines = path.read_text(
    encoding="utf-8"
).splitlines()

print("\n" + "=" * 70)
print("K-MEANS AREA ASSIGNMENT COLUMNS")
print("=" * 70)

for start, end in [(75, 130), (340, 375)]:

    for index in range(
        start,
        min(end, len(lines)) + 1
    ):
        print(
            f"{index:4d} | "
            f"{lines[index - 1]}"
        )

# --------------------------------------------------
# Identify current GUI layout location
# --------------------------------------------------

path = ROOT / "dashboard/urban_pulse_simple.py"

lines = path.read_text(
    encoding="utf-8"
).splitlines()

print("\n" + "=" * 70)
print("CURRENT GUI RESULTS LAYOUT")
print("=" * 70)

for index in range(
    1400,
    min(1465, len(lines)) + 1
):
    print(
        f"{index:4d} | "
        f"{lines[index - 1]}"
    )

print("\nSTATUS: INSPECTION COMPLETE")
print("No files or models modified.")


K-MEANS AREA ASSIGNMENT DATA FLOW

FILE: src/models/new_batch_area_kmeans.py
FUNCTION: analyze_new_batch_areas

RETURN STRUCTURES
Line 496: Name
Line 353: Name
Line 369: Name
Line 386: Name
Line 411: Name

FINAL RETURN CONTEXT
 484 |     ].map(descriptions)
 485 | 
 486 |     output.update({
 487 |         "status": "clustered",
 488 |         "reason": None,
 489 |         "model": model,
 490 |         "cluster_profiles": cluster_profiles,
 491 |         "silhouette": float(
 492 |             silhouette_score(features, labels)
 493 |         )
 494 |     })
 495 | 
 496 |     return output

FILE: src/integration/combined_batch_intelligence.py
FUNCTION: analyze_combined_daily_batch

RETURN STRUCTURES
Line 376: ['combined_records', 'selected_records', 'date_audit', 'area_summary', 'candidate_intelligence', 'kmeans_output', 'dbscan_output', 'stats']

FINAL RETURN CONTEXT
 364 |                 "candidate_complaints"
 365 |             ],
 366 |         "dbscan_below_reporting_threshold

In [24]:

from pathlib import Path
import ast
import os
import pandas as pd

ROOT = Path(
    r"D:\Urban-Infrastructure-Pulse-Intelligence"
)

gui_path = ROOT / "dashboard/urban_pulse_simple.py"

backup_path = ROOT / (
    "dashboard/urban_pulse_simple_before_phase_21_7.py.bak"
)

temp_path = ROOT / (
    "dashboard/urban_pulse_simple_phase21_7.tmp"
)

print("NEW-BATCH K-MEANS CLUSTER EXPLORER")
print("=" * 70)

assert gui_path.is_file()

original_bytes = gui_path.read_bytes()
original = original_bytes.decode("utf-8")

marker = "# PHASE 21.7C — K-MEANS CLUSTER EXPLORER"

assert marker not in original, (
    "Phase 21.7C is already present. "
    "No duplicate modification performed."
)

# =====================================================
# 1. EXTRACT ALL NEW-BATCH K-MEANS AREA ASSIGNMENTS
# =====================================================

helper_code = '''
# PHASE 21.7C — K-MEANS CLUSTER EXPLORER

def _pulse_new_kmeans_cluster_areas(integrated):
    """Read new-batch model assignments, not investigation flags."""

    import pandas as pd

    specifications = [
        (
            "area_id",
            "new_batch_cluster",
            "new_batch_description",
            "incoming_complaints",
        ),
        (
            "new_kmeans_area_id",
            "new_kmeans_cluster",
            "new_kmeans_description",
            "new_kmeans_area_complaints",
        ),
        (
            "kmeans_area_id",
            "kmeans_cluster_id",
            "kmeans_reporting_profile",
            None,
        ),
    ]

    candidates = []

    def visit(value, path="integrated", depth=0):
        if depth > 6:
            return

        if isinstance(value, pd.DataFrame):
            for area_col, id_col, label_col, count_col in specifications:

                required = {
                    area_col, id_col, label_col
                }

                if not required.issubset(value.columns):
                    continue

                cols = [area_col, id_col, label_col]
                has_counts = (
                    count_col is not None
                    and count_col in value.columns
                )

                if has_counts:
                    cols.append(count_col)

                frame = value[cols].copy()

                frame = frame.rename(columns={
                    area_col: "area_id",
                    id_col: "cluster_id",
                    label_col: "reporting_profile",
                })

                if has_counts:
                    frame = frame.rename(columns={
                        count_col: "complaints"
                    })
                else:
                    frame["complaints"] = pd.NA

                frame["area_id"] = (
                    frame["area_id"]
                    .astype("string")
                    .str.strip()
                    .str.upper()
                )

                frame["reporting_profile"] = (
                    frame["reporting_profile"]
                    .astype("string")
                    .str.strip()
                )

                frame["cluster_id"] = pd.to_numeric(
                    frame["cluster_id"],
                    errors="coerce",
                )

                frame["complaints"] = pd.to_numeric(
                    frame["complaints"],
                    errors="coerce",
                )

                frame = frame.dropna(
                    subset=[
                        "area_id",
                        "cluster_id",
                        "reporting_profile",
                    ]
                )

                frame = frame.loc[
                    frame["area_id"].str.len().gt(0)
                    &
                    frame["reporting_profile"].str.len().gt(0)
                ].copy()

                if frame.empty:
                    continue

                # A geographic area must have exactly
                # one cluster and profile in this batch.

                if (
                    frame.groupby("area_id")[
                        "cluster_id"
                    ].nunique().gt(1).any()
                ):
                    raise ValueError(
                        "Conflicting K-Means assignments "
                        "for one geographic area."
                    )

                if (
                    frame.groupby("area_id")[
                        "reporting_profile"
                    ].nunique().gt(1).any()
                ):
                    raise ValueError(
                        "Conflicting K-Means profile labels "
                        "for one geographic area."
                    )

                if (
                    frame.groupby("cluster_id")[
                        "reporting_profile"
                    ].nunique().gt(1).any()
                ):
                    raise ValueError(
                        "One cluster ID has conflicting "
                        "reporting-profile descriptions."
                    )

                frame = (
                    frame.sort_values("area_id")
                    .drop_duplicates("area_id")
                    .reset_index(drop=True)
                )

                priority = (
                    100 if "area_summary" in path else
                    50 if "new_kmeans" in path else
                    10
                )

                candidates.append((
                    len(frame),
                    priority,
                    path,
                    frame,
                ))

        elif isinstance(value, dict):
            for key, child in value.items():
                if isinstance(
                    child,
                    (dict, list, tuple, pd.DataFrame),
                ):
                    visit(
                        child,
                        f"{path}.{key}",
                        depth + 1,
                    )

        elif isinstance(value, (list, tuple)):
            if value and all(
                isinstance(row, dict) for row in value
            ):
                visit(
                    pd.DataFrame(value),
                    path,
                    depth + 1,
                )

    visit(integrated)

    if not candidates:
        return {
            "status": "unavailable",
            "reason": (
                "No complete new-batch K-Means "
                "area-assignment table was found."
            ),
            "areas": pd.DataFrame(),
        }

    candidates.sort(
        key=lambda item: (
            item[0],
            item[1],
        ),
        reverse=True,
    )

    _, _, source_name, areas = candidates[0]

    return {
        "status": "clustered",
        "source": source_name,
        "areas": areas,
    }


def _pulse_show_cluster_explorer(explorer):
    """Display every area in the selected K-Means cluster."""

    st.markdown("### Explore New-Batch K-Means Clusters")

    st.caption(
        "Select a one-day reporting pattern to see all "
        "geographic areas assigned to that cluster. "
        "These groups describe complaint types, "
        "not infrastructure damage or repair priorities."
    )

    if (
        not isinstance(explorer, dict)
        or explorer.get("status") != "clustered"
    ):
        reason = (
            explorer.get("reason", "No cluster data.")
            if isinstance(explorer, dict)
            else "No cluster data."
        )

        st.info(
            "New-batch K-Means cluster memberships "
            "are unavailable for this uploaded batch. "
            + str(reason)
        )
        return

    areas = explorer["areas"]

    if areas.empty:
        st.info("No assigned K-Means areas to display.")
        return

    profiles = (
        areas[["cluster_id", "reporting_profile"]]
        .drop_duplicates()
        .sort_values("cluster_id")
    )

    labels = {
        row.cluster_id: str(row.reporting_profile)
        for row in profiles.itertuples(index=False)
    }

    selected_cluster = st.selectbox(
        "Choose a K-Means reporting pattern",
        options=list(labels.keys()),
        format_func=lambda cluster_id: (
            labels[cluster_id]
        ),
        key="pulse_kmeans_cluster_explorer",
    )

    selected_areas = (
        areas.loc[
            areas["cluster_id"].eq(selected_cluster)
        ]
        .sort_values("area_id")
        .copy()
    )

    st.markdown(
        f"**{labels[selected_cluster]}**"
    )

    left, right = st.columns(2)

    with left:
        st.metric(
            "Areas in this cluster",
            len(selected_areas),
        )

    with right:
        if selected_areas["complaints"].notna().all():
            st.metric(
                "Complaints in these areas",
                int(selected_areas["complaints"].sum()),
            )
        else:
            st.metric(
                "Complaints in these areas",
                "Unavailable",
            )

    selected_areas["Borough / ZIP"] = (
        selected_areas["area_id"]
        .str.replace("_", " · ZIP ", n=1, regex=False)
        .str.title()
    )

    display_table = selected_areas[
        ["Borough / ZIP", "complaints"]
    ].rename(columns={
        "complaints": "Incoming complaints"
    })

    st.dataframe(
        display_table,
        hide_index=True,
        use_container_width=True,
    )

    st.caption(
        "All model-assigned areas in this cluster "
        "are shown, including areas without "
        "DBSCAN or anomaly investigation signals."
    )

'''

# =====================================================
# 2. VERIFY EXTRACTION WITH TEST ASSIGNMENTS
# =====================================================

parsed_helper = ast.parse(helper_code)

extract_function = next(
    node for node in parsed_helper.body
    if isinstance(node, ast.FunctionDef)
    and node.name == "_pulse_new_kmeans_cluster_areas"
)

scope = {"pd": pd}

exec(
    compile(
        ast.Module(
            body=[extract_function],
            type_ignores=[],
        ),
        "<cluster_explorer_test>",
        "exec",
    ),
    scope,
)

extract = scope[
    "_pulse_new_kmeans_cluster_areas"
]

# Simulate an area-level model result.
test_area_summary = pd.DataFrame({
    "area_id": [
        "BRONX_10451",
        "MANHATTAN_10003",
        "QUEENS_11005",
    ],
    "new_batch_cluster": [0, 1, 1],
    "new_batch_description": [
        "Road-heavy reporting",
        "Sewer-heavy reporting",
        "Sewer-heavy reporting",
    ],
    "incoming_complaints": [25, 25, 25],
})

test_result = extract({
    "new_kmeans": {
        "area_summary": test_area_summary
    }
})

assert test_result["status"] == "clustered"
assert len(test_result["areas"]) == 3

sewer_areas = test_result["areas"].loc[
    test_result["areas"][
        "reporting_profile"
    ].eq("Sewer-heavy reporting")
]

assert len(sewer_areas) == 2
assert sewer_areas["complaints"].sum() == 50

# Verify fallback to the combined record table.
combined_test = pd.DataFrame({
    "new_kmeans_area_id": [
        "BRONX_10451",
        "BRONX_10451",
        "MANHATTAN_10003",
    ],
    "new_kmeans_cluster": [0, 0, 1],
    "new_kmeans_description": [
        "Road-heavy reporting",
        "Road-heavy reporting",
        "Sewer-heavy reporting",
    ],
    "new_kmeans_area_complaints": [25, 25, 25],
})

fallback_result = extract({
    "combined": combined_test
})

assert fallback_result["status"] == "clustered"
assert len(fallback_result["areas"]) == 2

print("Area-table extraction test: PASSED")
print("Combined-table fallback test: PASSED")
print("Duplicate-area removal test: PASSED")

# =====================================================
# 3. INSERT THE EXPLORER FUNCTIONS
# =====================================================

function_anchor = "def _show_location_review("

assert original.count(function_anchor) == 1

updated = original.replace(
    function_anchor,
    helper_code + "\n" + function_anchor,
    1,
)

# =====================================================
# 4. STORE MODEL ASSIGNMENTS AFTER ANALYSIS
# =====================================================

session_anchor = (
    '                "feed": feed, "review": queue,'
)

assert updated.count(session_anchor) == 1

session_addition = (
    session_anchor
    + '\n'
    + '                "new_kmeans_explorer": '
      '_pulse_new_kmeans_cluster_areas('
      'result.get("integrated")),'
)

updated = updated.replace(
    session_anchor,
    session_addition,
    1,
)

# =====================================================
# 5. ADD SEPARATE GUI SECTION BEFORE INVESTIGATIONS
# =====================================================

display_anchor = (
    '    with st.expander('
    '"View exclusions and download full processing audit"):'
)

assert updated.count(display_anchor) == 1

display_addition = '''
    # New-batch K-Means cluster explorer
    _pulse_show_cluster_explorer(
        current.get("new_kmeans_explorer")
    )

    st.divider()

'''

updated = updated.replace(
    display_anchor,
    display_addition + display_anchor,
    1,
)

# =====================================================
# 6. VALIDATE GUI BEFORE SAVING
# =====================================================

ast.parse(updated)
compile(updated, str(gui_path), "exec")

checks = {
    "Explorer functions present":
        updated.count(
            "def _pulse_new_kmeans_cluster_areas"
        ) == 1,

    "Cluster dropdown present":
        updated.count(
            "Choose a K-Means reporting pattern"
        ) == 1,

    "Model assignments stored":
        updated.count(
            '"new_kmeans_explorer": '
        ) == 1,

    "Explorer added to results":
        updated.count(
            'current.get("new_kmeans_explorer")'
        ) == 1,

    "Investigation cards retained":
        "def _show_location_review(" in updated,

    "Investigation queue retained":
        "build_review_queue(feed)" in updated,

    "Historical comparisons retained":
        "comparable_day_comparison" in updated,

    "Upload pipeline retained":
        "process_uploaded_csv_bytes(" in updated,

    "Original source unchanged":
        gui_path.read_bytes() == original_bytes,
}

print("\nPRE-SAVE VALIDATION")

for name, passed in checks.items():
    print(f'{"PASS" if passed else "FAIL"}: {name}')

assert all(checks.values()), (
    "Validation failed. GUI not modified."
)

# =====================================================
# 7. BACKUP EXISTING GUI
# =====================================================

if backup_path.exists():
    assert backup_path.read_bytes() == original_bytes, (
        "Existing backup differs from current GUI. "
        "No changes made."
    )
else:
    with backup_path.open("xb") as file:
        file.write(original_bytes)

assert backup_path.read_bytes() == original_bytes

# =====================================================
# 8. SAVE WINDOWS-SAFE
# =====================================================

assert not temp_path.exists(), (
    "Temporary integration file already exists."
)

updated_bytes = updated.encode("utf-8")

with temp_path.open("xb") as file:
    file.write(updated_bytes)

assert temp_path.read_bytes() == updated_bytes
assert gui_path.read_bytes() == original_bytes

os.replace(temp_path, gui_path)

assert gui_path.read_bytes() == updated_bytes
assert backup_path.read_bytes() == original_bytes

print("\n" + "=" * 70)
print("GUI SOURCE UPDATE PASSED")
print("=" * 70)

print("Updated:", gui_path)
print("Backup:", backup_path)

print("\nAdded:")
print("- New-Batch K-Means Cluster Explorer")
print("- Dropdown populated from fitted cluster labels")
print("- Complete assigned-area list for each cluster")
print("- Per-area complaint counts when available")
print("- No dependence on investigation flags")

print("\nUnchanged:")
print("- Historical PCA and K-Means")
print("- Current K-Means model")
print("- Historical/current DBSCAN")
print("- Anomaly detection")
print("- Historical comparison analysis")
print("- Investigation rankings")

print("\nNEXT: STREAMLIT GUI ACCEPTANCE TEST")


NEW-BATCH K-MEANS CLUSTER EXPLORER
Area-table extraction test: PASSED
Combined-table fallback test: PASSED
Duplicate-area removal test: PASSED

PRE-SAVE VALIDATION
PASS: Explorer functions present
PASS: Cluster dropdown present
PASS: Model assignments stored
PASS: Explorer added to results
PASS: Investigation cards retained
PASS: Investigation queue retained
PASS: Historical comparisons retained
PASS: Upload pipeline retained
PASS: Original source unchanged

GUI SOURCE UPDATE PASSED
Updated: D:\Urban-Infrastructure-Pulse-Intelligence\dashboard\urban_pulse_simple.py
Backup: D:\Urban-Infrastructure-Pulse-Intelligence\dashboard\urban_pulse_simple_before_phase_21_7.py.bak

Added:
- New-Batch K-Means Cluster Explorer
- Dropdown populated from fitted cluster labels
- Complete assigned-area list for each cluster
- Per-area complaint counts when available
- No dependence on investigation flags

Unchanged:
- Historical PCA and K-Means
- Current K-Means model
- Historical/current DBSCAN
- Anomal

## Live dashboard regression check — consumption of Notebook 07 anomaly engine

The former Notebook 06 anomaly threshold diagnostics and file-editing patches have moved
out of this integration notebook. The existing full-upload regression test below remains
because it validates the **downstream** bulk upload, investigation queue and dashboard feed.
It does not retrain or patch the anomaly engine.


In [1]:

from pathlib import Path
import sys
import json
import pandas as pd

ROOT = Path(
    r"D:\Urban-Infrastructure-Pulse-Intelligence"
)

sys.path.insert(0, str(ROOT))

from src.integration.live_uploaded_batch_pipeline import (
    process_uploaded_csv_bytes
)

from src.integration.investigation_review_queue import (
    build_review_queue
)

print("FULL ANOMALY REGRESSION TEST")
print("=" * 70)

# --------------------------------------------------
# 1. Input paths
# --------------------------------------------------

input_path = ROOT / (
    "outputs/tests/phase_20_2e_gui_acceptance/"
    "gui_test_clean_500.csv"
)

old_feed_path = ROOT / (
    "outputs/tests/phase_19_2e_dashboard_feed/"
    "dashboard_feed.json"
)

model_path = ROOT / (
    "src/models/new_batch_anomaly_screen.py"
)

for path in [input_path, old_feed_path, model_path]:
    assert path.is_file(), f"Missing file: {path}"

assert (
    "historical_max_exceeded_below_strength_threshold"
    in model_path.read_text(encoding="utf-8")
), "Updated anomaly rule not found."

# --------------------------------------------------
# 2. Load original results
# --------------------------------------------------

with old_feed_path.open(
    "r", encoding="utf-8"
) as file:
    old_feed = json.load(file)

old_review = build_review_queue(old_feed)

old_anomaly_count = len(
    old_feed["anomaly_cards"]
)

old_investigation_count = len(
    old_review["queue"]
)

print("\n1. ORIGINAL PIPELINE")
print("Anomaly groups:", old_anomaly_count)
print("Investigation groups:", old_investigation_count)

# --------------------------------------------------
# 3. Re-run current pipeline using revised model
# --------------------------------------------------

print("\n2. RUNNING REVISED 500-REPORT PIPELINE")

result = process_uploaded_csv_bytes(
    csv_bytes=input_path.read_bytes(),
    reporting_day="2026-11-09",
    project_root=ROOT,
    synthetic_data=True,
)

assert result["status"] == "analyzed"

new_feed = result["feed"]
new_review = build_review_queue(new_feed)

new_anomaly_count = len(
    new_feed["anomaly_cards"]
)

new_investigation_count = len(
    new_review["queue"]
)

# --------------------------------------------------
# 4. Compare old versus revised results
# --------------------------------------------------

comparison = pd.DataFrame([
    {
        "Metric": "Anomaly candidate groups",
        "Before": old_anomaly_count,
        "After": new_anomaly_count,
    },
    {
        "Metric": "Investigation groups",
        "Before": old_investigation_count,
        "After": new_investigation_count,
    },
    {
        "Metric": "Review-first groups",
        "Before": old_review["summary"][
            "review_first_groups"
        ],
        "After": new_review["summary"][
            "review_first_groups"
        ],
    },
    {
        "Metric": "Spatial-only groups",
        "Before": old_review["summary"][
            "spatial_only_groups"
        ],
        "After": new_review["summary"][
            "spatial_only_groups"
        ],
    },
])

comparison["Change"] = (
    comparison["After"] - comparison["Before"]
)

print("\n3. BEFORE VS AFTER")
display(comparison)

# --------------------------------------------------
# 5. Validate complete pipeline
# --------------------------------------------------

selected_count = int(
    result["selection_stats"][
        "selected_for_new_models"
    ]
)

checks = {
    "Updated anomaly rule installed":
        "historical_max_exceeded_below_strength_threshold"
        in model_path.read_text(encoding="utf-8"),

    "500 reports submitted":
        result["uploaded_rows"] == 500,

    "500 reports retained in audit":
        len(result["backend_records"]) == 500,

    "500 reports selected for analysis":
        selected_count == 500,

    "500 complaints in dashboard":
        new_feed["summary"][
            "input_complaints"
        ] == 500,

    "Anomaly candidates did not increase":
        new_anomaly_count <= old_anomaly_count,

    "Review queue generated successfully":
        isinstance(new_review["queue"], list),

    "Investigation items are unique":
        len({
            item["review_order"]
            for item in new_review["queue"]
        }) == new_investigation_count,
}

print("\n4. VALIDATION RESULTS")

validation = pd.DataFrame([
    {"Test": name, "Passed": bool(passed)}
    for name, passed in checks.items()
])

display(validation)

assert validation["Passed"].all(), (
    "Full-pipeline validation failed."
)

print(
    "\nChecks passed:",
    int(validation["Passed"].sum()),
    "/",
    len(validation)
)

# --------------------------------------------------
# 6. Display revised anomaly candidate summaries
# --------------------------------------------------

print("\n5. REVISED ANOMALY CANDIDATES")

if new_feed["anomaly_cards"]:

    revised_cards = pd.DataFrame(
        new_feed["anomaly_cards"]
    )

    useful_columns = [
        col for col in [
            "area_id",
            "complaint_category",
            "incoming_reports",
            "seasonal_max",
            "all_weekday_max",
        ]
        if col in revised_cards.columns
    ]

    display(
        revised_cards[useful_columns]
        if useful_columns
        else revised_cards.head(20)
    )

else:
    print("No qualifying anomaly candidate groups.")

print("\n" + "=" * 70)
print("FULL PIPELINE TEST PASSED")
print("=" * 70)

print("Original anomaly groups:", old_anomaly_count)
print("Revised anomaly groups:", new_anomaly_count)

print(
    "Anomaly groups excluded:",
    old_anomaly_count - new_anomaly_count
)

print("Revised investigation groups:", new_investigation_count)
print("\nNo model or GUI source files were modified.")
print("No new result files were saved by this cell.")


FULL ANOMALY REGRESSION TEST

1. ORIGINAL PIPELINE
Anomaly groups: 15
Investigation groups: 23

2. RUNNING REVISED 500-REPORT PIPELINE


c:\Users\Administrator\miniconda3\envs\ml\Lib\site-packages\sklearn\cluster\_kmeans.py:1424: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=1.
  warnings.warn(



3. BEFORE VS AFTER


,Metric,Before,After,Change
0,Anomaly candidate groups,15,10,-5
1,Investigation groups,23,23,0
2,Review-first groups,15,10,-5
3,Spatial-only groups,8,13,5



4. VALIDATION RESULTS


,Test,Passed
0,Updated anomaly rule installed,True
1,500 reports submitted,True
2,500 reports retained in audit,True
3,500 reports selected for analysis,True
4,500 complaints in dashboard,True
5,Anomaly candidates did not increase,True
6,Review queue generated successfully,True
7,Investigation items are unique,True



Checks passed: 8 / 8

5. REVISED ANOMALY CANDIDATES


,area_id,complaint_category
0,BRONX_10451,STREET CONDITION
1,BRONX_10453,SEWER
2,BRONX_10454,STREET LIGHT CONDITION
3,BROOKLYN_11204,SEWER
4,MANHATTAN_10001,STREET CONDITION
5,MANHATTAN_10002,STREET CONDITION
6,MANHATTAN_10003,SEWER
7,QUEENS_11004,STREET CONDITION
8,STATEN ISLAND_10302,STREET CONDITION
9,STATEN ISLAND_10303,SEWER



FULL PIPELINE TEST PASSED
Original anomaly groups: 15
Revised anomaly groups: 10
Anomaly groups excluded: 5
Revised investigation groups: 23

No model or GUI source files were modified.
No new result files were saved by this cell.
